# Experiment 1 — Fixed-model equilibrium/bifurcation diagram.

In [ ]:
# ============================================================
# EXPERIMENT 01
# Fixed-threshold equilibrium / bifurcation structure
# ============================================================
#
# This notebook:
#   1. verifies the fixed-threshold reference model M0;
#   2. constructs the equilibrium branch parametrically by H;
#   3. classifies local stability from the analytical Jacobian;
#   4. identifies fold points;
#   5. saves ALL numerical outputs and high-resolution figures
#      to a dedicated Google Drive folder.
#
# IMPORTANT:
# This is NOT yet the adaptive-threshold bifurcation experiment.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
EXP_DIR = BASE_DIR / "Experiment_01_Fixed_Model_Bifurcation"

EXP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment folder:\n{EXP_DIR}")
print(f"Exists: {EXP_DIR.exists()}")

In [ ]:
import sys
import json
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import root, root_scalar

np.random.seed(20261006)

# Save software versions
versions_text = f"""
Experiment: Fixed-threshold equilibrium/bifurcation structure
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""

with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)

print(versions_text)

In [ ]:
# ============================================================
# EXPLORATORY PARAMETERS
# ============================================================
#
# These are dimensionless exploratory values.
# They are NOT empirical periodontal parameter estimates.
#
# Do not interpret this set biologically until the parameter
# landscape has been mapped.
# ============================================================

PARAMS = {
    "n": 4.0,          # steepness of host response F
    "q": 4.0,          # steepness of microbial response G
    "K_M": 0.50,       # midpoint/threshold of G
    "theta0": 0.60,    # fixed host response threshold
    "r": 1.0,          # relative microbial timescale

    # plotting / branch domain
    "H_min": 1e-5,
    "H_max": 1.0 - 1e-5,
    "N_branch": 20000,

    # physical/display interval for external forcing
    "u_display_min": 0.0,
    "u_display_max": 1.20,

    # tolerances
    "stability_tol": 1e-8,
    "fold_root_tol": 1e-12,
}

with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(PARAMS, f, indent=4)

print(json.dumps(PARAMS, indent=4))

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def F(S, theta, n):
    """
    Host response:
        F(S; theta) = S^n / (S^n + theta^n)

    Intended model domain: S >= 0, theta > 0.
    """
    S = np.asarray(S)
    return np.power(S, n) / (
        np.power(S, n) + np.power(theta, n)
    )


def dF_dS(S, theta, n):
    """
    Partial derivative dF/dS.
    """
    S = np.asarray(S)

    numerator = (
        n
        * np.power(theta, n)
        * np.power(S, n - 1)
    )

    denominator = (
        np.power(S, n) + np.power(theta, n)
    )**2

    return numerator / denominator


# ------------------------------------------------------------
# IMPORTANT MODEL-DEFINITION ASSUMPTION
# ------------------------------------------------------------
#
# The supplied "simulation and bifurcation map" specifies G(H)
# symbolically and refers to q and K_M, but the retrieved excerpt
# does not explicitly restate G's functional form.
#
# For this notebook we use the Hill form:
#
#       G(H) = H^q / (H^q + K_M^q)
#
# This must be checked against the final Methods/model derivation.
#
# If G differs in the final model, replace ONLY G() and dG_dH().
# ------------------------------------------------------------

def G(H, K_M, q):
    """
    Provisional Hill microbial-response function.
    """
    H = np.asarray(H)

    return np.power(H, q) / (
        np.power(H, q) + np.power(K_M, q)
    )


def dG_dH(H, K_M, q):
    """
    Analytical derivative dG/dH for the provisional Hill form.
    """
    H = np.asarray(H)

    numerator = (
        q
        * np.power(K_M, q)
        * np.power(H, q - 1)
    )

    denominator = (
        np.power(H, q) + np.power(K_M, q)
    )**2

    return numerator / denominator


def rhs_M0(t, y, u, p):
    """
    Fixed-threshold reference model M0.

    y = [H, M]
    """
    H, M = y

    S = M + u

    dH = -H + F(S, p["theta0"], p["n"])
    dM = p["r"] * (-M + G(H, p["K_M"], p["q"]))

    return np.array([dH, dM])


def jacobian_M0(H, M, u, p):
    """
    Analytical Jacobian of M0.
    """
    S = M + u

    FS = dF_dS(S, p["theta0"], p["n"])
    GH = dG_dH(H, p["K_M"], p["q"])
    r = p["r"]

    return np.array([
        [-1.0,     FS],
        [r * GH,  -r]
    ])

In [ ]:
# ============================================================
# PARAMETRIC EQUILIBRIUM BRANCH
# ============================================================

def equilibrium_M_from_H(H, p):
    """
    At equilibrium:
        M* = G(H*)
    """
    return G(H, p["K_M"], p["q"])


def equilibrium_u_from_H(H, p):
    """
    Scalar equilibrium representation:

        u(H) =
            theta0 * (H/(1-H))^(1/n)
            - G(H)

    Valid for 0 < H < 1.
    """
    n = p["n"]
    theta0 = p["theta0"]

    response_signal = theta0 * np.power(
        H / (1.0 - H),
        1.0 / n
    )

    return response_signal - G(H, p["K_M"], p["q"])


def du_dH(H, p):
    """
    Analytical derivative of the scalar branch u(H).

    Fold candidates satisfy:
        du/dH = 0
    """
    n = p["n"]
    theta0 = p["theta0"]

    a = np.power(
        H / (1.0 - H),
        1.0 / n
    )

    da_dH = a / (n * H * (1.0 - H))

    return (
        theta0 * da_dH
        - dG_dH(H, p["K_M"], p["q"])
    )

In [ ]:
# ============================================================
# BUILD COMPLETE M0 EQUILIBRIUM BRANCH
# ============================================================

H_branch = np.linspace(
    PARAMS["H_min"],
    PARAMS["H_max"],
    int(PARAMS["N_branch"])
)

M_branch = equilibrium_M_from_H(H_branch, PARAMS)
u_branch = equilibrium_u_from_H(H_branch, PARAMS)

eig1 = np.empty(len(H_branch), dtype=complex)
eig2 = np.empty(len(H_branch), dtype=complex)
max_real_eig = np.empty(len(H_branch))
detJ = np.empty(len(H_branch))
traceJ = np.empty(len(H_branch))
loop_gain = np.empty(len(H_branch))

for i, (H, M, u) in enumerate(
    zip(H_branch, M_branch, u_branch)
):
    J = jacobian_M0(H, M, u, PARAMS)

    eigvals = np.linalg.eigvals(J)
    eigvals = eigvals[np.argsort(eigvals.real)]

    eig1[i] = eigvals[0]
    eig2[i] = eigvals[1]

    max_real_eig[i] = np.max(eigvals.real)
    detJ[i] = np.linalg.det(J)
    traceJ[i] = np.trace(J)

    S = M + u

    loop_gain[i] = (
        dF_dS(S, PARAMS["theta0"], PARAMS["n"])
        * dG_dH(H, PARAMS["K_M"], PARAMS["q"])
    )

stable = max_real_eig < -PARAMS["stability_tol"]
unstable = max_real_eig > PARAMS["stability_tol"]
near_neutral = ~(stable | unstable)

branch_df = pd.DataFrame({
    "H_star": H_branch,
    "M_star": M_branch,
    "u": u_branch,

    "eig1_real": eig1.real,
    "eig1_imag": eig1.imag,

    "eig2_real": eig2.real,
    "eig2_imag": eig2.imag,

    "max_real_eigenvalue": max_real_eig,
    "trace_J": traceJ,
    "det_J": detJ,
    "F_S_times_G_H": loop_gain,

    "stable": stable,
    "unstable": unstable,
    "near_neutral": near_neutral
})

branch_df.to_csv(
    EXP_DIR / "equilibrium_branch.csv",
    index=False
)

print(branch_df.head())
print()
print(f"Saved: {EXP_DIR / 'equilibrium_branch.csv'}")

In [ ]:
# ============================================================
# FOLD DETECTION
# ============================================================
#
# Fold candidates are identified geometrically from:
#
#       du/dH = 0
#
# Each is then checked independently against:
#
#       F_S * G_H = 1
#
# and against the Jacobian eigenvalue approaching zero.
# ============================================================

H_scan = H_branch
derivative_scan = du_dH(H_scan, PARAMS)

fold_brackets = []

for i in range(len(H_scan) - 1):
    y1 = derivative_scan[i]
    y2 = derivative_scan[i + 1]

    if not (np.isfinite(y1) and np.isfinite(y2)):
        continue

    if y1 == 0:
        fold_brackets.append(
            (H_scan[i], H_scan[i])
        )

    elif y1 * y2 < 0:
        fold_brackets.append(
            (H_scan[i], H_scan[i + 1])
        )


fold_records = []

for a, b in fold_brackets:

    if a == b:
        Hf = a
    else:
        result = root_scalar(
            lambda h: du_dH(h, PARAMS),
            bracket=(a, b),
            method="brentq",
            xtol=PARAMS["fold_root_tol"]
        )
        Hf = result.root

    Mf = equilibrium_M_from_H(Hf, PARAMS)
    uf = equilibrium_u_from_H(Hf, PARAMS)
    Sf = Mf + uf

    Jf = jacobian_M0(Hf, Mf, uf, PARAMS)
    eigf = np.linalg.eigvals(Jf)

    FS = dF_dS(
        Sf,
        PARAMS["theta0"],
        PARAMS["n"]
    )

    GH = dG_dH(
        Hf,
        PARAMS["K_M"],
        PARAMS["q"]
    )

    fold_records.append({
        "H_fold": Hf,
        "M_fold": Mf,
        "u_fold": uf,

        "du_dH": du_dH(Hf, PARAMS),

        "F_S": FS,
        "G_H": GH,
        "F_S_times_G_H": FS * GH,
        "fold_condition_error": FS * GH - 1.0,

        "eig1_real": eigf[0].real,
        "eig1_imag": eigf[0].imag,

        "eig2_real": eigf[1].real,
        "eig2_imag": eigf[1].imag,

        "min_abs_eigenvalue": np.min(np.abs(eigf))
    })


fold_df = pd.DataFrame(fold_records)

fold_df.to_csv(
    EXP_DIR / "fold_points.csv",
    index=False
)

print(f"Detected fold candidates: {len(fold_df)}")
display(fold_df)

In [ ]:
# ============================================================
# VERIFY PARAMETRIC EQUILIBRIA WITH AN INDEPENDENT 2D ROOT SOLVER
# ============================================================

def steady_state_residual_M0(x, u, p):
    H, M = x

    return np.array([
        -H + F(M + u, p["theta0"], p["n"]),
        p["r"] * (-M + G(H, p["K_M"], p["q"]))
    ])


# Choose several locations distributed along the branch,
# avoiding the extreme endpoints.
sample_indices = np.linspace(
    int(0.05 * len(branch_df)),
    int(0.95 * len(branch_df)),
    12,
    dtype=int
)

equilibrium_check_records = []

for idx in sample_indices:

    H_exact = branch_df.loc[idx, "H_star"]
    M_exact = branch_df.loc[idx, "M_star"]
    u = branch_df.loc[idx, "u"]

    # Deliberately perturb the exact parametric equilibrium
    x0 = np.array([
        np.clip(H_exact + 0.01, 1e-4, 0.9999),
        np.clip(M_exact - 0.01, 1e-4, 0.9999)
    ])

    sol = root(
        lambda x: steady_state_residual_M0(
            x, u, PARAMS
        ),
        x0=x0
    )

    if sol.success:
        H_num, M_num = sol.x

        error = np.linalg.norm(
            np.array([H_num, M_num])
            - np.array([H_exact, M_exact])
        )

    else:
        H_num = np.nan
        M_num = np.nan
        error = np.nan

    equilibrium_check_records.append({
        "u": u,

        "H_parametric": H_exact,
        "M_parametric": M_exact,

        "H_root_solver": H_num,
        "M_root_solver": M_num,

        "solver_success": sol.success,
        "difference_norm": error
    })


equilibrium_check_df = pd.DataFrame(
    equilibrium_check_records
)

equilibrium_check_df.to_csv(
    EXP_DIR / "equilibrium_numerical_check.csv",
    index=False
)

display(equilibrium_check_df)

In [ ]:
# ============================================================
# ANALYTICAL VS FINITE-DIFFERENCE JACOBIAN
# ============================================================

def finite_difference_jacobian(func, x, eps=1e-6):
    x = np.asarray(x, dtype=float)

    f0 = np.asarray(func(x))
    J = np.zeros((len(f0), len(x)))

    for j in range(len(x)):
        dx = np.zeros_like(x)
        dx[j] = eps

        f_plus = np.asarray(func(x + dx))
        f_minus = np.asarray(func(x - dx))

        J[:, j] = (
            f_plus - f_minus
        ) / (2.0 * eps)

    return J


# Representative interior test state
H_test = 0.43
M_test = 0.37
u_test = 0.20

J_analytical = jacobian_M0(
    H_test,
    M_test,
    u_test,
    PARAMS
)

J_numeric = finite_difference_jacobian(
    lambda x: rhs_M0(
        0.0,
        x,
        u_test,
        PARAMS
    ),
    np.array([H_test, M_test])
)

J_difference = J_analytical - J_numeric

jacobian_max_abs_error = np.max(
    np.abs(J_difference)
)

print("Analytical Jacobian")
print(J_analytical)

print("\nFinite-difference Jacobian")
print(J_numeric)

print("\nDifference")
print(J_difference)

print(
    "\nMaximum absolute difference:",
    jacobian_max_abs_error
)

In [ ]:
# ============================================================
# BASIC DYNAMICAL DOMAIN CHECK
# ============================================================
#
# This is NOT the perturbation experiment.
# It is only a numerical sanity check that trajectories remain
# within the intended normalized state domain.
# ============================================================

u_check = 0.20

initial_conditions = [
    [0.01, 0.01],
    [0.10, 0.90],
    [0.90, 0.10],
    [0.50, 0.50],
    [0.99, 0.99],
]

domain_check_records = []

for y0 in initial_conditions:

    sol = solve_ivp(
        fun=lambda t, y: rhs_M0(
            t, y, u_check, PARAMS
        ),
        t_span=(0.0, 100.0),
        y0=y0,
        method="LSODA",
        rtol=1e-10,
        atol=1e-12,
        dense_output=False
    )

    H = sol.y[0]
    M = sol.y[1]

    domain_check_records.append({
        "H0": y0[0],
        "M0": y0[1],

        "solver_success": sol.success,

        "H_min": np.min(H),
        "H_max": np.max(H),

        "M_min": np.min(M),
        "M_max": np.max(M),

        "H_within_0_1":
            np.min(H) >= -1e-8
            and np.max(H) <= 1 + 1e-8,

        "M_within_0_1":
            np.min(M) >= -1e-8
            and np.max(M) <= 1 + 1e-8,
    })


domain_check_df = pd.DataFrame(
    domain_check_records
)

domain_check_df.to_csv(
    EXP_DIR / "domain_checks.csv",
    index=False
)

display(domain_check_df)

In [ ]:
# ============================================================
# NUMBER OF EQUILIBRIA AS A FUNCTION OF u
# ============================================================

def roots_in_H_for_u(u_target, p, N=10000):
    """
    Find all scalar roots of:

        u(H) - u_target = 0

    on 0 < H < 1.
    """

    H_grid = np.linspace(
        p["H_min"],
        p["H_max"],
        N
    )

    f_grid = (
        equilibrium_u_from_H(H_grid, p)
        - u_target
    )

    roots = []

    for i in range(len(H_grid) - 1):

        f1 = f_grid[i]
        f2 = f_grid[i + 1]

        if not (
            np.isfinite(f1)
            and np.isfinite(f2)
        ):
            continue

        if f1 * f2 < 0:

            result = root_scalar(
                lambda h:
                    equilibrium_u_from_H(h, p)
                    - u_target,

                bracket=(
                    H_grid[i],
                    H_grid[i + 1]
                ),

                method="brentq",
                xtol=1e-12
            )

            roots.append(result.root)

    # Remove numerical duplicates
    if len(roots) == 0:
        return np.array([])

    roots = np.array(sorted(roots))

    unique_roots = [roots[0]]

    for r_ in roots[1:]:
        if abs(r_ - unique_roots[-1]) > 1e-6:
            unique_roots.append(r_)

    return np.array(unique_roots)


u_count_grid = np.linspace(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"],
    250
)

count_records = []

for u_target in u_count_grid:

    roots_H = roots_in_H_for_u(
        u_target,
        PARAMS
    )

    count_records.append({
        "u": u_target,
        "n_equilibria": len(roots_H),
        "H_roots": ";".join(
            f"{x:.10f}" for x in roots_H
        )
    })


count_df = pd.DataFrame(count_records)

count_df.to_csv(
    EXP_DIR / "equilibrium_counts_vs_u.csv",
    index=False
)

display(count_df.head())

In [ ]:
# ============================================================
# FIGURE 1
# Fixed-model equilibrium / bifurcation diagram
# ============================================================

plot_df = branch_df[
    (branch_df["u"] >= PARAMS["u_display_min"])
    &
    (branch_df["u"] <= PARAMS["u_display_max"])
].copy()


fig, ax = plt.subplots(figsize=(8.0, 6.0))

# Stable branch
stable_y = np.where(
    plot_df["stable"],
    plot_df["H_star"],
    np.nan
)

ax.plot(
    plot_df["u"],
    stable_y,
    linewidth=2.4,
    label="Stable equilibrium"
)

# Unstable branch
unstable_y = np.where(
    ~plot_df["stable"],
    plot_df["H_star"],
    np.nan
)

ax.plot(
    plot_df["u"],
    unstable_y,
    linestyle="--",
    linewidth=2.0,
    label="Unstable equilibrium"
)

# Fold points
if len(fold_df) > 0:

    fold_plot = fold_df[
        (fold_df["u_fold"] >= PARAMS["u_display_min"])
        &
        (fold_df["u_fold"] <= PARAMS["u_display_max"])
    ]

    ax.scatter(
        fold_plot["u_fold"],
        fold_plot["H_fold"],
        s=65,
        zorder=5,
        label="Fold"
    )

ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Host inflammatory equilibrium, $H^*$",
    fontsize=12
)

ax.set_title(
    "Fixed-threshold host–microbial equilibrium structure",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

ax.set_ylim(0, 1)

ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

figure_path = (
    EXP_DIR
    / "fig01_fixed_model_bifurcation.jpg"
)

fig.savefig(
    figure_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(f"Saved high-resolution figure:\n{figure_path}")

In [ ]:
# ============================================================
# FIGURE 2
# Microbial equilibrium branch
# ============================================================

fig, ax = plt.subplots(figsize=(8.0, 6.0))

stable_y = np.where(
    plot_df["stable"],
    plot_df["M_star"],
    np.nan
)

unstable_y = np.where(
    ~plot_df["stable"],
    plot_df["M_star"],
    np.nan
)

ax.plot(
    plot_df["u"],
    stable_y,
    linewidth=2.4,
    label="Stable equilibrium"
)

ax.plot(
    plot_df["u"],
    unstable_y,
    linestyle="--",
    linewidth=2.0,
    label="Unstable equilibrium"
)

if len(fold_df) > 0:

    fold_plot = fold_df[
        (fold_df["u_fold"] >= PARAMS["u_display_min"])
        &
        (fold_df["u_fold"] <= PARAMS["u_display_max"])
    ]

    ax.scatter(
        fold_plot["u_fold"],
        fold_plot["M_fold"],
        s=65,
        zorder=5,
        label="Fold"
    )

ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Microbial functional equilibrium, $M^*$",
    fontsize=12
)

ax.set_title(
    "Microbial equilibrium along the fixed-threshold branch",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

ax.set_ylim(0, 1)

ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

figure_path = (
    EXP_DIR
    / "fig02_fixed_model_microbe_branch.jpg"
)

fig.savefig(
    figure_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(f"Saved high-resolution figure:\n{figure_path}")

In [ ]:
# ============================================================
# FIGURE 3
# Maximum real eigenvalue along the branch
# ============================================================

fig, ax = plt.subplots(figsize=(8.0, 6.0))

ax.plot(
    plot_df["u"],
    plot_df["max_real_eigenvalue"],
    linewidth=2.0
)

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.2
)

if len(fold_df) > 0:

    fold_plot = fold_df[
        (fold_df["u_fold"] >= PARAMS["u_display_min"])
        &
        (fold_df["u_fold"] <= PARAMS["u_display_max"])
    ]

    ax.scatter(
        fold_plot["u_fold"],
        np.zeros(len(fold_plot)),
        s=65,
        zorder=5,
        label="Fold"
    )

ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Maximum real part of Jacobian eigenvalues",
    fontsize=12
)

ax.set_title(
    "Local stability of fixed-model equilibria",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

if len(fold_df) > 0:
    ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

figure_path = (
    EXP_DIR
    / "fig03_fixed_model_stability.jpg"
)

fig.savefig(
    figure_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(f"Saved high-resolution figure:\n{figure_path}")

In [ ]:
# ============================================================
# EXPERIMENT 01 AUTOMATIC SUMMARY
# ============================================================

visible_folds = fold_df[
    (fold_df["u_fold"] >= PARAMS["u_display_min"])
    &
    (fold_df["u_fold"] <= PARAMS["u_display_max"])
].copy()

n_folds_visible = len(visible_folds)

if n_folds_visible == 0:
    regime_class = (
        "No folds detected in the specified u interval. "
        "The branch is monostable with respect to fold-mediated "
        "bistability over this interval."
    )

elif n_folds_visible == 2:
    regime_class = (
        "Two folds detected in the specified u interval. "
        "The fixed-threshold architecture contains an "
        "S-shaped equilibrium branch and a candidate bistable interval."
    )

else:
    regime_class = (
        f"{n_folds_visible} fold candidate(s) detected. "
        "Inspect the branch and fold diagnostics manually."
    )


max_equilibrium_solver_error = np.nanmax(
    equilibrium_check_df["difference_norm"]
)

all_domain_checks_pass = bool(
    domain_check_df["H_within_0_1"].all()
    and
    domain_check_df["M_within_0_1"].all()
)

if len(fold_df) > 0:
    max_fold_condition_error = np.max(
        np.abs(
            fold_df["fold_condition_error"]
        )
    )
else:
    max_fold_condition_error = np.nan


report = f"""
============================================================
EXPERIMENT 01
FIXED-THRESHOLD EQUILIBRIUM / BIFURCATION STRUCTURE
============================================================

Timestamp:
{datetime.now(timezone.utc).isoformat()}

MODEL
-----
M0:
    dH/dt = -H + F(M + u; theta0)
    dM/dt = r[-M + G(H)]

F(S;theta0):
    S^n / (S^n + theta0^n)

PROVISIONAL G(H):
    H^q / (H^q + K_M^q)

IMPORTANT:
The exact G(H) must be verified against the finalized
mathematical Methods before publication.

PARAMETERS
----------
{json.dumps(PARAMS, indent=4)}

RESULT
------
Number of detected fold points over full H branch:
{len(fold_df)}

Number of folds in displayed u interval:
{n_folds_visible}

Automatic regime classification:
{regime_class}

VERIFICATION
------------
Maximum analytical-vs-finite-difference Jacobian error:
{jacobian_max_abs_error:.6e}

Maximum numerical equilibrium-vs-parametric branch error:
{max_equilibrium_solver_error:.6e}

All tested H and M trajectories remained within intended
normalized domain:
{all_domain_checks_pass}

Maximum |F_S * G_H - 1| at detected folds:
{max_fold_condition_error}

INTERPRETATION BOUNDARY
-----------------------
This experiment characterizes ONLY the fixed-threshold null model.

It does NOT establish:
- adaptive signal interpretation,
- biological memory,
- adaptive-threshold-induced bistability,
- hysteresis,
- a cusp,
- or periodontal parameter calibration.

Those questions require subsequent experiments.
============================================================
"""

print(report)

with open(
    EXP_DIR / "verification_report.txt",
    "w"
) as f:
    f.write(report)

In [ ]:
# ============================================================
# README
# ============================================================

readme = """
EXPERIMENT 01 — FIXED-MODEL EQUILIBRIUM/BIFURCATION DIAGRAM
==========================================================

PURPOSE
-------
Establish the transition structure of the fixed-threshold
host–microbial reference model before introducing adaptive
response thresholds.

PRIMARY QUESTION
----------------
Can ordinary reciprocal nonlinear host–microbial feedback
already produce folds and bistability?

OUTPUTS
-------
parameters.json
    Exact exploratory parameters used.

software_versions.txt
    Reproducibility information.

equilibrium_branch.csv
    Complete parameterized equilibrium branch.

fold_points.csv
    Numerically identified fold candidates and independent
    fold-condition diagnostics.

equilibrium_numerical_check.csv
    Comparison of parametric and numerical equilibrium solutions.

domain_checks.csv
    Numerical state-domain checks.

equilibrium_counts_vs_u.csv
    Number of equilibria as a function of external forcing u.

verification_report.txt
    Automatic numerical verification summary.

fig01_fixed_model_bifurcation.jpg
    Main fixed-threshold H*(u) bifurcation diagram.

fig02_fixed_model_microbe_branch.jpg
    Corresponding M*(u) branch.

fig03_fixed_model_stability.jpg
    Local-stability diagnostic based on Jacobian eigenvalues.

IMPORTANT MODEL-DEFINITION ISSUE
--------------------------------
The currently retrieved source material specifies G(H)
symbolically but does not restate its complete functional
formula.

The present notebook therefore uses:

    G(H) = H^q / (H^q + K_M^q)

as a provisional Hill-response definition consistent with
the q and K_M parameterization.

This definition MUST be checked against the finalized Methods
before interpreting or publishing the numerical results.

SCOPE
-----
Experiment 01 concerns the FIXED-threshold model only.

No inference about adaptive thresholds, biological memory,
state-dependent interpretation, or hysteresis should be made
from this experiment alone.
"""

with open(
    EXP_DIR / "experiment_README.txt",
    "w"
) as f:
    f.write(readme)

print(readme)

In [ ]:
# ============================================================
# FINAL INVENTORY
# ============================================================

print("Files saved in:")
print(EXP_DIR)
print("\n---------------------------------------------")

for path in sorted(EXP_DIR.iterdir()):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"{path.name:45s} {size_kb:10.1f} KB")

In [ ]:
# ============================================================
# CATCH-UP EXPORT CELL
# Experiment 01 — save all numerical outputs already in memory
# ============================================================
#
# Add and run this ONE cell after the cells you already ran.
#
# It does NOT rerun the model or simulations.
# It simply exports numerical objects currently present
# in the Colab runtime.
#
# REQUIREMENT:
# The current Colab runtime must still contain the variables
# generated by the previous cells.
# ============================================================

from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
from datetime import datetime, timezone


# ------------------------------------------------------------
# 1. Locate experiment directory
# ------------------------------------------------------------

if "EXP_DIR" not in globals():
    EXP_DIR = Path(
        "/content/drive/MyDrive/"
        "Biosemiotic_Perio/"
        "Experiment_01_Fixed_Model_Bifurcation"
    )
else:
    EXP_DIR = Path(EXP_DIR)

EXP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Export directory:\n{EXP_DIR}\n")


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

export_log = []

def log_export(filename, status, note=""):
    export_log.append({
        "file": filename,
        "status": status,
        "note": note
    })

def save_df_if_present(var_name, filename):
    """
    Save a DataFrame if it exists in the current runtime.
    """
    if var_name in globals():
        obj = globals()[var_name]

        if isinstance(obj, pd.DataFrame):
            path = EXP_DIR / filename
            obj.to_csv(
                path,
                index=False,
                float_format="%.17g"
            )
            log_export(
                filename,
                "saved",
                f"source variable: {var_name}; rows={len(obj)}"
            )
            print(f"SAVED: {filename}")
            return True

    log_export(
        filename,
        "not saved",
        f"variable '{var_name}' not found as DataFrame"
    )
    print(f"SKIPPED: {filename} — {var_name} not available")
    return False


# ------------------------------------------------------------
# 3. Parameters: CSV + JSON
# ------------------------------------------------------------

if "PARAMS" in globals():

    parameters_df_export = pd.DataFrame([
        {
            "parameter": key,
            "value": value
        }
        for key, value in PARAMS.items()
    ])

    parameters_df_export.to_csv(
        EXP_DIR / "parameters.csv",
        index=False,
        float_format="%.17g"
    )

    with open(
        EXP_DIR / "parameters.json",
        "w"
    ) as f:
        json.dump(
            PARAMS,
            f,
            indent=4,
            default=str
        )

    log_export(
        "parameters.csv",
        "saved",
        f"{len(parameters_df_export)} parameters"
    )

    log_export(
        "parameters.json",
        "saved",
        "configuration copy"
    )

    print("SAVED: parameters.csv")
    print("SAVED: parameters.json")

else:
    log_export(
        "parameters.csv",
        "not saved",
        "PARAMS not available"
    )
    print("SKIPPED: parameters — PARAMS not available")


# ------------------------------------------------------------
# 4. Master numerical datasets
# ------------------------------------------------------------

save_df_if_present(
    "branch_df",
    "equilibrium_branch.csv"
)

save_df_if_present(
    "fold_df",
    "fold_points.csv"
)

save_df_if_present(
    "equilibrium_check_df",
    "equilibrium_numerical_check.csv"
)

save_df_if_present(
    "domain_check_df",
    "domain_checks.csv"
)

save_df_if_present(
    "count_df",
    "equilibrium_counts_vs_u.csv"
)


# ------------------------------------------------------------
# 5. Jacobian check CSV
# ------------------------------------------------------------

if (
    "J_analytical" in globals()
    and "J_numeric" in globals()
):

    J_analytical_export = np.asarray(
        J_analytical,
        dtype=float
    )

    J_numeric_export = np.asarray(
        J_numeric,
        dtype=float
    )

    J_difference_export = (
        J_analytical_export
        - J_numeric_export
    )

    state_names = ["H", "M"]

    jacobian_records = []

    for i, row_name in enumerate(state_names):
        for j, col_name in enumerate(state_names):

            jacobian_records.append({
                "test_H":
                    globals().get("H_test", np.nan),

                "test_M":
                    globals().get("M_test", np.nan),

                "test_u":
                    globals().get("u_test", np.nan),

                "row_variable":
                    row_name,

                "column_variable":
                    col_name,

                "analytical_value":
                    J_analytical_export[i, j],

                "finite_difference_value":
                    J_numeric_export[i, j],

                "difference":
                    J_difference_export[i, j],

                "absolute_difference":
                    abs(J_difference_export[i, j])
            })

    jacobian_check_export_df = pd.DataFrame(
        jacobian_records
    )

    jacobian_check_export_df.to_csv(
        EXP_DIR / "jacobian_check.csv",
        index=False,
        float_format="%.17g"
    )

    log_export(
        "jacobian_check.csv",
        "saved",
        "analytical vs finite-difference Jacobian"
    )

    print("SAVED: jacobian_check.csv")

else:
    log_export(
        "jacobian_check.csv",
        "not saved",
        "J_analytical and/or J_numeric unavailable"
    )

    print(
        "SKIPPED: jacobian_check.csv — "
        "Jacobian variables unavailable"
    )


# ------------------------------------------------------------
# 6. Reconstruct plotting dataset if necessary
# ------------------------------------------------------------

plot_df_export = None

if "plot_df" in globals():

    plot_df_export = plot_df.copy()

elif (
    "branch_df" in globals()
    and "PARAMS" in globals()
):

    plot_df_export = branch_df[
        (branch_df["u"] >= PARAMS["u_display_min"])
        &
        (branch_df["u"] <= PARAMS["u_display_max"])
    ].copy()


# ------------------------------------------------------------
# 7. Figure-specific source-data CSVs
# ------------------------------------------------------------

if plot_df_export is not None:

    # FIGURE 1
    fig01_cols = [
        c for c in [
            "u",
            "H_star",
            "M_star",
            "max_real_eigenvalue",
            "stable",
            "unstable",
            "near_neutral"
        ]
        if c in plot_df_export.columns
    ]

    fig01_data_export = plot_df_export[
        fig01_cols
    ].copy()

    fig01_data_export.to_csv(
        EXP_DIR
        / "fig01_fixed_model_bifurcation_data.csv",
        index=False,
        float_format="%.17g"
    )

    log_export(
        "fig01_fixed_model_bifurcation_data.csv",
        "saved",
        f"{len(fig01_data_export)} plotted branch points"
    )


    # FIGURE 2
    fig02_cols = [
        c for c in [
            "u",
            "M_star",
            "H_star",
            "stable",
            "unstable",
            "near_neutral"
        ]
        if c in plot_df_export.columns
    ]

    fig02_data_export = plot_df_export[
        fig02_cols
    ].copy()

    fig02_data_export.to_csv(
        EXP_DIR
        / "fig02_fixed_model_microbe_branch_data.csv",
        index=False,
        float_format="%.17g"
    )

    log_export(
        "fig02_fixed_model_microbe_branch_data.csv",
        "saved",
        f"{len(fig02_data_export)} plotted branch points"
    )


    # FIGURE 3
    fig03_cols = [
        c for c in [
            "u",
            "H_star",
            "M_star",
            "eig1_real",
            "eig1_imag",
            "eig2_real",
            "eig2_imag",
            "max_real_eigenvalue",
            "trace_J",
            "det_J",
            "F_S_times_G_H",
            "stable",
            "unstable",
            "near_neutral"
        ]
        if c in plot_df_export.columns
    ]

    fig03_data_export = plot_df_export[
        fig03_cols
    ].copy()

    fig03_data_export.to_csv(
        EXP_DIR
        / "fig03_fixed_model_stability_data.csv",
        index=False,
        float_format="%.17g"
    )

    log_export(
        "fig03_fixed_model_stability_data.csv",
        "saved",
        f"{len(fig03_data_export)} plotted branch points"
    )

    print(
        "SAVED: figure-specific numerical source CSVs"
    )

else:

    for fn in [
        "fig01_fixed_model_bifurcation_data.csv",
        "fig02_fixed_model_microbe_branch_data.csv",
        "fig03_fixed_model_stability_data.csv"
    ]:
        log_export(
            fn,
            "not saved",
            "plot_df/branch_df unavailable"
        )

    print(
        "SKIPPED: figure source-data CSVs — "
        "plotting dataset unavailable"
    )


# ------------------------------------------------------------
# 8. Fold data specifically used in figures
# ------------------------------------------------------------

if "fold_df" in globals():

    fold_figure_df = fold_df.copy()

    if "PARAMS" in globals():

        fold_figure_df = fold_figure_df[
            (
                fold_figure_df["u_fold"]
                >= PARAMS["u_display_min"]
            )
            &
            (
                fold_figure_df["u_fold"]
                <= PARAMS["u_display_max"]
            )
        ].copy()

    fold_figure_df.to_csv(
        EXP_DIR / "figure_fold_points.csv",
        index=False,
        float_format="%.17g"
    )

    log_export(
        "figure_fold_points.csv",
        "saved",
        f"{len(fold_figure_df)} visible fold points"
    )

    print("SAVED: figure_fold_points.csv")


# ------------------------------------------------------------
# 9. Numerical summary metrics
# ------------------------------------------------------------

summary_records = []

def add_metric(metric, value, units_or_type="dimensionless"):
    summary_records.append({
        "metric": metric,
        "value": value,
        "units_or_type": units_or_type
    })


if "branch_df" in globals():

    add_metric(
        "number_of_branch_points",
        len(branch_df),
        "count"
    )

    if "u" in branch_df.columns:
        add_metric(
            "branch_u_min",
            branch_df["u"].min()
        )

        add_metric(
            "branch_u_max",
            branch_df["u"].max()
        )

    if "H_star" in branch_df.columns:
        add_metric(
            "branch_H_min",
            branch_df["H_star"].min()
        )

        add_metric(
            "branch_H_max",
            branch_df["H_star"].max()
        )

    if "M_star" in branch_df.columns:
        add_metric(
            "branch_M_min",
            branch_df["M_star"].min()
        )

        add_metric(
            "branch_M_max",
            branch_df["M_star"].max()
        )


if "fold_df" in globals():

    add_metric(
        "number_of_detected_folds_full_branch",
        len(fold_df),
        "count"
    )

    if len(fold_df) > 0:

        if "fold_condition_error" in fold_df.columns:
            add_metric(
                "maximum_absolute_fold_condition_error",
                np.max(
                    np.abs(
                        fold_df[
                            "fold_condition_error"
                        ]
                    )
                )
            )

        if "min_abs_eigenvalue" in fold_df.columns:
            add_metric(
                "minimum_absolute_eigenvalue_at_fold",
                fold_df[
                    "min_abs_eigenvalue"
                ].min()
            )


if "equilibrium_check_df" in globals():

    if "difference_norm" in equilibrium_check_df.columns:
        add_metric(
            "maximum_equilibrium_solver_difference_norm",
            equilibrium_check_df[
                "difference_norm"
            ].max()
        )


if "jacobian_max_abs_error" in globals():

    add_metric(
        "jacobian_max_absolute_error",
        jacobian_max_abs_error
    )

elif (
    "J_analytical" in globals()
    and "J_numeric" in globals()
):

    add_metric(
        "jacobian_max_absolute_error",
        np.max(
            np.abs(
                np.asarray(J_analytical)
                -
                np.asarray(J_numeric)
            )
        )
    )


if "domain_check_df" in globals():

    if (
        "H_within_0_1" in domain_check_df.columns
        and
        "M_within_0_1" in domain_check_df.columns
    ):

        all_domain_checks_pass_export = bool(
            domain_check_df[
                "H_within_0_1"
            ].all()
            and
            domain_check_df[
                "M_within_0_1"
            ].all()
        )

        add_metric(
            "all_domain_checks_pass",
            int(all_domain_checks_pass_export),
            "boolean_0_1"
        )


if "count_df" in globals():

    if "n_equilibria" in count_df.columns:

        add_metric(
            "maximum_number_of_equilibria_observed",
            count_df["n_equilibria"].max(),
            "count"
        )

        n_three = (
            count_df["n_equilibria"] == 3
        ).sum()

        add_metric(
            "number_of_u_grid_points_with_three_equilibria",
            n_three,
            "count"
        )


if "PARAMS" in globals():

    if "u_display_min" in PARAMS:
        add_metric(
            "u_display_min",
            PARAMS["u_display_min"]
        )

    if "u_display_max" in PARAMS:
        add_metric(
            "u_display_max",
            PARAMS["u_display_max"]
        )


summary_metrics_export_df = pd.DataFrame(
    summary_records
)

summary_metrics_export_df.to_csv(
    EXP_DIR / "experiment_summary_metrics.csv",
    index=False,
    float_format="%.17g"
)

log_export(
    "experiment_summary_metrics.csv",
    "saved",
    f"{len(summary_metrics_export_df)} summary metrics"
)

print("SAVED: experiment_summary_metrics.csv")


# ------------------------------------------------------------
# 10. Export log / manifest
# ------------------------------------------------------------

export_manifest_df = pd.DataFrame(
    export_log
)

export_manifest_df.insert(
    0,
    "export_timestamp_utc",
    datetime.now(timezone.utc).isoformat()
)

export_manifest_df.to_csv(
    EXP_DIR / "export_manifest.csv",
    index=False
)

print("SAVED: export_manifest.csv")


# ------------------------------------------------------------
# 11. Inventory every file currently in the experiment folder
# ------------------------------------------------------------

inventory_records = []

for path in sorted(EXP_DIR.iterdir()):

    if path.is_file():

        inventory_records.append({
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "size_kB": path.stat().st_size / 1024.0
        })


inventory_df = pd.DataFrame(
    inventory_records
)

inventory_df.to_csv(
    EXP_DIR / "file_inventory.csv",
    index=False,
    float_format="%.17g"
)

print("SAVED: file_inventory.csv")


# ------------------------------------------------------------
# 12. Create one ZIP for convenient download/upload
# ------------------------------------------------------------

BASE_DIR_EXPORT = EXP_DIR.parent

zip_base = (
    BASE_DIR_EXPORT
    / f"{EXP_DIR.name}_EXPORT"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP_DIR)
)

print("\n================================================")
print("CATCH-UP EXPORT COMPLETE")
print("================================================")
print(f"\nExperiment folder:\n{EXP_DIR}")
print(f"\nZIP archive:\n{zip_path}")

print("\nFiles now present:\n")

for path in sorted(EXP_DIR.iterdir()):
    if path.is_file():
        print(
            f"{path.name:50s} "
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

print("\n================================================")
print(
    "Upload the ZIP, or at minimum the CSV files, "
    "here for analysis."
)
print("================================================")

# Experiment 2: the adaptive-model equilibrium/bifurcation diagram

In [ ]:
# ============================================================
# EXPERIMENT 02
# Adaptive-threshold equilibrium / bifurcation structure
# ============================================================
#
# IMPORTANT:
# This experiment writes ONLY to:
#
# MyDrive/Biosemiotic_Perio/
# Experiment_02_Adaptive_Model_Bifurcation/
#
# Experiment 01 is never overwritten.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP1_DIR = (
    BASE_DIR
    / "Experiment_01_Fixed_Model_Bifurcation"
)

EXP2_DIR = (
    BASE_DIR
    / "Experiment_02_Adaptive_Model_Bifurcation"
)

EXP2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment 1 directory:")
print(EXP1_DIR)

print("\nExperiment 2 directory:")
print(EXP2_DIR)

print("\nExperiment 2 exists:")
print(EXP2_DIR.exists())

In [ ]:
# ============================================================
# IMPORTS + LOAD EXPERIMENT 01 PARAMETERS
# ============================================================

import sys
import json
import platform
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import scipy
from scipy.optimize import root, root_scalar


# ------------------------------------------------------------
# Load Experiment 01 parameters
# ------------------------------------------------------------

exp1_parameter_file = (
    EXP1_DIR / "parameters.json"
)

if not exp1_parameter_file.exists():
    raise FileNotFoundError(
        "Experiment 01 parameters.json was not found at:\n"
        f"{exp1_parameter_file}\n\n"
        "Experiment 02 should use the exact same fixed-model "
        "parameters for a valid comparison."
    )


with open(exp1_parameter_file, "r") as f:
    PARAMS_EXP1 = json.load(f)


print("Loaded Experiment 01 parameters:")
print(json.dumps(PARAMS_EXP1, indent=4))


# ------------------------------------------------------------
# Save a copy INSIDE Experiment 02 for provenance
# ------------------------------------------------------------

exp1_base_df = pd.DataFrame([
    {
        "parameter": key,
        "experiment01_value": value
    }
    for key, value in PARAMS_EXP1.items()
])

exp1_base_df.to_csv(
    EXP2_DIR
    / "experiment01_parameters_used_as_base.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# EXPERIMENT 02 CONFIGURATION
# ============================================================
#
# alpha values are exploratory dimensionless values.
# They are NOT periodontal biological estimates.
#
# Experiment 3 will perform the dense two-parameter search.
# ============================================================

ALPHA_VALUES = np.array([
    0.0,
    0.5,
    1.0,
    2.0,
    3.0
], dtype=float)


# One exploratory threshold-memory timescale for stability.
#
# epsilon affects stability/time dynamics,
# NOT equilibrium branch geometry.
EPSILON = 0.10


PARAMS = PARAMS_EXP1.copy()

PARAMS["epsilon"] = EPSILON

# Make sure numeric values loaded from JSON are appropriate
PARAMS["n"] = float(PARAMS["n"])
PARAMS["q"] = float(PARAMS["q"])
PARAMS["K_M"] = float(PARAMS["K_M"])
PARAMS["theta0"] = float(PARAMS["theta0"])
PARAMS["r"] = float(PARAMS["r"])

PARAMS["H_min"] = float(
    PARAMS.get("H_min", 1e-5)
)

PARAMS["H_max"] = float(
    PARAMS.get("H_max", 1.0 - 1e-5)
)

PARAMS["N_branch"] = int(
    PARAMS.get("N_branch", 20000)
)

PARAMS["u_display_min"] = float(
    PARAMS.get("u_display_min", 0.0)
)

PARAMS["u_display_max"] = float(
    PARAMS.get("u_display_max", 1.20)
)

PARAMS["stability_tol"] = float(
    PARAMS.get("stability_tol", 1e-8)
)

PARAMS["fold_root_tol"] = float(
    PARAMS.get("fold_root_tol", 1e-12)
)


# Save alpha values separately
alpha_df = pd.DataFrame({
    "alpha": ALPHA_VALUES
})

alpha_df.to_csv(
    EXP2_DIR / "alpha_values.csv",
    index=False,
    float_format="%.17g"
)


# Save Experiment 2 parameter set as CSV
params2_records = [
    {
        "parameter": key,
        "value": value
    }
    for key, value in PARAMS.items()
]

params2_df = pd.DataFrame(params2_records)

params2_df.to_csv(
    EXP2_DIR / "experiment02_parameters.csv",
    index=False,
    float_format="%.17g"
)


# JSON version
experiment02_config = {
    "base_parameters_from_experiment01": PARAMS_EXP1,
    "epsilon": EPSILON,
    "alpha_values": ALPHA_VALUES.tolist()
}

with open(
    EXP2_DIR / "experiment02_parameters.json",
    "w"
) as f:
    json.dump(
        experiment02_config,
        f,
        indent=4
    )


print("Experiment 2 configuration:")
print(json.dumps(experiment02_config, indent=4))

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def F(S, theta, n):
    """
    Host-response Hill function.
    """
    S = np.asarray(S)

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(S, theta, n):
    """
    Partial derivative of F with respect to S.
    """
    S = np.asarray(S)

    numerator = (
        n
        * np.power(theta, n)
        * np.power(S, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


def dF_dtheta(S, theta, n):
    """
    Partial derivative of F with respect to theta.
    """
    S = np.asarray(S)

    numerator = (
        -n
        * np.power(S, n)
        * np.power(theta, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


# ------------------------------------------------------------
# SAME G(H) AS EXPERIMENT 01
# ------------------------------------------------------------

def G(H, K_M, q):
    H = np.asarray(H)

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(H, K_M, q):
    H = np.asarray(H)

    numerator = (
        q
        * np.power(K_M, q)
        * np.power(H, q - 1)
    )

    denominator = (
        np.power(H, q)
        +
        np.power(K_M, q)
    )**2

    return numerator / denominator

In [ ]:
# ============================================================
# ADAPTIVE MODEL M1
# ============================================================

def rhs_M1(t, y, u, alpha, p):
    """
    Adaptive-threshold model M1.

    State:
        y = [H, M, theta]
    """

    H, M, theta = y

    S = M + u

    dH = (
        -H
        + F(S, theta, p["n"])
    )

    dM = (
        p["r"]
        * (
            -M
            + G(H, p["K_M"], p["q"])
        )
    )

    dtheta = (
        p["epsilon"]
        * theta
        * (
            np.log(p["theta0"] / theta)
            - alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])


def jacobian_M1(H, M, theta, u, alpha, p):
    """
    Analytical Jacobian of M1.
    """

    S = M + u

    FS = dF_dS(
        S,
        theta,
        p["n"]
    )

    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )

    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )

    r = p["r"]
    eps = p["epsilon"]

    adaptive_bracket = (
        np.log(p["theta0"] / theta)
        - alpha * H
    )

    return np.array([
        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            r * GH,
            -r,
            0.0
        ],

        [
            -eps * alpha * theta,
            0.0,
            eps * (
                adaptive_bracket - 1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# ADAPTIVE EQUILIBRIUM PARAMETERIZATION
# ============================================================

def theta_star_from_H(H, alpha, p):
    """
    Adaptive equilibrium threshold:

        theta* = theta0 exp(-alpha H)
    """

    return (
        p["theta0"]
        * np.exp(-alpha * H)
    )


def M_star_from_H(H, p):
    """
    Microbial equilibrium:
        M* = G(H*)
    """

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def u_star_from_H(H, alpha, p):
    """
    Adaptive equilibrium branch:

        u(H;alpha)
        =
        theta0 exp(-alpha H)
        (H/(1-H))^(1/n)
        - G(H)
    """

    theta_star = theta_star_from_H(
        H,
        alpha,
        p
    )

    response_signal = (
        theta_star
        * np.power(
            H / (1.0 - H),
            1.0 / p["n"]
        )
    )

    return (
        response_signal
        -
        G(H, p["K_M"], p["q"])
    )


def du_dH_adaptive(H, alpha, p):
    """
    Analytical derivative of u(H;alpha).

    Adaptive fold candidates satisfy:
        du/dH = 0
    """

    n = p["n"]

    A = np.power(
        H / (1.0 - H),
        1.0 / n
    )

    dA_dH = (
        A
        /
        (
            n
            * H
            * (1.0 - H)
        )
    )

    theta_star = theta_star_from_H(
        H,
        alpha,
        p
    )

    return (
        theta_star
        * (
            dA_dH
            - alpha * A
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )

In [ ]:
# ============================================================
# BUILD ALL ADAPTIVE EQUILIBRIUM BRANCHES
# ============================================================

H_grid = np.linspace(
    PARAMS["H_min"],
    PARAMS["H_max"],
    PARAMS["N_branch"]
)

branch_records = []


for alpha in ALPHA_VALUES:

    M_values = M_star_from_H(
        H_grid,
        PARAMS
    )

    theta_values = theta_star_from_H(
        H_grid,
        alpha,
        PARAMS
    )

    u_values = u_star_from_H(
        H_grid,
        alpha,
        PARAMS
    )

    for H, M, theta, u in zip(
        H_grid,
        M_values,
        theta_values,
        u_values
    ):

        J = jacobian_M1(
            H,
            M,
            theta,
            u,
            alpha,
            PARAMS
        )

        eigvals = np.linalg.eigvals(J)

        # Sort reproducibly by real part
        eigvals = eigvals[
            np.argsort(eigvals.real)
        ]

        max_real = np.max(
            eigvals.real
        )

        stable = (
            max_real
            <
            -PARAMS["stability_tol"]
        )

        unstable = (
            max_real
            >
            PARAMS["stability_tol"]
        )

        near_neutral = (
            not stable
            and
            not unstable
        )

        S = M + u

        FS = dF_dS(
            S,
            theta,
            PARAMS["n"]
        )

        Ftheta = dF_dtheta(
            S,
            theta,
            PARAMS["n"]
        )

        GH = dG_dH(
            H,
            PARAMS["K_M"],
            PARAMS["q"]
        )

        adaptive_fold_gain = (
            FS * GH
            -
            alpha
            * theta
            * Ftheta
        )

        branch_records.append({

            "alpha": alpha,

            "H_star": H,
            "M_star": M,
            "theta_star": theta,
            "u": u,
            "S_star": S,

            "eig1_real": eigvals[0].real,
            "eig1_imag": eigvals[0].imag,

            "eig2_real": eigvals[1].real,
            "eig2_imag": eigvals[1].imag,

            "eig3_real": eigvals[2].real,
            "eig3_imag": eigvals[2].imag,

            "max_real_eigenvalue":
                max_real,

            "trace_J":
                np.trace(J),

            "det_J":
                np.linalg.det(J),

            "F_S":
                FS,

            "F_theta":
                Ftheta,

            "G_H":
                GH,

            "adaptive_fold_gain":
                adaptive_fold_gain,

            "adaptive_fold_condition_error":
                adaptive_fold_gain - 1.0,

            "stable":
                stable,

            "unstable":
                unstable,

            "near_neutral":
                near_neutral
        })


adaptive_branch_df = pd.DataFrame(
    branch_records
)

adaptive_branch_df.to_csv(
    EXP2_DIR
    / "adaptive_equilibrium_branches_all_alpha.csv",
    index=False,
    float_format="%.17g"
)

print(
    "Saved master adaptive branch dataset:"
)

print(
    EXP2_DIR
    / "adaptive_equilibrium_branches_all_alpha.csv"
)

display(
    adaptive_branch_df.head()
)

In [ ]:
# ============================================================
# DETECT ADAPTIVE FOLDS
# ============================================================

fold_records = []


for alpha in ALPHA_VALUES:

    derivative_values = du_dH_adaptive(
        H_grid,
        alpha,
        PARAMS
    )

    brackets = []

    for i in range(
        len(H_grid) - 1
    ):

        y1 = derivative_values[i]
        y2 = derivative_values[i + 1]

        if not (
            np.isfinite(y1)
            and
            np.isfinite(y2)
        ):
            continue

        if y1 == 0.0:

            brackets.append(
                (
                    H_grid[i],
                    H_grid[i]
                )
            )

        elif y1 * y2 < 0.0:

            brackets.append(
                (
                    H_grid[i],
                    H_grid[i + 1]
                )
            )


    for bracket_id, (a, b) in enumerate(
        brackets,
        start=1
    ):

        if a == b:
            Hf = a

        else:
            solution = root_scalar(
                lambda H:
                    du_dH_adaptive(
                        H,
                        alpha,
                        PARAMS
                    ),
                bracket=(a, b),
                method="brentq",
                xtol=PARAMS[
                    "fold_root_tol"
                ]
            )

            Hf = solution.root


        Mf = M_star_from_H(
            Hf,
            PARAMS
        )

        thetaf = theta_star_from_H(
            Hf,
            alpha,
            PARAMS
        )

        uf = u_star_from_H(
            Hf,
            alpha,
            PARAMS
        )

        Sf = Mf + uf


        Jf = jacobian_M1(
            Hf,
            Mf,
            thetaf,
            uf,
            alpha,
            PARAMS
        )

        eigf = np.linalg.eigvals(Jf)


        FS = dF_dS(
            Sf,
            thetaf,
            PARAMS["n"]
        )

        Ftheta = dF_dtheta(
            Sf,
            thetaf,
            PARAMS["n"]
        )

        GH = dG_dH(
            Hf,
            PARAMS["K_M"],
            PARAMS["q"]
        )


        fold_gain = (
            FS * GH
            -
            alpha
            * thetaf
            * Ftheta
        )


        fold_records.append({

            "alpha":
                alpha,

            "fold_id":
                bracket_id,

            "H_fold":
                Hf,

            "M_fold":
                Mf,

            "theta_fold":
                thetaf,

            "u_fold":
                uf,

            "S_fold":
                Sf,

            "du_dH":
                du_dH_adaptive(
                    Hf,
                    alpha,
                    PARAMS
                ),

            "F_S":
                FS,

            "F_theta":
                Ftheta,

            "G_H":
                GH,

            "adaptive_fold_gain":
                fold_gain,

            "fold_condition_error":
                fold_gain - 1.0,

            "eig1_real":
                eigf[0].real,

            "eig1_imag":
                eigf[0].imag,

            "eig2_real":
                eigf[1].real,

            "eig2_imag":
                eigf[1].imag,

            "eig3_real":
                eigf[2].real,

            "eig3_imag":
                eigf[2].imag,

            "minimum_absolute_eigenvalue":
                np.min(
                    np.abs(eigf)
                )
        })


adaptive_fold_df = pd.DataFrame(
    fold_records
)

adaptive_fold_df.to_csv(
    EXP2_DIR
    / "adaptive_fold_points_all_alpha.csv",
    index=False,
    float_format="%.17g"
)

print(
    f"Total detected folds: "
    f"{len(adaptive_fold_df)}"
)

display(adaptive_fold_df)

In [ ]:
# ============================================================
# alpha = 0 VS EXPERIMENT 01
# ============================================================

exp1_branch_path = (
    EXP1_DIR
    / "equilibrium_branch.csv"
)

if exp1_branch_path.exists():

    exp1_branch_df = pd.read_csv(
        exp1_branch_path
    )

    alpha0_df = (
        adaptive_branch_df[
            adaptive_branch_df["alpha"] == 0.0
        ]
        .reset_index(drop=True)
    )

    n_compare = min(
        len(exp1_branch_df),
        len(alpha0_df)
    )

    comparison_df = pd.DataFrame({

        "H_exp1":
            exp1_branch_df[
                "H_star"
            ].iloc[:n_compare].values,

        "H_alpha0":
            alpha0_df[
                "H_star"
            ].iloc[:n_compare].values,

        "M_exp1":
            exp1_branch_df[
                "M_star"
            ].iloc[:n_compare].values,

        "M_alpha0":
            alpha0_df[
                "M_star"
            ].iloc[:n_compare].values,

        "u_exp1":
            exp1_branch_df[
                "u"
            ].iloc[:n_compare].values,

        "u_alpha0":
            alpha0_df[
                "u"
            ].iloc[:n_compare].values
    })


    comparison_df[
        "absolute_H_difference"
    ] = np.abs(
        comparison_df["H_exp1"]
        -
        comparison_df["H_alpha0"]
    )

    comparison_df[
        "absolute_M_difference"
    ] = np.abs(
        comparison_df["M_exp1"]
        -
        comparison_df["M_alpha0"]
    )

    comparison_df[
        "absolute_u_difference"
    ] = np.abs(
        comparison_df["u_exp1"]
        -
        comparison_df["u_alpha0"]
    )


    comparison_df.to_csv(
        EXP2_DIR
        / "alpha0_vs_experiment01_check.csv",
        index=False,
        float_format="%.17g"
    )


    print(
        "Maximum alpha=0 vs Experiment 1 "
        "differences:"
    )

    print(
        "H:",
        comparison_df[
            "absolute_H_difference"
        ].max()
    )

    print(
        "M:",
        comparison_df[
            "absolute_M_difference"
        ].max()
    )

    print(
        "u:",
        comparison_df[
            "absolute_u_difference"
        ].max()
    )

else:

    comparison_df = pd.DataFrame()

    print(
        "Experiment 1 equilibrium_branch.csv "
        "not found; alpha=0 comparison skipped."
    )

In [ ]:
# ============================================================
# EQUILIBRIUM RESIDUAL CHECKS
# ============================================================

residual_records = []


for alpha in ALPHA_VALUES:

    branch_alpha = (
        adaptive_branch_df[
            adaptive_branch_df[
                "alpha"
            ] == alpha
        ]
        .reset_index(drop=True)
    )

    sample_indices = np.linspace(
        int(
            0.05
            * len(branch_alpha)
        ),
        int(
            0.95
            * len(branch_alpha)
        ),
        12,
        dtype=int
    )


    for idx in sample_indices:

        row = branch_alpha.iloc[idx]

        H = row["H_star"]
        M = row["M_star"]
        theta = row["theta_star"]
        u = row["u"]

        residual = rhs_M1(
            0.0,
            np.array([
                H,
                M,
                theta
            ]),
            u,
            alpha,
            PARAMS
        )

        residual_records.append({

            "alpha":
                alpha,

            "H_star":
                H,

            "M_star":
                M,

            "theta_star":
                theta,

            "u":
                u,

            "dH_residual":
                residual[0],

            "dM_residual":
                residual[1],

            "dtheta_residual":
                residual[2],

            "residual_norm":
                np.linalg.norm(
                    residual
                )
        })


residual_check_df = pd.DataFrame(
    residual_records
)

residual_check_df.to_csv(
    EXP2_DIR
    / "equilibrium_residual_checks.csv",
    index=False,
    float_format="%.17g"
)

display(residual_check_df)

In [ ]:
# ============================================================
# EQUILIBRIUM COUNTS AS FUNCTION OF u FOR EACH alpha
# ============================================================

def roots_in_H_for_u_alpha(
    u_target,
    alpha,
    p,
    N=8000
):

    H_test = np.linspace(
        p["H_min"],
        p["H_max"],
        N
    )

    values = (
        u_star_from_H(
            H_test,
            alpha,
            p
        )
        -
        u_target
    )

    roots = []

    for i in range(
        len(H_test) - 1
    ):

        f1 = values[i]
        f2 = values[i + 1]

        if not (
            np.isfinite(f1)
            and
            np.isfinite(f2)
        ):
            continue

        if f1 * f2 < 0.0:

            result = root_scalar(
                lambda H:
                    u_star_from_H(
                        H,
                        alpha,
                        p
                    )
                    -
                    u_target,

                bracket=(
                    H_test[i],
                    H_test[i + 1]
                ),

                method="brentq",
                xtol=1e-12
            )

            roots.append(
                result.root
            )


    if len(roots) == 0:
        return np.array([])


    roots = np.array(
        sorted(roots)
    )

    unique_roots = [
        roots[0]
    ]

    for value in roots[1:]:

        if (
            abs(
                value
                -
                unique_roots[-1]
            )
            >
            1e-6
        ):
            unique_roots.append(
                value
            )

    return np.array(
        unique_roots
    )


u_count_grid = np.linspace(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"],
    250
)


equilibrium_count_records = []


for alpha in ALPHA_VALUES:

    for u_target in u_count_grid:

        H_roots = roots_in_H_for_u_alpha(
            u_target,
            alpha,
            PARAMS
        )

        equilibrium_count_records.append({

            "alpha":
                alpha,

            "u":
                u_target,

            "n_equilibria":
                len(H_roots),

            "H_roots":
                ";".join(
                    f"{x:.17g}"
                    for x in H_roots
                )
        })


equilibrium_count_df = pd.DataFrame(
    equilibrium_count_records
)

equilibrium_count_df.to_csv(
    EXP2_DIR
    / "equilibrium_counts_vs_u_alpha.csv",
    index=False,
    float_format="%.17g"
)

display(
    equilibrium_count_df.head()
)

In [ ]:
# ============================================================
# SUMMARY BY alpha
# ============================================================

summary_records = []


for alpha in ALPHA_VALUES:

    branch_alpha = (
        adaptive_branch_df[
            adaptive_branch_df[
                "alpha"
            ] == alpha
        ]
    )

    folds_alpha = (
        adaptive_fold_df[
            adaptive_fold_df[
                "alpha"
            ] == alpha
        ]
        if len(adaptive_fold_df) > 0
        else pd.DataFrame()
    )

    counts_alpha = (
        equilibrium_count_df[
            equilibrium_count_df[
                "alpha"
            ] == alpha
        ]
    )


    fold_u_values = (
        sorted(
            folds_alpha[
                "u_fold"
            ].tolist()
        )
        if len(folds_alpha) > 0
        else []
    )


    summary_records.append({

        "alpha":
            alpha,

        "epsilon":
            PARAMS["epsilon"],

        "number_of_folds":
            len(folds_alpha),

        "fold_u_values":
            ";".join(
                f"{x:.17g}"
                for x in fold_u_values
            ),

        "minimum_fold_u":
            (
                min(fold_u_values)
                if fold_u_values
                else np.nan
            ),

        "maximum_fold_u":
            (
                max(fold_u_values)
                if fold_u_values
                else np.nan
            ),

        "fold_separation":
            (
                max(fold_u_values)
                -
                min(fold_u_values)
                if len(fold_u_values) >= 2
                else np.nan
            ),

        "maximum_number_of_equilibria":
            counts_alpha[
                "n_equilibria"
            ].max(),

        "number_of_u_grid_points_with_3_equilibria":
            (
                counts_alpha[
                    "n_equilibria"
                ] == 3
            ).sum(),

        "minimum_theta_star":
            branch_alpha[
                "theta_star"
            ].min(),

        "maximum_theta_star":
            branch_alpha[
                "theta_star"
            ].max(),

        "minimum_max_real_eigenvalue":
            branch_alpha[
                "max_real_eigenvalue"
            ].min(),

        "maximum_max_real_eigenvalue":
            branch_alpha[
                "max_real_eigenvalue"
            ].max(),

        "number_stable_branch_points":
            branch_alpha[
                "stable"
            ].sum(),

        "number_unstable_branch_points":
            branch_alpha[
                "unstable"
            ].sum()
    })


summary_by_alpha_df = pd.DataFrame(
    summary_records
)

summary_by_alpha_df.to_csv(
    EXP2_DIR
    / "experiment02_summary_by_alpha.csv",
    index=False,
    float_format="%.17g"
)

display(summary_by_alpha_df)

In [ ]:
# ============================================================
# FIGURE 1 SOURCE DATA
# ============================================================

fig01_data = adaptive_branch_df[
    (
        adaptive_branch_df["u"]
        >=
        PARAMS["u_display_min"]
    )
    &
    (
        adaptive_branch_df["u"]
        <=
        PARAMS["u_display_max"]
    )
][[
    "alpha",
    "u",
    "H_star",
    "M_star",
    "theta_star",
    "stable",
    "unstable",
    "near_neutral",
    "max_real_eigenvalue"
]].copy()


fig01_data.to_csv(
    EXP2_DIR
    / "fig01_adaptive_bifurcation_data.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# FIGURE 1
# Adaptive host equilibrium branches for selected alpha values
# ============================================================

fig, ax = plt.subplots(
    figsize=(9.0, 6.5)
)


for alpha in ALPHA_VALUES:

    df = (
        fig01_data[
            fig01_data[
                "alpha"
            ] == alpha
        ]
    )

    stable_H = np.where(
        df["stable"],
        df["H_star"],
        np.nan
    )

    unstable_H = np.where(
        ~df["stable"],
        df["H_star"],
        np.nan
    )


    stable_line, = ax.plot(
        df["u"],
        stable_H,
        linewidth=2.2,
        label=fr"$\alpha={alpha:g}$"
    )

    ax.plot(
        df["u"],
        unstable_H,
        linestyle="--",
        linewidth=1.8,
        color=stable_line.get_color()
    )


# Add fold locations
if len(adaptive_fold_df) > 0:

    visible_folds = adaptive_fold_df[
        (
            adaptive_fold_df[
                "u_fold"
            ]
            >=
            PARAMS[
                "u_display_min"
            ]
        )
        &
        (
            adaptive_fold_df[
                "u_fold"
            ]
            <=
            PARAMS[
                "u_display_max"
            ]
        )
    ]

    ax.scatter(
        visible_folds["u_fold"],
        visible_folds["H_fold"],
        s=45,
        zorder=5
    )


ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Host inflammatory equilibrium, $H^*$",
    fontsize=12
)

ax.set_title(
    "Adaptive-threshold equilibrium structure",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

ax.set_ylim(
    0,
    1
)

ax.legend(
    frameon=False,
    title="Adaptation strength"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig01_path = (
    EXP2_DIR
    / "fig01_adaptive_bifurcation.jpg"
)

fig.savefig(
    fig01_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved:\n{fig01_path}"
)

In [ ]:
# ============================================================
# FIGURE 2 SOURCE DATA
# ============================================================

fig02_data = fig01_data[[
    "alpha",
    "u",
    "H_star",
    "theta_star",
    "stable",
    "unstable",
    "near_neutral"
]].copy()

fig02_data.to_csv(
    EXP2_DIR
    / "fig02_adaptive_threshold_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 2
# theta* versus forcing
# ============================================================

fig, ax = plt.subplots(
    figsize=(9.0, 6.5)
)


for alpha in ALPHA_VALUES:

    df = fig02_data[
        fig02_data[
            "alpha"
        ] == alpha
    ]

    stable_theta = np.where(
        df["stable"],
        df["theta_star"],
        np.nan
    )

    unstable_theta = np.where(
        ~df["stable"],
        df["theta_star"],
        np.nan
    )


    stable_line, = ax.plot(
        df["u"],
        stable_theta,
        linewidth=2.2,
        label=fr"$\alpha={alpha:g}$"
    )

    ax.plot(
        df["u"],
        unstable_theta,
        linestyle="--",
        linewidth=1.8,
        color=stable_line.get_color()
    )


ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Adaptive response threshold, "
    r"$\theta^*$",
    fontsize=12
)

ax.set_title(
    "Equilibrium response-threshold adaptation",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

ax.legend(
    frameon=False,
    title="Adaptation strength"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig02_path = (
    EXP2_DIR
    / "fig02_adaptive_threshold.jpg"
)

fig.savefig(
    fig02_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved:\n{fig02_path}"
)

In [ ]:
# ============================================================
# FIGURE 3 SOURCE DATA
# ============================================================

fig03_data = fig01_data[[
    "alpha",
    "u",
    "H_star",
    "theta_star",
    "max_real_eigenvalue",
    "stable",
    "unstable"
]].copy()

fig03_data.to_csv(
    EXP2_DIR
    / "fig03_adaptive_stability_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 3
# Maximum real eigenvalue
# ============================================================

fig, ax = plt.subplots(
    figsize=(9.0, 6.5)
)


for alpha in ALPHA_VALUES:

    df = fig03_data[
        fig03_data[
            "alpha"
        ] == alpha
    ]

    ax.plot(
        df["u"],
        df[
            "max_real_eigenvalue"
        ],
        linewidth=1.8,
        label=fr"$\alpha={alpha:g}$"
    )


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.2
)

ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Maximum real part of Jacobian eigenvalues",
    fontsize=12
)

ax.set_title(
    "Local stability of adaptive-model equilibria",
    fontsize=13
)

ax.set_xlim(
    PARAMS["u_display_min"],
    PARAMS["u_display_max"]
)

ax.legend(
    frameon=False,
    title="Adaptation strength"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig03_path = (
    EXP2_DIR
    / "fig03_adaptive_stability.jpg"
)

fig.savefig(
    fig03_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved:\n{fig03_path}"
)

In [ ]:
# ============================================================
# EXPERIMENT 02 README
# ============================================================

readme = f"""
EXPERIMENT 02
ADAPTIVE-THRESHOLD EQUILIBRIUM / BIFURCATION STRUCTURE
======================================================

PURPOSE
-------
Determine how promoting the host-response threshold from
a fixed parameter theta0 to a dynamical state theta changes
the one-parameter equilibrium structure.

MODEL
-----
M1:

    dH/dt =
        -H + F(M + u; theta)

    dM/dt =
        r[-M + G(H)]

    dtheta/dt =
        epsilon * theta *
        [ln(theta0/theta) - alpha*H]

Equilibrium threshold:

    theta* = theta0 exp(-alpha H*)

Equilibrium branch:

    u(H;alpha)
    =
    theta0 exp(-alpha H)
    (H/(1-H))^(1/n)
    - G(H)

ADAPTATION STRENGTHS
--------------------
{ALPHA_VALUES.tolist()}

EPSILON USED FOR LOCAL STABILITY
--------------------------------
{EPSILON}

IMPORTANT:
epsilon does not change the equilibrium branch geometry,
but it can change dynamical stability.

QUESTIONS ADDRESSED
-------------------
1. Does adaptation move existing fold points?
2. Does the number of folds change at the selected alpha values?
3. Can adaptation generate a three-equilibrium interval in a
   parameter regime where alpha=0 does not?
4. How does theta* vary along the equilibrium branches?
5. Which branches are locally stable for the chosen timescales?

NOT YET ADDRESSED
-----------------
This experiment does NOT yet perform:

- dense two-parameter (u, alpha) continuation;
- precise estimation of a critical alpha;
- cusp localization;
- systematic r/epsilon timescale sweeps;
- same-signal/different-history experiments;
- recovery/rechallenge;
- forward/reverse hysteresis protocols.

Those belong to subsequent experiments.

MODEL-DEFINITION NOTE
---------------------
G(H) is kept identical to Experiment 01:

    G(H) = H^q / (H^q + K_M^q)

This should be replaced in BOTH experiments if the finalized
Methods specify a different microbial response function.

DATA POLICY
-----------
All numeric results are saved as CSV.

Every JPEG figure has a corresponding source-data CSV.

No Experiment 01 file is overwritten.
"""

with open(
    EXP2_DIR
    / "experiment02_README.txt",
    "w"
) as f:
    f.write(readme)

print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP EXPORT
# ============================================================

inventory_records = []


for path in sorted(
    EXP2_DIR.iterdir()
):

    if path.is_file():

        inventory_records.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                / 1024.0,

            "generated_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory_records
)

manifest_df.to_csv(
    EXP2_DIR
    / "experiment02_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_02_Adaptive_Model_"
    "Bifurcation_EXPORT"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP2_DIR)
)


print(
    "=============================================="
)

print(
    "EXPERIMENT 02 EXPORT COMPLETE"
)

print(
    "=============================================="
)

print(
    "\nExperiment 2 folder:"
)

print(
    EXP2_DIR
)

print(
    "\nZIP archive:"
)

print(
    zip_path
)

print(
    "\nFILES:\n"
)


for path in sorted(
    EXP2_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:55s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

In [ ]:
# ============================================================
# MANIFEST + ZIP EXPORT
# ============================================================

inventory_records = []


for path in sorted(
    EXP2_DIR.iterdir()
):

    if path.is_file():

        inventory_records.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                / 1024.0,

            "generated_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory_records
)

manifest_df.to_csv(
    EXP2_DIR
    / "experiment02_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_02_Adaptive_Model_"
    "Bifurcation_EXPORT"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP2_DIR)
)


print(
    "=============================================="
)

print(
    "EXPERIMENT 02 EXPORT COMPLETE"
)

print(
    "=============================================="
)

print(
    "\nExperiment 2 folder:"
)

print(
    EXP2_DIR
)

print(
    "\nZIP archive:"
)

print(
    zip_path
)

print(
    "\nFILES:\n"
)


for path in sorted(
    EXP2_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:55s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# Experiment 3: the two-parameter fold/regime map

In [ ]:
# ============================================================
# EXPERIMENT 03
# Two-parameter (u, alpha) fold / regime map
# ============================================================
#
# This experiment writes ONLY to:
#
# MyDrive/Biosemiotic_Perio/
# Experiment_03_u_alpha_Regime_Map/
#
# Experiments 1 and 2 remain untouched.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP1_DIR = (
    BASE_DIR
    / "Experiment_01_Fixed_Model_Bifurcation"
)

EXP2_DIR = (
    BASE_DIR
    / "Experiment_02_Adaptive_Model_Bifurcation"
)

EXP3_DIR = (
    BASE_DIR
    / "Experiment_03_u_alpha_Regime_Map"
)

EXP3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment 3 directory:")
print(EXP3_DIR)
print("\nExists:", EXP3_DIR.exists())

In [ ]:
# ============================================================
# IMPORTS + PARAMETER PROVENANCE
# ============================================================

import sys
import json
import shutil
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy
from scipy.optimize import root_scalar


# ------------------------------------------------------------
# Load Experiment 1 base parameters
# ------------------------------------------------------------

exp1_parameter_file = (
    EXP1_DIR / "parameters.json"
)

if not exp1_parameter_file.exists():
    raise FileNotFoundError(
        f"Missing Experiment 1 parameters:\n"
        f"{exp1_parameter_file}"
    )


with open(
    exp1_parameter_file,
    "r"
) as f:
    PARAMS_EXP1 = json.load(f)


# ------------------------------------------------------------
# Load Experiment 2 configuration
# ------------------------------------------------------------

exp2_parameter_file = (
    EXP2_DIR / "experiment02_parameters.json"
)

if not exp2_parameter_file.exists():
    raise FileNotFoundError(
        f"Missing Experiment 2 parameters:\n"
        f"{exp2_parameter_file}"
    )


with open(
    exp2_parameter_file,
    "r"
) as f:
    CONFIG_EXP2 = json.load(f)


print("Experiment 1 base parameters:")
print(json.dumps(PARAMS_EXP1, indent=4))

print("\nExperiment 2 configuration:")
print(json.dumps(CONFIG_EXP2, indent=4))

In [ ]:
# ============================================================
# EXPERIMENT 03 CONFIGURATION
# ============================================================

PARAMS = PARAMS_EXP1.copy()

PARAMS["n"] = float(PARAMS["n"])
PARAMS["q"] = float(PARAMS["q"])
PARAMS["K_M"] = float(PARAMS["K_M"])
PARAMS["theta0"] = float(PARAMS["theta0"])
PARAMS["r"] = float(PARAMS["r"])


# Keep epsilon identical to Experiment 2 for selected
# local-stability validation only.
EPSILON = float(
    CONFIG_EXP2.get(
        "epsilon",
        0.1
    )
)

PARAMS["epsilon"] = EPSILON


# ------------------------------------------------------------
# Dense alpha continuation grid
# ------------------------------------------------------------

ALPHA_MIN = 0.0
ALPHA_MAX = 5.0
N_ALPHA = 501

ALPHA_GRID = np.linspace(
    ALPHA_MIN,
    ALPHA_MAX,
    N_ALPHA
)


# ------------------------------------------------------------
# LOGIT coordinate for H
#
# x = log(H/(1-H))
# H = sigmoid(x)
#
# This fixes the boundary problem from Experiments 1 and 2.
# x = +/-30 corresponds to H extremely close to 0 or 1.
# ------------------------------------------------------------

X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 30001

X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)


# ------------------------------------------------------------
# Regime-map resolution
# ------------------------------------------------------------

N_U_MAP = 801


# ------------------------------------------------------------
# Numerical tolerances
# ------------------------------------------------------------

FOLD_XTOL = 1e-12
STABILITY_TOL = 1e-8


CONFIG3 = {
    "alpha_min": ALPHA_MIN,
    "alpha_max": ALPHA_MAX,
    "n_alpha": N_ALPHA,

    "x_min": X_MIN,
    "x_max": X_MAX,
    "n_x_scan": N_X_SCAN,

    "n_u_map": N_U_MAP,

    "fold_xtol": FOLD_XTOL,
    "stability_tol": STABILITY_TOL,

    "epsilon_for_stability_validation": EPSILON,

    "base_parameters": PARAMS_EXP1
}


with open(
    EXP3_DIR
    / "experiment03_parameters.json",
    "w"
) as f:
    json.dump(
        CONFIG3,
        f,
        indent=4
    )


config_rows = []

for key, value in CONFIG3.items():

    if key == "base_parameters":
        continue

    config_rows.append({
        "parameter": key,
        "value": value
    })


for key, value in PARAMS_EXP1.items():

    config_rows.append({
        "parameter":
            f"base_{key}",

        "value":
            value
    })


pd.DataFrame(
    config_rows
).to_csv(
    EXP3_DIR
    / "experiment03_parameters.csv",
    index=False
)


pd.DataFrame({
    "alpha": ALPHA_GRID
}).to_csv(
    EXP3_DIR
    / "alpha_grid.csv",
    index=False,
    float_format="%.17g"
)

print(json.dumps(CONFIG3, indent=4))

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def sigmoid(x):
    """
    Numerically stable enough for x in [-30, 30].
    """
    return 1.0 / (
        1.0 + np.exp(-x)
    )


def logit(H):
    return np.log(
        H / (1.0 - H)
    )


def F(S, theta, n):

    S = np.asarray(S)

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(S, theta, n):

    numerator = (
        n
        * np.power(theta, n)
        * np.power(S, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return (
        numerator
        / denominator
    )


def dF_dtheta(S, theta, n):

    numerator = (
        -n
        * np.power(S, n)
        * np.power(theta, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return (
        numerator
        / denominator
    )


# ------------------------------------------------------------
# SAME PROVISIONAL G(H) AS EXPERIMENTS 1 AND 2
# ------------------------------------------------------------

def G(H, K_M, q):

    H = np.asarray(H)

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(H, K_M, q):

    numerator = (
        q
        * np.power(K_M, q)
        * np.power(H, q - 1)
    )

    denominator = (
        np.power(H, q)
        +
        np.power(K_M, q)
    )**2

    return (
        numerator
        / denominator
    )


def d2G_dH2(H, K_M, q):
    """
    Analytical second derivative of the Hill G(H).
    """

    gp = dG_dH(
        H,
        K_M,
        q
    )

    term1 = (
        (q - 1.0)
        / H
    )

    term2 = (
        2.0
        * q
        * np.power(H, q - 1)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )

    return (
        gp
        * (
            term1
            - term2
        )
    )

In [ ]:
# ============================================================
# ADAPTIVE EQUILIBRIUM GEOMETRY
# ============================================================

def theta_star(H, alpha, p):

    return (
        p["theta0"]
        * np.exp(
            -alpha * H
        )
    )


def M_star(H, p):

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def response_factor_A(H, p):
    """
    A(H) = (H/(1-H))^(1/n)
    """

    return np.power(
        H / (1.0 - H),
        1.0 / p["n"]
    )


def u_star(H, alpha, p):
    """
    Equilibrium branch:

    u(H;alpha)
      =
      theta0 exp(-alpha H)
      (H/(1-H))^(1/n)
      - G(H)
    """

    return (
        theta_star(
            H,
            alpha,
            p
        )
        *
        response_factor_A(
            H,
            p
        )
        -
        M_star(
            H,
            p
        )
    )


def B_factor(H, p):
    """
    A'/A = 1/[n H(1-H)]
    """

    return (
        1.0
        /
        (
            p["n"]
            * H
            * (1.0 - H)
        )
    )


def dB_dH(H, p):

    return (
        (2.0 * H - 1.0)
        /
        (
            p["n"]
            * H**2
            * (1.0 - H)**2
        )
    )


def du_dH(H, alpha, p):

    A = response_factor_A(
        H,
        p
    )

    theta = theta_star(
        H,
        alpha,
        p
    )

    B = B_factor(
        H,
        p
    )

    return (
        theta
        * A
        * (
            B - alpha
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )


def d2u_dH2(H, alpha, p):
    """
    Second derivative used as a cusp / degeneracy diagnostic.
    """

    A = response_factor_A(
        H,
        p
    )

    theta = theta_star(
        H,
        alpha,
        p
    )

    B = B_factor(
        H,
        p
    )

    Bprime = dB_dH(
        H,
        p
    )

    return (
        theta
        * A
        * (
            (B - alpha)**2
            +
            Bprime
        )
        -
        d2G_dH2(
            H,
            p["K_M"],
            p["q"]
        )
    )

In [ ]:
# ============================================================
# FOLD CONTINUATION OVER alpha
# ============================================================

H_SCAN = sigmoid(
    X_SCAN
)

fold_records = []


for alpha_index, alpha in enumerate(
    ALPHA_GRID
):

    derivative_values = du_dH(
        H_SCAN,
        alpha,
        PARAMS
    )

    brackets = []


    for i in range(
        len(X_SCAN) - 1
    ):

        y1 = derivative_values[i]
        y2 = derivative_values[i + 1]

        if not (
            np.isfinite(y1)
            and
            np.isfinite(y2)
        ):
            continue

        if y1 == 0.0:

            brackets.append(
                (
                    X_SCAN[i],
                    X_SCAN[i]
                )
            )

        elif y1 * y2 < 0.0:

            brackets.append(
                (
                    X_SCAN[i],
                    X_SCAN[i + 1]
                )
            )


    fold_candidates = []


    for x_left, x_right in brackets:

        if x_left == x_right:

            x_fold = x_left

        else:

            solution = root_scalar(
                lambda x:
                    du_dH(
                        sigmoid(x),
                        alpha,
                        PARAMS
                    ),

                bracket=(
                    x_left,
                    x_right
                ),

                method="brentq",
                xtol=FOLD_XTOL
            )

            x_fold = solution.root


        H_fold = sigmoid(
            x_fold
        )

        u_fold = u_star(
            H_fold,
            alpha,
            PARAMS
        )

        M_fold = M_star(
            H_fold,
            PARAMS
        )

        theta_fold = theta_star(
            H_fold,
            alpha,
            PARAMS
        )

        second_derivative = (
            d2u_dH2(
                H_fold,
                alpha,
                PARAMS
            )
        )


        fold_candidates.append({

            "x_fold":
                x_fold,

            "H_fold":
                H_fold,

            "M_fold":
                M_fold,

            "theta_fold":
                theta_fold,

            "u_fold":
                u_fold,

            "du_dH":
                du_dH(
                    H_fold,
                    alpha,
                    PARAMS
                ),

            "d2u_dH2":
                second_derivative
        })


    # --------------------------------------------------------
    # Remove any accidental duplicate roots
    # --------------------------------------------------------

    fold_candidates = sorted(
        fold_candidates,
        key=lambda d:
            d["H_fold"]
    )

    unique_folds = []

    for candidate in fold_candidates:

        if len(unique_folds) == 0:

            unique_folds.append(
                candidate
            )

        else:

            previous = (
                unique_folds[-1]
            )

            if (
                abs(
                    candidate["H_fold"]
                    -
                    previous["H_fold"]
                )
                >
                1e-9
            ):

                unique_folds.append(
                    candidate
                )


    # --------------------------------------------------------
    # Label lower-H and higher-H folds
    # --------------------------------------------------------

    for fold_number, candidate in enumerate(
        unique_folds,
        start=1
    ):

        if len(unique_folds) == 2:

            if fold_number == 1:
                fold_type = (
                    "forward_low_state_loss"
                )

            else:
                fold_type = (
                    "reverse_high_state_loss"
                )

        else:

            fold_type = (
                f"unclassified_fold_{fold_number}"
            )


        fold_records.append({

            "alpha_index":
                alpha_index,

            "alpha":
                alpha,

            "n_folds_at_alpha":
                len(unique_folds),

            "fold_number":
                fold_number,

            "fold_type":
                fold_type,

            **candidate
        })


fold_curves_df = pd.DataFrame(
    fold_records
)

fold_curves_df.to_csv(
    EXP3_DIR
    / "fold_curves.csv",
    index=False,
    float_format="%.17g"
)

print(
    "Fold continuation complete."
)

print(
    "Total fold points:",
    len(fold_curves_df)
)

display(
    fold_curves_df.head(10)
)

In [ ]:
# ============================================================
# FOLD SUMMARY BY alpha
# ============================================================

summary_records = []


for alpha in ALPHA_GRID:

    subset = fold_curves_df[
        np.isclose(
            fold_curves_df["alpha"],
            alpha
        )
    ].sort_values(
        "H_fold"
    )


    record = {
        "alpha":
            alpha,

        "number_of_folds":
            len(subset)
    }


    if len(subset) >= 1:

        low_fold = subset.iloc[0]

        record.update({

            "H_forward_fold":
                low_fold["H_fold"],

            "M_forward_fold":
                low_fold["M_fold"],

            "theta_forward_fold":
                low_fold["theta_fold"],

            "u_forward_fold":
                low_fold["u_fold"],

            "d2u_forward_fold":
                low_fold["d2u_dH2"]
        })


    else:

        record.update({

            "H_forward_fold":
                np.nan,

            "M_forward_fold":
                np.nan,

            "theta_forward_fold":
                np.nan,

            "u_forward_fold":
                np.nan,

            "d2u_forward_fold":
                np.nan
        })


    if len(subset) >= 2:

        high_fold = subset.iloc[-1]

        record.update({

            "H_reverse_fold":
                high_fold["H_fold"],

            "M_reverse_fold":
                high_fold["M_fold"],

            "theta_reverse_fold":
                high_fold["theta_fold"],

            "u_reverse_fold":
                high_fold["u_fold"],

            "d2u_reverse_fold":
                high_fold["d2u_dH2"]
        })


    else:

        record.update({

            "H_reverse_fold":
                np.nan,

            "M_reverse_fold":
                np.nan,

            "theta_reverse_fold":
                np.nan,

            "u_reverse_fold":
                np.nan,

            "d2u_reverse_fold":
                np.nan
        })


    if len(subset) >= 2:

        u_forward = (
            record[
                "u_forward_fold"
            ]
        )

        u_reverse = (
            record[
                "u_reverse_fold"
            ]
        )

        full_width = (
            u_forward
            -
            u_reverse
        )

        positive_domain_width = max(
            0.0,
            u_forward
            -
            max(
                0.0,
                u_reverse
            )
        )

    else:

        full_width = np.nan
        positive_domain_width = np.nan


    record[
        "full_signed_fold_separation"
    ] = full_width

    record[
        "nonnegative_u_coexistence_width"
    ] = positive_domain_width


    summary_records.append(
        record
    )


fold_summary_df = pd.DataFrame(
    summary_records
)

fold_summary_df.to_csv(
    EXP3_DIR
    / "fold_summary_by_alpha.csv",
    index=False,
    float_format="%.17g"
)

display(
    fold_summary_df.head()
)

In [ ]:
# ============================================================
# CUSP / FOLD-DEGENERACY DIAGNOSTICS
# ============================================================

cusp_records = []


# ------------------------------------------------------------
# Diagnostic 1:
# change in number of folds as alpha changes
# ------------------------------------------------------------

fold_counts = (
    fold_summary_df[
        "number_of_folds"
    ].values
)


for i in range(
    len(ALPHA_GRID) - 1
):

    if (
        fold_counts[i]
        !=
        fold_counts[i + 1]
    ):

        cusp_records.append({

            "diagnostic":
                "fold_count_change",

            "alpha_left":
                ALPHA_GRID[i],

            "alpha_right":
                ALPHA_GRID[i + 1],

            "fold_count_left":
                fold_counts[i],

            "fold_count_right":
                fold_counts[i + 1],

            "possible_cusp":
                True
        })


# ------------------------------------------------------------
# Diagnostic 2:
# smallest |d2u/dH2| found on any fold
#
# At an ordinary fold:
# du/dH = 0, d2u/dH2 != 0
#
# At a cusp:
# du/dH = 0, d2u/dH2 = 0
# ------------------------------------------------------------

if len(fold_curves_df) > 0:

    min_d2_row = fold_curves_df.loc[
        fold_curves_df[
            "d2u_dH2"
        ].abs().idxmin()
    ]

    cusp_records.append({

        "diagnostic":
            "minimum_absolute_second_derivative",

        "alpha_left":
            min_d2_row["alpha"],

        "alpha_right":
            min_d2_row["alpha"],

        "fold_count_left":
            min_d2_row[
                "n_folds_at_alpha"
            ],

        "fold_count_right":
            min_d2_row[
                "n_folds_at_alpha"
            ],

        "H_fold":
            min_d2_row[
                "H_fold"
            ],

        "u_fold":
            min_d2_row[
                "u_fold"
            ],

        "d2u_dH2":
            min_d2_row[
                "d2u_dH2"
            ],

        "absolute_d2u_dH2":
            abs(
                min_d2_row[
                    "d2u_dH2"
                ]
            ),

        "possible_cusp":
            False
    })


cusp_diagnostics_df = pd.DataFrame(
    cusp_records
)

cusp_diagnostics_df.to_csv(
    EXP3_DIR
    / "cusp_diagnostics.csv",
    index=False,
    float_format="%.17g"
)

display(cusp_diagnostics_df)

In [ ]:
# ============================================================
# BUILD THE (u, alpha) REGIME MAP
# ============================================================

valid_folds = fold_summary_df[
    fold_summary_df[
        "number_of_folds"
    ] >= 2
].copy()


if len(valid_folds) == 0:

    raise RuntimeError(
        "No two-fold alpha slices were detected."
    )


minimum_reverse = (
    valid_folds[
        "u_reverse_fold"
    ].min()
)

maximum_forward = (
    valid_folds[
        "u_forward_fold"
    ].max()
)


u_span = (
    maximum_forward
    -
    minimum_reverse
)

u_margin = (
    0.10
    * u_span
)


U_MAP_MIN = (
    minimum_reverse
    -
    u_margin
)

U_MAP_MAX = (
    maximum_forward
    +
    u_margin
)


U_GRID = np.linspace(
    U_MAP_MIN,
    U_MAP_MAX,
    N_U_MAP
)


regime_records = []


for _, row in fold_summary_df.iterrows():

    alpha = row["alpha"]

    u_forward = (
        row[
            "u_forward_fold"
        ]
    )

    u_reverse = (
        row[
            "u_reverse_fold"
        ]
    )


    for u in U_GRID:

        if (
            np.isfinite(u_forward)
            and
            np.isfinite(u_reverse)
        ):

            lower = min(
                u_forward,
                u_reverse
            )

            upper = max(
                u_forward,
                u_reverse
            )

            if (
                lower
                <
                u
                <
                upper
            ):

                equilibrium_geometry = (
                    "three_equilibria"
                )

                equilibrium_code = 3

            else:

                equilibrium_geometry = (
                    "one_equilibrium"
                )

                equilibrium_code = 1

        else:

            equilibrium_geometry = (
                "undetermined"
            )

            equilibrium_code = np.nan


        regime_records.append({

            "alpha":
                alpha,

            "u":
                u,

            "equilibrium_code":
                equilibrium_code,

            "equilibrium_geometry":
                equilibrium_geometry,

            "u_is_nonnegative":
                u >= 0.0,

            "u_forward_fold":
                u_forward,

            "u_reverse_fold":
                u_reverse
        })


regime_map_df = pd.DataFrame(
    regime_records
)


regime_map_df.to_csv(
    EXP3_DIR
    / "regime_map.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Regime map rows:",
    len(regime_map_df)
)

print(
    "u range:",
    U_MAP_MIN,
    "to",
    U_MAP_MAX
)

In [ ]:
# ============================================================
# INDEPENDENT ROOT COUNT USING LOGIT H
# ============================================================

def roots_for_u_alpha(
    u_target,
    alpha,
    p,
    x_min=X_MIN,
    x_max=X_MAX,
    n_scan=N_X_SCAN
):

    x_grid = np.linspace(
        x_min,
        x_max,
        n_scan
    )

    H_grid = sigmoid(
        x_grid
    )

    values = (
        u_star(
            H_grid,
            alpha,
            p
        )
        -
        u_target
    )

    roots_x = []


    for i in range(
        len(x_grid) - 1
    ):

        f1 = values[i]
        f2 = values[i + 1]

        if not (
            np.isfinite(f1)
            and
            np.isfinite(f2)
        ):
            continue

        if f1 * f2 < 0.0:

            solution = root_scalar(

                lambda x:
                    u_star(
                        sigmoid(x),
                        alpha,
                        p
                    )
                    -
                    u_target,

                bracket=(
                    x_grid[i],
                    x_grid[i + 1]
                ),

                method="brentq",
                xtol=1e-12
            )

            roots_x.append(
                solution.root
            )


    if len(roots_x) == 0:

        return np.array([])


    roots_H = np.array([
        sigmoid(x)
        for x in roots_x
    ])


    roots_H = np.sort(
        roots_H
    )


    unique_roots = [
        roots_H[0]
    ]


    for H in roots_H[1:]:

        if (
            abs(
                H
                -
                unique_roots[-1]
            )
            >
            1e-10
        ):

            unique_roots.append(
                H
            )


    return np.array(
        unique_roots
    )

In [ ]:
# ============================================================
# ADAPTIVE JACOBIAN
# Used ONLY for selected stability checks
# ============================================================

def jacobian_M1(
    H,
    M,
    theta,
    u,
    alpha,
    p
):

    S = M + u

    FS = dF_dS(
        S,
        theta,
        p["n"]
    )

    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )

    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )

    r = p["r"]
    eps = p["epsilon"]


    adaptive_bracket = (
        np.log(
            p["theta0"]
            /
            theta
        )
        -
        alpha * H
    )


    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            r * GH,
            -r,
            0.0
        ],

        [
            -eps
            * alpha
            * theta,

            0.0,

            eps
            * (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# ROOT-COUNT + STABILITY VALIDATION
# ============================================================

VALIDATION_ALPHAS = [
    0.0,
    1.0,
    3.0,
    5.0
]


root_validation_records = []
stability_records = []


for alpha in VALIDATION_ALPHAS:

    row = fold_summary_df.iloc[
        (
            fold_summary_df[
                "alpha"
            ]
            -
            alpha
        ).abs().argsort()[:1]
    ].iloc[0]


    uf = row[
        "u_forward_fold"
    ]

    ur = row[
        "u_reverse_fold"
    ]


    test_u_values = [

        ur - 0.10,

        ur + 0.25
        * (
            uf - ur
        ),

        0.0,

        ur + 0.75
        * (
            uf - ur
        ),

        uf + 0.10
    ]


    for u_target in test_u_values:

        H_roots = roots_for_u_alpha(
            u_target,
            alpha,
            PARAMS
        )


        root_validation_records.append({

            "alpha":
                alpha,

            "u":
                u_target,

            "number_of_roots":
                len(H_roots),

            "H_roots":
                ";".join(
                    f"{x:.17g}"
                    for x in H_roots
                )
        })


        for root_number, H in enumerate(
            H_roots,
            start=1
        ):

            M = M_star(
                H,
                PARAMS
            )

            theta = theta_star(
                H,
                alpha,
                PARAMS
            )

            J = jacobian_M1(
                H,
                M,
                theta,
                u_target,
                alpha,
                PARAMS
            )

            eigvals = np.linalg.eigvals(
                J
            )

            max_real = np.max(
                eigvals.real
            )


            stability_records.append({

                "alpha":
                    alpha,

                "u":
                    u_target,

                "root_number":
                    root_number,

                "H_star":
                    H,

                "M_star":
                    M,

                "theta_star":
                    theta,

                "eig1_real":
                    eigvals[0].real,

                "eig1_imag":
                    eigvals[0].imag,

                "eig2_real":
                    eigvals[1].real,

                "eig2_imag":
                    eigvals[1].imag,

                "eig3_real":
                    eigvals[2].real,

                "eig3_imag":
                    eigvals[2].imag,

                "max_real_eigenvalue":
                    max_real,

                "stable":
                    max_real
                    <
                    -STABILITY_TOL
            })


root_count_validation_df = pd.DataFrame(
    root_validation_records
)

stability_validation_df = pd.DataFrame(
    stability_records
)


root_count_validation_df.to_csv(
    EXP3_DIR
    / "root_count_validation.csv",
    index=False,
    float_format="%.17g"
)


stability_validation_df.to_csv(
    EXP3_DIR
    / "stability_validation.csv",
    index=False,
    float_format="%.17g"
)


display(
    root_count_validation_df
)

display(
    stability_validation_df.head(20)
)

In [ ]:
# ============================================================
# FIGURE 1 SOURCE DATA
# ============================================================

fig01_data = regime_map_df.copy()

fig01_data.to_csv(
    EXP3_DIR
    / "fig01_u_alpha_regime_map_data.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# FIGURE 1
# (u, alpha) equilibrium-regime map
# ============================================================

pivot = (
    fig01_data
    .pivot(
        index="alpha",
        columns="u",
        values="equilibrium_code"
    )
)


u_values = (
    pivot.columns.values
)

alpha_values = (
    pivot.index.values
)

Z = pivot.values


fig, ax = plt.subplots(
    figsize=(9.5, 7.0)
)


mesh = ax.pcolormesh(
    u_values,
    alpha_values,
    Z,
    shading="auto"
)


# Overlay fold curves
ax.plot(
    fold_summary_df[
        "u_forward_fold"
    ],
    fold_summary_df[
        "alpha"
    ],
    linewidth=2.3,
    label="Forward fold"
)

ax.plot(
    fold_summary_df[
        "u_reverse_fold"
    ],
    fold_summary_df[
        "alpha"
    ],
    linewidth=2.3,
    label="Reverse fold"
)


# u = 0 reference
ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.2,
    label="$u=0$"
)


ax.set_xlabel(
    "External forcing, $u$",
    fontsize=12
)

ax.set_ylabel(
    "Adaptation strength, $\\alpha$",
    fontsize=12
)

ax.set_title(
    "Two-parameter equilibrium regime map",
    fontsize=13
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig01_path = (
    EXP3_DIR
    / "fig01_u_alpha_regime_map.jpg"
)

fig.savefig(
    fig01_path,
    format="jpg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved:\n{fig01_path}"
)

In [ ]:
# ============================================================
# FIGURE 2 SOURCE DATA
# ============================================================

fig02_data = fold_summary_df[[
    "alpha",
    "u_forward_fold",
    "u_reverse_fold",
    "number_of_folds"
]].copy()


fig02_data.to_csv(
    EXP3_DIR
    / "fig02_switching_thresholds_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 2
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)


ax.plot(
    fig02_data["alpha"],
    fig02_data[
        "u_forward_fold"
    ],
    linewidth=2.3,
    label="Forward switching threshold"
)


ax.plot(
    fig02_data["alpha"],
    fig02_data[
        "u_reverse_fold"
    ],
    linewidth=2.3,
    label="Reverse switching threshold"
)


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$",
    fontsize=12
)

ax.set_ylabel(
    "Fold forcing, $u$",
    fontsize=12
)

ax.set_title(
    "Adaptive reorganization of switching thresholds",
    fontsize=13
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig02_path = (
    EXP3_DIR
    / "fig02_switching_thresholds.jpg"
)

fig.savefig(
    fig02_path,
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3 SOURCE DATA
# ============================================================

fig03_data = fold_summary_df[[
    "alpha",
    "full_signed_fold_separation",
    "nonnegative_u_coexistence_width"
]].copy()


fig03_data.to_csv(
    EXP3_DIR
    / "fig03_hysteresis_widths_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 3
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)


ax.plot(
    fig03_data["alpha"],
    fig03_data[
        "full_signed_fold_separation"
    ],
    linewidth=2.3,
    label="Full signed fold separation"
)


ax.plot(
    fig03_data["alpha"],
    fig03_data[
        "nonnegative_u_coexistence_width"
    ],
    linewidth=2.3,
    label="Coexistence width for $u\\geq0$"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$",
    fontsize=12
)

ax.set_ylabel(
    "Width in forcing space",
    fontsize=12
)

ax.set_title(
    "Full versus nonnegative-forcing transition width",
    fontsize=13
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig03_path = (
    EXP3_DIR
    / "fig03_hysteresis_widths.jpg"
)

fig.savefig(
    fig03_path,
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 4 SOURCE DATA
# ============================================================

fig04_data = fold_summary_df[[
    "alpha",

    "H_forward_fold",
    "H_reverse_fold",

    "theta_forward_fold",
    "theta_reverse_fold",

    "M_forward_fold",
    "M_reverse_fold"
]].copy()


fig04_data.to_csv(
    EXP3_DIR
    / "fig04_fold_state_coordinates_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 4
# Host coordinate of forward/reverse folds
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)


ax.plot(
    fig04_data["alpha"],
    fig04_data[
        "H_forward_fold"
    ],
    linewidth=2.3,
    label="Forward fold $H^*$"
)


ax.plot(
    fig04_data["alpha"],
    fig04_data[
        "H_reverse_fold"
    ],
    linewidth=2.3,
    label="Reverse fold $H^*$"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$",
    fontsize=12
)

ax.set_ylabel(
    "Host-state coordinate at fold, $H^*$",
    fontsize=12
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "State-space displacement of the fold points",
    fontsize=13
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig04_path = (
    EXP3_DIR
    / "fig04_fold_state_coordinates.jpg"
)

fig.savefig(
    fig04_path,
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 03 SUMMARY
# ============================================================

n_alpha_with_two_folds = int(
    (
        fold_summary_df[
            "number_of_folds"
        ] == 2
    ).sum()
)


n_alpha_without_two_folds = int(
    len(
        fold_summary_df
    )
    -
    n_alpha_with_two_folds
)


alpha0_row = (
    fold_summary_df
    .iloc[
        (
            fold_summary_df[
                "alpha"
            ]
            -
            0.0
        ).abs().argsort()[:1]
    ]
    .iloc[0]
)


alpha_max_row = (
    fold_summary_df
    .iloc[
        (
            fold_summary_df[
                "alpha"
            ]
            -
            ALPHA_MAX
        ).abs().argsort()[:1]
    ]
    .iloc[0]
)


summary_rows = [

    {
        "metric":
            "alpha_min",

        "value":
            ALPHA_MIN
    },

    {
        "metric":
            "alpha_max",

        "value":
            ALPHA_MAX
    },

    {
        "metric":
            "number_alpha_slices",

        "value":
            N_ALPHA
    },

    {
        "metric":
            "alpha_slices_with_exactly_two_folds",

        "value":
            n_alpha_with_two_folds
    },

    {
        "metric":
            "alpha_slices_without_exactly_two_folds",

        "value":
            n_alpha_without_two_folds
    },

    {
        "metric":
            "forward_fold_u_at_alpha_0",

        "value":
            alpha0_row[
                "u_forward_fold"
            ]
    },

    {
        "metric":
            "reverse_fold_u_at_alpha_0",

        "value":
            alpha0_row[
                "u_reverse_fold"
            ]
    },

    {
        "metric":
            f"forward_fold_u_at_alpha_{ALPHA_MAX:g}",

        "value":
            alpha_max_row[
                "u_forward_fold"
            ]
    },

    {
        "metric":
            f"reverse_fold_u_at_alpha_{ALPHA_MAX:g}",

        "value":
            alpha_max_row[
                "u_reverse_fold"
            ]
    },

    {
        "metric":
            "minimum_absolute_d2u_dH2_at_any_fold",

        "value":
            fold_curves_df[
                "d2u_dH2"
            ].abs().min()
    },

    {
        "metric":
            "maximum_full_signed_fold_separation",

        "value":
            fold_summary_df[
                "full_signed_fold_separation"
            ].max()
    },

    {
        "metric":
            "minimum_nonnegative_u_coexistence_width",

        "value":
            fold_summary_df[
                "nonnegative_u_coexistence_width"
            ].min()
    },

    {
        "metric":
            "maximum_nonnegative_u_coexistence_width",

        "value":
            fold_summary_df[
                "nonnegative_u_coexistence_width"
            ].max()
    }
]


experiment03_summary_df = pd.DataFrame(
    summary_rows
)


experiment03_summary_df.to_csv(
    EXP3_DIR
    / "experiment03_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    experiment03_summary_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
EXPERIMENT 03
TWO-PARAMETER (u, alpha) FOLD / REGIME MAP
==========================================

PURPOSE
-------
Map how adaptive response strength alpha reorganizes the
equilibrium transition structure of the adaptive-threshold
host-microbial model.

The fixed-threshold reference model from Experiment 01
already contains two saddle-node folds.

Therefore Experiment 03 does NOT assume that adaptation
creates bistability.

Instead it asks:

1. How do the two fold curves move in the (u, alpha) plane?
2. Does the forward switching threshold decrease with alpha?
3. Does the reverse switching threshold move to increasingly
   negative forcing?
4. Does the full fold separation grow or shrink?
5. How does the answer differ if the admissible domain is
   restricted to u >= 0?
6. Do the two folds merge or disappear?
7. Is there evidence of a cusp for alpha in
   [{ALPHA_MIN}, {ALPHA_MAX}]?

BOUNDARY CORRECTION
-------------------
Experiments 01 and 02 parameterized equilibria using a
uniform H grid with finite cutoffs near H=0 and H=1.

That could miss equilibria extremely close to the boundaries.

Experiment 03 instead uses:

    x = log(H/(1-H))
    H = 1/(1 + exp(-x))

with:

    x in [{X_MIN}, {X_MAX}]

This permits reliable exploration very close to both H=0
and H=1.

PRIMARY OUTPUTS
---------------
fold_curves.csv
    Dense fold continuation over alpha.

fold_summary_by_alpha.csv
    Forward and reverse fold positions and widths.

regime_map.csv
    Full (u, alpha) classification as one versus three
    equilibrium geometry.

cusp_diagnostics.csv
    Fold-count and second-derivative diagnostics.

root_count_validation.csv
    Independent root counting using the logit coordinate.

stability_validation.csv
    Selected local-stability checks using epsilon={EPSILON}.

FIGURES
-------
fig01_u_alpha_regime_map.jpg
    Main two-parameter regime map.

fig02_switching_thresholds.jpg
    Forward and reverse fold thresholds versus alpha.

fig03_hysteresis_widths.jpg
    Full signed fold separation versus coexistence width
    when u is restricted to nonnegative values.

fig04_fold_state_coordinates.jpg
    State coordinates of the fold points.

Every JPEG has a matching source-data CSV.

IMPORTANT INTERPRETATION LIMIT
------------------------------
The two-parameter equilibrium geometry is independent of
r and epsilon.

The selected stability checks use:

    r = {PARAMS["r"]}
    epsilon = {EPSILON}

Systematic timescale exploration belongs to a later
experiment.

MODEL-DEFINITION NOTE
---------------------
The same provisional microbial Hill response is retained:

    G(H) = H^q / (H^q + K_M^q)

with:

    q = {PARAMS["q"]}
    K_M = {PARAMS["K_M"]}

This must remain identical across Experiments 01-03 unless
the underlying Methods model is formally changed.
"""


with open(
    EXP3_DIR
    / "experiment03_README.txt",
    "w"
) as f:
    f.write(readme)


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory_records = []


for path in sorted(
    EXP3_DIR.iterdir()
):

    if path.is_file():

        inventory_records.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                / 1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory_records
)


manifest_df.to_csv(
    EXP3_DIR
    / "experiment03_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    / "Experiment_03_u_alpha_Regime_Map_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP3_DIR)
)


print(
    "=============================================="
)

print(
    "EXPERIMENT 03 EXPORT COMPLETE"
)

print(
    "=============================================="
)

print(
    "\nExperiment folder:"
)

print(
    EXP3_DIR
)

print(
    "\nZIP archive:"
)

print(
    zip_path
)

print(
    "\nFILES:\n"
)


for path in sorted(
    EXP3_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:55s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# Experiment 4

In [ ]:
# ============================================================
# EXPERIMENT 04
# Same signal / different history in a monostable regime
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP1_DIR = (
    BASE_DIR
    / "Experiment_01_Fixed_Model_Bifurcation"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

EXP4_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment 4 directory:")
print(EXP4_DIR)

In [ ]:
# ============================================================
# IMPORTS + BASELINE PARAMETER PROVENANCE
# ============================================================

import json
import shutil
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar


# ------------------------------------------------------------
# Load original Experiment 1 parameter set only as a
# reference point for the monostable search.
# ------------------------------------------------------------

parameter_file = (
    EXP1_DIR / "parameters.json"
)

if not parameter_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{parameter_file}"
    )

with open(parameter_file, "r") as f:
    BASE_PARAMS = json.load(f)

print("Reference parameter set:")
print(json.dumps(BASE_PARAMS, indent=4))

In [ ]:
# ============================================================
# EXPERIMENT 04 DESIGN
# ============================================================

# Adaptive strength:
# pre-specified rather than chosen after seeing the effect.
ALPHA = 1.0

# Memory / adaptation timescale
EPSILON = 0.1

# Positive forcing interval in which the selected parameter
# set MUST remain monostable.
U_MONO_MIN = 0.02
U_MONO_MAX = 1.50

# History protocol
U_LOW = 0.05
U_HIGH = 0.40

T_BASELINE = 150.0
T_HISTORY = 200.0

# Same-S probe
T_PROBE = 2.0

# Final convergence test
U_RETURN = U_LOW
T_RETURN = 200.0

# Probe construction
PROBE_ADMISSIBILITY_MARGIN = 0.05

# Numerical settings
X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 20001

N_U_SCREEN = 61

STABILITY_TOL = 1e-7
STEADY_STATE_TOL = 1e-7

RTOL = 1e-10
ATOL = 1e-12


experiment04_design = {
    "alpha": ALPHA,
    "epsilon": EPSILON,

    "u_monostable_min": U_MONO_MIN,
    "u_monostable_max": U_MONO_MAX,

    "u_low_history": U_LOW,
    "u_high_history": U_HIGH,

    "t_baseline": T_BASELINE,
    "t_history": T_HISTORY,
    "t_probe": T_PROBE,

    "u_return": U_RETURN,
    "t_return": T_RETURN,

    "probe_admissibility_margin":
        PROBE_ADMISSIBILITY_MARGIN,

    "x_min": X_MIN,
    "x_max": X_MAX,
    "n_x_scan": N_X_SCAN,

    "n_u_screen": N_U_SCREEN,

    "stability_tol": STABILITY_TOL,
    "steady_state_tol": STEADY_STATE_TOL,

    "rtol": RTOL,
    "atol": ATOL
}


with open(
    EXP4_DIR
    / "experiment04_design.json",
    "w"
) as f:

    json.dump(
        experiment04_design,
        f,
        indent=4
    )


pd.DataFrame(
    [
        {
            "parameter": k,
            "value": v
        }
        for k, v
        in experiment04_design.items()
    ]
).to_csv(
    EXP4_DIR
    / "experiment04_design.csv",
    index=False
)

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


def F(S, theta, n):

    S = np.asarray(S)

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(S, theta, n):

    numerator = (
        n
        * np.power(theta, n)
        * np.power(S, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


def dF_dtheta(S, theta, n):

    numerator = (
        -n
        * np.power(S, n)
        * np.power(theta, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


# ------------------------------------------------------------
# SAME PROVISIONAL G(H)
# ------------------------------------------------------------

def G(H, K_M, q):

    H = np.asarray(H)

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(H, K_M, q):

    numerator = (
        q
        * np.power(K_M, q)
        * np.power(H, q - 1)
    )

    denominator = (
        np.power(H, q)
        +
        np.power(K_M, q)
    )**2

    return numerator / denominator


def rhs_M1(
    t,
    y,
    u,
    alpha,
    p
):

    H, M, theta = y

    S = M + u

    dH = (
        -H
        +
        F(
            S,
            theta,
            p["n"]
        )
    )

    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )

    dtheta = (
        p["epsilon"]
        * theta
        *
        (
            np.log(
                p["theta0"]
                / theta
            )
            -
            alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])

In [ ]:
# ============================================================
# EQUILIBRIUM + JACOBIAN
# ============================================================

def theta_star(
    H,
    alpha,
    p
):

    return (
        p["theta0"]
        *
        np.exp(
            -alpha * H
        )
    )


def M_star(H, p):

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def u_star(
    H,
    alpha,
    p
):

    theta = theta_star(
        H,
        alpha,
        p
    )

    return (
        theta
        *
        np.power(
            H / (1.0 - H),
            1.0 / p["n"]
        )
        -
        M_star(
            H,
            p
        )
    )


def du_dH(
    H,
    alpha,
    p
):

    A = np.power(
        H / (1.0 - H),
        1.0 / p["n"]
    )

    B = (
        1.0
        /
        (
            p["n"]
            * H
            * (1.0 - H)
        )
    )

    theta = theta_star(
        H,
        alpha,
        p
    )

    return (
        theta
        * A
        * (
            B - alpha
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )


def jacobian_M1(
    H,
    M,
    theta,
    u,
    alpha,
    p
):

    S = M + u

    FS = dF_dS(
        S,
        theta,
        p["n"]
    )

    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )

    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )

    r = p["r"]
    eps = p["epsilon"]

    adaptive_bracket = (
        np.log(
            p["theta0"]
            / theta
        )
        -
        alpha * H
    )

    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            r * GH,
            -r,
            0.0
        ],

        [
            -eps
            * alpha
            * theta,

            0.0,

            eps
            * (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# EQUILIBRIUM ROOT FINDING
# ============================================================

X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)

H_SCAN = sigmoid(
    X_SCAN
)


def find_equilibrium_roots(
    u_target,
    alpha,
    p
):

    values = (
        u_star(
            H_SCAN,
            alpha,
            p
        )
        -
        u_target
    )

    indices = np.where(
        values[:-1]
        *
        values[1:]
        <
        0.0
    )[0]

    roots = []


    for i in indices:

        result = root_scalar(

            lambda x:
                (
                    u_star(
                        sigmoid(x),
                        alpha,
                        p
                    )
                    -
                    u_target
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",
            xtol=1e-12
        )

        roots.append(
            sigmoid(
                result.root
            )
        )


    return np.asarray(
        roots
    )


def equilibrium_state(
    u_target,
    alpha,
    p
):

    roots = find_equilibrium_roots(
        u_target,
        alpha,
        p
    )

    if len(roots) != 1:

        raise RuntimeError(
            f"Expected exactly one equilibrium at "
            f"u={u_target}, but found {len(roots)}."
        )

    H = roots[0]

    M = M_star(
        H,
        p
    )

    theta = theta_star(
        H,
        alpha,
        p
    )

    return np.array([
        H,
        M,
        theta
    ])

In [ ]:
# ============================================================
# MONOSTABLE PARAMETER SCREEN
# ============================================================
#
# Goal:
#
# Find a parameter regime with exactly ONE equilibrium
# throughout the positive-forcing interval relevant to
# Experiment 04.
#
# Candidate ranges remain within the exploratory domains
# proposed in the project.
# ============================================================

N_CANDIDATES = [
    2.0,
    3.0,
    4.0
]

Q_CANDIDATES = [
    1.0,
    2.0,
    3.0,
    4.0
]

K_CANDIDATES = [
    0.5,
    0.7,
    0.9
]

THETA0_CANDIDATES = [
    0.6,
    0.8,
    1.0
]


U_SCREEN = np.linspace(
    U_MONO_MIN,
    U_MONO_MAX,
    N_U_SCREEN
)


BASE_N = float(
    BASE_PARAMS["n"]
)

BASE_Q = float(
    BASE_PARAMS["q"]
)

BASE_K = float(
    BASE_PARAMS["K_M"]
)

BASE_THETA0 = float(
    BASE_PARAMS["theta0"]
)

BASE_R = float(
    BASE_PARAMS["r"]
)


screen_records = []


for n in N_CANDIDATES:

    for q in Q_CANDIDATES:

        for K_M in K_CANDIDATES:

            for theta0 in THETA0_CANDIDATES:

                p = {
                    "n": n,
                    "q": q,
                    "K_M": K_M,
                    "theta0": theta0,
                    "r": BASE_R,
                    "epsilon": EPSILON
                }


                # --------------------------------------------
                # Count equilibria over the entire positive
                # forcing screen.
                # --------------------------------------------

                root_counts = []

                for u in U_SCREEN:

                    roots = (
                        find_equilibrium_roots(
                            u,
                            ALPHA,
                            p
                        )
                    )

                    root_counts.append(
                        len(roots)
                    )


                unique_root_counts = sorted(
                    set(root_counts)
                )

                geometry_pass = (
                    unique_root_counts == [1]
                )


                # --------------------------------------------
                # Detect folds whose u-coordinate lies inside
                # the actual Experiment 04 forcing interval.
                # --------------------------------------------

                derivative = du_dH(
                    H_SCAN,
                    ALPHA,
                    p
                )

                fold_indices = np.where(
                    derivative[:-1]
                    *
                    derivative[1:]
                    <
                    0.0
                )[0]

                relevant_fold_u = []


                for i in fold_indices:

                    result = root_scalar(

                        lambda x:
                            du_dH(
                                sigmoid(x),
                                ALPHA,
                                p
                            ),

                        bracket=(
                            X_SCAN[i],
                            X_SCAN[i + 1]
                        ),

                        method="brentq",
                        xtol=1e-12
                    )

                    Hf = sigmoid(
                        result.root
                    )

                    uf = u_star(
                        Hf,
                        ALPHA,
                        p
                    )

                    if (
                        U_MONO_MIN
                        <=
                        uf
                        <=
                        U_MONO_MAX
                    ):

                        relevant_fold_u.append(
                            uf
                        )


                fold_pass = (
                    len(
                        relevant_fold_u
                    )
                    ==
                    0
                )


                # --------------------------------------------
                # Local stability check
                # only if equilibrium geometry passes.
                # --------------------------------------------

                stability_pass = False
                worst_max_real_eigenvalue = np.nan
                min_stability_margin = np.nan


                if (
                    geometry_pass
                    and
                    fold_pass
                ):

                    max_real_values = []

                    stability_u_values = (
                        np.linspace(
                            U_MONO_MIN,
                            U_MONO_MAX,
                            21
                        )
                    )


                    for u in stability_u_values:

                        state = equilibrium_state(
                            u,
                            ALPHA,
                            p
                        )

                        H, M, theta = state

                        eigvals = np.linalg.eigvals(
                            jacobian_M1(
                                H,
                                M,
                                theta,
                                u,
                                ALPHA,
                                p
                            )
                        )

                        max_real_values.append(
                            np.max(
                                eigvals.real
                            )
                        )


                    worst_max_real_eigenvalue = (
                        max(
                            max_real_values
                        )
                    )

                    min_stability_margin = (
                        -worst_max_real_eigenvalue
                    )

                    stability_pass = (
                        worst_max_real_eigenvalue
                        <
                        -STABILITY_TOL
                    )


                # --------------------------------------------
                # Objective distance from Experiment 1
                # baseline.
                #
                # Used ONLY after scientific inclusion
                # criteria have been passed.
                # --------------------------------------------

                distance_from_baseline = np.sqrt(

                    (
                        (n - BASE_N)
                        / 2.0
                    )**2

                    +

                    (
                        (q - BASE_Q)
                        / 3.0
                    )**2

                    +

                    (
                        (K_M - BASE_K)
                        / 0.4
                    )**2

                    +

                    (
                        (
                            theta0
                            -
                            BASE_THETA0
                        )
                        /
                        0.4
                    )**2
                )


                screen_records.append({

                    "n":
                        n,

                    "q":
                        q,

                    "K_M":
                        K_M,

                    "theta0":
                        theta0,

                    "r":
                        BASE_R,

                    "alpha":
                        ALPHA,

                    "epsilon":
                        EPSILON,

                    "minimum_root_count":
                        min(
                            root_counts
                        ),

                    "maximum_root_count":
                        max(
                            root_counts
                        ),

                    "unique_root_counts":
                        ";".join(
                            str(x)
                            for x
                            in unique_root_counts
                        ),

                    "number_relevant_folds":
                        len(
                            relevant_fold_u
                        ),

                    "relevant_fold_u":
                        ";".join(
                            f"{x:.17g}"
                            for x
                            in relevant_fold_u
                        ),

                    "geometry_pass":
                        geometry_pass,

                    "fold_pass":
                        fold_pass,

                    "worst_max_real_eigenvalue":
                        worst_max_real_eigenvalue,

                    "minimum_stability_margin":
                        min_stability_margin,

                    "stability_pass":
                        stability_pass,

                    "monostable_pass":
                        (
                            geometry_pass
                            and
                            fold_pass
                            and
                            stability_pass
                        ),

                    "distance_from_baseline":
                        distance_from_baseline
                })


screen_df = pd.DataFrame(
    screen_records
)


screen_df.to_csv(
    EXP4_DIR
    / "monostable_parameter_screen.csv",
    index=False,
    float_format="%.17g"
)


passing_df = screen_df[
    screen_df[
        "monostable_pass"
    ]
].copy()


if len(passing_df) == 0:

    raise RuntimeError(
        "No candidate passed the pre-specified "
        "monostability screen.\n"
        "Do NOT continue by manually choosing a parameter "
        "set. Inspect monostable_parameter_screen.csv first."
    )


passing_df = passing_df.sort_values(
    by=[
        "distance_from_baseline",
        "minimum_stability_margin"
    ],
    ascending=[
        True,
        False
    ]
)


selected = passing_df.iloc[0]


print(
    "Number of passing monostable parameter sets:",
    len(passing_df)
)

print(
    "\nAutomatically selected nearest "
    "monostable parameter set:"
)

display(
    selected.to_frame().T
)

In [ ]:
# ============================================================
# SELECTED MONOSTABLE PARAMETER SET
# ============================================================

PARAMS = {

    "n":
        float(
            selected["n"]
        ),

    "q":
        float(
            selected["q"]
        ),

    "K_M":
        float(
            selected["K_M"]
        ),

    "theta0":
        float(
            selected["theta0"]
        ),

    "r":
        float(
            selected["r"]
        ),

    "epsilon":
        EPSILON
}


selected_df = pd.DataFrame([
    {
        "parameter":
            key,

        "value":
            value
    }
    for key, value
    in PARAMS.items()
])


selected_df["alpha"] = ALPHA


selected_df.to_csv(
    EXP4_DIR
    / "selected_monostable_parameters.csv",
    index=False,
    float_format="%.17g"
)


with open(
    EXP4_DIR
    / "selected_monostable_parameters.json",
    "w"
) as f:

    json.dump(
        {
            **PARAMS,
            "alpha": ALPHA
        },
        f,
        indent=4
    )


print(PARAMS)

In [ ]:
# ============================================================
# DENSE MONOSTABILITY VALIDATION
# ============================================================

U_VALIDATE = np.linspace(
    U_MONO_MIN,
    U_MONO_MAX,
    151
)


validation_records = []


for u in U_VALIDATE:

    roots = find_equilibrium_roots(
        u,
        ALPHA,
        PARAMS
    )


    for root_number, H in enumerate(
        roots,
        start=1
    ):

        M = M_star(
            H,
            PARAMS
        )

        theta = theta_star(
            H,
            ALPHA,
            PARAMS
        )

        eigvals = np.linalg.eigvals(
            jacobian_M1(
                H,
                M,
                theta,
                u,
                ALPHA,
                PARAMS
            )
        )

        max_real = np.max(
            eigvals.real
        )


        validation_records.append({

            "u":
                u,

            "number_of_equilibria":
                len(roots),

            "root_number":
                root_number,

            "H_star":
                H,

            "M_star":
                M,

            "theta_star":
                theta,

            "max_real_eigenvalue":
                max_real,

            "stable":
                max_real
                <
                -STABILITY_TOL
        })


monostability_validation_df = pd.DataFrame(
    validation_records
)


monostability_validation_df.to_csv(
    EXP4_DIR
    / "monostability_validation.csv",
    index=False,
    float_format="%.17g"
)


assert (
    monostability_validation_df[
        "number_of_equilibria"
    ].eq(1).all()
), (
    "Dense validation found multistability. "
    "Stop before running the history experiment."
)


assert (
    monostability_validation_df[
        "stable"
    ].all()
), (
    "Dense validation found an unstable equilibrium. "
    "Stop before running the history experiment."
)


print(
    "Dense monostability validation PASSED."
)

In [ ]:
# ============================================================
# COMMON BASELINE
# ============================================================

initial_state = np.array([

    0.05,

    float(
        G(
            0.05,
            PARAMS["K_M"],
            PARAMS["q"]
        )
    ),

    PARAMS["theta0"]
])


t_baseline = np.linspace(
    0.0,
    T_BASELINE,
    3001
)


sol_baseline = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_LOW,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_BASELINE
    ),

    initial_state,

    t_eval=
        t_baseline,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


baseline_state = (
    sol_baseline.y[:, -1]
)


baseline_residual = np.linalg.norm(

    rhs_M1(
        0.0,
        baseline_state,
        U_LOW,
        ALPHA,
        PARAMS
    )
)


baseline_df = pd.DataFrame({

    "time":
        sol_baseline.t,

    "H":
        sol_baseline.y[0],

    "M":
        sol_baseline.y[1],

    "theta":
        sol_baseline.y[2],

    "u":
        U_LOW
})


baseline_df.to_csv(
    EXP4_DIR
    / "common_baseline_trajectory.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Baseline endpoint:",
    baseline_state
)

print(
    "Baseline residual norm:",
    baseline_residual
)

In [ ]:
# ============================================================
# TWO HISTORIES
# ============================================================

t_history = np.linspace(
    0.0,
    T_HISTORY,
    4001
)


# ------------------------------------------------------------
# HISTORY A
# Continued low forcing
# ------------------------------------------------------------

sol_A = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_LOW,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_HISTORY
    ),

    baseline_state,

    t_eval=
        t_history,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


# ------------------------------------------------------------
# HISTORY B
# Prolonged high forcing
# ------------------------------------------------------------

sol_B = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_HIGH,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_HISTORY
    ),

    baseline_state,

    t_eval=
        t_history,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


state_A = sol_A.y[:, -1]
state_B = sol_B.y[:, -1]


H_A, M_A, theta_A = state_A
H_B, M_B, theta_B = state_B


residual_A = np.linalg.norm(
    rhs_M1(
        0.0,
        state_A,
        U_LOW,
        ALPHA,
        PARAMS
    )
)


residual_B = np.linalg.norm(
    rhs_M1(
        0.0,
        state_B,
        U_HIGH,
        ALPHA,
        PARAMS
    )
)


history_A_df = pd.DataFrame({

    "time":
        sol_A.t,

    "H":
        sol_A.y[0],

    "M":
        sol_A.y[1],

    "theta":
        sol_A.y[2],

    "u":
        U_LOW,

    "history":
        "A_low_previous_forcing"
})


history_B_df = pd.DataFrame({

    "time":
        sol_B.t,

    "H":
        sol_B.y[0],

    "M":
        sol_B.y[1],

    "theta":
        sol_B.y[2],

    "u":
        U_HIGH,

    "history":
        "B_prolonged_high_forcing"
})


history_A_df.to_csv(
    EXP4_DIR
    / "history_A_trajectory.csv",
    index=False,
    float_format="%.17g"
)


history_B_df.to_csv(
    EXP4_DIR
    / "history_B_trajectory.csv",
    index=False,
    float_format="%.17g"
)


history_endpoints_df = pd.DataFrame([

    {
        "history":
            "A_low_previous_forcing",

        "H":
            H_A,

        "M":
            M_A,

        "theta":
            theta_A,

        "previous_u":
            U_LOW,

        "equilibrium_residual_norm":
            residual_A
    },

    {
        "history":
            "B_prolonged_high_forcing",

        "H":
            H_B,

        "M":
            M_B,

        "theta":
            theta_B,

        "previous_u":
            U_HIGH,

        "equilibrium_residual_norm":
            residual_B
    }
])


history_endpoints_df.to_csv(
    EXP4_DIR
    / "history_endpoints.csv",
    index=False,
    float_format="%.17g"
)


display(
    history_endpoints_df
)


print(
    "\nTheta difference, B - A:",
    theta_B - theta_A
)

In [ ]:
# ============================================================
# CHOOSE THE SAME PHYSICAL SIGNAL S*
# ============================================================

S_mid_threshold = np.sqrt(
    theta_A
    *
    theta_B
)


S_admissible_floor = (
    max(
        M_A,
        M_B
    )
    +
    PROBE_ADMISSIBILITY_MARGIN
)


S_STAR = max(
    S_mid_threshold,
    S_admissible_floor
)


u_A_probe_initial = (
    S_STAR
    -
    M_A
)

u_B_probe_initial = (
    S_STAR
    -
    M_B
)


if (
    u_A_probe_initial
    <
    0.0
    or
    u_B_probe_initial
    <
    0.0
):

    raise RuntimeError(
        "The matched-S probe would require negative u."
    )


if (
    u_A_probe_initial
    >
    U_MONO_MAX
    or
    u_B_probe_initial
    >
    U_MONO_MAX
):

    raise RuntimeError(
        "The probe lies outside the pre-validated "
        "monostable forcing interval."
    )


probe_definition_df = pd.DataFrame([{

    "S_star":
        S_STAR,

    "S_mid_threshold":
        S_mid_threshold,

    "S_admissible_floor":
        S_admissible_floor,

    "M_A":
        M_A,

    "M_B":
        M_B,

    "theta_A":
        theta_A,

    "theta_B":
        theta_B,

    "u_A_at_probe_start":
        u_A_probe_initial,

    "u_B_at_probe_start":
        u_B_probe_initial
}])


probe_definition_df.to_csv(
    EXP4_DIR
    / "probe_definition.csv",
    index=False,
    float_format="%.17g"
)


display(probe_definition_df)

In [ ]:
# ============================================================
# SAME SIGNAL -> RESPONSE MAPPING
# ============================================================

R_A = float(
    F(
        S_STAR,
        theta_A,
        PARAMS["n"]
    )
)

R_B = float(
    F(
        S_STAR,
        theta_B,
        PARAMS["n"]
    )
)


delta_R = (
    R_B
    -
    R_A
)


# Native instantaneous H derivatives
dH_A_native = (
    -H_A
    +
    R_A
)

dH_B_native = (
    -H_B
    +
    R_B
)


# ------------------------------------------------------------
# Matched-state theta-only control
#
# Same H, same M, same S.
# ONLY theta differs.
# ------------------------------------------------------------

H_REFERENCE = H_A
M_REFERENCE = M_A


dH_A_theta_only = (
    -H_REFERENCE
    +
    R_A
)

dH_B_theta_only = (
    -H_REFERENCE
    +
    R_B
)


response_mapping_df = pd.DataFrame([{

    "S_star":
        S_STAR,

    "theta_A":
        theta_A,

    "theta_B":
        theta_B,

    "R_A_same_signal":
        R_A,

    "R_B_same_signal":
        R_B,

    "delta_R_B_minus_A":
        delta_R,

    "H_A_native":
        H_A,

    "H_B_native":
        H_B,

    "native_dH_A":
        dH_A_native,

    "native_dH_B":
        dH_B_native,

    "native_delta_dH_B_minus_A":
        dH_B_native
        -
        dH_A_native,

    "matched_H_reference":
        H_REFERENCE,

    "matched_M_reference":
        M_REFERENCE,

    "theta_only_dH_A":
        dH_A_theta_only,

    "theta_only_dH_B":
        dH_B_theta_only,

    "theta_only_delta_dH_B_minus_A":
        dH_B_theta_only
        -
        dH_A_theta_only
}])


response_mapping_df.to_csv(
    EXP4_DIR
    / "same_signal_response_mapping.csv",
    index=False,
    float_format="%.17g"
)


display(
    response_mapping_df
)

In [ ]:
# ============================================================
# SIGNAL-CLAMPED PROBE
# ============================================================

def rhs_signal_clamp(
    t,
    y,
    S_star,
    alpha,
    p
):

    H, M, theta = y

    # Dynamic forcing required to keep physical signal fixed
    u_dynamic = (
        S_star
        -
        M
    )

    # Host sees exactly the prescribed signal
    dH = (
        -H
        +
        F(
            S_star,
            theta,
            p["n"]
        )
    )

    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )

    dtheta = (
        p["epsilon"]
        * theta
        *
        (
            np.log(
                p["theta0"]
                / theta
            )
            -
            alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])


t_probe = np.linspace(
    0.0,
    T_PROBE,
    1001
)

In [ ]:
# ============================================================
# NATIVE HISTORY STATES UNDER SAME SIGNAL
# ============================================================

native_probe_A = solve_ivp(

    lambda t, y:
        rhs_signal_clamp(
            t,
            y,
            S_STAR,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_PROBE
    ),

    state_A,

    t_eval=
        t_probe,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


native_probe_B = solve_ivp(

    lambda t, y:
        rhs_signal_clamp(
            t,
            y,
            S_STAR,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_PROBE
    ),

    state_B,

    t_eval=
        t_probe,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


def probe_dataframe(
    solution,
    label
):

    H = solution.y[0]
    M = solution.y[1]
    theta = solution.y[2]

    u_dynamic = (
        S_STAR
        -
        M
    )

    R = F(
        S_STAR,
        theta,
        PARAMS["n"]
    )

    return pd.DataFrame({

        "time":
            solution.t,

        "history":
            label,

        "H":
            H,

        "M":
            M,

        "theta":
            theta,

        "S":
            S_STAR,

        "u_dynamic":
            u_dynamic,

        "response_mapping_R":
            R
    })


native_A_df = probe_dataframe(
    native_probe_A,
    "A_low_history"
)

native_B_df = probe_dataframe(
    native_probe_B,
    "B_high_history"
)


native_probe_df = pd.concat(
    [
        native_A_df,
        native_B_df
    ],
    ignore_index=True
)


native_probe_df.to_csv(
    EXP4_DIR
    / "matched_signal_probe_native_states.csv",
    index=False,
    float_format="%.17g"
)


minimum_probe_u = (
    native_probe_df[
        "u_dynamic"
    ].min()
)


if minimum_probe_u < -1e-8:

    raise RuntimeError(
        "The exact signal clamp required negative u "
        "during the probe. Inspect the saved data."
    )


print(
    "Minimum u required during native signal clamp:",
    minimum_probe_u
)

In [ ]:
# ============================================================
# THETA-ONLY MATCHED-STATE CONTROL
# ============================================================

theta_only_state_A = np.array([

    H_REFERENCE,

    M_REFERENCE,

    theta_A
])


theta_only_state_B = np.array([

    H_REFERENCE,

    M_REFERENCE,

    theta_B
])


theta_probe_A = solve_ivp(

    lambda t, y:
        rhs_signal_clamp(
            t,
            y,
            S_STAR,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_PROBE
    ),

    theta_only_state_A,

    t_eval=
        t_probe,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


theta_probe_B = solve_ivp(

    lambda t, y:
        rhs_signal_clamp(
            t,
            y,
            S_STAR,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_PROBE
    ),

    theta_only_state_B,

    t_eval=
        t_probe,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


theta_A_df = probe_dataframe(
    theta_probe_A,
    "theta_A_low_history"
)

theta_B_df = probe_dataframe(
    theta_probe_B,
    "theta_B_high_history"
)


theta_only_probe_df = pd.concat(
    [
        theta_A_df,
        theta_B_df
    ],
    ignore_index=True
)


theta_only_probe_df.to_csv(
    EXP4_DIR
    / "matched_signal_probe_theta_only_control.csv",
    index=False,
    float_format="%.17g"
)


minimum_theta_control_u = (
    theta_only_probe_df[
        "u_dynamic"
    ].min()
)


if minimum_theta_control_u < -1e-8:

    raise RuntimeError(
        "Theta-only signal clamp required negative u."
    )

In [ ]:
# ============================================================
# PROBE METRICS
# ============================================================

def trapezoid_integral(
    y,
    x
):

    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(
            y,
            x
        )

    return np.trapz(
        y,
        x
    )


def calculate_probe_metrics(
    df,
    experiment_type
):

    records = []


    for label, g in df.groupby(
        "history"
    ):

        g = g.sort_values(
            "time"
        )

        t = g["time"].values
        H = g["H"].values

        R = (
            g[
                "response_mapping_R"
            ].values
        )

        H0 = H[0]

        peak_index = np.argmax(
            H
        )

        peak_H = H[
            peak_index
        ]

        t_peak = t[
            peak_index
        ]


        records.append({

            "experiment_type":
                experiment_type,

            "history":
                label,

            "H_start":
                H0,

            "H_end":
                H[-1],

            "H_peak":
                peak_H,

            "response_amplitude":
                peak_H - H0,

            "response_latency_to_peak":
                t_peak,

            "AUC_delta_H":
                trapezoid_integral(
                    H - H0,
                    t
                ),

            "AUC_response_mapping_R":
                trapezoid_integral(
                    R,
                    t
                ),

            "R_start":
                R[0],

            "R_end":
                R[-1],

            "theta_start":
                g[
                    "theta"
                ].iloc[0],

            "theta_end":
                g[
                    "theta"
                ].iloc[-1],

            "u_min":
                g[
                    "u_dynamic"
                ].min(),

            "u_max":
                g[
                    "u_dynamic"
                ].max()
        })


    return pd.DataFrame(
        records
    )


native_metrics_df = (
    calculate_probe_metrics(
        native_probe_df,
        "native_history_states"
    )
)


theta_only_metrics_df = (
    calculate_probe_metrics(
        theta_only_probe_df,
        "theta_only_matched_state"
    )
)


probe_metrics_df = pd.concat(
    [
        native_metrics_df,
        theta_only_metrics_df
    ],
    ignore_index=True
)


probe_metrics_df.to_csv(
    EXP4_DIR
    / "probe_metrics.csv",
    index=False,
    float_format="%.17g"
)


display(
    probe_metrics_df
)

In [ ]:
# ============================================================
# SAME LONG-TERM EQUILIBRIUM TEST
# ============================================================

end_probe_A = (
    native_probe_A.y[:, -1]
)

end_probe_B = (
    native_probe_B.y[:, -1]
)


t_return = np.linspace(
    0.0,
    T_RETURN,
    4001
)


return_A = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_RETURN,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_RETURN
    ),

    end_probe_A,

    t_eval=
        t_return,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


return_B = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_RETURN,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_RETURN
    ),

    end_probe_B,

    t_eval=
        t_return,

    method=
        "LSODA",

    rtol=
        RTOL,

    atol=
        ATOL
)


expected_equilibrium = (
    equilibrium_state(
        U_RETURN,
        ALPHA,
        PARAMS
    )
)


final_A = (
    return_A.y[:, -1]
)

final_B = (
    return_B.y[:, -1]
)


A_to_expected = np.linalg.norm(
    final_A
    -
    expected_equilibrium
)


B_to_expected = np.linalg.norm(
    final_B
    -
    expected_equilibrium
)


A_to_B = np.linalg.norm(
    final_A
    -
    final_B
)


recovery_A_df = pd.DataFrame({

    "time":
        return_A.t,

    "history":
        "A",

    "H":
        return_A.y[0],

    "M":
        return_A.y[1],

    "theta":
        return_A.y[2],

    "u":
        U_RETURN
})


recovery_B_df = pd.DataFrame({

    "time":
        return_B.t,

    "history":
        "B",

    "H":
        return_B.y[0],

    "M":
        return_B.y[1],

    "theta":
        return_B.y[2],

    "u":
        U_RETURN
})


recovery_df = pd.concat(
    [
        recovery_A_df,
        recovery_B_df
    ],
    ignore_index=True
)


recovery_df.to_csv(
    EXP4_DIR
    / "same_equilibrium_recovery_trajectories.csv",
    index=False,
    float_format="%.17g"
)


recovery_summary_df = pd.DataFrame([{

    "expected_H":
        expected_equilibrium[0],

    "expected_M":
        expected_equilibrium[1],

    "expected_theta":
        expected_equilibrium[2],

    "final_H_A":
        final_A[0],

    "final_M_A":
        final_A[1],

    "final_theta_A":
        final_A[2],

    "final_H_B":
        final_B[0],

    "final_M_B":
        final_B[1],

    "final_theta_B":
        final_B[2],

    "distance_A_to_expected":
        A_to_expected,

    "distance_B_to_expected":
        B_to_expected,

    "distance_A_to_B":
        A_to_B
}])


recovery_summary_df.to_csv(
    EXP4_DIR
    / "same_equilibrium_recovery_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    recovery_summary_df
)

In [ ]:
# ============================================================
# FIGURE 1 SOURCE DATA
# ============================================================

fig01_data = pd.concat(
    [
        history_A_df,
        history_B_df
    ],
    ignore_index=True
)


fig01_data.to_csv(
    EXP4_DIR
    / "fig01_history_threshold_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 1
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


for label, g in fig01_data.groupby(
    "history"
):

    ax.plot(
        g["time"],
        g["theta"],
        linewidth=2.2,
        label=label
    )


ax.set_xlabel(
    "History duration"
)

ax.set_ylabel(
    "Adaptive response threshold, $\\theta$"
)

ax.set_title(
    "Different histories generate different response states"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP4_DIR
    / "fig01_history_threshold.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2 SOURCE DATA
# ============================================================

S_MAX_FIGURE = max(
    1.2 * S_STAR,
    1.2 * PARAMS["theta0"]
)


S_GRID = np.linspace(
    0.0,
    S_MAX_FIGURE,
    1000
)


fig02_data = pd.DataFrame({

    "S":
        S_GRID,

    "R_history_A":
        F(
            S_GRID,
            theta_A,
            PARAMS["n"]
        ),

    "R_history_B":
        F(
            S_GRID,
            theta_B,
            PARAMS["n"]
        )
})


fig02_data.to_csv(
    EXP4_DIR
    / "fig02_same_signal_response_mapping_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 2
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig02_data["S"],
    fig02_data[
        "R_history_A"
    ],
    linewidth=2.2,
    label="History A"
)


ax.plot(
    fig02_data["S"],
    fig02_data[
        "R_history_B"
    ],
    linewidth=2.2,
    label="History B"
)


ax.axvline(
    S_STAR,
    linestyle="--",
    linewidth=1.2,
    label="$S^*$"
)


ax.scatter(
    [S_STAR, S_STAR],
    [R_A, R_B],
    s=55,
    zorder=5
)


ax.set_xlabel(
    "Physical signal, $S$"
)

ax.set_ylabel(
    "Host response mapping, $F(S,\\theta)$"
)

ax.set_title(
    "Same signal, different response after different histories"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP4_DIR
    / "fig02_same_signal_response_mapping.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3 SOURCE DATA
# ============================================================

fig03_data = (
    theta_only_probe_df.copy()
)


fig03_data.to_csv(
    EXP4_DIR
    / "fig03_theta_only_probe_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 3
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


for label, g in fig03_data.groupby(
    "history"
):

    ax.plot(
        g["time"],
        g["H"],
        linewidth=2.2,
        label=label
    )


ax.set_xlabel(
    "Time after matched-signal probe"
)

ax.set_ylabel(
    "Host response state, $H$"
)

ax.set_title(
    "Theta-only causal control under identical signal"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP4_DIR
    / "fig03_theta_only_probe.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 4 SOURCE DATA
# ============================================================

fig04_data = (
    native_probe_df.copy()
)


fig04_data.to_csv(
    EXP4_DIR
    / "fig04_native_same_signal_probe_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 4
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


for label, g in fig04_data.groupby(
    "history"
):

    ax.plot(
        g["time"],
        g["H"],
        linewidth=2.2,
        label=label
    )


ax.set_xlabel(
    "Time after matched-signal probe"
)

ax.set_ylabel(
    "Host state, $H$"
)

ax.set_title(
    "Natural history-dependent responses under the same signal"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP4_DIR
    / "fig04_native_same_signal_probe.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 5 SOURCE DATA
# ============================================================

fig05_data = (
    recovery_df.copy()
)


fig05_data.to_csv(
    EXP4_DIR
    / "fig05_same_equilibrium_recovery_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 5
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


for label, g in fig05_data.groupby(
    "history"
):

    ax.plot(
        g["time"],
        g["H"],
        linewidth=2.2,
        label=f"History {label}"
    )


ax.axhline(
    expected_equilibrium[0],
    linestyle="--",
    linewidth=1.2,
    label="Unique equilibrium"
)


ax.set_xlabel(
    "Time after return to common forcing"
)

ax.set_ylabel(
    "Host state, $H$"
)

ax.set_title(
    "Different histories converge to the same long-term state"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP4_DIR
    / "fig05_same_equilibrium_recovery.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 04 SUMMARY
# ============================================================

theta_difference = (
    theta_B
    -
    theta_A
)


relative_theta_change = (
    (
        theta_B
        -
        theta_A
    )
    /
    theta_A
)


theta_only_metrics_A = (
    theta_only_metrics_df[
        theta_only_metrics_df[
            "history"
        ]
        ==
        "theta_A_low_history"
    ].iloc[0]
)


theta_only_metrics_B = (
    theta_only_metrics_df[
        theta_only_metrics_df[
            "history"
        ]
        ==
        "theta_B_high_history"
    ].iloc[0]
)


summary_rows = [

    {
        "metric":
            "selected_n",

        "value":
            PARAMS["n"]
    },

    {
        "metric":
            "selected_q",

        "value":
            PARAMS["q"]
    },

    {
        "metric":
            "selected_K_M",

        "value":
            PARAMS["K_M"]
    },

    {
        "metric":
            "selected_theta0",

        "value":
            PARAMS["theta0"]
    },

    {
        "metric":
            "alpha",

        "value":
            ALPHA
    },

    {
        "metric":
            "epsilon",

        "value":
            EPSILON
    },

    {
        "metric":
            "number_of_monostable_parameter_sets_found",

        "value":
            len(passing_df)
    },

    {
        "metric":
            "theta_A_after_low_history",

        "value":
            theta_A
    },

    {
        "metric":
            "theta_B_after_high_history",

        "value":
            theta_B
    },

    {
        "metric":
            "theta_difference_B_minus_A",

        "value":
            theta_difference
    },

    {
        "metric":
            "relative_theta_difference",

        "value":
            relative_theta_change
    },

    {
        "metric":
            "S_star",

        "value":
            S_STAR
    },

    {
        "metric":
            "R_A_same_signal",

        "value":
            R_A
    },

    {
        "metric":
            "R_B_same_signal",

        "value":
            R_B
    },

    {
        "metric":
            "delta_R_B_minus_A",

        "value":
            delta_R
    },

    {
        "metric":
            "theta_only_immediate_delta_dH_B_minus_A",

        "value":
            (
                dH_B_theta_only
                -
                dH_A_theta_only
            )
    },

    {
        "metric":
            "theta_only_peak_H_difference_B_minus_A",

        "value":
            (
                theta_only_metrics_B[
                    "H_peak"
                ]
                -
                theta_only_metrics_A[
                    "H_peak"
                ]
            )
    },

    {
        "metric":
            "theta_only_AUC_delta_H_difference_B_minus_A",

        "value":
            (
                theta_only_metrics_B[
                    "AUC_delta_H"
                ]
                -
                theta_only_metrics_A[
                    "AUC_delta_H"
                ]
            )
    },

    {
        "metric":
            "final_distance_between_histories",

        "value":
            A_to_B
    },

    {
        "metric":
            "history_A_distance_to_unique_equilibrium",

        "value":
            A_to_expected
    },

    {
        "metric":
            "history_B_distance_to_unique_equilibrium",

        "value":
            B_to_expected
    }
]


experiment04_summary_df = pd.DataFrame(
    summary_rows
)


experiment04_summary_df.to_csv(
    EXP4_DIR
    / "experiment04_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    experiment04_summary_df
)

In [ ]:
# ============================================================
# DECISION FLAGS
# ============================================================

monostable_confirmed = bool(
    monostability_validation_df[
        "number_of_equilibria"
    ].eq(1).all()

    and

    monostability_validation_df[
        "stable"
    ].all()
)


history_thresholds_differ = bool(
    abs(
        theta_A
        -
        theta_B
    )
    >
    1e-6
)


same_signal_response_differs = bool(
    abs(
        delta_R
    )
    >
    1e-6
)


same_final_equilibrium = bool(
    A_to_B
    <
    1e-5
)


decision_df = pd.DataFrame([{

    "monostable_regime_confirmed":
        monostable_confirmed,

    "history_dependent_theta_confirmed":
        history_thresholds_differ,

    "same_signal_response_mapping_differs":
        same_signal_response_differs,

    "both_histories_converge_to_same_equilibrium":
        same_final_equilibrium,

    "all_required_conditions":
        (
            monostable_confirmed
            and
            history_thresholds_differ
            and
            same_signal_response_differs
            and
            same_final_equilibrium
        )
}])


decision_df.to_csv(
    EXP4_DIR
    / "experiment04_decision_flags.csv",
    index=False
)


display(decision_df)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
EXPERIMENT 04
SAME SIGNAL / DIFFERENT HISTORY IN A MONOSTABLE REGIME
=======================================================

PRIMARY QUESTION
----------------
Can two systems exposed to exactly the same present physical
signal produce different host response mappings because their
previous histories have left different adaptive thresholds?

MODEL
-----
Adaptive model M1.

alpha   = {ALPHA}
epsilon = {EPSILON}

MONOSTABILITY REQUIREMENT
-------------------------
The Experiment 1-3 baseline was deliberately NOT reused as
the primary Experiment 4 regime because that parameter set is
bistable.

A pre-specified parameter grid was screened.

A candidate was accepted only if:

1. exactly one equilibrium existed throughout
   u in [{U_MONO_MIN}, {U_MONO_MAX}];

2. no fold occurred in that interval;

3. the unique equilibrium was locally stable throughout
   the sampled interval.

Among all passing candidates, the parameter set closest to
the Experiment 1 baseline was selected automatically.

SELECTED PARAMETERS
-------------------
n      = {PARAMS["n"]}
q      = {PARAMS["q"]}
K_M    = {PARAMS["K_M"]}
theta0 = {PARAMS["theta0"]}
r      = {PARAMS["r"]}
alpha  = {ALPHA}
epsilon= {EPSILON}

HISTORY PROTOCOL
----------------
Both systems first shared a common low-forcing baseline.

History A:
continued low forcing u = {U_LOW}

History B:
prolonged higher forcing u = {U_HIGH}

Both were then exposed to the same physical signal:

S* = {S_STAR}

Because S = M + u, equal u would not necessarily mean equal
signal.

Therefore the probe was defined in signal space rather than
forcing space.

PRIMARY RESPONSE-MAPPING TEST
-----------------------------
The direct comparison is:

R_A = F(S*, theta_A)
R_B = F(S*, theta_B)

This isolates the adaptive response mapping.

NATIVE-STATE PROBE
------------------
The actual history endpoints were exposed to the same
signal-clamped probe.

These trajectories contain the full consequences of history:
H, M and theta may all initially differ.

THETA-ONLY CONTROL
------------------
A stricter causal control was also performed.

Both systems were initialized with the same H and M, while
retaining the two history-derived values theta_A and theta_B.

Thus the only initial difference was theta.

Both were then exposed to the same clamped physical signal.

LONG-TERM CONVERGENCE
---------------------
After the probe both systems were returned to the same
forcing u = {U_RETURN}.

Because the selected regime is monostable, they should
eventually converge to the same equilibrium.

INTERPRETATION
--------------
If:

1. the parameter regime is monostable,
2. theta_A != theta_B,
3. F(S*,theta_A) != F(S*,theta_B), and
4. both systems later converge to the same equilibrium,

then the model demonstrates transient history-dependent
response without requiring alternative stable states.

MODEL-DEFINITION NOTE
---------------------
The microbial response remains the same provisional function
used in Experiments 1-3:

G(H) = H^q / (H^q + K_M^q)

Any future formal change to G(H) requires rerunning all four
experiments.

DATA POLICY
-----------
All numerical outputs are saved as CSV.

Every figure has a matching source-data CSV.

All Experiment 04 files are isolated from Experiments 1-3.
"""


with open(
    EXP4_DIR
    / "experiment04_README.txt",
    "w"
) as f:

    f.write(readme)


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    EXP4_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                / 1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    EXP4_DIR
    / "experiment04_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_04_Same_Signal_"
    "Different_History_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP4_DIR)
)


print(
    "=========================================="
)

print(
    "EXPERIMENT 04 EXPORT COMPLETE"
)

print(
    "=========================================="
)

print(
    "\nFolder:"
)

print(
    EXP4_DIR
)

print(
    "\nZIP:"
)

print(
    zip_path
)

print(
    "\nFILES:\n"
)


for path in sorted(
    EXP4_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:60s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# Experiment 5: recovery–rechallenge

In [ ]:
 # ============================================================
# EXPERIMENT 05
# Recovery–rechallenge memory experiment
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

EXP5_DIR = (
    BASE_DIR
    / "Experiment_05_Recovery_Rechallenge_Memory"
)

EXP5_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment 5 directory:")
print(EXP5_DIR)

In [ ]:
# ============================================================
# IMPORTS + LOAD EXPERIMENT 04 PARAMETERS
# ============================================================

import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar


# ------------------------------------------------------------
# Load the exact monostable parameter set selected in Exp 4
# ------------------------------------------------------------

param_file = (
    EXP4_DIR
    / "selected_monostable_parameters.json"
)

design_file = (
    EXP4_DIR
    / "experiment04_design.json"
)

probe_file = (
    EXP4_DIR
    / "probe_definition.csv"
)


if not param_file.exists():
    raise FileNotFoundError(
        f"Missing:\n{param_file}"
    )

if not design_file.exists():
    raise FileNotFoundError(
        f"Missing:\n{design_file}"
    )

if not probe_file.exists():
    raise FileNotFoundError(
        f"Missing:\n{probe_file}"
    )


with open(param_file, "r") as f:
    exp4_params = json.load(f)

with open(design_file, "r") as f:
    exp4_design = json.load(f)

probe_exp4_df = pd.read_csv(
    probe_file
)


PARAMS = {
    "n": float(exp4_params["n"]),
    "q": float(exp4_params["q"]),
    "K_M": float(exp4_params["K_M"]),
    "theta0": float(exp4_params["theta0"]),
    "r": float(exp4_params["r"]),
    "epsilon": float(exp4_params["epsilon"])
}

ALPHA = float(
    exp4_params["alpha"]
)

U_LOW = float(
    exp4_design["u_low_history"]
)

U_COND = float(
    exp4_design["u_high_history"]
)

S_STAR = float(
    probe_exp4_df[
        "S_star"
    ].iloc[0]
)


print("Experiment 4 parameters:")
print(PARAMS)

print("\nalpha =", ALPHA)
print("baseline/recovery u =", U_LOW)
print("conditioning u =", U_COND)
print("rechallenge S* =", S_STAR)

In [ ]:
# ============================================================
# EXPERIMENT 05 PROTOCOL
# ============================================================

# Same baseline and conditioning durations as Experiment 4
T_BASELINE = 150.0
T_CONDITIONING = 200.0

# Short standardized rechallenge
T_PROBE = 2.0


# ------------------------------------------------------------
# Recovery intervals
#
# Includes T=0 for immediate rechallenge, then increasingly
# long intervals.
# ------------------------------------------------------------

RECOVERY_TIMES = np.array([
    0.0,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0,
    40.0,
    80.0,
    160.0
])


RTOL = 1e-10
ATOL = 1e-12


CONFIG5 = {
    "alpha": ALPHA,
    "u_baseline": U_LOW,
    "u_conditioning": U_COND,
    "S_star": S_STAR,

    "t_baseline": T_BASELINE,
    "t_conditioning": T_CONDITIONING,
    "t_probe": T_PROBE,

    "recovery_times":
        RECOVERY_TIMES.tolist(),

    "rtol": RTOL,
    "atol": ATOL,

    "parameters": PARAMS
}


with open(
    EXP5_DIR
    / "experiment05_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG5,
        f,
        indent=4
    )


config_rows = []

for key, value in CONFIG5.items():

    if key == "parameters":
        continue

    if isinstance(
        value,
        list
    ):
        value = ";".join(
            str(x)
            for x in value
        )

    config_rows.append({
        "parameter": key,
        "value": value
    })


for key, value in PARAMS.items():

    config_rows.append({
        "parameter":
            f"model_{key}",

        "value":
            value
    })


pd.DataFrame(
    config_rows
).to_csv(
    EXP5_DIR
    / "experiment05_parameters.csv",
    index=False
)


pd.DataFrame({
    "recovery_time":
        RECOVERY_TIMES
}).to_csv(
    EXP5_DIR
    / "recovery_times.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def F(S, theta, n):

    S = np.asarray(S)

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def G(H, K_M, q):

    H = np.asarray(H)

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def rhs_M1(
    t,
    y,
    u,
    alpha,
    p
):

    H, M, theta = y

    S = M + u

    dH = (
        -H
        +
        F(
            S,
            theta,
            p["n"]
        )
    )

    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )

    dtheta = (
        p["epsilon"]
        * theta
        *
        (
            np.log(
                p["theta0"]
                / theta
            )
            -
            alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])


def rhs_signal_clamp(
    t,
    y,
    S_star,
    alpha,
    p
):

    H, M, theta = y

    # Required external forcing to maintain exact physical S
    u_dynamic = (
        S_star
        -
        M
    )

    dH = (
        -H
        +
        F(
            S_star,
            theta,
            p["n"]
        )
    )

    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )

    dtheta = (
        p["epsilon"]
        * theta
        *
        (
            np.log(
                p["theta0"]
                / theta
            )
            -
            alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])

In [ ]:
# ============================================================
# COMMON BASELINE
# ============================================================

initial_state = np.array([

    0.05,

    float(
        G(
            0.05,
            PARAMS["K_M"],
            PARAMS["q"]
        )
    ),

    PARAMS["theta0"]
])


t_baseline = np.linspace(
    0.0,
    T_BASELINE,
    3001
)


baseline_solution = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_LOW,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_BASELINE
    ),

    initial_state,

    t_eval=t_baseline,

    method="LSODA",

    rtol=RTOL,
    atol=ATOL
)


baseline_state = (
    baseline_solution.y[:, -1]
)


baseline_residual = np.linalg.norm(

    rhs_M1(
        0.0,
        baseline_state,
        U_LOW,
        ALPHA,
        PARAMS
    )
)


baseline_df = pd.DataFrame({

    "time":
        baseline_solution.t,

    "H":
        baseline_solution.y[0],

    "M":
        baseline_solution.y[1],

    "theta":
        baseline_solution.y[2],

    "u":
        U_LOW
})


baseline_df.to_csv(
    EXP5_DIR
    / "baseline_trajectory.csv",
    index=False,
    float_format="%.17g"
)


print("Baseline state:")
print(baseline_state)

print(
    "Baseline residual:",
    baseline_residual
)

In [ ]:
# ============================================================
# CONDITIONING PHASE
# ============================================================

t_conditioning = np.linspace(
    0.0,
    T_CONDITIONING,
    4001
)


conditioning_solution = solve_ivp(

    lambda t, y:
        rhs_M1(
            t,
            y,
            U_COND,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_CONDITIONING
    ),

    baseline_state,

    t_eval=t_conditioning,

    method="LSODA",

    rtol=RTOL,
    atol=ATOL
)


conditioned_state = (
    conditioning_solution.y[:, -1]
)


conditioning_residual = np.linalg.norm(

    rhs_M1(
        0.0,
        conditioned_state,
        U_COND,
        ALPHA,
        PARAMS
    )
)


conditioning_df = pd.DataFrame({

    "time":
        conditioning_solution.t,

    "H":
        conditioning_solution.y[0],

    "M":
        conditioning_solution.y[1],

    "theta":
        conditioning_solution.y[2],

    "u":
        U_COND
})


conditioning_df.to_csv(
    EXP5_DIR
    / "conditioning_trajectory.csv",
    index=False,
    float_format="%.17g"
)


print("Conditioned state:")
print(conditioned_state)

print(
    "Conditioning residual:",
    conditioning_residual
)

In [ ]:
# ============================================================
# RECOVERY TRAJECTORIES
# ============================================================

recovery_state_records = []
recovery_trajectory_frames = []


for T_recovery in RECOVERY_TIMES:

    # --------------------------------------------------------
    # T = 0 means immediate rechallenge:
    # no recovery integration is necessary.
    # --------------------------------------------------------

    if T_recovery == 0.0:

        recovered_state = (
            conditioned_state.copy()
        )

        temp_df = pd.DataFrame({

            "recovery_time":
                [0.0],

            "time_within_recovery":
                [0.0],

            "H":
                [conditioned_state[0]],

            "M":
                [conditioned_state[1]],

            "theta":
                [conditioned_state[2]],

            "u":
                [U_LOW]
        })


    else:

        n_points = max(
            101,
            int(
                20
                *
                T_recovery
            )
            +
            1
        )


        t_eval = np.linspace(
            0.0,
            T_recovery,
            n_points
        )


        solution = solve_ivp(

            lambda t, y:
                rhs_M1(
                    t,
                    y,
                    U_LOW,
                    ALPHA,
                    PARAMS
                ),

            (
                0.0,
                T_recovery
            ),

            conditioned_state,

            t_eval=t_eval,

            method="LSODA",

            rtol=RTOL,
            atol=ATOL
        )


        recovered_state = (
            solution.y[:, -1]
        )


        temp_df = pd.DataFrame({

            "recovery_time":
                T_recovery,

            "time_within_recovery":
                solution.t,

            "H":
                solution.y[0],

            "M":
                solution.y[1],

            "theta":
                solution.y[2],

            "u":
                U_LOW
        })


    recovery_trajectory_frames.append(
        temp_df
    )


    H_rec, M_rec, theta_rec = (
        recovered_state
    )


    R_rec = float(
        F(
            S_STAR,
            theta_rec,
            PARAMS["n"]
        )
    )


    recovery_state_records.append({

        "recovery_time":
            T_recovery,

        "H_before_probe":
            H_rec,

        "M_before_probe":
            M_rec,

        "theta_before_probe":
            theta_rec,

        "R_at_same_signal":
            R_rec,

        "u_required_at_probe_start":
            S_STAR
            -
            M_rec,

        "distance_from_baseline_state":
            np.linalg.norm(
                recovered_state
                -
                baseline_state
            )
    })


all_recovery_trajectories_df = (
    pd.concat(
        recovery_trajectory_frames,
        ignore_index=True
    )
)


recovery_states_df = pd.DataFrame(
    recovery_state_records
)


all_recovery_trajectories_df.to_csv(
    EXP5_DIR
    / "all_recovery_trajectories.csv",
    index=False,
    float_format="%.17g"
)


recovery_states_df.to_csv(
    EXP5_DIR
    / "recovery_states_before_probe.csv",
    index=False,
    float_format="%.17g"
)


display(
    recovery_states_df
)

In [ ]:
# ============================================================
# UNCONDITIONED REFERENCE PROBE
# ============================================================

t_probe = np.linspace(
    0.0,
    T_PROBE,
    1001
)


reference_probe = solve_ivp(

    lambda t, y:
        rhs_signal_clamp(
            t,
            y,
            S_STAR,
            ALPHA,
            PARAMS
        ),

    (
        0.0,
        T_PROBE
    ),

    baseline_state,

    t_eval=t_probe,

    method="LSODA",

    rtol=RTOL,
    atol=ATOL
)


reference_probe_df = pd.DataFrame({

    "time":
        reference_probe.t,

    "H":
        reference_probe.y[0],

    "M":
        reference_probe.y[1],

    "theta":
        reference_probe.y[2],

    "S":
        S_STAR,

    "u_dynamic":
        S_STAR
        -
        reference_probe.y[1],

    "response_mapping_R":
        F(
            S_STAR,
            reference_probe.y[2],
            PARAMS["n"]
        )
})


reference_probe_df.to_csv(
    EXP5_DIR
    / "reference_unconditioned_probe.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# RECOVERY–RECHALLENGE SERIES
# ============================================================

probe_frames = []
probe_summary_records = []


for _, recovery_row in (
    recovery_states_df.iterrows()
):

    T_recovery = float(
        recovery_row[
            "recovery_time"
        ]
    )


    recovered_state = np.array([

        recovery_row[
            "H_before_probe"
        ],

        recovery_row[
            "M_before_probe"
        ],

        recovery_row[
            "theta_before_probe"
        ]
    ])


    solution = solve_ivp(

        lambda t, y:
            rhs_signal_clamp(
                t,
                y,
                S_STAR,
                ALPHA,
                PARAMS
            ),

        (
            0.0,
            T_PROBE
        ),

        recovered_state,

        t_eval=t_probe,

        method="LSODA",

        rtol=RTOL,
        atol=ATOL
    )


    H = solution.y[0]
    M = solution.y[1]
    theta = solution.y[2]

    R = F(
        S_STAR,
        theta,
        PARAMS["n"]
    )

    u_dynamic = (
        S_STAR
        -
        M
    )


    frame = pd.DataFrame({

        "recovery_time":
            T_recovery,

        "probe_time":
            solution.t,

        "H":
            H,

        "M":
            M,

        "theta":
            theta,

        "S":
            S_STAR,

        "u_dynamic":
            u_dynamic,

        "response_mapping_R":
            R
    })


    probe_frames.append(
        frame
    )


    H0 = H[0]

    peak_index = np.argmax(
        H
    )


    probe_summary_records.append({

        "recovery_time":
            T_recovery,

        "H_start":
            H0,

        "H_end":
            H[-1],

        "H_peak":
            H[peak_index],

        "delta_H_max":
            H[peak_index]
            -
            H0,

        "time_to_H_peak":
            solution.t[
                peak_index
            ],

        "theta_start":
            theta[0],

        "theta_end":
            theta[-1],

        "R_start":
            R[0],

        "R_end":
            R[-1],

        "u_probe_min":
            np.min(
                u_dynamic
            ),

        "u_probe_max":
            np.max(
                u_dynamic
            )
    })


all_probe_trajectories_df = (
    pd.concat(
        probe_frames,
        ignore_index=True
    )
)


probe_summary_df = pd.DataFrame(
    probe_summary_records
)


all_probe_trajectories_df.to_csv(
    EXP5_DIR
    / "all_rechallenge_trajectories.csv",
    index=False,
    float_format="%.17g"
)


probe_summary_df.to_csv(
    EXP5_DIR
    / "rechallenge_summary_raw.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# RESPONSE METRICS
# ============================================================

def integrate_trapezoid(
    y,
    x
):

    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(
            y,
            x
        )

    return np.trapz(
        y,
        x
    )


# ------------------------------------------------------------
# Reference metrics
# ------------------------------------------------------------

ref_t = (
    reference_probe_df[
        "time"
    ].values
)

ref_H = (
    reference_probe_df[
        "H"
    ].values
)

ref_R = (
    reference_probe_df[
        "response_mapping_R"
    ].values
)


REF_H0 = ref_H[0]

REF_DELTA_H_MAX = (
    np.max(
        ref_H
    )
    -
    REF_H0
)

REF_AUC_DELTA_H = (
    integrate_trapezoid(
        ref_H
        -
        REF_H0,
        ref_t
    )
)

REF_R_START = (
    ref_R[0]
)

REF_AUC_R = (
    integrate_trapezoid(
        ref_R,
        ref_t
    )
)


metric_records = []


for T_recovery, g in (
    all_probe_trajectories_df
    .groupby(
        "recovery_time"
    )
):

    g = g.sort_values(
        "probe_time"
    )

    t = (
        g[
            "probe_time"
        ].values
    )

    H = (
        g["H"].values
    )

    R = (
        g[
            "response_mapping_R"
        ].values
    )

    theta = (
        g[
            "theta"
        ].values
    )


    H0 = H[0]

    delta_H_max = (
        np.max(H)
        -
        H0
    )

    auc_delta_H = (
        integrate_trapezoid(
            H - H0,
            t
        )
    )

    auc_R = (
        integrate_trapezoid(
            R,
            t
        )
    )


    metric_records.append({

        "recovery_time":
            T_recovery,

        "theta_before_probe":
            theta[0],

        "theta_deficit_vs_baseline":
            (
                baseline_state[2]
                -
                theta[0]
            ),

        "R_same_signal_before_probe":
            R[0],

        "R_excess_vs_unconditioned":
            (
                R[0]
                -
                REF_R_START
            ),

        "delta_H_max":
            delta_H_max,

        "delta_H_max_excess_vs_reference":
            (
                delta_H_max
                -
                REF_DELTA_H_MAX
            ),

        "AUC_delta_H":
            auc_delta_H,

        "AUC_delta_H_excess_vs_reference":
            (
                auc_delta_H
                -
                REF_AUC_DELTA_H
            ),

        "AUC_response_mapping_R":
            auc_R,

        "AUC_R_excess_vs_reference":
            (
                auc_R
                -
                REF_AUC_R
            )
    })


memory_metrics_df = pd.DataFrame(
    metric_records
)


memory_metrics_df.to_csv(
    EXP5_DIR
    / "memory_metrics_vs_recovery_time.csv",
    index=False,
    float_format="%.17g"
)


display(
    memory_metrics_df
)

In [ ]:
# ============================================================
# THETA-ONLY MEMORY CONTROL
# ============================================================

H_REFERENCE = (
    baseline_state[0]
)

M_REFERENCE = (
    baseline_state[1]
)


theta_only_records = []


for _, row in (
    recovery_states_df.iterrows()
):

    T_recovery = float(
        row[
            "recovery_time"
        ]
    )

    theta_T = float(
        row[
            "theta_before_probe"
        ]
    )


    R_theta_T = float(
        F(
            S_STAR,
            theta_T,
            PARAMS["n"]
        )
    )


    dH_theta_T = (
        -H_REFERENCE
        +
        R_theta_T
    )


    R_reference = float(
        F(
            S_STAR,
            baseline_state[2],
            PARAMS["n"]
        )
    )


    dH_reference = (
        -H_REFERENCE
        +
        R_reference
    )


    theta_only_records.append({

        "recovery_time":
            T_recovery,

        "theta_history":
            theta_T,

        "theta_reference":
            baseline_state[2],

        "R_history_same_S":
            R_theta_T,

        "R_reference_same_S":
            R_reference,

        "delta_R_history_minus_reference":
            (
                R_theta_T
                -
                R_reference
            ),

        "matched_state_dH_history":
            dH_theta_T,

        "matched_state_dH_reference":
            dH_reference,

        "matched_state_delta_dH":
            (
                dH_theta_T
                -
                dH_reference
            )
    })


theta_only_memory_df = pd.DataFrame(
    theta_only_records
)


theta_only_memory_df.to_csv(
    EXP5_DIR
    / "theta_only_memory_control.csv",
    index=False,
    float_format="%.17g"
)


display(
    theta_only_memory_df
)

In [ ]:
# ============================================================
# APPARENT MEMORY HALF-LIFE
# MODEL-FREE INTERPOLATION
# ============================================================

def apparent_half_life(
    times,
    values
):
    """
    Find first time at which a positive memory measure falls
    to <= 50% of its T=0 value.

    Linear interpolation is used between sampled times.

    No exponential recovery model is assumed.
    """

    times = np.asarray(
        times,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )


    initial = values[0]

    if (
        not np.isfinite(initial)
        or
        initial <= 0.0
    ):

        return np.nan


    target = (
        0.5
        *
        initial
    )


    for i in range(
        1,
        len(times)
    ):

        y_prev = values[i - 1]
        y_now = values[i]

        if (
            y_prev > target
            and
            y_now <= target
        ):

            t_prev = times[i - 1]
            t_now = times[i]

            fraction = (
                target
                -
                y_prev
            ) / (
                y_now
                -
                y_prev
            )

            return (
                t_prev
                +
                fraction
                *
                (
                    t_now
                    -
                    t_prev
                )
            )


    return np.nan


T_HALF_R = apparent_half_life(

    memory_metrics_df[
        "recovery_time"
    ],

    memory_metrics_df[
        "R_excess_vs_unconditioned"
    ]
)


T_HALF_THETA = apparent_half_life(

    memory_metrics_df[
        "recovery_time"
    ],

    memory_metrics_df[
        "theta_deficit_vs_baseline"
    ]
)


T_HALF_AUC = apparent_half_life(

    memory_metrics_df[
        "recovery_time"
    ],

    memory_metrics_df[
        "AUC_delta_H_excess_vs_reference"
    ]
)


half_life_df = pd.DataFrame([

    {
        "memory_measure":
            "same_signal_response_mapping_excess",

        "apparent_half_life":
            T_HALF_R
    },

    {
        "memory_measure":
            "theta_deficit",

        "apparent_half_life":
            T_HALF_THETA
    },

    {
        "memory_measure":
            "AUC_delta_H_excess",

        "apparent_half_life":
            T_HALF_AUC
    }
])


half_life_df.to_csv(
    EXP5_DIR
    / "apparent_memory_half_lives.csv",
    index=False,
    float_format="%.17g"
)


display(
    half_life_df
)

In [ ]:
# ============================================================
# MONOTONICITY DIAGNOSTICS
# ============================================================

def nonincreasing(
    x,
    tol=1e-10
):

    x = np.asarray(
        x,
        dtype=float
    )

    return bool(
        np.all(
            np.diff(x)
            <=
            tol
        )
    )


monotonicity_df = pd.DataFrame([

    {
        "measure":
            "theta_deficit_vs_baseline",

        "monotonic_nonincreasing":
            nonincreasing(
                memory_metrics_df[
                    "theta_deficit_vs_baseline"
                ]
            )
    },

    {
        "measure":
            "R_excess_vs_unconditioned",

        "monotonic_nonincreasing":
            nonincreasing(
                memory_metrics_df[
                    "R_excess_vs_unconditioned"
                ]
            )
    },

    {
        "measure":
            "delta_H_max_excess_vs_reference",

        "monotonic_nonincreasing":
            nonincreasing(
                memory_metrics_df[
                    "delta_H_max_excess_vs_reference"
                ]
            )
    },

    {
        "measure":
            "AUC_delta_H_excess_vs_reference",

        "monotonic_nonincreasing":
            nonincreasing(
                memory_metrics_df[
                    "AUC_delta_H_excess_vs_reference"
                ]
            )
    }
])


monotonicity_df.to_csv(
    EXP5_DIR
    / "memory_decay_monotonicity.csv",
    index=False
)


display(
    monotonicity_df
)

In [ ]:
# ============================================================
# FIGURE 1 SOURCE DATA
# ============================================================

fig01_data = memory_metrics_df[[
    "recovery_time",
    "theta_before_probe",
    "theta_deficit_vs_baseline"
]].copy()


fig01_data.to_csv(
    EXP5_DIR
    / "fig01_theta_recovery_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 1
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig01_data[
        "recovery_time"
    ],
    fig01_data[
        "theta_before_probe"
    ],
    marker="o",
    linewidth=2.2
)


ax.axhline(
    baseline_state[2],
    linestyle="--",
    linewidth=1.2,
    label="Recovered baseline threshold"
)


if np.isfinite(
    T_HALF_THETA
):

    ax.axvline(
        T_HALF_THETA,
        linestyle=":",
        linewidth=1.2,
        label="Apparent half-recovery"
    )


ax.set_xlabel(
    "Recovery interval, $T_{recovery}$"
)

ax.set_ylabel(
    "Threshold before rechallenge, $\\theta(T)$"
)

ax.set_title(
    "Recovery of the adaptive response threshold"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig01_theta_recovery.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2 SOURCE DATA
# ============================================================

fig02_data = memory_metrics_df[[
    "recovery_time",
    "R_same_signal_before_probe",
    "R_excess_vs_unconditioned"
]].copy()


fig02_data.to_csv(
    EXP5_DIR
    / "fig02_same_signal_memory_decay_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 2
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig02_data[
        "recovery_time"
    ],
    fig02_data[
        "R_excess_vs_unconditioned"
    ],
    marker="o",
    linewidth=2.2
)


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1
)


if np.isfinite(
    T_HALF_R
):

    ax.axvline(
        T_HALF_R,
        linestyle=":",
        linewidth=1.2,
        label="Apparent response-memory half-life"
    )


ax.set_xlabel(
    "Recovery interval, $T_{recovery}$"
)

ax.set_ylabel(
    "Excess response to identical signal, "
    "$\\Delta R(S^*)$"
)

ax.set_title(
    "Decay of history-dependent response to the same signal"
)


if np.isfinite(
    T_HALF_R
):

    ax.legend(
        frameon=False
    )


ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig02_same_signal_memory_decay.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3 SOURCE DATA
# ============================================================

fig03_data = memory_metrics_df[[
    "recovery_time",
    "delta_H_max",
    "delta_H_max_excess_vs_reference"
]].copy()


fig03_data.to_csv(
    EXP5_DIR
    / "fig03_peak_response_memory_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 3
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig03_data[
        "recovery_time"
    ],
    fig03_data[
        "delta_H_max_excess_vs_reference"
    ],
    marker="o",
    linewidth=2.2
)


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1
)


ax.set_xlabel(
    "Recovery interval, $T_{recovery}$"
)

ax.set_ylabel(
    "Excess peak response, "
    "$\\Delta H_{max}$"
)

ax.set_title(
    "Recovery-time dependence of rechallenge amplitude"
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig03_peak_response_memory.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 4 SOURCE DATA
# ============================================================

fig04_data = memory_metrics_df[[
    "recovery_time",
    "AUC_delta_H",
    "AUC_delta_H_excess_vs_reference"
]].copy()


fig04_data.to_csv(
    EXP5_DIR
    / "fig04_auc_memory_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 4
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig04_data[
        "recovery_time"
    ],
    fig04_data[
        "AUC_delta_H_excess_vs_reference"
    ],
    marker="o",
    linewidth=2.2
)


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1
)


if np.isfinite(
    T_HALF_AUC
):

    ax.axvline(
        T_HALF_AUC,
        linestyle=":",
        linewidth=1.2,
        label="Apparent AUC half-life"
    )


ax.set_xlabel(
    "Recovery interval, $T_{recovery}$"
)

ax.set_ylabel(
    "Excess $H$-response AUC"
)

ax.set_title(
    "Decay of integrated rechallenge response"
)


if np.isfinite(
    T_HALF_AUC
):

    ax.legend(
        frameon=False
    )


ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig04_auc_memory.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 5 SOURCE DATA
# ============================================================

DISPLAY_RECOVERY_TIMES = [
    0.0,
    2.0,
    10.0,
    40.0,
    160.0
]


fig05_data = (
    all_probe_trajectories_df[
        all_probe_trajectories_df[
            "recovery_time"
        ].isin(
            DISPLAY_RECOVERY_TIMES
        )
    ]
    .copy()
)


fig05_data.to_csv(
    EXP5_DIR
    / "fig05_selected_rechallenge_trajectories_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 5
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


for T_recovery, g in (
    fig05_data.groupby(
        "recovery_time"
    )
):

    ax.plot(
        g[
            "probe_time"
        ],
        g["H"],
        linewidth=2.0,
        label=f"T = {T_recovery:g}"
    )


ax.set_xlabel(
    "Time after identical rechallenge"
)

ax.set_ylabel(
    "Host response state, $H$"
)

ax.set_title(
    "Rechallenge response after progressively longer recovery"
)

ax.legend(
    frameon=False,
    title="Recovery interval"
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig05_selected_rechallenge_trajectories.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 6 SOURCE DATA
# ============================================================

fig06_data = theta_only_memory_df.copy()


fig06_data.to_csv(
    EXP5_DIR
    / "fig06_theta_only_memory_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 6
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.3)
)


ax.plot(
    fig06_data[
        "recovery_time"
    ],
    fig06_data[
        "matched_state_delta_dH"
    ],
    marker="o",
    linewidth=2.2
)


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1
)


ax.set_xlabel(
    "Recovery interval, $T_{recovery}$"
)

ax.set_ylabel(
    "Theta-only excess immediate response"
)

ax.set_title(
    "Decay of adaptive-state memory at matched H, M and S"
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP5_DIR
    / "fig06_theta_only_memory.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 05 SUMMARY
# ============================================================

immediate_row = (
    memory_metrics_df
    .sort_values(
        "recovery_time"
    )
    .iloc[0]
)


longest_row = (
    memory_metrics_df
    .sort_values(
        "recovery_time"
    )
    .iloc[-1]
)


summary_rows = [

    {
        "metric":
            "alpha",

        "value":
            ALPHA
    },

    {
        "metric":
            "epsilon",

        "value":
            PARAMS["epsilon"]
    },

    {
        "metric":
            "S_star",

        "value":
            S_STAR
    },

    {
        "metric":
            "conditioning_u",

        "value":
            U_COND
    },

    {
        "metric":
            "recovery_u",

        "value":
            U_LOW
    },

    {
        "metric":
            "theta_immediately_after_conditioning",

        "value":
            immediate_row[
                "theta_before_probe"
            ]
    },

    {
        "metric":
            "same_signal_response_excess_at_T0",

        "value":
            immediate_row[
                "R_excess_vs_unconditioned"
            ]
    },

    {
        "metric":
            "same_signal_response_excess_at_longest_recovery",

        "value":
            longest_row[
                "R_excess_vs_unconditioned"
            ]
    },

    {
        "metric":
            "peak_response_excess_at_T0",

        "value":
            immediate_row[
                "delta_H_max_excess_vs_reference"
            ]
    },

    {
        "metric":
            "AUC_excess_at_T0",

        "value":
            immediate_row[
                "AUC_delta_H_excess_vs_reference"
            ]
    },

    {
        "metric":
            "apparent_R_memory_half_life",

        "value":
            T_HALF_R
    },

    {
        "metric":
            "apparent_theta_deficit_half_life",

        "value":
            T_HALF_THETA
    },

    {
        "metric":
            "apparent_AUC_memory_half_life",

        "value":
            T_HALF_AUC
    }
]


experiment05_summary_df = pd.DataFrame(
    summary_rows
)


experiment05_summary_df.to_csv(
    EXP5_DIR
    / "experiment05_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    experiment05_summary_df
)

In [ ]:
# ============================================================
# EXPERIMENT 05 DECISION FLAGS
# ============================================================

initial_memory_present = bool(

    immediate_row[
        "R_excess_vs_unconditioned"
    ]
    >
    1e-6
)


memory_declines = bool(

    longest_row[
        "R_excess_vs_unconditioned"
    ]
    <
    immediate_row[
        "R_excess_vs_unconditioned"
    ]
)


substantial_recovery = bool(

    abs(
        longest_row[
            "R_excess_vs_unconditioned"
        ]
    )
    <
    0.10
    *
    abs(
        immediate_row[
            "R_excess_vs_unconditioned"
        ]
    )
)


half_life_resolved = bool(
    np.isfinite(
        T_HALF_R
    )
)


decision_flags_df = pd.DataFrame([{

    "initial_memory_present":
        initial_memory_present,

    "same_signal_memory_declines_with_recovery":
        memory_declines,

    "less_than_10_percent_memory_remaining_at_longest_recovery":
        substantial_recovery,

    "apparent_response_memory_half_life_resolved":
        half_life_resolved
}])


decision_flags_df.to_csv(
    EXP5_DIR
    / "experiment05_decision_flags.csv",
    index=False
)


display(
    decision_flags_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
EXPERIMENT 05
RECOVERY–RECHALLENGE MEMORY
============================

PURPOSE
-------
Quantify how long the altered signal-response mapping produced
by prior conditioning persists after the conditioning stimulus
is removed.

MODEL
-----
Uses exactly the monostable parameter regime selected in
Experiment 04.

n       = {PARAMS["n"]}
q       = {PARAMS["q"]}
K_M     = {PARAMS["K_M"]}
theta0  = {PARAMS["theta0"]}
r       = {PARAMS["r"]}
alpha   = {ALPHA}
epsilon = {PARAMS["epsilon"]}

PROTOCOL
--------
baseline
    ->
conditioning
    ->
variable recovery interval
    ->
identical same-S rechallenge

Baseline/recovery forcing:
    u = {U_LOW}

Conditioning forcing:
    u = {U_COND}

Physical rechallenge signal:
    S* = {S_STAR}

Recovery intervals:
    {RECOVERY_TIMES.tolist()}

IMPORTANT
---------
The rechallenge is identical in PHYSICAL SIGNAL space.

Because:
    S = M + u

the required external forcing is dynamically adjusted during
the probe so that every rechallenge satisfies:

    S(t) = S*

This avoids confusing microbial-state differences with signal
differences.

PRIMARY MEMORY VARIABLES
------------------------
1. theta(T_recovery)

2. response mapping:
       F(S*, theta(T_recovery))

3. excess same-signal response relative to the fully recovered
   control

4. peak H-response amplitude

5. H-response AUC

6. theta-only matched-state response

APPARENT MEMORY HALF-LIFE
-------------------------
A model-free apparent half-life is calculated by interpolation
as the recovery time at which the excess memory measure falls
to 50% of its immediate post-conditioning value.

No exponential decay model is assumed.

This is important because H, M and theta remain dynamically
coupled, so the complete system need not recover as a single
exponential.

INTERPRETATION
--------------
If the same-signal response remains elevated after the
conditioning input has been removed, then the model exhibits
transient memory.

If the excess response progressively returns to zero as the
recovery interval increases, the model demonstrates reversible
history dependence with a measurable persistence timescale.

The phenomenological memory timescale should not be interpreted
as a biological periodontal time constant until the model has
been calibrated against experimental data.

DATA POLICY
-----------
All numerical results are saved as CSV.

Every figure has a matching source-data CSV.

All Experiment 05 outputs are isolated from Experiments 01-04.
"""


with open(
    EXP5_DIR
    / "experiment05_README.txt",
    "w"
) as f:

    f.write(
        readme
    )


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP EXPORT
# ============================================================

inventory = []


for path in sorted(
    EXP5_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                / 1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    EXP5_DIR
    / "experiment05_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_05_Recovery_Rechallenge_Memory_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP5_DIR)
)


print(
    "============================================"
)

print(
    "EXPERIMENT 05 EXPORT COMPLETE"
)

print(
    "============================================"
)

print("\nFolder:")
print(EXP5_DIR)

print("\nZIP:")
print(zip_path)

print("\nFILES:\n")


for path in sorted(
    EXP5_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:60s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# Experiment 6:dynamic lag versus equilibrium hysteresis

In [ ]:
# ============================================================
# EXPERIMENT 06
# Dynamic lag versus equilibrium hysteresis
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP1_DIR = (
    BASE_DIR
    / "Experiment_01_Fixed_Model_Bifurcation"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

EXP6_DIR = (
    BASE_DIR
    / "Experiment_06_Dynamic_Lag_vs_Equilibrium_Hysteresis"
)

EXP6_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment 6 directory:")
print(EXP6_DIR)

In [ ]:
# ============================================================
# IMPORTS + PARAMETER PROVENANCE
# ============================================================

import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar


# ------------------------------------------------------------
# Experiment 4 monostable parameter set
# ------------------------------------------------------------

exp4_parameter_file = (
    EXP4_DIR
    / "selected_monostable_parameters.json"
)

if not exp4_parameter_file.exists():
    raise FileNotFoundError(
        f"Missing:\n{exp4_parameter_file}"
    )

with open(
    exp4_parameter_file,
    "r"
) as f:
    EXP4_PARAMS = json.load(f)


MONO_PARAMS = {
    "n": float(EXP4_PARAMS["n"]),
    "q": float(EXP4_PARAMS["q"]),
    "K_M": float(EXP4_PARAMS["K_M"]),
    "theta0": float(EXP4_PARAMS["theta0"]),
    "r": float(EXP4_PARAMS["r"]),
    "epsilon": float(EXP4_PARAMS["epsilon"])
}

ALPHA = float(
    EXP4_PARAMS["alpha"]
)


# ------------------------------------------------------------
# Experiment 1 parameters used only as the reference point
# for ranking candidate bistable parameter sets.
# ------------------------------------------------------------

exp1_parameter_file = (
    EXP1_DIR
    / "parameters.json"
)

with open(
    exp1_parameter_file,
    "r"
) as f:
    EXP1_PARAMS = json.load(f)


print("Monostable parameters:")
print(MONO_PARAMS)

print("\nalpha =", ALPHA)

print("\nExperiment 1 reference parameters:")
print(EXP1_PARAMS)

In [ ]:
# ============================================================
# EXPERIMENT 06 DESIGN
# ============================================================

U_MIN = 0.05
U_MAX = 0.80

DELTA_U = (
    U_MAX
    -
    U_MIN
)


# ------------------------------------------------------------
# Duration of ONE leg:
# U_MIN -> U_MAX
#
# Full triangular cycle duration = 2*T_ramp
# ------------------------------------------------------------

RAMP_DURATIONS = np.array([
    10.0,
    25.0,
    50.0,
    100.0,
    200.0,
    400.0,
    800.0
])


RAMP_SPEEDS = (
    DELTA_U
    /
    RAMP_DURATIONS
)


# ------------------------------------------------------------
# Operational crossing threshold.
#
# This is a secondary metric.
#
# Primary switch detection will use maximum |dH/du|.
# ------------------------------------------------------------

H_CROSS = 0.50


# ------------------------------------------------------------
# Numerical resolution
# ------------------------------------------------------------

N_RAMP_POINTS = 8001

X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 30001

FOLD_XTOL = 1e-12

RTOL = 1e-10
ATOL = 1e-12

STABILITY_TOL = 1e-8

# Ignore a small fraction of each ramp edge when locating
# maximum |dH/du|.
EDGE_TRIM_FRACTION = 0.01


CONFIG6 = {
    "alpha": ALPHA,

    "u_min": U_MIN,
    "u_max": U_MAX,

    "H_cross": H_CROSS,

    "ramp_durations":
        RAMP_DURATIONS.tolist(),

    "ramp_speeds":
        RAMP_SPEEDS.tolist(),

    "n_ramp_points":
        N_RAMP_POINTS,

    "x_min": X_MIN,
    "x_max": X_MAX,
    "n_x_scan": N_X_SCAN,

    "fold_xtol":
        FOLD_XTOL,

    "edge_trim_fraction":
        EDGE_TRIM_FRACTION,

    "rtol": RTOL,
    "atol": ATOL
}


with open(
    EXP6_DIR
    / "experiment06_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG6,
        f,
        indent=4
    )


config_rows = []

for key, value in CONFIG6.items():

    if isinstance(value, list):
        value = ";".join(
            str(x)
            for x in value
        )

    config_rows.append({
        "parameter": key,
        "value": value
    })


for key, value in MONO_PARAMS.items():

    config_rows.append({
        "parameter":
            f"monostable_{key}",

        "value":
            value
    })


pd.DataFrame(
    config_rows
).to_csv(
    EXP6_DIR
    / "experiment06_parameters.csv",
    index=False
)

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


def F(S, theta, n):

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(S, theta, n):

    numerator = (
        n
        *
        np.power(theta, n)
        *
        np.power(S, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


def dF_dtheta(S, theta, n):

    numerator = (
        -n
        *
        np.power(S, n)
        *
        np.power(theta, n - 1)
    )

    denominator = (
        np.power(S, n)
        +
        np.power(theta, n)
    )**2

    return numerator / denominator


# ------------------------------------------------------------
# Same provisional G(H) as Experiments 1-5
# ------------------------------------------------------------

def G(H, K_M, q):

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(H, K_M, q):

    numerator = (
        q
        *
        np.power(K_M, q)
        *
        np.power(H, q - 1)
    )

    denominator = (
        np.power(H, q)
        +
        np.power(K_M, q)
    )**2

    return numerator / denominator


def theta_star(
    H,
    alpha,
    p
):

    return (
        p["theta0"]
        *
        np.exp(
            -alpha * H
        )
    )


def M_star(
    H,
    p
):

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def u_star(
    H,
    alpha,
    p
):

    theta = theta_star(
        H,
        alpha,
        p
    )

    return (
        theta
        *
        np.power(
            H / (1.0 - H),
            1.0 / p["n"]
        )
        -
        M_star(
            H,
            p
        )
    )


def du_dH(
    H,
    alpha,
    p
):

    A = np.power(
        H / (1.0 - H),
        1.0 / p["n"]
    )

    B = (
        1.0
        /
        (
            p["n"]
            *
            H
            *
            (1.0 - H)
        )
    )

    theta = theta_star(
        H,
        alpha,
        p
    )

    return (
        theta
        *
        A
        *
        (
            B - alpha
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )

In [ ]:
# ============================================================
# ADAPTIVE MODEL
# ============================================================

def rhs_M1_time_dependent_u(
    t,
    y,
    u_function,
    alpha,
    p
):

    H, M, theta = y

    u = float(
        u_function(t)
    )

    S = (
        M + u
    )

    dH = (
        -H
        +
        F(
            S,
            theta,
            p["n"]
        )
    )

    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )

    dtheta = (
        p["epsilon"]
        *
        theta
        *
        (
            np.log(
                p["theta0"]
                /
                theta
            )
            -
            alpha * H
        )
    )

    return np.array([
        dH,
        dM,
        dtheta
    ])


def jacobian_M1(
    H,
    M,
    theta,
    u,
    alpha,
    p
):

    S = (
        M + u
    )

    FS = dF_dS(
        S,
        theta,
        p["n"]
    )

    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )

    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )

    r = p["r"]
    eps = p["epsilon"]

    adaptive_bracket = (
        np.log(
            p["theta0"]
            /
            theta
        )
        -
        alpha * H
    )


    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            r * GH,
            -r,
            0.0
        ],

        [
            -eps
            *
            alpha
            *
            theta,

            0.0,

            eps
            *
            (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# EQUILIBRIUM ROOT FINDER
# ============================================================

X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)

H_SCAN = sigmoid(
    X_SCAN
)


def equilibrium_roots(
    u_target,
    alpha,
    p
):

    values = (
        u_star(
            H_SCAN,
            alpha,
            p
        )
        -
        u_target
    )

    sign_change_indices = np.where(
        values[:-1]
        *
        values[1:]
        <
        0.0
    )[0]


    roots = []


    for i in sign_change_indices:

        result = root_scalar(

            lambda x:
                (
                    u_star(
                        sigmoid(x),
                        alpha,
                        p
                    )
                    -
                    u_target
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )

        roots.append(
            sigmoid(
                result.root
            )
        )


    return np.asarray(
        roots
    )


def state_from_H(
    H,
    alpha,
    p
):

    return np.array([

        H,

        M_star(
            H,
            p
        ),

        theta_star(
            H,
            alpha,
            p
        )
    ])

In [ ]:
# ============================================================
# FOLD DETECTOR
# ============================================================

def detect_folds(
    alpha,
    p
):

    derivative = du_dH(
        H_SCAN,
        alpha,
        p
    )


    indices = np.where(
        derivative[:-1]
        *
        derivative[1:]
        <
        0.0
    )[0]


    folds = []


    for i in indices:

        result = root_scalar(

            lambda x:
                du_dH(
                    sigmoid(x),
                    alpha,
                    p
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",
            xtol=FOLD_XTOL
        )


        Hf = sigmoid(
            result.root
        )


        folds.append({

            "H_fold":
                Hf,

            "M_fold":
                M_star(
                    Hf,
                    p
                ),

            "theta_fold":
                theta_star(
                    Hf,
                    alpha,
                    p
                ),

            "u_fold":
                u_star(
                    Hf,
                    alpha,
                    p
                )
        })


    return sorted(
        folds,
        key=lambda d:
            d["H_fold"]
    )

In [ ]:
# ============================================================
# MONOSTABLE CONTROL VALIDATION
# ============================================================

U_VALIDATION = np.linspace(
    U_MIN,
    U_MAX,
    201
)


mono_validation_records = []


for u in U_VALIDATION:

    roots = equilibrium_roots(
        u,
        ALPHA,
        MONO_PARAMS
    )

    mono_validation_records.append({

        "u":
            u,

        "number_of_equilibria":
            len(roots),

        "H_roots":
            ";".join(
                f"{x:.17g}"
                for x in roots
            )
    })


mono_validation_df = pd.DataFrame(
    mono_validation_records
)


mono_validation_df.to_csv(
    EXP6_DIR
    / "monostable_ramp_range_validation.csv",
    index=False,
    float_format="%.17g"
)


if not (
    mono_validation_df[
        "number_of_equilibria"
    ].eq(1).all()
):

    raise RuntimeError(
        "Experiment 4 regime is not monostable "
        "throughout the Experiment 6 ramp interval."
    )


print(
    "Monostable control validation PASSED."
)

In [ ]:
# ============================================================
# POSITIVE-u BISTABLE COMPARATOR SCREEN
# ============================================================
#
# NO hand-tuning after seeing results.
#
# alpha, r and epsilon are kept the same as Experiment 4.
# ============================================================

N_CANDIDATES = [
    2.0,
    3.0,
    4.0,
    5.0,
    6.0
]

Q_CANDIDATES = [
    1.0,
    2.0,
    3.0,
    4.0,
    5.0,
    6.0
]

K_CANDIDATES = [
    0.30,
    0.50,
    0.70,
    0.90
]

THETA0_CANDIDATES = [
    0.40,
    0.60,
    0.80,
    1.00
]


REFERENCE_N = float(
    EXP1_PARAMS["n"]
)

REFERENCE_Q = float(
    EXP1_PARAMS["q"]
)

REFERENCE_K = float(
    EXP1_PARAMS["K_M"]
)

REFERENCE_THETA0 = float(
    EXP1_PARAMS["theta0"]
)


screen_records = []


for n in N_CANDIDATES:

    for q in Q_CANDIDATES:

        for K_M in K_CANDIDATES:

            for theta0 in THETA0_CANDIDATES:

                p = {

                    "n": n,
                    "q": q,

                    "K_M":
                        K_M,

                    "theta0":
                        theta0,

                    "r":
                        MONO_PARAMS[
                            "r"
                        ],

                    "epsilon":
                        MONO_PARAMS[
                            "epsilon"
                        ]
                }


                folds = detect_folds(
                    ALPHA,
                    p
                )


                positive_domain_pass = False

                u_reverse = np.nan
                u_forward = np.nan

                H_reverse = np.nan
                H_forward = np.nan

                three_root_pass = False
                stability_pattern_pass = False

                root_count_at_min = np.nan
                root_count_at_max = np.nan


                if len(folds) == 2:

                    # lower-H fold = forward low-state loss
                    low_H_fold = folds[0]

                    # higher-H fold = reverse high-state loss
                    high_H_fold = folds[1]


                    u_forward = (
                        low_H_fold[
                            "u_fold"
                        ]
                    )

                    u_reverse = (
                        high_H_fold[
                            "u_fold"
                        ]
                    )

                    H_forward = (
                        low_H_fold[
                            "H_fold"
                        ]
                    )

                    H_reverse = (
                        high_H_fold[
                            "H_fold"
                        ]
                    )


                    positive_domain_pass = bool(

                        U_MIN
                        <
                        u_reverse
                        <
                        u_forward
                        <
                        U_MAX
                    )


                    if positive_domain_pass:

                        roots_min = (
                            equilibrium_roots(
                                U_MIN,
                                ALPHA,
                                p
                            )
                        )

                        roots_max = (
                            equilibrium_roots(
                                U_MAX,
                                ALPHA,
                                p
                            )
                        )

                        root_count_at_min = (
                            len(
                                roots_min
                            )
                        )

                        root_count_at_max = (
                            len(
                                roots_max
                            )
                        )


                        u_middle = (
                            0.5
                            *
                            (
                                u_reverse
                                +
                                u_forward
                            )
                        )


                        roots_middle = (
                            equilibrium_roots(
                                u_middle,
                                ALPHA,
                                p
                            )
                        )


                        three_root_pass = (
                            len(
                                roots_middle
                            )
                            ==
                            3
                        )


                        if three_root_pass:

                            stability_flags = []


                            for H in roots_middle:

                                M = M_star(
                                    H,
                                    p
                                )

                                theta = theta_star(
                                    H,
                                    ALPHA,
                                    p
                                )


                                eigvals = (
                                    np.linalg.eigvals(
                                        jacobian_M1(
                                            H,
                                            M,
                                            theta,
                                            u_middle,
                                            ALPHA,
                                            p
                                        )
                                    )
                                )


                                stability_flags.append(

                                    np.max(
                                        eigvals.real
                                    )
                                    <
                                    -STABILITY_TOL
                                )


                            stability_pattern_pass = (

                                stability_flags
                                ==
                                [
                                    True,
                                    False,
                                    True
                                ]
                            )


                # --------------------------------------------
                # Ranking is applied ONLY after the candidate
                # passes the scientific criteria.
                # --------------------------------------------

                distance_from_reference = np.sqrt(

                    (
                        (n - REFERENCE_N)
                        /
                        2.0
                    )**2

                    +

                    (
                        (q - REFERENCE_Q)
                        /
                        3.0
                    )**2

                    +

                    (
                        (K_M - REFERENCE_K)
                        /
                        0.4
                    )**2

                    +

                    (
                        (
                            theta0
                            -
                            REFERENCE_THETA0
                        )
                        /
                        0.4
                    )**2
                )


                pass_all = bool(

                    positive_domain_pass

                    and

                    three_root_pass

                    and

                    stability_pattern_pass

                    and

                    root_count_at_min == 1

                    and

                    root_count_at_max == 1
                )


                screen_records.append({

                    "n":
                        n,

                    "q":
                        q,

                    "K_M":
                        K_M,

                    "theta0":
                        theta0,

                    "alpha":
                        ALPHA,

                    "r":
                        p["r"],

                    "epsilon":
                        p[
                            "epsilon"
                        ],

                    "number_of_folds":
                        len(folds),

                    "u_reverse_fold":
                        u_reverse,

                    "u_forward_fold":
                        u_forward,

                    "H_reverse_fold":
                        H_reverse,

                    "H_forward_fold":
                        H_forward,

                    "root_count_at_u_min":
                        root_count_at_min,

                    "root_count_at_u_max":
                        root_count_at_max,

                    "positive_domain_pass":
                        positive_domain_pass,

                    "three_root_pass":
                        three_root_pass,

                    "stability_pattern_pass":
                        stability_pattern_pass,

                    "bistable_comparator_pass":
                        pass_all,

                    "distance_from_experiment01_reference":
                        distance_from_reference
                })


bistable_screen_df = pd.DataFrame(
    screen_records
)


bistable_screen_df.to_csv(
    EXP6_DIR
    / "bistable_parameter_screen.csv",
    index=False,
    float_format="%.17g"
)


passing_bistable_df = (
    bistable_screen_df[
        bistable_screen_df[
            "bistable_comparator_pass"
        ]
    ]
    .copy()
)


if len(
    passing_bistable_df
) == 0:

    raise RuntimeError(
        "No predeclared candidate produced a complete "
        "positive-u bistable hysteresis interval.\n\n"
        "STOP HERE rather than hand-tuning parameters.\n"
        "Inspect bistable_parameter_screen.csv."
    )


passing_bistable_df = (
    passing_bistable_df
    .sort_values(
        "distance_from_experiment01_reference"
    )
)


selected_bistable = (
    passing_bistable_df.iloc[0]
)


display(
    selected_bistable.to_frame().T
)

In [ ]:
# ============================================================
# SELECTED BISTABLE PARAMETER SET
# ============================================================

BISTABLE_PARAMS = {

    "n":
        float(
            selected_bistable[
                "n"
            ]
        ),

    "q":
        float(
            selected_bistable[
                "q"
            ]
        ),

    "K_M":
        float(
            selected_bistable[
                "K_M"
            ]
        ),

    "theta0":
        float(
            selected_bistable[
                "theta0"
            ]
        ),

    "r":
        float(
            selected_bistable[
                "r"
            ]
        ),

    "epsilon":
        float(
            selected_bistable[
                "epsilon"
            ]
        )
}


selected_bistable_df = pd.DataFrame([

    {
        "parameter":
            key,

        "value":
            value
    }

    for key, value
    in BISTABLE_PARAMS.items()
])


selected_bistable_df[
    "alpha"
] = ALPHA


selected_bistable_df.to_csv(
    EXP6_DIR
    / "selected_bistable_parameters.csv",
    index=False,
    float_format="%.17g"
)


with open(
    EXP6_DIR
    / "selected_bistable_parameters.json",
    "w"
) as f:

    json.dump(
        {
            **BISTABLE_PARAMS,
            "alpha": ALPHA
        },
        f,
        indent=4
    )


print(BISTABLE_PARAMS)

In [ ]:
# ============================================================
# EQUILIBRIUM REFERENCES
# ============================================================

bistable_folds = detect_folds(
    ALPHA,
    BISTABLE_PARAMS
)


forward_fold = (
    bistable_folds[0]
)

reverse_fold = (
    bistable_folds[1]
)


U_EQ_FORWARD = float(
    forward_fold[
        "u_fold"
    ]
)

U_EQ_REVERSE = float(
    reverse_fold[
        "u_fold"
    ]
)

EQUILIBRIUM_HYSTERESIS_WIDTH = (
    U_EQ_FORWARD
    -
    U_EQ_REVERSE
)


reference_metrics_df = pd.DataFrame([

    {
        "regime":
            "monostable_dynamic_lag_control",

        "equilibrium_forward_fold":
            np.nan,

        "equilibrium_reverse_fold":
            np.nan,

        "equilibrium_hysteresis_width":
            0.0
    },

    {
        "regime":
            "bistable_equilibrium_hysteresis",

        "equilibrium_forward_fold":
            U_EQ_FORWARD,

        "equilibrium_reverse_fold":
            U_EQ_REVERSE,

        "equilibrium_hysteresis_width":
            EQUILIBRIUM_HYSTERESIS_WIDTH
    }
])


reference_metrics_df.to_csv(
    EXP6_DIR
    / "equilibrium_reference_metrics.csv",
    index=False,
    float_format="%.17g"
)


display(
    reference_metrics_df
)

In [ ]:
# ============================================================
# EQUILIBRIUM H=0.5 CROSSING REFERENCES
# ============================================================

U_HCROSS_MONO = float(

    u_star(
        H_CROSS,
        ALPHA,
        MONO_PARAMS
    )
)


U_HCROSS_BISTABLE = float(

    u_star(
        H_CROSS,
        ALPHA,
        BISTABLE_PARAMS
    )
)


crossing_reference_df = pd.DataFrame([

    {
        "regime":
            "monostable_dynamic_lag_control",

        "H_cross":
            H_CROSS,

        "equilibrium_u_at_H_cross":
            U_HCROSS_MONO
    },

    {
        "regime":
            "bistable_equilibrium_hysteresis",

        "H_cross":
            H_CROSS,

        "equilibrium_u_at_H_cross":
            U_HCROSS_BISTABLE
    }
])


crossing_reference_df.to_csv(
    EXP6_DIR
    / "H_cross_equilibrium_references.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# INITIAL STATES
# ============================================================

def unique_low_state_at_u_min(
    p
):

    roots = equilibrium_roots(
        U_MIN,
        ALPHA,
        p
    )


    if len(roots) != 1:

        raise RuntimeError(
            f"Expected one equilibrium at U_MIN={U_MIN}, "
            f"found {len(roots)}."
        )


    return state_from_H(
        roots[0],
        ALPHA,
        p
    )


MONO_INITIAL_STATE = (
    unique_low_state_at_u_min(
        MONO_PARAMS
    )
)


BISTABLE_INITIAL_STATE = (
    unique_low_state_at_u_min(
        BISTABLE_PARAMS
    )
)


print(
    "Monostable initial state:",
    MONO_INITIAL_STATE
)

print(
    "Bistable initial state:",
    BISTABLE_INITIAL_STATE
)

In [ ]:
# ============================================================
# TRIANGULAR FORCING
# ============================================================

def triangular_forcing(
    T_ramp
):

    speed = (
        DELTA_U
        /
        T_ramp
    )


    def u_function(t):

        if t <= T_ramp:

            return (
                U_MIN
                +
                speed * t
            )

        else:

            return (
                U_MAX
                -
                speed
                *
                (
                    t
                    -
                    T_ramp
                )
            )


    return (
        u_function,
        speed
    )

In [ ]:
# ============================================================
# SINGLE RAMP-CYCLE RUNNER
# ============================================================

def run_ramp_cycle(
    regime_name,
    p,
    initial_state,
    T_ramp
):

    u_function, speed = (
        triangular_forcing(
            T_ramp
        )
    )


    total_time = (
        2.0
        *
        T_ramp
    )


    t_eval = np.linspace(
        0.0,
        total_time,
        N_RAMP_POINTS
    )


    solution = solve_ivp(

        lambda t, y:
            rhs_M1_time_dependent_u(
                t,
                y,
                u_function,
                ALPHA,
                p
            ),

        (
            0.0,
            total_time
        ),

        initial_state,

        t_eval=t_eval,

        method="LSODA",

        rtol=RTOL,
        atol=ATOL
    )


    u_values = np.array([
        u_function(t)
        for t in solution.t
    ])


    phase = np.where(
        solution.t
        <=
        T_ramp,
        "up",
        "down"
    )


    return pd.DataFrame({

        "regime":
            regime_name,

        "ramp_duration":
            T_ramp,

        "ramp_speed":
            speed,

        "time":
            solution.t,

        "phase":
            phase,

        "u":
            u_values,

        "H":
            solution.y[0],

        "M":
            solution.y[1],

        "theta":
            solution.y[2],

        "S":
            solution.y[1]
            +
            u_values
    })

In [ ]:
# ============================================================
# RUN ALL RATE EXPERIMENTS
# ============================================================

trajectory_frames = []


for T_ramp in RAMP_DURATIONS:

    print(
        f"Running monostable T_ramp={T_ramp:g}"
    )

    trajectory_frames.append(

        run_ramp_cycle(
            "monostable_dynamic_lag_control",
            MONO_PARAMS,
            MONO_INITIAL_STATE,
            T_ramp
        )
    )


    print(
        f"Running bistable T_ramp={T_ramp:g}"
    )

    trajectory_frames.append(

        run_ramp_cycle(
            "bistable_equilibrium_hysteresis",
            BISTABLE_PARAMS,
            BISTABLE_INITIAL_STATE,
            T_ramp
        )
    )


all_ramp_trajectories_df = (
    pd.concat(
        trajectory_frames,
        ignore_index=True
    )
)


all_ramp_trajectories_df.to_csv(
    EXP6_DIR
    / "all_ramp_trajectories.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Saved",
    len(
        all_ramp_trajectories_df
    ),
    "trajectory rows."
)

In [ ]:
# ============================================================
# THRESHOLD CROSSING HELPER
# ============================================================

def interpolate_H_crossing(
    u,
    H,
    H_cross
):

    u = np.asarray(
        u,
        dtype=float
    )

    H = np.asarray(
        H,
        dtype=float
    )


    difference = (
        H
        -
        H_cross
    )


    crossing_indices = np.where(

        difference[:-1]
        *
        difference[1:]
        <=
        0.0

    )[0]


    if len(
        crossing_indices
    ) == 0:

        return np.nan


    i = crossing_indices[0]


    H1 = H[i]
    H2 = H[i + 1]

    u1 = u[i]
    u2 = u[i + 1]


    if H2 == H1:

        return (
            0.5
            *
            (
                u1 + u2
            )
        )


    fraction = (
        H_cross
        -
        H1
    ) / (
        H2
        -
        H1
    )


    return (
        u1
        +
        fraction
        *
        (
            u2
            -
            u1
        )
    )

In [ ]:
# ============================================================
# RAMP METRICS
# ============================================================

def branch_switch_point(
    branch_df
):

    branch_df = (
        branch_df
        .sort_values(
            "time"
        )
        .reset_index(
            drop=True
        )
    )


    u = (
        branch_df[
            "u"
        ].values
    )

    H = (
        branch_df[
            "H"
        ].values
    )


    dH_du = np.gradient(
        H,
        u
    )


    n = len(
        branch_df
    )

    trim = max(
        2,
        int(
            EDGE_TRIM_FRACTION
            *
            n
        )
    )


    valid_indices = np.arange(
        trim,
        n - trim
    )


    if len(
        valid_indices
    ) == 0:

        return (
            np.nan,
            np.nan
        )


    local_index = (
        np.argmax(
            np.abs(
                dH_du[
                    valid_indices
                ]
            )
        )
    )


    index = (
        valid_indices[
            local_index
        ]
    )


    return (
        u[index],
        dH_du[index]
    )


def calculate_loop_area(
    up_df,
    down_df,
    variable
):

    up = (
        up_df
        .sort_values(
            "u"
        )
    )

    down = (
        down_df
        .sort_values(
            "u"
        )
    )


    u_common = np.linspace(
        max(
            up["u"].min(),
            down["u"].min()
        ),

        min(
            up["u"].max(),
            down["u"].max()
        ),

        3000
    )


    y_up = np.interp(
        u_common,
        up["u"],
        up[variable]
    )


    y_down = np.interp(
        u_common,
        down["u"],
        down[variable]
    )


    difference = np.abs(
        y_up
        -
        y_down
    )


    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(
            difference,
            u_common
        )

    return np.trapz(
        difference,
        u_common
    )


metric_records = []


for (
    regime,
    T_ramp
), g in (
    all_ramp_trajectories_df
    .groupby(
        [
            "regime",
            "ramp_duration"
        ]
    )
):

    up = (
        g[
            g[
                "phase"
            ]
            ==
            "up"
        ]
        .copy()
    )

    down = (
        g[
            g[
                "phase"
            ]
            ==
            "down"
        ]
        .copy()
    )


    ramp_speed = float(
        g[
            "ramp_speed"
        ].iloc[0]
    )


    u_up_slope, slope_up = (
        branch_switch_point(
            up
        )
    )

    u_down_slope, slope_down = (
        branch_switch_point(
            down
        )
    )


    u_up_cross = (
        interpolate_H_crossing(
            up["u"].values,
            up["H"].values,
            H_CROSS
        )
    )


    # reverse the down leg so search proceeds from low u upward
    down_sorted_u = (
        down
        .sort_values(
            "u"
        )
    )

    u_down_cross = (
        interpolate_H_crossing(
            down_sorted_u[
                "u"
            ].values,

            down_sorted_u[
                "H"
            ].values,

            H_CROSS
        )
    )


    slope_width = (
        u_up_slope
        -
        u_down_slope
    )


    crossing_width = (

        u_up_cross
        -
        u_down_cross

        if (
            np.isfinite(
                u_up_cross
            )
            and
            np.isfinite(
                u_down_cross
            )
        )

        else
        np.nan
    )


    metric_records.append({

        "regime":
            regime,

        "ramp_duration":
            T_ramp,

        "ramp_speed":
            ramp_speed,

        "u_up_max_abs_dHdu":
            u_up_slope,

        "u_down_max_abs_dHdu":
            u_down_slope,

        "slope_based_hysteresis_width":
            slope_width,

        "dHdu_at_up_switch":
            slope_up,

        "dHdu_at_down_switch":
            slope_down,

        "u_up_H_cross":
            u_up_cross,

        "u_down_H_cross":
            u_down_cross,

        "H_cross_based_width":
            crossing_width,

        "H_loop_area":
            calculate_loop_area(
                up,
                down,
                "H"
            ),

        "M_loop_area":
            calculate_loop_area(
                up,
                down,
                "M"
            ),

        "theta_loop_area":
            calculate_loop_area(
                up,
                down,
                "theta"
            )
    })


ramp_metrics_df = pd.DataFrame(
    metric_records
)


ramp_metrics_df.to_csv(
    EXP6_DIR
    / "ramp_metrics.csv",
    index=False,
    float_format="%.17g"
)


display(
    ramp_metrics_df
)

In [ ]:
# ============================================================
# QUASI-STATIC COMPARISON
# ============================================================

comparison_records = []


for regime, g in (
    ramp_metrics_df.groupby(
        "regime"
    )
):

    slowest = (
        g.sort_values(
            "ramp_speed"
        )
        .iloc[0]
    )


    if regime == (
        "monostable_dynamic_lag_control"
    ):

        expected_width = 0.0

        expected_up = (
            U_HCROSS_MONO
        )

        expected_down = (
            U_HCROSS_MONO
        )

    else:

        expected_width = (
            EQUILIBRIUM_HYSTERESIS_WIDTH
        )

        expected_up = (
            U_EQ_FORWARD
        )

        expected_down = (
            U_EQ_REVERSE
        )


    comparison_records.append({

        "regime":
            regime,

        "slowest_ramp_speed":
            slowest[
                "ramp_speed"
            ],

        "slowest_ramp_duration":
            slowest[
                "ramp_duration"
            ],

        "observed_slope_width":
            slowest[
                "slope_based_hysteresis_width"
            ],

        "expected_quasistatic_width":
            expected_width,

        "width_error":
            (
                slowest[
                    "slope_based_hysteresis_width"
                ]
                -
                expected_width
            ),

        "observed_u_up":
            slowest[
                "u_up_max_abs_dHdu"
            ],

        "expected_u_up":
            expected_up,

        "observed_u_down":
            slowest[
                "u_down_max_abs_dHdu"
            ],

        "expected_u_down":
            expected_down
    })


quasistatic_comparison_df = pd.DataFrame(
    comparison_records
)


quasistatic_comparison_df.to_csv(
    EXP6_DIR
    / "quasistatic_comparison.csv",
    index=False,
    float_format="%.17g"
)


display(
    quasistatic_comparison_df
)

In [ ]:
# ============================================================
# EXPLORATORY SLOW-RATE EXTRAPOLATION
# ============================================================

extrapolation_records = []


for regime, g in (
    ramp_metrics_df.groupby(
        "regime"
    )
):

    slow = (
        g.sort_values(
            "ramp_speed"
        )
        .head(4)
        .copy()
    )


    x = (
        slow[
            "ramp_speed"
        ].values
    )

    y = (
        slow[
            "slope_based_hysteresis_width"
        ].values
    )


    valid = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )


    x = x[valid]
    y = y[valid]


    if len(x) >= 3:

        coefficients = np.polyfit(
            x,
            y,
            1
        )

        slope = (
            coefficients[0]
        )

        intercept = (
            coefficients[1]
        )


        predicted = (
            slope * x
            +
            intercept
        )


        ss_res = np.sum(
            (
                y
                -
                predicted
            )**2
        )


        ss_tot = np.sum(
            (
                y
                -
                np.mean(y)
            )**2
        )


        r_squared = (

            1.0
            -
            ss_res
            /
            ss_tot

            if ss_tot > 0.0

            else
            np.nan
        )


    else:

        slope = np.nan
        intercept = np.nan
        r_squared = np.nan


    expected_intercept = (

        0.0

        if regime
        ==
        "monostable_dynamic_lag_control"

        else
        EQUILIBRIUM_HYSTERESIS_WIDTH
    )


    extrapolation_records.append({

        "regime":
            regime,

        "number_of_slowest_rates_used":
            len(x),

        "linear_slope":
            slope,

        "extrapolated_width_at_zero_speed":
            intercept,

        "expected_quasistatic_width":
            expected_intercept,

        "extrapolation_error":
            (
                intercept
                -
                expected_intercept
            ),

        "R_squared":
            r_squared
    })


slow_rate_extrapolation_df = (
    pd.DataFrame(
        extrapolation_records
    )
)


slow_rate_extrapolation_df.to_csv(
    EXP6_DIR
    / "slow_rate_extrapolation.csv",
    index=False,
    float_format="%.17g"
)


display(
    slow_rate_extrapolation_df
)

In [ ]:
# ============================================================
# FIGURE 1 SOURCE DATA
# ============================================================

fig01_data = (
    all_ramp_trajectories_df[
        all_ramp_trajectories_df[
            "regime"
        ]
        ==
        "monostable_dynamic_lag_control"
    ]
    .copy()
)


fig01_data.to_csv(
    EXP6_DIR
    / "fig01_monostable_H_u_loops_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 1
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for speed, g in (
    fig01_data.groupby(
        "ramp_speed"
    )
):

    g = g.sort_values(
        "time"
    )

    ax.plot(
        g["u"],
        g["H"],
        linewidth=1.8,
        label=f"v={speed:.4g}"
    )


ax.set_xlabel(
    "External forcing, $u$"
)

ax.set_ylabel(
    "Host state, $H$"
)

ax.set_title(
    "Monostable regime: rate-dependent dynamic lag"
)

ax.legend(
    frameon=False,
    title="Ramp speed"
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig01_monostable_H_u_loops.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2 SOURCE DATA
# ============================================================

fig02_data = (
    all_ramp_trajectories_df[
        all_ramp_trajectories_df[
            "regime"
        ]
        ==
        "bistable_equilibrium_hysteresis"
    ]
    .copy()
)


fig02_data.to_csv(
    EXP6_DIR
    / "fig02_bistable_H_u_loops_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 2
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for speed, g in (
    fig02_data.groupby(
        "ramp_speed"
    )
):

    g = g.sort_values(
        "time"
    )

    ax.plot(
        g["u"],
        g["H"],
        linewidth=1.8,
        label=f"v={speed:.4g}"
    )


ax.axvline(
    U_EQ_FORWARD,
    linestyle="--",
    linewidth=1.2,
    label="Equilibrium forward fold"
)


ax.axvline(
    U_EQ_REVERSE,
    linestyle=":",
    linewidth=1.2,
    label="Equilibrium reverse fold"
)


ax.set_xlabel(
    "External forcing, $u$"
)

ax.set_ylabel(
    "Host state, $H$"
)

ax.set_title(
    "Bistable regime: equilibrium hysteresis plus dynamic lag"
)

ax.legend(
    frameon=False,
    title="Ramp speed"
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig02_bistable_H_u_loops.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3 SOURCE DATA
# ============================================================

fig03_data = (
    ramp_metrics_df[[
        "regime",
        "ramp_duration",
        "ramp_speed",
        "slope_based_hysteresis_width",
        "H_cross_based_width"
    ]]
    .copy()
)


fig03_data.to_csv(
    EXP6_DIR
    / "fig03_hysteresis_width_vs_speed_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 3
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for regime, g in (
    fig03_data.groupby(
        "regime"
    )
):

    g = g.sort_values(
        "ramp_speed"
    )

    ax.plot(
        g["ramp_speed"],
        g[
            "slope_based_hysteresis_width"
        ],
        marker="o",
        linewidth=2.0,
        label=regime
    )


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1,
    label="Zero-width limit"
)


ax.axhline(
    EQUILIBRIUM_HYSTERESIS_WIDTH,
    linestyle=":",
    linewidth=1.2,
    label="Bistable equilibrium fold separation"
)


ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "Apparent hysteresis width, $\\Delta u$"
)

ax.set_title(
    "Dynamic lag versus equilibrium hysteresis"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig03_hysteresis_width_vs_speed.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 4 SOURCE DATA
# ============================================================

fig04_data = (
    ramp_metrics_df[[
        "regime",
        "ramp_speed",
        "H_loop_area",
        "M_loop_area",
        "theta_loop_area"
    ]]
    .copy()
)


fig04_data.to_csv(
    EXP6_DIR
    / "fig04_loop_area_vs_speed_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 4
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for regime, g in (
    fig04_data.groupby(
        "regime"
    )
):

    g = g.sort_values(
        "ramp_speed"
    )

    ax.plot(
        g["ramp_speed"],
        g["H_loop_area"],
        marker="o",
        linewidth=2.0,
        label=regime
    )


ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "Area between ascending and descending $H(u)$ branches"
)

ax.set_title(
    "Rate dependence of loop area"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig04_loop_area_vs_speed.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 5 SOURCE DATA
# ============================================================

fig05_data = (
    fig01_data[[
        "ramp_duration",
        "ramp_speed",
        "time",
        "phase",
        "u",
        "theta"
    ]]
    .copy()
)


fig05_data.to_csv(
    EXP6_DIR
    / "fig05_monostable_theta_u_loops_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 5
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for speed, g in (
    fig05_data.groupby(
        "ramp_speed"
    )
):

    g = g.sort_values(
        "time"
    )

    ax.plot(
        g["u"],
        g["theta"],
        linewidth=1.8,
        label=f"v={speed:.4g}"
    )


ax.set_xlabel(
    "External forcing, $u$"
)

ax.set_ylabel(
    "Adaptive threshold, $\\theta$"
)

ax.set_title(
    "Adaptive-threshold lag in the monostable regime"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig05_monostable_theta_u_loops.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 6 SOURCE DATA
# ============================================================

fig06_data = (
    fig02_data[[
        "ramp_duration",
        "ramp_speed",
        "time",
        "phase",
        "u",
        "theta"
    ]]
    .copy()
)


fig06_data.to_csv(
    EXP6_DIR
    / "fig06_bistable_theta_u_loops_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE 6
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for speed, g in (
    fig06_data.groupby(
        "ramp_speed"
    )
):

    g = g.sort_values(
        "time"
    )

    ax.plot(
        g["u"],
        g["theta"],
        linewidth=1.8,
        label=f"v={speed:.4g}"
    )


ax.set_xlabel(
    "External forcing, $u$"
)

ax.set_ylabel(
    "Adaptive threshold, $\\theta$"
)

ax.set_title(
    "Adaptive-threshold trajectories in the bistable regime"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6_DIR
    / "fig06_bistable_theta_u_loops.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 06 SUMMARY
# ============================================================

mono_metrics = (
    ramp_metrics_df[
        ramp_metrics_df[
            "regime"
        ]
        ==
        "monostable_dynamic_lag_control"
    ]
    .sort_values(
        "ramp_speed"
    )
)


bi_metrics = (
    ramp_metrics_df[
        ramp_metrics_df[
            "regime"
        ]
        ==
        "bistable_equilibrium_hysteresis"
    ]
    .sort_values(
        "ramp_speed"
    )
)


mono_slowest = (
    mono_metrics.iloc[0]
)

bi_slowest = (
    bi_metrics.iloc[0]
)


summary_rows = [

    {
        "metric":
            "number_of_positive_u_bistable_candidates",

        "value":
            len(
                passing_bistable_df
            )
    },

    {
        "metric":
            "equilibrium_bistable_forward_fold",

        "value":
            U_EQ_FORWARD
    },

    {
        "metric":
            "equilibrium_bistable_reverse_fold",

        "value":
            U_EQ_REVERSE
    },

    {
        "metric":
            "equilibrium_bistable_width",

        "value":
            EQUILIBRIUM_HYSTERESIS_WIDTH
    },

    {
        "metric":
            "slowest_ramp_speed",

        "value":
            mono_slowest[
                "ramp_speed"
            ]
    },

    {
        "metric":
            "monostable_width_at_slowest_speed",

        "value":
            mono_slowest[
                "slope_based_hysteresis_width"
            ]
    },

    {
        "metric":
            "bistable_width_at_slowest_speed",

        "value":
            bi_slowest[
                "slope_based_hysteresis_width"
            ]
    },

    {
        "metric":
            "bistable_slowest_width_minus_equilibrium_width",

        "value":
            (
                bi_slowest[
                    "slope_based_hysteresis_width"
                ]
                -
                EQUILIBRIUM_HYSTERESIS_WIDTH
            )
    },

    {
        "metric":
            "monostable_loop_area_at_slowest_speed",

        "value":
            mono_slowest[
                "H_loop_area"
            ]
    },

    {
        "metric":
            "bistable_loop_area_at_slowest_speed",

        "value":
            bi_slowest[
                "H_loop_area"
            ]
    }
]


experiment06_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


experiment06_summary_df.to_csv(
    EXP6_DIR
    / "experiment06_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    experiment06_summary_df
)

In [ ]:
# ============================================================
# DECISION FLAGS
# ============================================================

mono_width_fast = float(
    mono_metrics.iloc[-1][
        "slope_based_hysteresis_width"
    ]
)

mono_width_slow = float(
    mono_metrics.iloc[0][
        "slope_based_hysteresis_width"
    ]
)


bi_width_fast = float(
    bi_metrics.iloc[-1][
        "slope_based_hysteresis_width"
    ]
)

bi_width_slow = float(
    bi_metrics.iloc[0][
        "slope_based_hysteresis_width"
    ]
)


monostable_width_shrinks = bool(

    abs(
        mono_width_slow
    )
    <
    abs(
        mono_width_fast
    )
)


bistable_width_approaches_fold_width = bool(

    abs(
        bi_width_slow
        -
        EQUILIBRIUM_HYSTERESIS_WIDTH
    )
    <
    0.10
    *
    EQUILIBRIUM_HYSTERESIS_WIDTH
)


bistable_width_remains_finite = bool(

    bi_width_slow
    >
    0.5
    *
    EQUILIBRIUM_HYSTERESIS_WIDTH
)


decision_flags_df = pd.DataFrame([{

    "monostable_apparent_width_shrinks_as_ramp_slows":
        monostable_width_shrinks,

    "bistable_slowest_width_within_10_percent_of_equilibrium_fold_width":
        bistable_width_approaches_fold_width,

    "bistable_slowest_width_remains_finite":
        bistable_width_remains_finite
}])


decision_flags_df.to_csv(
    EXP6_DIR
    / "experiment06_decision_flags.csv",
    index=False
)


display(
    decision_flags_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
EXPERIMENT 06
DYNAMIC LAG VERSUS EQUILIBRIUM HYSTERESIS
==========================================

PRIMARY QUESTION
----------------
Can rate-dependent loops caused by finite adaptation speed be
distinguished from genuine equilibrium hysteresis?

CORE DIAGNOSTIC
---------------
For multiple ramp speeds:

    U_MIN -> U_MAX -> U_MIN

measure apparent forward and reverse switching points.

If the regime is monostable:

    apparent hysteresis width -> 0

as ramp speed -> 0.

If the equilibrium system is bistable:

    apparent hysteresis width

should approach the finite separation between its two
equilibrium folds.

REGIME A
--------
Monostable adaptive regime selected in Experiment 04.

Parameters:
{MONO_PARAMS}

alpha:
{ALPHA}

Any up/down loop in this regime is necessarily dynamic.

REGIME B
--------
Positive-forcing bistable adaptive comparator.

The comparator was NOT selected manually.

A predeclared parameter grid was screened and a candidate was
accepted only if:

    U_MIN < reverse fold < forward fold < U_MAX

with:

- one equilibrium at U_MIN;
- one equilibrium at U_MAX;
- three equilibria between the folds;
- stable / unstable / stable local stability pattern.

The passing candidate closest to the Experiment 01 reference
parameter set was chosen automatically.

Selected comparator:
{BISTABLE_PARAMS}

Equilibrium forward fold:
{U_EQ_FORWARD}

Equilibrium reverse fold:
{U_EQ_REVERSE}

Equilibrium hysteresis width:
{EQUILIBRIUM_HYSTERESIS_WIDTH}

RAMP RANGE
----------
U_MIN = {U_MIN}
U_MAX = {U_MAX}

Ramp durations:
{RAMP_DURATIONS.tolist()}

Ramp speeds:
{RAMP_SPEEDS.tolist()}

PRIMARY SWITCHING METRIC
------------------------
Maximum absolute slope:

    max |dH/du|

on each upward and downward ramp leg.

A secondary H = {H_CROSS} crossing metric is also saved.

LOOP AREAS
----------
For each ramp speed the areas between the upward and downward
branches are calculated for:

    H(u)
    M(u)
    theta(u)

INTERPRETATION
--------------
A loop seen at finite forcing speed is NOT sufficient evidence
of equilibrium hysteresis.

The relevant distinction is the slow-ramp limit:

MONOSTABLE:
    width -> 0

BISTABLE:
    width -> equilibrium fold separation > 0

The slow-rate linear extrapolation is exploratory only and is
not assumed to be a mechanistic law.

MODEL-DEFINITION NOTE
---------------------
The same provisional microbial response function is retained:

    G(H) = H^q / (H^q + K_M^q)

Changing this function later requires rerunning the model
experiments.

DATA POLICY
-----------
All numerical outputs are CSV.

Every JPEG has a figure-specific CSV.

Experiment 06 is stored separately from Experiments 01-05.
"""


with open(
    EXP6_DIR
    / "experiment06_README.txt",
    "w"
) as f:

    f.write(
        readme
    )


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    EXP6_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                /
                1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    EXP6_DIR
    / "experiment06_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_06_Dynamic_Lag_vs_Equilibrium_Hysteresis_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP6_DIR)
)


print(
    "=============================================="
)

print(
    "EXPERIMENT 06 EXPORT COMPLETE"
)

print(
    "=============================================="
)

print("\nFolder:")
print(EXP6_DIR)

print("\nZIP:")
print(zip_path)

print("\nFILES:\n")


for path in sorted(
    EXP6_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:65s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# Experiment 6B

In [ ]:
# ============================================================
# EXPERIMENT 06B
# Quasi-static convergence correction
# ============================================================

from pathlib import Path
import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp


EXP6B_DIR = (
    BASE_DIR
    / "Experiment_06B_Quasistatic_Hysteresis_Convergence"
)

EXP6B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Reload the bistable comparator selected in Experiment 6.
# DO NOT rescreen or retune parameters.
# ------------------------------------------------------------

bistable_parameter_file = (
    EXP6_DIR
    / "selected_bistable_parameters.json"
)

if not bistable_parameter_file.exists():
    raise FileNotFoundError(
        f"Missing:\n{bistable_parameter_file}"
    )


with open(
    bistable_parameter_file,
    "r"
) as f:

    saved_bistable = json.load(f)


BISTABLE_PARAMS = {

    "n":
        float(saved_bistable["n"]),

    "q":
        float(saved_bistable["q"]),

    "K_M":
        float(saved_bistable["K_M"]),

    "theta0":
        float(saved_bistable["theta0"]),

    "r":
        float(saved_bistable["r"]),

    "epsilon":
        float(saved_bistable["epsilon"])
}


# ------------------------------------------------------------
# Load exact equilibrium reference values from Experiment 6
# ------------------------------------------------------------

reference_df = pd.read_csv(
    EXP6_DIR
    / "equilibrium_reference_metrics.csv"
)


bistable_reference = (
    reference_df[
        reference_df["regime"]
        ==
        "bistable_equilibrium_hysteresis"
    ]
    .iloc[0]
)


U_EQ_FORWARD = float(
    bistable_reference[
        "equilibrium_forward_fold"
    ]
)

U_EQ_REVERSE = float(
    bistable_reference[
        "equilibrium_reverse_fold"
    ]
)

EQ_WIDTH = float(
    bistable_reference[
        "equilibrium_hysteresis_width"
    ]
)


print("Experiment 6B directory:")
print(EXP6B_DIR)

print("\nMonostable parameters:")
print(MONO_PARAMS)

print("\nBistable parameters:")
print(BISTABLE_PARAMS)

print("\nEquilibrium reference:")
print("forward fold =", U_EQ_FORWARD)
print("reverse fold =", U_EQ_REVERSE)
print("width =", EQ_WIDTH)

In [ ]:
# ============================================================
# CORRECTED RAMP PROTOCOL
# ============================================================

# Crucial correction:
# give the reverse switch substantial positive-u room
# below its equilibrium fold at ~0.063.

U_MIN_6B = 0.01
U_MAX_6B = 0.80

DELTA_U_6B = (
    U_MAX_6B
    -
    U_MIN_6B
)


# ------------------------------------------------------------
# Focus on the slower-rate regime.
#
# Experiment 6 already characterized the fast transients.
# ------------------------------------------------------------

RAMP_DURATIONS_6B = np.array([
    200.0,
    400.0,
    800.0,
    1600.0,
    3200.0,
    6400.0
])


RAMP_SPEEDS_6B = (
    DELTA_U_6B
    /
    RAMP_DURATIONS_6B
)


# Primary operational switch marker
H_CROSS_6B = 0.50


# Dense enough output in forcing space
N_RAMP_POINTS_6B = 12001


RTOL_6B = 1e-10
ATOL_6B = 1e-12


CONFIG6B = {

    "u_min":
        U_MIN_6B,

    "u_max":
        U_MAX_6B,

    "delta_u":
        DELTA_U_6B,

    "H_cross":
        H_CROSS_6B,

    "ramp_durations":
        RAMP_DURATIONS_6B.tolist(),

    "ramp_speeds":
        RAMP_SPEEDS_6B.tolist(),

    "n_ramp_points":
        N_RAMP_POINTS_6B,

    "rtol":
        RTOL_6B,

    "atol":
        ATOL_6B,

    "alpha":
        ALPHA,

    "equilibrium_forward_fold":
        U_EQ_FORWARD,

    "equilibrium_reverse_fold":
        U_EQ_REVERSE,

    "equilibrium_hysteresis_width":
        EQ_WIDTH
}


with open(
    EXP6B_DIR
    / "experiment06B_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG6B,
        f,
        indent=4
    )


pd.DataFrame([
    {
        "ramp_duration":
            T,

        "ramp_speed":
            v
    }

    for T, v in zip(
        RAMP_DURATIONS_6B,
        RAMP_SPEEDS_6B
    )
]).to_csv(
    EXP6B_DIR
    / "ramp_protocol.csv",
    index=False,
    float_format="%.17g"
)


print(
    pd.DataFrame({
        "T_ramp": RAMP_DURATIONS_6B,
        "speed": RAMP_SPEEDS_6B
    })
)

In [ ]:
# ============================================================
# INITIAL-STATE VALIDATION
# ============================================================

def unique_equilibrium_state_at_u(
    u,
    p
):

    roots = equilibrium_roots(
        u,
        ALPHA,
        p
    )


    if len(roots) != 1:

        raise RuntimeError(
            f"Expected exactly one equilibrium at "
            f"u={u}, but found {len(roots)}."
        )


    return state_from_H(
        roots[0],
        ALPHA,
        p
    )


MONO_INITIAL_6B = (
    unique_equilibrium_state_at_u(
        U_MIN_6B,
        MONO_PARAMS
    )
)


BISTABLE_INITIAL_6B = (
    unique_equilibrium_state_at_u(
        U_MIN_6B,
        BISTABLE_PARAMS
    )
)


endpoint_validation_df = pd.DataFrame([

    {
        "regime":
            "monostable",

        "u":
            U_MIN_6B,

        "H":
            MONO_INITIAL_6B[0],

        "M":
            MONO_INITIAL_6B[1],

        "theta":
            MONO_INITIAL_6B[2]
    },

    {
        "regime":
            "bistable",

        "u":
            U_MIN_6B,

        "H":
            BISTABLE_INITIAL_6B[0],

        "M":
            BISTABLE_INITIAL_6B[1],

        "theta":
            BISTABLE_INITIAL_6B[2]
    }
])


endpoint_validation_df.to_csv(
    EXP6B_DIR
    / "initial_state_validation.csv",
    index=False,
    float_format="%.17g"
)


display(
    endpoint_validation_df
)

In [ ]:
# ============================================================
# CORRECTED TRIANGULAR RAMP
# ============================================================

def triangular_forcing_6B(
    T_ramp
):

    speed = (
        DELTA_U_6B
        /
        T_ramp
    )


    def u_function(t):

        if t <= T_ramp:

            return (
                U_MIN_6B
                +
                speed * t
            )

        return (
            U_MAX_6B
            -
            speed
            *
            (
                t
                -
                T_ramp
            )
        )


    return (
        u_function,
        speed
    )


def run_ramp_6B(
    regime,
    p,
    initial_state,
    T_ramp
):

    u_function, speed = (
        triangular_forcing_6B(
            T_ramp
        )
    )


    total_time = (
        2.0
        *
        T_ramp
    )


    t_eval = np.linspace(
        0.0,
        total_time,
        N_RAMP_POINTS_6B
    )


    solution = solve_ivp(

        lambda t, y:
            rhs_M1_time_dependent_u(
                t,
                y,
                u_function,
                ALPHA,
                p
            ),

        (
            0.0,
            total_time
        ),

        initial_state,

        t_eval=
            t_eval,

        method=
            "LSODA",

        rtol=
            RTOL_6B,

        atol=
            ATOL_6B
    )


    if not solution.success:

        raise RuntimeError(
            f"ODE failure for {regime}, "
            f"T_ramp={T_ramp}:\n"
            f"{solution.message}"
        )


    u_values = np.array([
        u_function(t)
        for t in solution.t
    ])


    phase = np.where(
        solution.t
        <=
        T_ramp,
        "up",
        "down"
    )


    return pd.DataFrame({

        "regime":
            regime,

        "ramp_duration":
            T_ramp,

        "ramp_speed":
            speed,

        "time":
            solution.t,

        "phase":
            phase,

        "u":
            u_values,

        "H":
            solution.y[0],

        "M":
            solution.y[1],

        "theta":
            solution.y[2],

        "S":
            solution.y[1]
            +
            u_values
    })

In [ ]:
# ============================================================
# RUN EXPERIMENT 06B
# ============================================================

trajectory_frames_6B = []


for T_ramp in RAMP_DURATIONS_6B:

    print(
        f"Monostable: T_ramp={T_ramp:g}"
    )

    trajectory_frames_6B.append(

        run_ramp_6B(
            "monostable_dynamic_lag",
            MONO_PARAMS,
            MONO_INITIAL_6B,
            T_ramp
        )
    )


    print(
        f"Bistable:   T_ramp={T_ramp:g}"
    )

    trajectory_frames_6B.append(

        run_ramp_6B(
            "bistable_equilibrium_hysteresis",
            BISTABLE_PARAMS,
            BISTABLE_INITIAL_6B,
            T_ramp
        )
    )


all_ramps_6B_df = pd.concat(
    trajectory_frames_6B,
    ignore_index=True
)


all_ramps_6B_df.to_csv(
    EXP6B_DIR
    / "all_ramp_trajectories_6B.csv",
    index=False,
    float_format="%.17g"
)


print(
    "\nRows saved:",
    len(all_ramps_6B_df)
)

In [ ]:
# ============================================================
# DIRECTIONAL H=0.5 CROSSING
# ============================================================

def directional_H_crossing(
    branch_df,
    direction,
    H_cross=0.5
):

    g = (
        branch_df
        .sort_values(
            "time"
        )
        .reset_index(
            drop=True
        )
    )


    H = g["H"].values
    u = g["u"].values


    if direction == "up":

        indices = np.where(
            (H[:-1] < H_cross)
            &
            (H[1:] >= H_cross)
        )[0]


    elif direction == "down":

        indices = np.where(
            (H[:-1] > H_cross)
            &
            (H[1:] <= H_cross)
        )[0]


    else:

        raise ValueError(
            "direction must be 'up' or 'down'"
        )


    if len(indices) == 0:

        return np.nan


    i = indices[0]


    H1 = H[i]
    H2 = H[i + 1]

    u1 = u[i]
    u2 = u[i + 1]


    if H2 == H1:

        return (
            0.5
            *
            (
                u1 + u2
            )
        )


    fraction = (
        H_cross
        -
        H1
    ) / (
        H2
        -
        H1
    )


    return (
        u1
        +
        fraction
        *
        (
            u2
            -
            u1
        )
    )

In [ ]:
# ============================================================
# LOOP AREA
# ============================================================

def loop_area_6B(
    up_df,
    down_df,
    variable
):

    up = (
        up_df
        .sort_values("u")
    )

    down = (
        down_df
        .sort_values("u")
    )


    u_common = np.linspace(
        U_MIN_6B,
        U_MAX_6B,
        4000
    )


    y_up = np.interp(
        u_common,
        up["u"],
        up[variable]
    )


    y_down = np.interp(
        u_common,
        down["u"],
        down[variable]
    )


    difference = np.abs(
        y_up
        -
        y_down
    )


    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(
            difference,
            u_common
        )


    return np.trapz(
        difference,
        u_common
    )

In [ ]:
# ============================================================
# CORRECTED RAMP METRICS
# ============================================================

metric_records_6B = []


for (
    regime,
    T_ramp
), g in (
    all_ramps_6B_df
    .groupby([
        "regime",
        "ramp_duration"
    ])
):

    up = (
        g[
            g["phase"]
            ==
            "up"
        ]
        .copy()
    )


    down = (
        g[
            g["phase"]
            ==
            "down"
        ]
        .copy()
    )


    speed = float(
        g[
            "ramp_speed"
        ].iloc[0]
    )


    u_up = directional_H_crossing(
        up,
        "up",
        H_CROSS_6B
    )


    u_down = directional_H_crossing(
        down,
        "down",
        H_CROSS_6B
    )


    forward_complete = bool(
        np.isfinite(
            u_up
        )
    )


    reverse_complete = bool(
        np.isfinite(
            u_down
        )
    )


    cycle_complete = bool(
        forward_complete
        and
        reverse_complete
    )


    if cycle_complete:

        width = (
            u_up
            -
            u_down
        )

    else:

        width = np.nan


    # --------------------------------------------------------
    # Boundary clearances
    # --------------------------------------------------------

    up_upper_clearance = (

        U_MAX_6B
        -
        u_up

        if np.isfinite(
            u_up
        )

        else np.nan
    )


    down_lower_clearance = (

        u_down
        -
        U_MIN_6B

        if np.isfinite(
            u_down
        )

        else np.nan
    )


    if regime == (
        "bistable_equilibrium_hysteresis"
    ):

        forward_reference = (
            U_EQ_FORWARD
        )

        reverse_reference = (
            U_EQ_REVERSE
        )

        width_reference = (
            EQ_WIDTH
        )

    else:

        # For a monostable system both dynamic crossings
        # should converge toward the same equilibrium crossing.
        forward_reference = np.nan
        reverse_reference = np.nan
        width_reference = 0.0


    metric_records_6B.append({

        "regime":
            regime,

        "ramp_duration":
            T_ramp,

        "ramp_speed":
            speed,

        "forward_transition_complete":
            forward_complete,

        "reverse_transition_complete":
            reverse_complete,

        "complete_cycle":
            cycle_complete,

        "u_up_H05":
            u_up,

        "u_down_H05":
            u_down,

        "H05_hysteresis_width":
            width,

        "upper_boundary_clearance":
            up_upper_clearance,

        "lower_boundary_clearance":
            down_lower_clearance,

        "equilibrium_forward_reference":
            forward_reference,

        "equilibrium_reverse_reference":
            reverse_reference,

        "equilibrium_width_reference":
            width_reference,

        "forward_switch_error":
            (
                u_up
                -
                forward_reference

                if (
                    np.isfinite(
                        u_up
                    )
                    and
                    np.isfinite(
                        forward_reference
                    )
                )

                else np.nan
            ),

        "reverse_switch_error":
            (
                u_down
                -
                reverse_reference

                if (
                    np.isfinite(
                        u_down
                    )
                    and
                    np.isfinite(
                        reverse_reference
                    )
                )

                else np.nan
            ),

        "width_error":
            (
                width
                -
                width_reference

                if np.isfinite(
                    width
                )

                else np.nan
            ),

        "H_loop_area":
            loop_area_6B(
                up,
                down,
                "H"
            ),

        "M_loop_area":
            loop_area_6B(
                up,
                down,
                "M"
            ),

        "theta_loop_area":
            loop_area_6B(
                up,
                down,
                "theta"
            )
    })


metrics_6B_df = pd.DataFrame(
    metric_records_6B
)


metrics_6B_df.to_csv(
    EXP6B_DIR
    / "ramp_metrics_6B.csv",
    index=False,
    float_format="%.17g"
)


display(
    metrics_6B_df
)

In [ ]:
# ============================================================
# CONVERGENCE DIAGNOSTICS
# ============================================================

convergence_records = []


for regime, g in (
    metrics_6B_df.groupby(
        "regime"
    )
):

    g = (
        g
        .sort_values(
            "ramp_speed",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    previous_width = np.nan


    for _, row in g.iterrows():

        width = (
            row[
                "H05_hysteresis_width"
            ]
        )


        if np.isfinite(
            previous_width
        ) and np.isfinite(
            width
        ):

            successive_change = (
                width
                -
                previous_width
            )

        else:

            successive_change = np.nan


        if regime == (
            "bistable_equilibrium_hysteresis"
        ):

            target_width = (
                EQ_WIDTH
            )

            absolute_target_error = (

                abs(
                    width
                    -
                    target_width
                )

                if np.isfinite(
                    width
                )

                else np.nan
            )


            relative_target_error = (

                absolute_target_error
                /
                target_width

                if np.isfinite(
                    absolute_target_error
                )

                else np.nan
            )

        else:

            target_width = 0.0

            absolute_target_error = (

                abs(width)

                if np.isfinite(
                    width
                )

                else np.nan
            )

            relative_target_error = np.nan


        convergence_records.append({

            "regime":
                regime,

            "ramp_duration":
                row[
                    "ramp_duration"
                ],

            "ramp_speed":
                row[
                    "ramp_speed"
                ],

            "complete_cycle":
                row[
                    "complete_cycle"
                ],

            "observed_width":
                width,

            "target_width":
                target_width,

            "absolute_target_error":
                absolute_target_error,

            "relative_target_error":
                relative_target_error,

            "change_from_previous_faster_ramp":
                successive_change
        })


        previous_width = width


convergence_6B_df = pd.DataFrame(
    convergence_records
)


convergence_6B_df.to_csv(
    EXP6B_DIR
    / "quasistatic_convergence_diagnostics.csv",
    index=False,
    float_format="%.17g"
)


display(
    convergence_6B_df
)

In [ ]:
# ============================================================
# FIGURE 1
# Hysteresis width versus forcing speed
# ============================================================

fig01_data = (
    metrics_6B_df[[
        "regime",
        "ramp_duration",
        "ramp_speed",
        "complete_cycle",
        "H05_hysteresis_width"
    ]]
    .copy()
)


fig01_data.to_csv(
    EXP6B_DIR
    / "fig01_corrected_width_vs_speed_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for regime, g in (
    fig01_data.groupby(
        "regime"
    )
):

    valid = g[
        g["complete_cycle"]
    ].sort_values(
        "ramp_speed"
    )


    ax.plot(
        valid[
            "ramp_speed"
        ],
        valid[
            "H05_hysteresis_width"
        ],
        marker="o",
        linewidth=2.1,
        label=regime
    )


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1,
    label="Monostable equilibrium limit"
)


ax.axhline(
    EQ_WIDTH,
    linestyle=":",
    linewidth=1.3,
    label="Bistable equilibrium fold separation"
)


ax.set_xscale(
    "log"
)


ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "$H=0.5$ switching-width, $\\Delta u$"
)

ax.set_title(
    "Quasi-static convergence of dynamic and equilibrium hysteresis"
)


ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6B_DIR
    / "fig01_corrected_width_vs_speed.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2
# Bistable switching points versus ramp speed
# ============================================================

fig02_data = (
    metrics_6B_df[
        metrics_6B_df[
            "regime"
        ]
        ==
        "bistable_equilibrium_hysteresis"
    ][[
        "ramp_duration",
        "ramp_speed",
        "complete_cycle",
        "u_up_H05",
        "u_down_H05",
        "forward_switch_error",
        "reverse_switch_error"
    ]]
    .copy()
)


fig02_data.to_csv(
    EXP6B_DIR
    / "fig02_bistable_switch_convergence_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


valid = (
    fig02_data[
        fig02_data[
            "complete_cycle"
        ]
    ]
    .sort_values(
        "ramp_speed"
    )
)


ax.plot(
    valid["ramp_speed"],
    valid["u_up_H05"],
    marker="o",
    linewidth=2.1,
    label="Forward dynamic switch"
)


ax.plot(
    valid["ramp_speed"],
    valid["u_down_H05"],
    marker="o",
    linewidth=2.1,
    label="Reverse dynamic switch"
)


ax.axhline(
    U_EQ_FORWARD,
    linestyle="--",
    linewidth=1.2,
    label="Forward equilibrium fold"
)


ax.axhline(
    U_EQ_REVERSE,
    linestyle=":",
    linewidth=1.2,
    label="Reverse equilibrium fold"
)


ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "Switching forcing, $u$"
)

ax.set_title(
    "Approach of dynamic switching points to equilibrium folds"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

fig.tight_layout()


fig.savefig(
    EXP6B_DIR
    / "fig02_bistable_switch_convergence.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3
# Representative corrected loops
# ============================================================

DISPLAY_DURATIONS = [
    400.0,
    1600.0,
    6400.0
]


fig03_data = (
    all_ramps_6B_df[
        all_ramps_6B_df[
            "ramp_duration"
        ].isin(
            DISPLAY_DURATIONS
        )
    ]
    .copy()
)


fig03_data.to_csv(
    EXP6B_DIR
    / "fig03_selected_H_u_loops_data.csv",
    index=False,
    float_format="%.17g"
)


# Separate figure for each regime
for regime in [
    "monostable_dynamic_lag",
    "bistable_equilibrium_hysteresis"
]:

    fig, ax = plt.subplots(
        figsize=(8.8, 6.5)
    )


    subset = fig03_data[
        fig03_data[
            "regime"
        ]
        ==
        regime
    ]


    for duration, g in (
        subset.groupby(
            "ramp_duration"
        )
    ):

        g = g.sort_values(
            "time"
        )

        ax.plot(
            g["u"],
            g["H"],
            linewidth=2.0,
            label=f"T={duration:g}"
        )


    if regime == (
        "bistable_equilibrium_hysteresis"
    ):

        ax.axvline(
            U_EQ_FORWARD,
            linestyle="--",
            linewidth=1.1
        )

        ax.axvline(
            U_EQ_REVERSE,
            linestyle=":",
            linewidth=1.1
        )


    ax.axhline(
        H_CROSS_6B,
        linestyle="--",
        linewidth=0.9
    )


    ax.set_xlabel(
        "External forcing, $u$"
    )

    ax.set_ylabel(
        "Host state, $H$"
    )

    ax.set_title(
        regime.replace(
            "_",
            " "
        )
    )

    ax.legend(
        frameon=False
    )

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    fig.tight_layout()


    filename = (
        "fig03_monostable_corrected_loops.jpg"
        if regime
        ==
        "monostable_dynamic_lag"

        else
        "fig04_bistable_corrected_loops.jpg"
    )


    fig.savefig(
        EXP6B_DIR
        / filename,
        dpi=600,
        format="jpg",
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

mono = (
    metrics_6B_df[
        metrics_6B_df[
            "regime"
        ]
        ==
        "monostable_dynamic_lag"
    ]
    .sort_values(
        "ramp_speed"
    )
)


bi = (
    metrics_6B_df[
        metrics_6B_df[
            "regime"
        ]
        ==
        "bistable_equilibrium_hysteresis"
    ]
    .sort_values(
        "ramp_speed"
    )
)


mono_slowest = mono.iloc[0]
bi_slowest = bi.iloc[0]


bi_relative_width_error = (

    abs(
        bi_slowest[
            "H05_hysteresis_width"
        ]
        -
        EQ_WIDTH
    )
    /
    EQ_WIDTH
)


summary_6B_df = pd.DataFrame([

    {
        "metric":
            "equilibrium_bistable_width",

        "value":
            EQ_WIDTH
    },

    {
        "metric":
            "slowest_ramp_duration",

        "value":
            bi_slowest[
                "ramp_duration"
            ]
    },

    {
        "metric":
            "slowest_ramp_speed",

        "value":
            bi_slowest[
                "ramp_speed"
            ]
    },

    {
        "metric":
            "monostable_width_slowest",

        "value":
            mono_slowest[
                "H05_hysteresis_width"
            ]
    },

    {
        "metric":
            "monostable_H_loop_area_slowest",

        "value":
            mono_slowest[
                "H_loop_area"
            ]
    },

    {
        "metric":
            "bistable_width_slowest",

        "value":
            bi_slowest[
                "H05_hysteresis_width"
            ]
    },

    {
        "metric":
            "bistable_relative_width_error_slowest",

        "value":
            bi_relative_width_error
    },

    {
        "metric":
            "bistable_forward_switch_slowest",

        "value":
            bi_slowest[
                "u_up_H05"
            ]
    },

    {
        "metric":
            "bistable_reverse_switch_slowest",

        "value":
            bi_slowest[
                "u_down_H05"
            ]
    },

    {
        "metric":
            "bistable_H_loop_area_slowest",

        "value":
            bi_slowest[
                "H_loop_area"
            ]
    }
])


summary_6B_df.to_csv(
    EXP6B_DIR
    / "experiment06B_summary.csv",
    index=False,
    float_format="%.17g"
)


all_cycles_complete = bool(
    metrics_6B_df[
        "complete_cycle"
    ].all()
)


bistable_within_5pct = bool(
    bi_relative_width_error
    <
    0.05
)


mono_width_shrinks = bool(

    abs(
        mono.iloc[0][
            "H05_hysteresis_width"
        ]
    )

    <

    abs(
        mono.iloc[-1][
            "H05_hysteresis_width"
        ]
    )
)


decision_6B_df = pd.DataFrame([{

    "all_forward_and_reverse_transitions_completed":
        all_cycles_complete,

    "monostable_width_shrinks_with_slower_forcing":
        mono_width_shrinks,

    "bistable_slowest_width_within_5_percent_of_equilibrium":
        bistable_within_5pct,

    "bistable_slowest_relative_error":
        bi_relative_width_error
}])


decision_6B_df.to_csv(
    EXP6B_DIR
    / "experiment06B_decision_flags.csv",
    index=False
)


display(
    summary_6B_df
)

display(
    decision_6B_df
)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    EXP6B_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                /
                1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    EXP6B_DIR
    / "experiment06B_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Experiment_06B_Quasistatic_Hysteresis_Convergence_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(EXP6B_DIR)
)


print(
    "============================================="
)

print(
    "EXPERIMENT 06B COMPLETE"
)

print(
    "============================================="
)

print("\nFolder:")
print(EXP6B_DIR)

print("\nZIP:")
print(zip_path)

print("\nFiles:")


for path in sorted(
    EXP6B_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:65s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

In [ ]:
# ============================================================
# 6B.15
# PREDECLARED T_RAMP = 12800 EXTENSION
# ============================================================

T_EXTRA = 12800.0

extra_speed = (
    DELTA_U_6B
    /
    T_EXTRA
)

print(
    "Running additional quasi-static test:"
)

print(
    "T_ramp =", T_EXTRA
)

print(
    "ramp speed =", extra_speed
)


# ------------------------------------------------------------
# Run ONLY the new duration
# ------------------------------------------------------------

extra_mono_df = run_ramp_6B(
    "monostable_dynamic_lag",
    MONO_PARAMS,
    MONO_INITIAL_6B,
    T_EXTRA
)


extra_bi_df = run_ramp_6B(
    "bistable_equilibrium_hysteresis",
    BISTABLE_PARAMS,
    BISTABLE_INITIAL_6B,
    T_EXTRA
)


extra_12800_df = pd.concat(
    [
        extra_mono_df,
        extra_bi_df
    ],
    ignore_index=True
)


extra_12800_df.to_csv(
    EXP6B_DIR
    / "ramp_trajectories_T12800.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Saved:",
    EXP6B_DIR
    / "ramp_trajectories_T12800.csv"
)

In [ ]:
# ============================================================
# 6B.16
# COMBINE ORIGINAL 6B DATA + T=12800
# ============================================================

existing_trajectory_file = (
    EXP6B_DIR
    / "all_ramp_trajectories_6B.csv"
)


if not existing_trajectory_file.exists():

    raise FileNotFoundError(
        f"Missing:\n{existing_trajectory_file}"
    )


existing_ramps_df = pd.read_csv(
    existing_trajectory_file
)


# Prevent accidental duplication if this cell is rerun
existing_ramps_df = (
    existing_ramps_df[
        existing_ramps_df[
            "ramp_duration"
        ]
        !=
        T_EXTRA
    ]
    .copy()
)


all_ramps_6B_extended_df = pd.concat(
    [
        existing_ramps_df,
        extra_12800_df
    ],
    ignore_index=True
)


all_ramps_6B_extended_df.to_csv(
    EXP6B_DIR
    / "all_ramp_trajectories_6B_plus_T12800.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Combined durations:"
)

print(
    sorted(
        all_ramps_6B_extended_df[
            "ramp_duration"
        ].unique()
    )
)

In [ ]:
# ============================================================
# 6B.17
# RECALCULATE METRICS WITH T=12800
# ============================================================

metric_records_extended = []


for (
    regime,
    T_ramp
), g in (
    all_ramps_6B_extended_df
    .groupby(
        [
            "regime",
            "ramp_duration"
        ]
    )
):

    up = (
        g[
            g["phase"]
            ==
            "up"
        ]
        .copy()
    )

    down = (
        g[
            g["phase"]
            ==
            "down"
        ]
        .copy()
    )


    speed = float(
        g[
            "ramp_speed"
        ].iloc[0]
    )


    u_up = directional_H_crossing(
        up,
        "up",
        H_CROSS_6B
    )

    u_down = directional_H_crossing(
        down,
        "down",
        H_CROSS_6B
    )


    forward_complete = bool(
        np.isfinite(
            u_up
        )
    )

    reverse_complete = bool(
        np.isfinite(
            u_down
        )
    )

    complete_cycle = bool(
        forward_complete
        and
        reverse_complete
    )


    width = (
        u_up
        -
        u_down

        if complete_cycle

        else np.nan
    )


    if regime == (
        "bistable_equilibrium_hysteresis"
    ):

        forward_reference = (
            U_EQ_FORWARD
        )

        reverse_reference = (
            U_EQ_REVERSE
        )

        width_reference = (
            EQ_WIDTH
        )

    else:

        forward_reference = np.nan
        reverse_reference = np.nan

        width_reference = 0.0


    metric_records_extended.append({

        "regime":
            regime,

        "ramp_duration":
            T_ramp,

        "ramp_speed":
            speed,

        "forward_transition_complete":
            forward_complete,

        "reverse_transition_complete":
            reverse_complete,

        "complete_cycle":
            complete_cycle,

        "u_up_H05":
            u_up,

        "u_down_H05":
            u_down,

        "H05_hysteresis_width":
            width,

        "upper_boundary_clearance":
            (
                U_MAX_6B
                -
                u_up

                if np.isfinite(
                    u_up
                )

                else np.nan
            ),

        "lower_boundary_clearance":
            (
                u_down
                -
                U_MIN_6B

                if np.isfinite(
                    u_down
                )

                else np.nan
            ),

        "equilibrium_forward_reference":
            forward_reference,

        "equilibrium_reverse_reference":
            reverse_reference,

        "equilibrium_width_reference":
            width_reference,

        "forward_switch_error":
            (
                u_up
                -
                forward_reference

                if (
                    np.isfinite(u_up)
                    and
                    np.isfinite(
                        forward_reference
                    )
                )

                else np.nan
            ),

        "reverse_switch_error":
            (
                u_down
                -
                reverse_reference

                if (
                    np.isfinite(u_down)
                    and
                    np.isfinite(
                        reverse_reference
                    )
                )

                else np.nan
            ),

        "width_error":
            (
                width
                -
                width_reference

                if np.isfinite(
                    width
                )

                else np.nan
            ),

        "H_loop_area":
            loop_area_6B(
                up,
                down,
                "H"
            ),

        "M_loop_area":
            loop_area_6B(
                up,
                down,
                "M"
            ),

        "theta_loop_area":
            loop_area_6B(
                up,
                down,
                "theta"
            )
    })


metrics_6B_extended_df = pd.DataFrame(
    metric_records_extended
)


metrics_6B_extended_df.to_csv(
    EXP6B_DIR
    / "ramp_metrics_6B_plus_T12800.csv",
    index=False,
    float_format="%.17g"
)


display(
    metrics_6B_extended_df
    .sort_values(
        [
            "regime",
            "ramp_duration"
        ]
    )
)

In [ ]:
# ============================================================
# 6B.18
# FINAL QUASI-STATIC CONVERGENCE CHECK
# ============================================================

mono_extended = (
    metrics_6B_extended_df[
        metrics_6B_extended_df[
            "regime"
        ]
        ==
        "monostable_dynamic_lag"
    ]
    .sort_values(
        "ramp_speed"
    )
)


bi_extended = (
    metrics_6B_extended_df[
        metrics_6B_extended_df[
            "regime"
        ]
        ==
        "bistable_equilibrium_hysteresis"
    ]
    .sort_values(
        "ramp_speed"
    )
)


mono_slowest = (
    mono_extended.iloc[0]
)

bi_slowest = (
    bi_extended.iloc[0]
)


bi_relative_error = (

    abs(
        bi_slowest[
            "H05_hysteresis_width"
        ]
        -
        EQ_WIDTH
    )

    /
    EQ_WIDTH
)


final_summary_df = pd.DataFrame([

    {
        "metric":
            "slowest_ramp_duration",

        "value":
            T_EXTRA
    },

    {
        "metric":
            "slowest_ramp_speed",

        "value":
            extra_speed
    },

    {
        "metric":
            "monostable_width_T12800",

        "value":
            mono_slowest[
                "H05_hysteresis_width"
            ]
    },

    {
        "metric":
            "monostable_H_loop_area_T12800",

        "value":
            mono_slowest[
                "H_loop_area"
            ]
    },

    {
        "metric":
            "bistable_forward_switch_T12800",

        "value":
            bi_slowest[
                "u_up_H05"
            ]
    },

    {
        "metric":
            "bistable_forward_fold",

        "value":
            U_EQ_FORWARD
    },

    {
        "metric":
            "bistable_reverse_switch_T12800",

        "value":
            bi_slowest[
                "u_down_H05"
            ]
    },

    {
        "metric":
            "bistable_reverse_fold",

        "value":
            U_EQ_REVERSE
    },

    {
        "metric":
            "bistable_width_T12800",

        "value":
            bi_slowest[
                "H05_hysteresis_width"
            ]
    },

    {
        "metric":
            "bistable_equilibrium_width",

        "value":
            EQ_WIDTH
    },

    {
        "metric":
            "bistable_relative_width_error_T12800",

        "value":
            bi_relative_error
    },

    {
        "metric":
            "bistable_H_loop_area_T12800",

        "value":
            bi_slowest[
                "H_loop_area"
            ]
    }
])


final_summary_df.to_csv(
    EXP6B_DIR
    / "experiment06B_T12800_summary.csv",
    index=False,
    float_format="%.17g"
)


final_flags_df = pd.DataFrame([{

    "T12800_monostable_cycle_complete":
        bool(
            mono_slowest[
                "complete_cycle"
            ]
        ),

    "T12800_bistable_cycle_complete":
        bool(
            bi_slowest[
                "complete_cycle"
            ]
        ),

    "bistable_width_within_5_percent_of_equilibrium":
        bool(
            bi_relative_error
            <
            0.05
        ),

    "bistable_width_within_3_percent_of_equilibrium":
        bool(
            bi_relative_error
            <
            0.03
        ),

    "bistable_relative_width_error":
        bi_relative_error
}])


final_flags_df.to_csv(
    EXP6B_DIR
    / "experiment06B_T12800_decision_flags.csv",
    index=False
)


display(
    final_summary_df
)

display(
    final_flags_df
)

In [ ]:
# ============================================================
# 6B.19
# FINAL WIDTH-VS-SPEED FIGURE INCLUDING T=12800
# ============================================================

fig05_data = (
    metrics_6B_extended_df[[
        "regime",
        "ramp_duration",
        "ramp_speed",
        "complete_cycle",
        "H05_hysteresis_width"
    ]]
    .copy()
)


fig05_data.to_csv(
    EXP6B_DIR
    / "fig05_final_width_vs_speed_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


for regime, g in (
    fig05_data.groupby(
        "regime"
    )
):

    valid = (
        g[
            g[
                "complete_cycle"
            ]
        ]
        .sort_values(
            "ramp_speed"
        )
    )


    ax.plot(
        valid[
            "ramp_speed"
        ],
        valid[
            "H05_hysteresis_width"
        ],
        marker="o",
        linewidth=2.1,
        label=regime
    )


ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.1,
    label="Monostable equilibrium limit"
)


ax.axhline(
    EQ_WIDTH,
    linestyle=":",
    linewidth=1.3,
    label="Bistable equilibrium fold separation"
)


ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "$H=0.5$ switching width, $\\Delta u$"
)

ax.set_title(
    "Dynamic lag versus equilibrium hysteresis: quasi-static limit"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP6B_DIR
    / "fig05_final_width_vs_speed.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 6B.20
# FINAL BISTABLE SWITCH CONVERGENCE FIGURE
# ============================================================

fig06_data = (
    bi_extended[[
        "ramp_duration",
        "ramp_speed",
        "complete_cycle",
        "u_up_H05",
        "u_down_H05",
        "forward_switch_error",
        "reverse_switch_error"
    ]]
    .copy()
)


fig06_data.to_csv(
    EXP6B_DIR
    / "fig06_final_bistable_switch_convergence_data.csv",
    index=False,
    float_format="%.17g"
)


valid = (
    fig06_data[
        fig06_data[
            "complete_cycle"
        ]
    ]
    .sort_values(
        "ramp_speed"
    )
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.5)
)


ax.plot(
    valid[
        "ramp_speed"
    ],
    valid[
        "u_up_H05"
    ],
    marker="o",
    linewidth=2.1,
    label="Forward dynamic switch"
)


ax.plot(
    valid[
        "ramp_speed"
    ],
    valid[
        "u_down_H05"
    ],
    marker="o",
    linewidth=2.1,
    label="Reverse dynamic switch"
)


ax.axhline(
    U_EQ_FORWARD,
    linestyle="--",
    linewidth=1.2,
    label="Forward equilibrium fold"
)


ax.axhline(
    U_EQ_REVERSE,
    linestyle=":",
    linewidth=1.2,
    label="Reverse equilibrium fold"
)


ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "Ramp speed, $|du/dt|$"
)

ax.set_ylabel(
    "Switching forcing, $u$"
)

ax.set_title(
    "Dynamic switching points approach equilibrium folds"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    EXP6B_DIR
    / "fig06_final_bistable_switch_convergence.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

# Robustness analysis R1

In [ ]:
# ============================================================
# ROBUSTNESS R1
# alpha-epsilon memory map
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

R1_DIR = (
    BASE_DIR
    / "Robustness_R1_alpha_epsilon_memory_map"
)

R1_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Robustness R1 folder:")
print(R1_DIR)

In [ ]:
# ============================================================
# IMPORTS + EXPERIMENT 04 PROVENANCE
# ============================================================

import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar


# ------------------------------------------------------------
# Required Experiment 4 files
# ------------------------------------------------------------

PARAM_FILE = (
    EXP4_DIR
    / "selected_monostable_parameters.json"
)

DESIGN_FILE = (
    EXP4_DIR
    / "experiment04_design.json"
)

PROBE_FILE = (
    EXP4_DIR
    / "probe_definition.csv"
)


for f in [
    PARAM_FILE,
    DESIGN_FILE,
    PROBE_FILE
]:
    if not f.exists():
        raise FileNotFoundError(
            f"Missing required Experiment 4 file:\n{f}"
        )


with open(
    PARAM_FILE,
    "r"
) as f:
    exp4_params = json.load(f)


with open(
    DESIGN_FILE,
    "r"
) as f:
    exp4_design = json.load(f)


probe_exp4_df = pd.read_csv(
    PROBE_FILE
)


# ------------------------------------------------------------
# Structural parameters stay fixed.
#
# alpha and epsilon will be varied.
# ------------------------------------------------------------

BASE_PARAMS = {

    "n":
        float(
            exp4_params["n"]
        ),

    "q":
        float(
            exp4_params["q"]
        ),

    "K_M":
        float(
            exp4_params["K_M"]
        ),

    "theta0":
        float(
            exp4_params["theta0"]
        ),

    "r":
        float(
            exp4_params["r"]
        )
}


U_LOW = float(
    exp4_design[
        "u_low_history"
    ]
)

U_HIGH = float(
    exp4_design[
        "u_high_history"
    ]
)

S_STAR = float(
    probe_exp4_df[
        "S_star"
    ].iloc[0]
)


# Same durations used in Experiment 4
T_BASELINE = 150.0
T_HISTORY = 200.0


print("Structural parameters:")
print(BASE_PARAMS)

print("\nExperiment 4 protocol:")
print("U_LOW =", U_LOW)
print("U_HIGH =", U_HIGH)
print("S_STAR =", S_STAR)
print("T_BASELINE =", T_BASELINE)
print("T_HISTORY =", T_HISTORY)

In [ ]:
# ============================================================
# ROBUSTNESS GRID
# ============================================================

ALPHA_VALUES = np.linspace(
    0.0,
    3.0,
    31
)


EPSILON_VALUES = np.logspace(
    -2.0,
    0.0,
    17
)


# ------------------------------------------------------------
# Autonomous forcing interval used to confirm monostability
# ------------------------------------------------------------

U_MONO_MIN = 0.02
U_MONO_MAX = 1.50

U_SCREEN = np.linspace(
    U_MONO_MIN,
    U_MONO_MAX,
    61
)


# ------------------------------------------------------------
# Recovery schedule
#
# One recovery integration is run per parameter pair and
# sampled at these times.
# ------------------------------------------------------------

RECOVERY_TIMES = np.array([
    0.0,
    0.25,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0,
    40.0,
    80.0,
    160.0,
    320.0,
    640.0,
    1280.0
])


# ------------------------------------------------------------
# Equilibrium/root numerical settings
# ------------------------------------------------------------

X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 12001

STABILITY_TOL = 1e-7
FOLD_ROOT_TOL = 1e-12


# ------------------------------------------------------------
# ODE tolerances
# ------------------------------------------------------------

RTOL = 1e-9
ATOL = 1e-11


CONFIG_R1 = {

    "alpha_values":
        ALPHA_VALUES.tolist(),

    "epsilon_values":
        EPSILON_VALUES.tolist(),

    "u_monostability_min":
        U_MONO_MIN,

    "u_monostability_max":
        U_MONO_MAX,

    "n_u_screen":
        len(U_SCREEN),

    "recovery_times":
        RECOVERY_TIMES.tolist(),

    "u_low":
        U_LOW,

    "u_high":
        U_HIGH,

    "S_star":
        S_STAR,

    "t_baseline":
        T_BASELINE,

    "t_history":
        T_HISTORY,

    "structural_parameters":
        BASE_PARAMS,

    "rtol":
        RTOL,

    "atol":
        ATOL
}


with open(
    R1_DIR
    / "R1_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG_R1,
        f,
        indent=4
    )


parameter_rows = []

for key, value in BASE_PARAMS.items():

    parameter_rows.append({
        "parameter":
            key,

        "value":
            value
    })


parameter_rows.extend([

    {
        "parameter":
            "u_low",

        "value":
            U_LOW
    },

    {
        "parameter":
            "u_high",

        "value":
            U_HIGH
    },

    {
        "parameter":
            "S_star",

        "value":
            S_STAR
    },

    {
        "parameter":
            "t_baseline",

        "value":
            T_BASELINE
    },

    {
        "parameter":
            "t_history",

        "value":
            T_HISTORY
    }
])


pd.DataFrame(
    parameter_rows
).to_csv(
    R1_DIR
    / "R1_parameters.csv",
    index=False,
    float_format="%.17g"
)


pd.DataFrame({
    "alpha":
        ALPHA_VALUES
}).to_csv(
    R1_DIR
    / "alpha_grid.csv",
    index=False,
    float_format="%.17g"
)


pd.DataFrame({
    "epsilon":
        EPSILON_VALUES
}).to_csv(
    R1_DIR
    / "epsilon_grid.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


def F(
    S,
    theta,
    n
):

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(
    S,
    theta,
    n
):

    return (
        n
        *
        np.power(theta, n)
        *
        np.power(S, n - 1)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )**2
    )


def dF_dtheta(
    S,
    theta,
    n
):

    return (
        -n
        *
        np.power(S, n)
        *
        np.power(theta, n - 1)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )**2
    )


# ------------------------------------------------------------
# IMPORTANT
# Same provisional microbial-response function used throughout
# Experiments 1-6.
# ------------------------------------------------------------

def G(
    H,
    K_M,
    q
):

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(
    H,
    K_M,
    q
):

    return (
        q
        *
        np.power(K_M, q)
        *
        np.power(H, q - 1)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )**2
    )


def theta_equilibrium(
    H,
    alpha,
    p
):

    return (
        p["theta0"]
        *
        np.exp(
            -alpha * H
        )
    )


def M_equilibrium(
    H,
    p
):

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def u_equilibrium(
    H,
    alpha,
    p
):

    theta = theta_equilibrium(
        H,
        alpha,
        p
    )

    return (
        theta
        *
        np.power(
            H
            /
            (1.0 - H),
            1.0 / p["n"]
        )
        -
        M_equilibrium(
            H,
            p
        )
    )


def du_dH(
    H,
    alpha,
    p
):

    power_term = np.power(
        H
        /
        (1.0 - H),
        1.0 / p["n"]
    )

    derivative_log_term = (
        1.0
        /
        (
            p["n"]
            *
            H
            *
            (1.0 - H)
        )
    )

    theta = theta_equilibrium(
        H,
        alpha,
        p
    )

    return (
        theta
        *
        power_term
        *
        (
            derivative_log_term
            -
            alpha
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )

In [ ]:
# ============================================================
# ADAPTIVE MODEL
# ============================================================

def rhs_M1(
    t,
    y,
    u,
    alpha,
    epsilon,
    p
):

    H, M, theta = y

    S = (
        M + u
    )


    dH = (
        -H
        +
        F(
            S,
            theta,
            p["n"]
        )
    )


    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )


    dtheta = (
        epsilon
        *
        theta
        *
        (
            np.log(
                p["theta0"]
                /
                theta
            )
            -
            alpha * H
        )
    )


    return np.array([
        dH,
        dM,
        dtheta
    ])


def jacobian_M1(
    H,
    M,
    theta,
    u,
    alpha,
    epsilon,
    p
):

    S = (
        M + u
    )


    FS = dF_dS(
        S,
        theta,
        p["n"]
    )


    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )


    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )


    adaptive_bracket = (
        np.log(
            p["theta0"]
            /
            theta
        )
        -
        alpha * H
    )


    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            p["r"] * GH,
            -p["r"],
            0.0
        ],

        [
            -epsilon
            *
            alpha
            *
            theta,

            0.0,

            epsilon
            *
            (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# EQUILIBRIUM GEOMETRY
# ============================================================

X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)

H_SCAN = sigmoid(
    X_SCAN
)


def equilibrium_roots(
    u_target,
    alpha,
    p
):

    values = (
        u_equilibrium(
            H_SCAN,
            alpha,
            p
        )
        -
        u_target
    )


    indices = np.where(
        values[:-1]
        *
        values[1:]
        <
        0.0
    )[0]


    roots = []


    for i in indices:

        result = root_scalar(

            lambda x:
                (
                    u_equilibrium(
                        sigmoid(x),
                        alpha,
                        p
                    )
                    -
                    u_target
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )


        roots.append(
            sigmoid(
                result.root
            )
        )


    return np.asarray(
        roots
    )


def detect_folds(
    alpha,
    p
):

    derivative = du_dH(
        H_SCAN,
        alpha,
        p
    )


    indices = np.where(
        derivative[:-1]
        *
        derivative[1:]
        <
        0.0
    )[0]


    folds = []


    for i in indices:

        result = root_scalar(

            lambda x:
                du_dH(
                    sigmoid(x),
                    alpha,
                    p
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=FOLD_ROOT_TOL
        )


        Hf = sigmoid(
            result.root
        )


        folds.append({

            "H_fold":
                Hf,

            "u_fold":
                u_equilibrium(
                    Hf,
                    alpha,
                    p
                )
        })


    return folds

In [ ]:
# ============================================================
# STRUCTURAL MONOSTABILITY SCREEN
# ============================================================

structural_records = []

# Cache equilibrium roots for later stability analysis
equilibrium_cache = {}


for alpha in ALPHA_VALUES:

    print(
        f"Structural screen alpha={alpha:.3f}"
    )


    folds = detect_folds(
        alpha,
        BASE_PARAMS
    )


    fold_us_in_interval = [

        float(
            f["u_fold"]
        )

        for f in folds

        if (
            U_MONO_MIN
            <
            f["u_fold"]
            <
            U_MONO_MAX
        )
    ]


    root_counts = []


    for u in U_SCREEN:

        roots = equilibrium_roots(
            u,
            alpha,
            BASE_PARAMS
        )


        equilibrium_cache[
            (
                float(alpha),
                float(u)
            )
        ] = roots


        root_counts.append(
            len(roots)
        )


    root_counts = np.asarray(
        root_counts
    )


    exactly_one_everywhere = bool(
        np.all(
            root_counts == 1
        )
    )


    no_fold_in_interval = bool(
        len(
            fold_us_in_interval
        )
        ==
        0
    )


    structurally_monostable = bool(
        exactly_one_everywhere
        and
        no_fold_in_interval
    )


    structural_records.append({

        "alpha":
            alpha,

        "min_equilibrium_count":
            int(
                np.min(
                    root_counts
                )
            ),

        "max_equilibrium_count":
            int(
                np.max(
                    root_counts
                )
            ),

        "number_of_folds_total":
            len(folds),

        "number_of_folds_in_u_interval":
            len(
                fold_us_in_interval
            ),

        "fold_u_values_in_interval":
            ";".join(
                f"{x:.17g}"
                for x
                in fold_us_in_interval
            ),

        "exactly_one_equilibrium_at_all_sampled_u":
            exactly_one_everywhere,

        "no_fold_in_screen_interval":
            no_fold_in_interval,

        "structurally_monostable":
            structurally_monostable
    })


structural_df = pd.DataFrame(
    structural_records
)


structural_df.to_csv(
    R1_DIR
    / "structural_monostability_by_alpha.csv",
    index=False,
    float_format="%.17g"
)


display(
    structural_df
)

In [ ]:
# ============================================================
# SIMULATION HELPERS
# ============================================================

def integrate_constant_u(
    y0,
    u,
    duration,
    alpha,
    epsilon,
    p,
    t_eval=None
):

    if duration == 0.0:

        class DummySolution:
            pass

        sol = DummySolution()

        sol.t = np.array([
            0.0
        ])

        sol.y = np.asarray(
            y0,
            dtype=float
        ).reshape(
            3,
            1
        )

        sol.success = True

        return sol


    if t_eval is None:

        t_eval = np.array([
            duration
        ])


    solution = solve_ivp(

        lambda t, y:
            rhs_M1(
                t,
                y,
                u,
                alpha,
                epsilon,
                p
            ),

        (
            0.0,
            duration
        ),

        y0,

        t_eval=
            t_eval,

        method=
            "LSODA",

        rtol=
            RTOL,

        atol=
            ATOL
    )


    if not solution.success:

        raise RuntimeError(
            solution.message
        )


    return solution


def apparent_half_life(
    times,
    values
):

    times = np.asarray(
        times,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )


    initial = values[0]


    if (
        not np.isfinite(initial)
        or
        initial <= 1e-12
    ):

        return np.nan


    target = (
        0.5
        *
        initial
    )


    for i in range(
        1,
        len(times)
    ):

        y1 = values[i - 1]
        y2 = values[i]


        if (
            y1 > target
            and
            y2 <= target
        ):

            t1 = times[i - 1]
            t2 = times[i]


            if y2 == y1:

                return float(
                    t2
                )


            fraction = (
                target
                -
                y1
            ) / (
                y2
                -
                y1
            )


            return float(
                t1
                +
                fraction
                *
                (
                    t2
                    -
                    t1
                )
            )


    return np.nan


def nonincreasing(
    values,
    tol=1e-9
):

    values = np.asarray(
        values,
        dtype=float
    )

    return bool(
        np.all(
            np.diff(values)
            <=
            tol
        )
    )

In [ ]:
# ============================================================
# LOCAL-STABILITY SCREEN
# ============================================================

stability_records = []


structural_lookup = (
    structural_df
    .set_index(
        "alpha"
    )[
        "structurally_monostable"
    ]
    .to_dict()
)


for alpha in ALPHA_VALUES:

    structural_mono = bool(
        structural_lookup[
            alpha
        ]
    )


    for epsilon in EPSILON_VALUES:

        max_real_part = np.nan

        any_complex = False
        oscillatory_instability = False


        if structural_mono:

            local_maxima = []


            for u in U_SCREEN:

                roots = (
                    equilibrium_cache[
                        (
                            float(alpha),
                            float(u)
                        )
                    ]
                )


                if len(roots) != 1:

                    raise RuntimeError(
                        "Cache inconsistency."
                    )


                H = float(
                    roots[0]
                )


                M = M_equilibrium(
                    H,
                    BASE_PARAMS
                )


                theta = theta_equilibrium(
                    H,
                    alpha,
                    BASE_PARAMS
                )


                eigvals = np.linalg.eigvals(

                    jacobian_M1(
                        H,
                        M,
                        theta,
                        u,
                        alpha,
                        epsilon,
                        BASE_PARAMS
                    )
                )


                local_maxima.append(
                    np.max(
                        eigvals.real
                    )
                )


                if np.any(
                    np.abs(
                        eigvals.imag
                    )
                    >
                    1e-8
                ):
                    any_complex = True


                if np.any(
                    (
                        eigvals.real
                        >
                        STABILITY_TOL
                    )
                    &
                    (
                        np.abs(
                            eigvals.imag
                        )
                        >
                        1e-8
                    )
                ):
                    oscillatory_instability = True


            max_real_part = float(
                np.max(
                    local_maxima
                )
            )


        stable_monostable = bool(

            structural_mono

            and

            np.isfinite(
                max_real_part
            )

            and

            max_real_part
            <
            -STABILITY_TOL
        )


        stability_records.append({

            "alpha":
                alpha,

            "epsilon":
                epsilon,

            "structurally_monostable":
                structural_mono,

            "max_real_eigenvalue_across_u":
                max_real_part,

            "stable_monostable":
                stable_monostable,

            "complex_eigenvalues_encountered":
                any_complex,

            "oscillatory_instability_detected":
                oscillatory_instability
        })


stability_df = pd.DataFrame(
    stability_records
)


stability_df.to_csv(
    R1_DIR
    / "alpha_epsilon_stability_screen.csv",
    index=False,
    float_format="%.17g"
)


display(
    stability_df.head()
)

In [ ]:
# ============================================================
# MAIN alpha-epsilon MEMORY SWEEP
# ============================================================

RESULTS_CHECKPOINT = (
    R1_DIR
    / "R1_grid_results_checkpoint.csv"
)

RECOVERY_CHECKPOINT = (
    R1_DIR
    / "R1_recovery_curves_checkpoint.csv"
)


# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

if RESULTS_CHECKPOINT.exists():

    results_existing = pd.read_csv(
        RESULTS_CHECKPOINT
    )

    completed_pairs = set(
        results_existing[
            "pair_id"
        ].astype(str)
    )

    results_records = (
        results_existing
        .to_dict(
            "records"
        )
    )

else:

    completed_pairs = set()
    results_records = []


if RECOVERY_CHECKPOINT.exists():

    recovery_existing = pd.read_csv(
        RECOVERY_CHECKPOINT
    )

    recovery_records = (
        recovery_existing
        .to_dict(
            "records"
        )
    )

else:

    recovery_records = []


# ------------------------------------------------------------
# Common Experiment 4 initial state
# ------------------------------------------------------------

H_INITIAL = 0.05

M_INITIAL = float(
    G(
        H_INITIAL,
        BASE_PARAMS["K_M"],
        BASE_PARAMS["q"]
    )
)

THETA_INITIAL = (
    BASE_PARAMS[
        "theta0"
    ]
)


Y_INITIAL = np.array([
    H_INITIAL,
    M_INITIAL,
    THETA_INITIAL
])


# ------------------------------------------------------------
# Sweep
# ------------------------------------------------------------

for alpha_index, alpha in enumerate(
    ALPHA_VALUES
):

    print(
        "\n"
        "========================================"
    )

    print(
        f"alpha {alpha_index + 1}/{len(ALPHA_VALUES)} "
        f"= {alpha:.3f}"
    )

    print(
        "========================================"
    )


    # Exact low/high equilibrium roots are independent
    # of epsilon.
    roots_low = equilibrium_roots(
        U_LOW,
        alpha,
        BASE_PARAMS
    )

    roots_high = equilibrium_roots(
        U_HIGH,
        alpha,
        BASE_PARAMS
    )


    low_unique = (
        len(roots_low)
        ==
        1
    )

    high_unique = (
        len(roots_high)
        ==
        1
    )


    if low_unique:

        H_low_eq = float(
            roots_low[0]
        )

        theta_low_eq = float(
            theta_equilibrium(
                H_low_eq,
                alpha,
                BASE_PARAMS
            )
        )

        R_low_eq = float(
            F(
                S_STAR,
                theta_low_eq,
                BASE_PARAMS["n"]
            )
        )

    else:

        H_low_eq = np.nan
        theta_low_eq = np.nan
        R_low_eq = np.nan


    if high_unique:

        H_high_eq = float(
            roots_high[0]
        )

        theta_high_eq = float(
            theta_equilibrium(
                H_high_eq,
                alpha,
                BASE_PARAMS
            )
        )

        R_high_eq = float(
            F(
                S_STAR,
                theta_high_eq,
                BASE_PARAMS["n"]
            )
        )

    else:

        H_high_eq = np.nan
        theta_high_eq = np.nan
        R_high_eq = np.nan


    if (
        np.isfinite(
            R_low_eq
        )
        and
        np.isfinite(
            R_high_eq
        )
    ):

        equilibrium_response_shift_potential = (
            R_high_eq
            -
            R_low_eq
        )

    else:

        equilibrium_response_shift_potential = np.nan


    for epsilon in EPSILON_VALUES:

        pair_id = (
            f"alpha_{alpha:.6f}"
            f"_epsilon_{epsilon:.10f}"
        )


        if pair_id in completed_pairs:

            print(
                f"Skipping completed {pair_id}"
            )

            continue


        print(
            f"  epsilon={epsilon:.5f}"
        )


        # ====================================================
        # 1. Common low-forcing baseline
        # ====================================================

        baseline_solution = integrate_constant_u(

            Y_INITIAL,

            U_LOW,

            T_BASELINE,

            alpha,

            epsilon,

            BASE_PARAMS
        )


        baseline_state = (
            baseline_solution.y[
                :,
                -1
            ]
        )


        # ====================================================
        # 2. History A:
        #    continued low forcing
        # ====================================================

        history_A_solution = integrate_constant_u(

            baseline_state,

            U_LOW,

            T_HISTORY,

            alpha,

            epsilon,

            BASE_PARAMS
        )


        state_A = (
            history_A_solution.y[
                :,
                -1
            ]
        )


        # ====================================================
        # 3. History B:
        #    prolonged high forcing
        # ====================================================

        history_B_solution = integrate_constant_u(

            baseline_state,

            U_HIGH,

            T_HISTORY,

            alpha,

            epsilon,

            BASE_PARAMS
        )


        state_B = (
            history_B_solution.y[
                :,
                -1
            ]
        )


        H_A, M_A, theta_A = state_A
        H_B, M_B, theta_B = state_B


        # ====================================================
        # Same physical signal response mapping
        # ====================================================

        R_A = float(
            F(
                S_STAR,
                theta_A,
                BASE_PARAMS["n"]
            )
        )


        R_B = float(
            F(
                S_STAR,
                theta_B,
                BASE_PARAMS["n"]
            )
        )


        delta_R_protocol = (
            R_B
            -
            R_A
        )


        # Relative to exact low equilibrium
        delta_R_B_vs_low_eq = (

            R_B
            -
            R_low_eq

            if np.isfinite(
                R_low_eq
            )

            else np.nan
        )


        # ====================================================
        # 4. Recovery from conditioned History B
        # ====================================================

        recovery_solution = integrate_constant_u(

            state_B,

            U_LOW,

            float(
                RECOVERY_TIMES[-1]
            ),

            alpha,

            epsilon,

            BASE_PARAMS,

            t_eval=
                RECOVERY_TIMES
        )


        H_rec = (
            recovery_solution.y[0]
        )

        M_rec = (
            recovery_solution.y[1]
        )

        theta_rec = (
            recovery_solution.y[2]
        )


        R_rec = F(
            S_STAR,
            theta_rec,
            BASE_PARAMS["n"]
        )


        if np.isfinite(
            R_low_eq
        ):

            memory_excess = (
                R_rec
                -
                R_low_eq
            )

        else:

            memory_excess = (
                np.full_like(
                    R_rec,
                    np.nan
                )
            )


        half_life = apparent_half_life(
            RECOVERY_TIMES,
            memory_excess
        )


        recovery_monotonic = (

            nonincreasing(
                memory_excess
            )

            if np.all(
                np.isfinite(
                    memory_excess
                )
            )

            else False
        )


        # ----------------------------------------------------
        # Fixed-time persistence values
        # ----------------------------------------------------

        def value_at_time(
            time_target
        ):

            index = int(
                np.where(
                    np.isclose(
                        RECOVERY_TIMES,
                        time_target
                    )
                )[0][0]
            )

            return float(
                memory_excess[
                    index
                ]
            )


        memory_T20 = (
            value_at_time(
                20.0
            )
        )

        memory_T80 = (
            value_at_time(
                80.0
            )
        )

        memory_T320 = (
            value_at_time(
                320.0
            )
        )


        initial_memory = float(
            memory_excess[0]
        )


        if (
            np.isfinite(
                initial_memory
            )
            and
            initial_memory
            >
            1e-12
        ):

            fraction_remaining_T20 = (
                memory_T20
                /
                initial_memory
            )

            fraction_remaining_T80 = (
                memory_T80
                /
                initial_memory
            )

            fraction_remaining_T320 = (
                memory_T320
                /
                initial_memory
            )

        else:

            fraction_remaining_T20 = np.nan
            fraction_remaining_T80 = np.nan
            fraction_remaining_T320 = np.nan


        # ----------------------------------------------------
        # Conditioning completion relative to equilibrium
        # potential.
        # ----------------------------------------------------

        if (
            np.isfinite(
                equilibrium_response_shift_potential
            )
            and
            abs(
                equilibrium_response_shift_potential
            )
            >
            1e-12
        ):

            conditioning_completion_fraction = (

                delta_R_B_vs_low_eq

                /
                equilibrium_response_shift_potential
            )

        else:

            conditioning_completion_fraction = np.nan


        # ====================================================
        # Residual diagnostics
        # ====================================================

        baseline_residual = float(
            np.linalg.norm(
                rhs_M1(
                    0.0,
                    baseline_state,
                    U_LOW,
                    alpha,
                    epsilon,
                    BASE_PARAMS
                )
            )
        )


        history_A_residual = float(
            np.linalg.norm(
                rhs_M1(
                    0.0,
                    state_A,
                    U_LOW,
                    alpha,
                    epsilon,
                    BASE_PARAMS
                )
            )
        )


        history_B_residual = float(
            np.linalg.norm(
                rhs_M1(
                    0.0,
                    state_B,
                    U_HIGH,
                    alpha,
                    epsilon,
                    BASE_PARAMS
                )
            )
        )


        # ====================================================
        # Stability classification from precomputed screen
        # ====================================================

        stability_row = stability_df[
            (
                np.isclose(
                    stability_df["alpha"],
                    alpha
                )
            )
            &
            (
                np.isclose(
                    stability_df["epsilon"],
                    epsilon
                )
            )
        ].iloc[0]


        stable_monostable = bool(
            stability_row[
                "stable_monostable"
            ]
        )


        structurally_monostable = bool(
            stability_row[
                "structurally_monostable"
            ]
        )


        # ====================================================
        # Main result record
        # ====================================================

        results_records.append({

            "pair_id":
                pair_id,

            "alpha":
                alpha,

            "epsilon":
                epsilon,

            "structurally_monostable":
                structurally_monostable,

            "stable_monostable":
                stable_monostable,

            "max_real_eigenvalue_across_u":
                stability_row[
                    "max_real_eigenvalue_across_u"
                ],

            "oscillatory_instability_detected":
                stability_row[
                    "oscillatory_instability_detected"
                ],

            "H_A":
                H_A,

            "M_A":
                M_A,

            "theta_A":
                theta_A,

            "H_B":
                H_B,

            "M_B":
                M_B,

            "theta_B":
                theta_B,

            "theta_difference_B_minus_A":
                theta_B
                -
                theta_A,

            "R_A_same_signal":
                R_A,

            "R_B_same_signal":
                R_B,

            "delta_R_protocol":
                delta_R_protocol,

            "H_low_equilibrium":
                H_low_eq,

            "theta_low_equilibrium":
                theta_low_eq,

            "R_low_equilibrium":
                R_low_eq,

            "H_high_equilibrium":
                H_high_eq,

            "theta_high_equilibrium":
                theta_high_eq,

            "R_high_equilibrium":
                R_high_eq,

            "equilibrium_response_shift_potential":
                equilibrium_response_shift_potential,

            "delta_R_B_vs_low_equilibrium":
                delta_R_B_vs_low_eq,

            "conditioning_completion_fraction":
                conditioning_completion_fraction,

            "initial_memory_excess":
                initial_memory,

            "memory_excess_T20":
                memory_T20,

            "memory_excess_T80":
                memory_T80,

            "memory_excess_T320":
                memory_T320,

            "fraction_memory_remaining_T20":
                fraction_remaining_T20,

            "fraction_memory_remaining_T80":
                fraction_remaining_T80,

            "fraction_memory_remaining_T320":
                fraction_remaining_T320,

            "apparent_memory_half_life":
                half_life,

            "epsilon_times_half_life":
                (
                    epsilon
                    *
                    half_life

                    if np.isfinite(
                        half_life
                    )

                    else np.nan
                ),

            "memory_decay_monotonic":
                recovery_monotonic,

            "baseline_residual":
                baseline_residual,

            "history_A_residual":
                history_A_residual,

            "history_B_residual":
                history_B_residual
        })


        # ====================================================
        # Full recovery-curve record
        # ====================================================

        for (
            t_rec,
            H_t,
            M_t,
            theta_t,
            R_t,
            memory_t
        ) in zip(

            RECOVERY_TIMES,
            H_rec,
            M_rec,
            theta_rec,
            R_rec,
            memory_excess
        ):

            recovery_records.append({

                "pair_id":
                    pair_id,

                "alpha":
                    alpha,

                "epsilon":
                    epsilon,

                "recovery_time":
                    t_rec,

                "H":
                    H_t,

                "M":
                    M_t,

                "theta":
                    theta_t,

                "R_same_signal":
                    R_t,

                "memory_excess_vs_low_equilibrium":
                    memory_t
            })


        completed_pairs.add(
            pair_id
        )


    # --------------------------------------------------------
    # Checkpoint after every alpha row
    # --------------------------------------------------------

    results_checkpoint_df = (
        pd.DataFrame(
            results_records
        )
        .drop_duplicates(
            subset=[
                "pair_id"
            ],
            keep="last"
        )
        .sort_values(
            [
                "alpha",
                "epsilon"
            ]
        )
    )


    recovery_checkpoint_df = (
        pd.DataFrame(
            recovery_records
        )
        .drop_duplicates(
            subset=[
                "pair_id",
                "recovery_time"
            ],
            keep="last"
        )
        .sort_values(
            [
                "alpha",
                "epsilon",
                "recovery_time"
            ]
        )
    )


    results_checkpoint_df.to_csv(
        RESULTS_CHECKPOINT,
        index=False,
        float_format="%.17g"
    )


    recovery_checkpoint_df.to_csv(
        RECOVERY_CHECKPOINT,
        index=False,
        float_format="%.17g"
    )


    print(
        f"Checkpoint saved after alpha={alpha:.3f}"
    )


print(
    "\nR1 sweep complete."
)

In [ ]:
# ============================================================
# FINALIZE RAW R1 DATA
# ============================================================

grid_results_df = pd.read_csv(
    RESULTS_CHECKPOINT
)


recovery_curves_df = pd.read_csv(
    RECOVERY_CHECKPOINT
)


grid_results_df = (
    grid_results_df
    .sort_values(
        [
            "alpha",
            "epsilon"
        ]
    )
    .reset_index(
        drop=True
    )
)


recovery_curves_df = (
    recovery_curves_df
    .sort_values(
        [
            "alpha",
            "epsilon",
            "recovery_time"
        ]
    )
    .reset_index(
        drop=True
    )
)


grid_results_df.to_csv(
    R1_DIR
    / "R1_alpha_epsilon_grid_results.csv",
    index=False,
    float_format="%.17g"
)


recovery_curves_df.to_csv(
    R1_DIR
    / "R1_all_recovery_curves.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Grid rows:",
    len(grid_results_df)
)

print(
    "Recovery rows:",
    len(recovery_curves_df)
)

In [ ]:
# ============================================================
# INTERNAL VALIDATION
# ============================================================

alpha_zero_df = (
    grid_results_df[
        np.isclose(
            grid_results_df[
                "alpha"
            ],
            0.0
        )
    ]
)


MAX_ABS_DELTA_R_ALPHA0 = float(
    np.nanmax(
        np.abs(
            alpha_zero_df[
                "delta_R_protocol"
            ]
        )
    )
)


MAX_ABS_THETA_DIFF_ALPHA0 = float(
    np.nanmax(
        np.abs(
            alpha_zero_df[
                "theta_difference_B_minus_A"
            ]
        )
    )
)


NUMBER_STABLE_MONOSTABLE = int(
    grid_results_df[
        "stable_monostable"
    ].sum()
)


NUMBER_OSCILLATORY_INSTABILITY = int(
    grid_results_df[
        "oscillatory_instability_detected"
    ].sum()
)


NUMBER_HALF_LIFE_RESOLVED = int(
    grid_results_df[
        "apparent_memory_half_life"
    ].notna().sum()
)


validation_df = pd.DataFrame([

    {
        "metric":
            "maximum_absolute_delta_R_at_alpha_0",

        "value":
            MAX_ABS_DELTA_R_ALPHA0
    },

    {
        "metric":
            "maximum_absolute_theta_difference_at_alpha_0",

        "value":
            MAX_ABS_THETA_DIFF_ALPHA0
    },

    {
        "metric":
            "number_of_parameter_pairs",

        "value":
            len(
                grid_results_df
            )
    },

    {
        "metric":
            "number_of_stable_monostable_pairs",

        "value":
            NUMBER_STABLE_MONOSTABLE
    },

    {
        "metric":
            "number_of_oscillatory_instability_pairs",

        "value":
            NUMBER_OSCILLATORY_INSTABILITY
    },

    {
        "metric":
            "number_of_resolved_memory_half_lives",

        "value":
            NUMBER_HALF_LIFE_RESOLVED
    }
])


validation_df.to_csv(
    R1_DIR
    / "R1_validation_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    validation_df
)

In [ ]:
# ============================================================
# PRIMARY ROBUSTNESS SUBSET
# ============================================================

primary_df = (
    grid_results_df[
        grid_results_df[
            "stable_monostable"
        ]
    ]
    .copy()
)


primary_df.to_csv(
    R1_DIR
    / "R1_stable_monostable_subset.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Stable monostable parameter pairs:",
    len(primary_df)
)

In [ ]:
# ============================================================
# FIGURE R1.1 SOURCE DATA
# ============================================================

fig01_data = (
    grid_results_df[[
        "alpha",
        "epsilon",
        "stable_monostable",
        "delta_R_protocol"
    ]]
    .copy()
)


fig01_data.to_csv(
    R1_DIR
    / "figR1_01_same_signal_memory_map_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# MATRIX
# ============================================================

pivot = (
    primary_df
    .pivot(
        index="epsilon",
        columns="alpha",
        values="delta_R_protocol"
    )
    .sort_index(
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(10.0, 6.8)
)


image = ax.imshow(

    pivot.values,

    origin="lower",

    aspect="auto",

    extent=[
        pivot.columns.min(),
        pivot.columns.max(),
        np.log10(
            pivot.index.min()
        ),
        np.log10(
            pivot.index.max()
        )
    ]
)


cbar = fig.colorbar(
    image,
    ax=ax
)

cbar.set_label(
    "Same-signal response difference, $\\Delta R$"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$"
)

ax.set_ylabel(
    "$\\log_{10}(\\varepsilon)$"
)

ax.set_title(
    "Robustness of history-dependent response mapping"
)


fig.tight_layout()


fig.savefig(
    R1_DIR
    / "figR1_01_same_signal_memory_map.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R1.2 SOURCE DATA
# ============================================================

fig02_data = (
    grid_results_df[[
        "alpha",
        "epsilon",
        "stable_monostable",
        "apparent_memory_half_life"
    ]]
    .copy()
)


fig02_data.to_csv(
    R1_DIR
    / "figR1_02_memory_half_life_map_data.csv",
    index=False,
    float_format="%.17g"
)


half_life_primary = (
    primary_df[
        primary_df[
            "apparent_memory_half_life"
        ].notna()
    ]
)


pivot_half = (
    half_life_primary
    .pivot(
        index="epsilon",
        columns="alpha",
        values="apparent_memory_half_life"
    )
    .sort_index()
)


fig, ax = plt.subplots(
    figsize=(10.0, 6.8)
)


image = ax.imshow(

    pivot_half.values,

    origin="lower",

    aspect="auto",

    extent=[
        pivot_half.columns.min(),
        pivot_half.columns.max(),
        np.log10(
            pivot_half.index.min()
        ),
        np.log10(
            pivot_half.index.max()
        )
    ]
)


cbar = fig.colorbar(
    image,
    ax=ax
)

cbar.set_label(
    "Apparent memory half-life"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$"
)

ax.set_ylabel(
    "$\\log_{10}(\\varepsilon)$"
)

ax.set_title(
    "Persistence timescale of same-signal response memory"
)


fig.tight_layout()


fig.savefig(
    R1_DIR
    / "figR1_02_memory_half_life_map.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R1.3 SOURCE DATA
# ============================================================

fig03_data = (
    grid_results_df[[
        "alpha",
        "epsilon",
        "stable_monostable",
        "fraction_memory_remaining_T20"
    ]]
    .copy()
)


fig03_data.to_csv(
    R1_DIR
    / "figR1_03_memory_fraction_T20_data.csv",
    index=False,
    float_format="%.17g"
)


persistence_df = (
    primary_df[
        primary_df[
            "fraction_memory_remaining_T20"
        ].notna()
    ]
)


pivot_persistence = (
    persistence_df
    .pivot(
        index="epsilon",
        columns="alpha",
        values="fraction_memory_remaining_T20"
    )
    .sort_index()
)


fig, ax = plt.subplots(
    figsize=(10.0, 6.8)
)


image = ax.imshow(

    pivot_persistence.values,

    origin="lower",

    aspect="auto",

    vmin=0.0,
    vmax=1.0,

    extent=[
        pivot_persistence.columns.min(),
        pivot_persistence.columns.max(),
        np.log10(
            pivot_persistence.index.min()
        ),
        np.log10(
            pivot_persistence.index.max()
        )
    ]
)


cbar = fig.colorbar(
    image,
    ax=ax
)

cbar.set_label(
    "Fraction of initial memory remaining at $T=20$"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$"
)

ax.set_ylabel(
    "$\\log_{10}(\\varepsilon)$"
)

ax.set_title(
    "Persistence of history-dependent response after recovery"
)


fig.tight_layout()


fig.savefig(
    R1_DIR
    / "figR1_03_memory_fraction_T20.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R1.4 SOURCE DATA
# ============================================================

fig04_data = (
    grid_results_df[[
        "alpha",
        "epsilon",
        "stable_monostable",
        "conditioning_completion_fraction"
    ]]
    .copy()
)


fig04_data.to_csv(
    R1_DIR
    / "figR1_04_conditioning_completion_data.csv",
    index=False,
    float_format="%.17g"
)


conditioning_df = (
    primary_df[
        primary_df[
            "conditioning_completion_fraction"
        ].notna()
    ]
)


pivot_conditioning = (
    conditioning_df
    .pivot(
        index="epsilon",
        columns="alpha",
        values="conditioning_completion_fraction"
    )
    .sort_index()
)


fig, ax = plt.subplots(
    figsize=(10.0, 6.8)
)


image = ax.imshow(

    pivot_conditioning.values,

    origin="lower",

    aspect="auto",

    extent=[
        pivot_conditioning.columns.min(),
        pivot_conditioning.columns.max(),
        np.log10(
            pivot_conditioning.index.min()
        ),
        np.log10(
            pivot_conditioning.index.max()
        )
    ]
)


cbar = fig.colorbar(
    image,
    ax=ax
)

cbar.set_label(
    "Fraction of equilibrium response shift acquired"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$"
)

ax.set_ylabel(
    "$\\log_{10}(\\varepsilon)$"
)

ax.set_title(
    "Acquisition of adaptive response shift during fixed conditioning"
)


fig.tight_layout()


fig.savefig(
    R1_DIR
    / "figR1_04_conditioning_completion.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R1.5 SOURCE DATA
# ============================================================

fig05_data = (
    grid_results_df[[
        "alpha",
        "epsilon",
        "stable_monostable",
        "epsilon_times_half_life"
    ]]
    .copy()
)


fig05_data.to_csv(
    R1_DIR
    / "figR1_05_scaled_half_life_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


for alpha in [
    0.5,
    1.0,
    1.5,
    2.0
]:

    g = primary_df[
        np.isclose(
            primary_df[
                "alpha"
            ],
            alpha
        )
    ].sort_values(
        "epsilon"
    )


    if len(g) == 0:
        continue


    ax.plot(
        g["epsilon"],
        g[
            "epsilon_times_half_life"
        ],
        marker="o",
        linewidth=1.8,
        label=f"$\\alpha={alpha:g}$"
    )


ax.set_xscale(
    "log"
)


ax.set_xlabel(
    "$\\varepsilon$"
)

ax.set_ylabel(
    "$\\varepsilon T_{1/2}$"
)

ax.set_title(
    "Scaling of the apparent memory timescale"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


fig.tight_layout()


fig.savefig(
    R1_DIR
    / "figR1_05_scaled_half_life.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# R1 SUMMARY
# ============================================================

stable_primary = (
    grid_results_df[
        grid_results_df[
            "stable_monostable"
        ]
    ]
)


nonzero_memory_primary = (
    stable_primary[
        np.abs(
            stable_primary[
                "delta_R_protocol"
            ]
        )
        >
        1e-6
    ]
)


summary_rows = [

    {
        "metric":
            "number_parameter_pairs_total",

        "value":
            len(
                grid_results_df
            )
    },

    {
        "metric":
            "number_stable_monostable_pairs",

        "value":
            len(
                stable_primary
            )
    },

    {
        "metric":
            "fraction_stable_monostable_pairs",

        "value":
            (
                len(
                    stable_primary
                )
                /
                len(
                    grid_results_df
                )
            )
    },

    {
        "metric":
            "number_stable_monostable_pairs_with_numerically_nonzero_memory",

        "value":
            len(
                nonzero_memory_primary
            )
    },

    {
        "metric":
            "fraction_stable_monostable_pairs_with_numerically_nonzero_memory",

        "value":
            (
                len(
                    nonzero_memory_primary
                )
                /
                len(
                    stable_primary
                )

                if len(
                    stable_primary
                )
                >
                0

                else np.nan
            )
    },

    {
        "metric":
            "maximum_delta_R_in_stable_monostable_region",

        "value":
            stable_primary[
                "delta_R_protocol"
            ].max()
    },

    {
        "metric":
            "median_delta_R_in_stable_monostable_region",

        "value":
            stable_primary[
                "delta_R_protocol"
            ].median()
    },

    {
        "metric":
            "maximum_resolved_memory_half_life",

        "value":
            stable_primary[
                "apparent_memory_half_life"
            ].max()
    },

    {
        "metric":
            "median_resolved_memory_half_life",

        "value":
            stable_primary[
                "apparent_memory_half_life"
            ].median()
    },

    {
        "metric":
            "maximum_absolute_delta_R_at_alpha_0",

        "value":
            MAX_ABS_DELTA_R_ALPHA0
    },

    {
        "metric":
            "number_oscillatory_instability_pairs",

        "value":
            NUMBER_OSCILLATORY_INSTABILITY
    }
]


R1_summary_df = pd.DataFrame(
    summary_rows
)


R1_summary_df.to_csv(
    R1_DIR
    / "R1_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    R1_summary_df
)

In [ ]:
# ============================================================
# R1 DECISION FLAGS
# ============================================================

alpha0_negative_control_pass = bool(
    MAX_ABS_DELTA_R_ALPHA0
    <
    1e-6
)


memory_exists_outside_bistability = bool(

    np.any(

        (
            grid_results_df[
                "stable_monostable"
            ]
        )

        &

        (
            np.abs(
                grid_results_df[
                    "delta_R_protocol"
                ]
            )
            >
            1e-6
        )
    )
)


multiple_alpha_values_support_memory = bool(

    grid_results_df[
        (
            grid_results_df[
                "stable_monostable"
            ]
        )
        &
        (
            np.abs(
                grid_results_df[
                    "delta_R_protocol"
                ]
            )
            >
            1e-6
        )
    ][
        "alpha"
    ].nunique()
    >
    1
)


multiple_epsilon_values_support_memory = bool(

    grid_results_df[
        (
            grid_results_df[
                "stable_monostable"
            ]
        )
        &
        (
            np.abs(
                grid_results_df[
                    "delta_R_protocol"
                ]
            )
            >
            1e-6
        )
    ][
        "epsilon"
    ].nunique()
    >
    1
)


decision_df = pd.DataFrame([{

    "alpha0_negative_control_pass":
        alpha0_negative_control_pass,

    "history_dependent_response_exists_in_stable_monostable_region":
        memory_exists_outside_bistability,

    "memory_occurs_for_multiple_alpha_values":
        multiple_alpha_values_support_memory,

    "memory_occurs_for_multiple_epsilon_values":
        multiple_epsilon_values_support_memory,

    "oscillatory_instability_detected_anywhere":
        bool(
            NUMBER_OSCILLATORY_INSTABILITY
            >
            0
        )
}])


decision_df.to_csv(
    R1_DIR
    / "R1_decision_flags.csv",
    index=False
)


display(
    decision_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
ROBUSTNESS R1
ALPHA-EPSILON MEMORY MAP
=========================

PURPOSE
-------
Test whether the history-dependent same-signal response found
in Experiments 4 and 5 is confined to the single original
(alpha, epsilon) choice or persists over a broader region of
adaptive-strength / adaptive-timescale space.

STRUCTURAL PARAMETERS
---------------------
n      = {BASE_PARAMS["n"]}
q      = {BASE_PARAMS["q"]}
K_M    = {BASE_PARAMS["K_M"]}
theta0 = {BASE_PARAMS["theta0"]}
r      = {BASE_PARAMS["r"]}

These are the structural parameters selected automatically in
Experiment 4 for the monostable proof-of-principle regime.

PARAMETER SWEEP
---------------
alpha:
    0 to 3
    {len(ALPHA_VALUES)} linearly spaced values

epsilon:
    0.01 to 1
    {len(EPSILON_VALUES)} logarithmically spaced values

The ranges are exploratory computational ranges and are not
biologically calibrated periodontal parameter bounds.

PROTOCOL
--------
The physical protocol is held fixed across the full grid.

Low history forcing:
    u = {U_LOW}

High conditioning forcing:
    u = {U_HIGH}

Baseline duration:
    {T_BASELINE}

History duration:
    {T_HISTORY}

Same physical probe signal:
    S* = {S_STAR}

No pair-specific optimization of forcing, conditioning duration,
or probe signal is performed.

PRIMARY OUTCOME
---------------
Same-signal response difference:

    delta_R = F(S*, theta_B) - F(S*, theta_A)

where A and B have different previous forcing histories.

PRIMARY INTERPRETATION REGION
-----------------------------
The main robustness analysis is restricted to parameter pairs
that remain:

    1. structurally monostable across
       u = [{U_MONO_MIN}, {U_MONO_MAX}]

and

    2. locally stable across that forcing interval.

Points outside this region are retained in the raw output but
are not used to claim memory without multistability.

MEMORY PERSISTENCE
------------------
After high-history conditioning, the model is returned to the
low forcing condition.

The response mapping at the identical physical signal S* is
evaluated at recovery times:

    {RECOVERY_TIMES.tolist()}

Memory excess is defined relative to the exact low-forcing
equilibrium response mapping for that alpha:

    DeltaR_memory(T)
    =
    F(S*, theta(T))
    -
    F(S*, theta_low_equilibrium)

The apparent memory half-life is obtained by interpolation and
does not assume exponential decay.

IMPORTANT ACQUISITION/DECAY DISTINCTION
----------------------------------------
epsilon affects both:

    - how rapidly the adaptive state is acquired during the
      fixed-duration conditioning period;

and

    - how rapidly it is lost during recovery.

Therefore the notebook separately records the fraction of the
equilibrium response shift actually acquired during the
200-time-unit conditioning period.

NEGATIVE CONTROL
----------------
alpha = 0 should eliminate history-dependent threshold
adaptation.

The maximum numerical same-signal response difference in the
alpha = 0 row is saved explicitly as a validation metric.

MODEL-DEFINITION CAVEAT
-----------------------
The same provisional microbial response function is retained:

    G(H) = H^q / (H^q + K_M^q)

This robustness analysis tests parameter robustness within that
model definition.

Functional-form robustness is reserved for Robustness R3.

DATA POLICY
-----------
Every numerical result used in a table, figure, summary, or
interpretation is saved as machine-readable CSV.

Every figure has a companion figure-data CSV.

Checkpoint CSVs are written after every alpha row so the sweep
can resume after an interrupted Colab session.
"""


with open(
    R1_DIR
    / "R1_README.txt",
    "w"
) as f:

    f.write(
        readme
    )


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    R1_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                /
                1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    R1_DIR
    / "R1_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Robustness_R1_alpha_epsilon_memory_map_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(R1_DIR)
)


print(
    "============================================"
)

print(
    "ROBUSTNESS R1 EXPORT COMPLETE"
)

print(
    "============================================"
)

print("\nFolder:")
print(R1_DIR)

print("\nZIP:")
print(zip_path)

print("\nFiles:\n")


for path in sorted(
    R1_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:65s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# R2 analysis

In [ ]:
# ============================================================
# ROBUSTNESS R2
# Multivariate structural robustness
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

R2_DIR = (
    BASE_DIR
    / "Robustness_R2_multivariate_structural_sample"
)

R2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("R2 directory:")
print(R2_DIR)

In [ ]:
# ============================================================
# IMPORTS + EXPERIMENT 04 PROVENANCE
# ============================================================

import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.stats import qmc, spearmanr


PARAM_FILE = (
    EXP4_DIR
    / "selected_monostable_parameters.json"
)

DESIGN_FILE = (
    EXP4_DIR
    / "experiment04_design.json"
)

PROBE_FILE = (
    EXP4_DIR
    / "probe_definition.csv"
)


for path in [
    PARAM_FILE,
    DESIGN_FILE,
    PROBE_FILE
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required Experiment 4 file:\n{path}"
        )


with open(
    PARAM_FILE,
    "r"
) as f:

    exp4_params = json.load(f)


with open(
    DESIGN_FILE,
    "r"
) as f:

    exp4_design = json.load(f)


probe_exp4_df = pd.read_csv(
    PROBE_FILE
)


EXP4_REFERENCE = {

    "n":
        float(exp4_params["n"]),

    "q":
        float(exp4_params["q"]),

    "K_M":
        float(exp4_params["K_M"]),

    "theta0":
        float(exp4_params["theta0"]),

    "r":
        float(exp4_params["r"]),

    "alpha":
        float(exp4_params["alpha"]),

    "epsilon":
        float(exp4_params["epsilon"])
}


U_LOW = float(
    exp4_design[
        "u_low_history"
    ]
)

U_HIGH = float(
    exp4_design[
        "u_high_history"
    ]
)

S_STAR = float(
    probe_exp4_df[
        "S_star"
    ].iloc[0]
)


T_CONDITIONING = 200.0


print("Experiment 4 reference:")
print(EXP4_REFERENCE)

print("\nProtocol:")
print("U_LOW =", U_LOW)
print("U_HIGH =", U_HIGH)
print("S_STAR =", S_STAR)
print("T_CONDITIONING =", T_CONDITIONING)

In [ ]:
# ============================================================
# PRESPECIFIED STRUCTURAL ROBUSTNESS DOMAIN
# ============================================================

N_SAMPLES = 1000

RNG_SEED = 20261006


# ------------------------------------------------------------
# Linear ranges
# ------------------------------------------------------------

RANGES_LINEAR = {

    "n":
        (1.5, 2.5),

    "q":
        (0.75, 1.50),

    "K_M":
        (0.75, 1.00),

    "theta0":
        (0.80, 1.00),

    "alpha":
        (0.50, 1.50)
}


# ------------------------------------------------------------
# Log-uniform ranges
# ------------------------------------------------------------

RANGES_LOG10 = {

    "r":
        (
            np.log10(0.30),
            np.log10(3.00)
        ),

    "epsilon":
        (
            np.log10(0.03),
            np.log10(0.30)
        )
}


# ------------------------------------------------------------
# Equilibrium-classification interval
# ------------------------------------------------------------

U_SCREEN_MIN = 0.02
U_SCREEN_MAX = 1.50

U_SCREEN = np.linspace(
    U_SCREEN_MIN,
    U_SCREEN_MAX,
    41
)


U_STABILITY = np.linspace(
    U_SCREEN_MIN,
    U_SCREEN_MAX,
    21
)


# ------------------------------------------------------------
# Recovery grid
# ------------------------------------------------------------

RECOVERY_TIMES = np.array([
    0.0,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0,
    40.0,
    80.0,
    160.0,
    320.0
])


# ------------------------------------------------------------
# Numerical settings
# ------------------------------------------------------------

X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 8001

STABILITY_TOL = 1e-7

RTOL = 1e-9
ATOL = 1e-11


CONFIG_R2 = {

    "n_samples":
        N_SAMPLES,

    "rng_seed":
        RNG_SEED,

    "linear_ranges":
        RANGES_LINEAR,

    "log10_ranges":
        RANGES_LOG10,

    "u_screen_min":
        U_SCREEN_MIN,

    "u_screen_max":
        U_SCREEN_MAX,

    "u_low":
        U_LOW,

    "u_high":
        U_HIGH,

    "S_star":
        S_STAR,

    "conditioning_duration":
        T_CONDITIONING,

    "recovery_times":
        RECOVERY_TIMES.tolist(),

    "rtol":
        RTOL,

    "atol":
        ATOL
}


with open(
    R2_DIR
    / "R2_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG_R2,
        f,
        indent=4
    )

In [ ]:
# ============================================================
# FIXED-SEED LATIN HYPERCUBE
# ============================================================

PARAMETER_ORDER = [
    "n",
    "q",
    "K_M",
    "theta0",
    "alpha",
    "r",
    "epsilon"
]


sampler = qmc.LatinHypercube(
    d=len(PARAMETER_ORDER),
    seed=RNG_SEED
)


unit_sample = sampler.random(
    n=N_SAMPLES
)


sample_df = pd.DataFrame({
    "sample_id":
        np.arange(
            1,
            N_SAMPLES + 1
        )
})


# ------------------------------------------------------------
# Linear parameters
# ------------------------------------------------------------

for column_index, parameter in enumerate(
    [
        "n",
        "q",
        "K_M",
        "theta0",
        "alpha"
    ]
):

    lo, hi = (
        RANGES_LINEAR[
            parameter
        ]
    )


    sample_df[
        parameter
    ] = (
        lo
        +
        unit_sample[
            :,
            column_index
        ]
        *
        (
            hi
            -
            lo
        )
    )


# ------------------------------------------------------------
# Log parameters
# ------------------------------------------------------------

for column_index, parameter in [

    (5, "r"),
    (6, "epsilon")

]:

    log_lo, log_hi = (
        RANGES_LOG10[
            parameter
        ]
    )


    log_values = (
        log_lo
        +
        unit_sample[
            :,
            column_index
        ]
        *
        (
            log_hi
            -
            log_lo
        )
    )


    sample_df[
        parameter
    ] = np.power(
        10.0,
        log_values
    )


sample_df.to_csv(
    R2_DIR
    / "R2_parameter_sample.csv",
    index=False,
    float_format="%.17g"
)


display(
    sample_df.head()
)

print(
    "\nNumber of sampled sets:",
    len(sample_df)
)

In [ ]:
# ============================================================
# MODEL FUNCTIONS
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


def F(
    S,
    theta,
    n
):

    return (
        np.power(S, n)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )
    )


def dF_dS(
    S,
    theta,
    n
):

    return (
        n
        *
        np.power(theta, n)
        *
        np.power(S, n - 1.0)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )**2
    )


def dF_dtheta(
    S,
    theta,
    n
):

    return (
        -n
        *
        np.power(S, n)
        *
        np.power(theta, n - 1.0)
        /
        (
            np.power(S, n)
            +
            np.power(theta, n)
        )**2
    )


# ------------------------------------------------------------
# Same provisional G(H) used in Experiments 1-6 and R1.
# ------------------------------------------------------------

def G(
    H,
    K_M,
    q
):

    return (
        np.power(H, q)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )
    )


def dG_dH(
    H,
    K_M,
    q
):

    return (
        q
        *
        np.power(K_M, q)
        *
        np.power(H, q - 1.0)
        /
        (
            np.power(H, q)
            +
            np.power(K_M, q)
        )**2
    )


def theta_equilibrium(
    H,
    p
):

    return (
        p["theta0"]
        *
        np.exp(
            -p["alpha"]
            *
            H
        )
    )


def M_equilibrium(
    H,
    p
):

    return G(
        H,
        p["K_M"],
        p["q"]
    )


def u_equilibrium(
    H,
    p
):

    theta = theta_equilibrium(
        H,
        p
    )


    return (
        theta
        *
        np.power(
            H
            /
            (1.0 - H),
            1.0 / p["n"]
        )
        -
        M_equilibrium(
            H,
            p
        )
    )


def du_dH(
    H,
    p
):

    power_term = np.power(
        H
        /
        (1.0 - H),
        1.0 / p["n"]
    )


    derivative_log_term = (
        1.0
        /
        (
            p["n"]
            *
            H
            *
            (1.0 - H)
        )
    )


    theta = theta_equilibrium(
        H,
        p
    )


    return (
        theta
        *
        power_term
        *
        (
            derivative_log_term
            -
            p["alpha"]
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"]
        )
    )

In [ ]:
# ============================================================
# ADAPTIVE MODEL
# ============================================================

def rhs_M1(
    t,
    y,
    u,
    p
):

    H, M, theta = y

    S = (
        M
        +
        u
    )


    dH = (
        -H
        +
        F(
            S,
            theta,
            p["n"]
        )
    )


    dM = (
        p["r"]
        *
        (
            -M
            +
            G(
                H,
                p["K_M"],
                p["q"]
            )
        )
    )


    dtheta = (
        p["epsilon"]
        *
        theta
        *
        (
            np.log(
                p["theta0"]
                /
                theta
            )
            -
            p["alpha"]
            *
            H
        )
    )


    return np.array([
        dH,
        dM,
        dtheta
    ])


def jacobian_M1(
    H,
    M,
    theta,
    u,
    p
):

    S = (
        M
        +
        u
    )


    FS = dF_dS(
        S,
        theta,
        p["n"]
    )


    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"]
    )


    GH = dG_dH(
        H,
        p["K_M"],
        p["q"]
    )


    adaptive_bracket = (
        np.log(
            p["theta0"]
            /
            theta
        )
        -
        p["alpha"]
        *
        H
    )


    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            p["r"] * GH,
            -p["r"],
            0.0
        ],

        [
            -p["epsilon"]
            *
            p["alpha"]
            *
            theta,

            0.0,

            p["epsilon"]
            *
            (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# EQUILIBRIUM GEOMETRY
# ============================================================

X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)

H_SCAN = sigmoid(
    X_SCAN
)


def branch_arrays(
    p
):

    u_branch = u_equilibrium(
        H_SCAN,
        p
    )


    derivative_branch = du_dH(
        H_SCAN,
        p
    )


    return (
        u_branch,
        derivative_branch
    )


def approximate_root_count(
    u_target,
    u_branch
):

    values = (
        u_branch
        -
        u_target
    )


    return int(
        np.sum(
            values[:-1]
            *
            values[1:]
            <
            0.0
        )
    )


def exact_equilibrium_roots(
    u_target,
    p,
    u_branch=None
):

    if u_branch is None:

        u_branch = u_equilibrium(
            H_SCAN,
            p
        )


    values = (
        u_branch
        -
        u_target
    )


    indices = np.where(
        values[:-1]
        *
        values[1:]
        <
        0.0
    )[0]


    roots = []


    for i in indices:

        result = root_scalar(

            lambda x:
                (
                    u_equilibrium(
                        sigmoid(x),
                        p
                    )
                    -
                    u_target
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )


        roots.append(
            sigmoid(
                result.root
            )
        )


    return np.asarray(
        roots
    )


def fold_u_values(
    p,
    derivative_branch=None
):

    if derivative_branch is None:

        derivative_branch = du_dH(
            H_SCAN,
            p
        )


    indices = np.where(
        derivative_branch[:-1]
        *
        derivative_branch[1:]
        <
        0.0
    )[0]


    folds = []


    for i in indices:

        result = root_scalar(

            lambda x:
                du_dH(
                    sigmoid(x),
                    p
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )


        Hf = sigmoid(
            result.root
        )


        folds.append(
            float(
                u_equilibrium(
                    Hf,
                    p
                )
            )
        )


    return folds

In [ ]:
# ============================================================
# STRUCTURAL CLASSIFICATION
# ============================================================

STRUCTURAL_CHECKPOINT = (
    R2_DIR
    / "R2_structural_checkpoint.csv"
)


if STRUCTURAL_CHECKPOINT.exists():

    existing = pd.read_csv(
        STRUCTURAL_CHECKPOINT
    )

    completed_structural = set(
        existing[
            "sample_id"
        ].astype(int)
    )

    structural_records = (
        existing.to_dict(
            "records"
        )
    )

else:

    completed_structural = set()
    structural_records = []


for row_number, row in sample_df.iterrows():

    sample_id = int(
        row["sample_id"]
    )


    if sample_id in completed_structural:

        continue


    p = {

        "n":
            float(row["n"]),

        "q":
            float(row["q"]),

        "K_M":
            float(row["K_M"]),

        "theta0":
            float(row["theta0"]),

        "alpha":
            float(row["alpha"]),

        "r":
            float(row["r"]),

        "epsilon":
            float(row["epsilon"])
    }


    u_branch, derivative_branch = (
        branch_arrays(
            p
        )
    )


    root_counts = np.array([

        approximate_root_count(
            u,
            u_branch
        )

        for u in U_SCREEN
    ])


    folds = fold_u_values(
        p,
        derivative_branch
    )


    folds_in_interval = [

        u_fold

        for u_fold in folds

        if (
            U_SCREEN_MIN
            <
            u_fold
            <
            U_SCREEN_MAX
        )
    ]


    exactly_one_equilibrium = bool(
        np.all(
            root_counts == 1
        )
    )


    no_fold_in_interval = bool(
        len(
            folds_in_interval
        )
        ==
        0
    )


    structurally_monostable = bool(
        exactly_one_equilibrium
        and
        no_fold_in_interval
    )


    multistable_detected = bool(

        (
            np.max(
                root_counts
            )
            >=
            3
        )

        or

        (
            len(
                folds_in_interval
            )
            >
            0
        )
    )


    structural_records.append({

        "sample_id":
            sample_id,

        **{
            key:
                float(row[key])

            for key in PARAMETER_ORDER
        },

        "minimum_root_count":
            int(
                np.min(
                    root_counts
                )
            ),

        "maximum_root_count":
            int(
                np.max(
                    root_counts
                )
            ),

        "number_of_folds_total":
            len(folds),

        "number_of_folds_in_interval":
            len(
                folds_in_interval
            ),

        "fold_u_values_in_interval":
            ";".join(
                f"{x:.17g}"
                for x
                in folds_in_interval
            ),

        "structurally_monostable":
            structurally_monostable,

        "multistable_detected":
            multistable_detected
    })


    completed_structural.add(
        sample_id
    )


    if (
        sample_id % 50 == 0
        or
        sample_id == N_SAMPLES
    ):

        checkpoint_df = (
            pd.DataFrame(
                structural_records
            )
            .drop_duplicates(
                subset=[
                    "sample_id"
                ],
                keep="last"
            )
            .sort_values(
                "sample_id"
            )
        )


        checkpoint_df.to_csv(
            STRUCTURAL_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


        print(
            "Structural checkpoint:",
            sample_id,
            "/",
            N_SAMPLES
        )


structural_df = (
    pd.DataFrame(
        structural_records
    )
    .drop_duplicates(
        subset=[
            "sample_id"
        ],
        keep="last"
    )
    .sort_values(
        "sample_id"
    )
    .reset_index(
        drop=True
    )
)


structural_df.to_csv(
    R2_DIR
    / "R2_structural_classification.csv",
    index=False,
    float_format="%.17g"
)


print("\nStructural classification complete.")

print(
    structural_df[
        [
            "structurally_monostable",
            "multistable_detected"
        ]
    ].value_counts()
)

In [ ]:
# ============================================================
# LOCAL STABILITY SCREEN
# ============================================================

STABILITY_CHECKPOINT = (
    R2_DIR
    / "R2_stability_checkpoint.csv"
)


if STABILITY_CHECKPOINT.exists():

    existing = pd.read_csv(
        STABILITY_CHECKPOINT
    )

    completed_stability = set(
        existing[
            "sample_id"
        ].astype(int)
    )

    stability_records = (
        existing.to_dict(
            "records"
        )
    )

else:

    completed_stability = set()
    stability_records = []


mono_candidates = structural_df[
    structural_df[
        "structurally_monostable"
    ]
]


for index, row in mono_candidates.iterrows():

    sample_id = int(
        row["sample_id"]
    )


    if sample_id in completed_stability:

        continue


    p = {

        key:
            float(row[key])

        for key in PARAMETER_ORDER
    }


    u_branch, _ = branch_arrays(
        p
    )


    maximum_real_part = (
        -np.inf
    )

    complex_encountered = False

    oscillatory_instability = False

    unresolved = False


    for u in U_STABILITY:

        roots = exact_equilibrium_roots(
            u,
            p,
            u_branch
        )


        if len(roots) != 1:

            unresolved = True
            break


        H = float(
            roots[0]
        )

        M = float(
            M_equilibrium(
                H,
                p
            )
        )

        theta = float(
            theta_equilibrium(
                H,
                p
            )
        )


        eigvals = np.linalg.eigvals(

            jacobian_M1(
                H,
                M,
                theta,
                u,
                p
            )
        )


        maximum_real_part = max(
            maximum_real_part,
            float(
                np.max(
                    eigvals.real
                )
            )
        )


        if np.any(
            np.abs(
                eigvals.imag
            )
            >
            1e-8
        ):

            complex_encountered = True


        if np.any(
            (
                eigvals.real
                >
                STABILITY_TOL
            )
            &
            (
                np.abs(
                    eigvals.imag
                )
                >
                1e-8
            )
        ):

            oscillatory_instability = True


    stable_monostable = bool(

        not unresolved

        and

        np.isfinite(
            maximum_real_part
        )

        and

        maximum_real_part
        <
        -STABILITY_TOL
    )


    stability_records.append({

        "sample_id":
            sample_id,

        "maximum_real_eigenvalue":
            (
                maximum_real_part
                if np.isfinite(
                    maximum_real_part
                )
                else np.nan
            ),

        "complex_eigenvalues_encountered":
            complex_encountered,

        "oscillatory_instability_detected":
            oscillatory_instability,

        "stability_screen_unresolved":
            unresolved,

        "stable_monostable":
            stable_monostable
    })


    completed_stability.add(
        sample_id
    )


    if (
        len(
            completed_stability
        )
        %
        50
        ==
        0
    ):

        pd.DataFrame(
            stability_records
        ).to_csv(
            STABILITY_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


stability_df = pd.DataFrame(
    stability_records
)


stability_df.to_csv(
    R2_DIR
    / "R2_monostable_stability.csv",
    index=False,
    float_format="%.17g"
)


structural_full_df = (
    structural_df
    .merge(
        stability_df,
        on="sample_id",
        how="left"
    )
)


structural_full_df[
    "stable_monostable"
] = (
    structural_full_df[
        "stable_monostable"
    ]
    .fillna(False)
)


structural_full_df.to_csv(
    R2_DIR
    / "R2_structural_and_stability_classification.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Stable monostable sets:",
    int(
        structural_full_df[
            "stable_monostable"
        ].sum()
    )
)

In [ ]:
# ============================================================
# SIMULATION HELPERS
# ============================================================

def integrate_constant_u(
    y0,
    u,
    duration,
    p,
    t_eval=None
):

    if t_eval is None:

        t_eval = np.array([
            duration
        ])


    solution = solve_ivp(

        lambda t, y:
            rhs_M1(
                t,
                y,
                u,
                p
            ),

        (
            0.0,
            duration
        ),

        y0,

        t_eval=
            t_eval,

        method=
            "LSODA",

        rtol=
            RTOL,

        atol=
            ATOL
    )


    if not solution.success:

        raise RuntimeError(
            solution.message
        )


    return solution


def apparent_half_life_from_peak(
    times,
    values
):

    """
    Half-life measured from the maximum observed post-
    conditioning memory value.

    This handles the small early overshoot found in R1.
    """

    times = np.asarray(
        times,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )


    finite = np.isfinite(
        values
    )


    if not np.all(
        finite
    ):

        return np.nan


    peak_index = int(
        np.argmax(
            values
        )
    )


    peak_value = float(
        values[
            peak_index
        ]
    )


    if peak_value <= 1e-12:

        return np.nan


    target = (
        0.5
        *
        peak_value
    )


    for i in range(
        peak_index + 1,
        len(times)
    ):

        previous = (
            values[
                i - 1
            ]
        )

        current = (
            values[
                i
            ]
        )


        if (
            previous > target
            and
            current <= target
        ):

            t1 = (
                times[
                    i - 1
                ]
            )

            t2 = (
                times[
                    i
                ]
            )


            if current == previous:

                return float(
                    t2
                )


            fraction = (
                target
                -
                previous
            ) / (
                current
                -
                previous
            )


            return float(
                t1
                +
                fraction
                *
                (
                    t2
                    -
                    t1
                )
            )


    return np.nan

In [ ]:
# ============================================================
# EXPERIMENT 4 REFERENCE CONTROL
# ============================================================

p_ref = dict(
    EXP4_REFERENCE
)


u_branch_ref, _ = (
    branch_arrays(
        p_ref
    )
)


roots_low_ref = (
    exact_equilibrium_roots(
        U_LOW,
        p_ref,
        u_branch_ref
    )
)


if len(
    roots_low_ref
) != 1:

    raise RuntimeError(
        "Reference model is not uniquely equilibrated "
        "at U_LOW."
    )


H_A_ref = float(
    roots_low_ref[0]
)

M_A_ref = float(
    M_equilibrium(
        H_A_ref,
        p_ref
    )
)

theta_A_ref = float(
    theta_equilibrium(
        H_A_ref,
        p_ref
    )
)


state_A_ref = np.array([
    H_A_ref,
    M_A_ref,
    theta_A_ref
])


solution_B_ref = (
    integrate_constant_u(
        state_A_ref,
        U_HIGH,
        T_CONDITIONING,
        p_ref
    )
)


state_B_ref = (
    solution_B_ref.y[
        :,
        -1
    ]
)


R_A_ref = float(
    F(
        S_STAR,
        theta_A_ref,
        p_ref["n"]
    )
)


R_B_ref = float(
    F(
        S_STAR,
        state_B_ref[2],
        p_ref["n"]
    )
)


reference_control_df = pd.DataFrame([{

    **EXP4_REFERENCE,

    "H_A":
        H_A_ref,

    "M_A":
        M_A_ref,

    "theta_A":
        theta_A_ref,

    "H_B":
        state_B_ref[0],

    "M_B":
        state_B_ref[1],

    "theta_B":
        state_B_ref[2],

    "R_A_same_signal":
        R_A_ref,

    "R_B_same_signal":
        R_B_ref,

    "delta_R":
        (
            R_B_ref
            -
            R_A_ref
        )
}])


reference_control_df.to_csv(
    R2_DIR
    / "R2_experiment04_reference_control.csv",
    index=False,
    float_format="%.17g"
)


display(
    reference_control_df
)

In [ ]:
# ============================================================
# STRUCTURAL ROBUSTNESS HISTORY EXPERIMENT
# ============================================================

HISTORY_CHECKPOINT = (
    R2_DIR
    / "R2_history_checkpoint.csv"
)

RECOVERY_CHECKPOINT = (
    R2_DIR
    / "R2_recovery_checkpoint.csv"
)


if HISTORY_CHECKPOINT.exists():

    history_existing = pd.read_csv(
        HISTORY_CHECKPOINT
    )

    completed_history = set(
        history_existing[
            "sample_id"
        ].astype(int)
    )

    history_records = (
        history_existing.to_dict(
            "records"
        )
    )

else:

    completed_history = set()
    history_records = []


if RECOVERY_CHECKPOINT.exists():

    recovery_existing = pd.read_csv(
        RECOVERY_CHECKPOINT
    )

    recovery_records = (
        recovery_existing.to_dict(
            "records"
        )
    )

else:

    recovery_records = []


stable_sets = structural_full_df[
    structural_full_df[
        "stable_monostable"
    ]
].copy()


for run_index, row in stable_sets.iterrows():

    sample_id = int(
        row["sample_id"]
    )


    if sample_id in completed_history:

        continue


    p = {

        key:
            float(row[key])

        for key in PARAMETER_ORDER
    }


    print(
        "Running sample",
        sample_id
    )


    u_branch, _ = branch_arrays(
        p
    )


    # ========================================================
    # Exact low-forcing equilibrium = History A
    # ========================================================

    roots_low = exact_equilibrium_roots(
        U_LOW,
        p,
        u_branch
    )


    if len(roots_low) != 1:

        print(
            "Skipping unresolved low equilibrium:",
            sample_id
        )

        continue


    H_A = float(
        roots_low[0]
    )

    M_A = float(
        M_equilibrium(
            H_A,
            p
        )
    )

    theta_A = float(
        theta_equilibrium(
            H_A,
            p
        )
    )


    state_A = np.array([
        H_A,
        M_A,
        theta_A
    ])


    R_A = float(
        F(
            S_STAR,
            theta_A,
            p["n"]
        )
    )


    # ========================================================
    # Exact high equilibrium = conditioning target
    # ========================================================

    roots_high = exact_equilibrium_roots(
        U_HIGH,
        p,
        u_branch
    )


    if len(roots_high) == 1:

        H_high_eq = float(
            roots_high[0]
        )

        theta_high_eq = float(
            theta_equilibrium(
                H_high_eq,
                p
            )
        )

        R_high_eq = float(
            F(
                S_STAR,
                theta_high_eq,
                p["n"]
            )
        )

        equilibrium_shift_potential = (
            R_high_eq
            -
            R_A
        )

    else:

        H_high_eq = np.nan
        theta_high_eq = np.nan
        R_high_eq = np.nan
        equilibrium_shift_potential = np.nan


    # ========================================================
    # History B: fixed high conditioning
    # ========================================================

    solution_B = integrate_constant_u(
        state_A,
        U_HIGH,
        T_CONDITIONING,
        p
    )


    state_B = (
        solution_B.y[
            :,
            -1
        ]
    )


    H_B, M_B, theta_B = (
        state_B
    )


    R_B = float(
        F(
            S_STAR,
            theta_B,
            p["n"]
        )
    )


    delta_R = (
        R_B
        -
        R_A
    )


    # ========================================================
    # Same-physical-signal probe feasibility
    # ========================================================

    u_required_A = (
        S_STAR
        -
        M_A
    )

    u_required_B = (
        S_STAR
        -
        M_B
    )


    probe_admissible = bool(

        0.0
        <=
        u_required_A
        <=
        U_SCREEN_MAX

        and

        0.0
        <=
        u_required_B
        <=
        U_SCREEN_MAX
    )


    # ========================================================
    # Conditioning completion
    # ========================================================

    if (
        np.isfinite(
            equilibrium_shift_potential
        )
        and
        abs(
            equilibrium_shift_potential
        )
        >
        1e-12
    ):

        conditioning_completion = (
            delta_R
            /
            equilibrium_shift_potential
        )

    else:

        conditioning_completion = np.nan


    # ========================================================
    # Recovery
    # ========================================================

    recovery_solution = integrate_constant_u(

        state_B,

        U_LOW,

        float(
            RECOVERY_TIMES[-1]
        ),

        p,

        t_eval=
            RECOVERY_TIMES
    )


    H_rec = (
        recovery_solution.y[0]
    )

    M_rec = (
        recovery_solution.y[1]
    )

    theta_rec = (
        recovery_solution.y[2]
    )


    R_rec = F(
        S_STAR,
        theta_rec,
        p["n"]
    )


    memory_excess = (
        R_rec
        -
        R_A
    )


    peak_index = int(
        np.argmax(
            memory_excess
        )
    )


    peak_memory = float(
        memory_excess[
            peak_index
        ]
    )

    peak_memory_time = float(
        RECOVERY_TIMES[
            peak_index
        ]
    )


    half_life_peak = (
        apparent_half_life_from_peak(
            RECOVERY_TIMES,
            memory_excess
        )
    )


    idx20 = int(
        np.where(
            np.isclose(
                RECOVERY_TIMES,
                20.0
            )
        )[0][0]
    )


    idx80 = int(
        np.where(
            np.isclose(
                RECOVERY_TIMES,
                80.0
            )
        )[0][0]
    )


    memory_T20 = float(
        memory_excess[
            idx20
        ]
    )

    memory_T80 = float(
        memory_excess[
            idx80
        ]
    )


    if peak_memory > 1e-12:

        fraction_peak_remaining_T20 = (
            memory_T20
            /
            peak_memory
        )

        fraction_peak_remaining_T80 = (
            memory_T80
            /
            peak_memory
        )

    else:

        fraction_peak_remaining_T20 = np.nan
        fraction_peak_remaining_T80 = np.nan


    # ========================================================
    # Residual at conditioning endpoint
    # ========================================================

    conditioning_residual = float(
        np.linalg.norm(
            rhs_M1(
                0.0,
                state_B,
                U_HIGH,
                p
            )
        )
    )


    # ========================================================
    # Save summary row
    # ========================================================

    history_records.append({

        "sample_id":
            sample_id,

        **p,

        "H_A":
            H_A,

        "M_A":
            M_A,

        "theta_A":
            theta_A,

        "H_B":
            H_B,

        "M_B":
            M_B,

        "theta_B":
            theta_B,

        "theta_difference_B_minus_A":
            theta_B
            -
            theta_A,

        "R_A_same_signal":
            R_A,

        "R_B_same_signal":
            R_B,

        "delta_R_same_signal":
            delta_R,

        "u_required_A_for_S_star":
            u_required_A,

        "u_required_B_for_S_star":
            u_required_B,

        "same_signal_probe_admissible":
            probe_admissible,

        "H_high_equilibrium":
            H_high_eq,

        "theta_high_equilibrium":
            theta_high_eq,

        "R_high_equilibrium":
            R_high_eq,

        "equilibrium_response_shift_potential":
            equilibrium_shift_potential,

        "conditioning_completion_fraction":
            conditioning_completion,

        "peak_memory_excess":
            peak_memory,

        "time_of_peak_memory":
            peak_memory_time,

        "memory_excess_T20":
            memory_T20,

        "memory_excess_T80":
            memory_T80,

        "fraction_peak_memory_remaining_T20":
            fraction_peak_remaining_T20,

        "fraction_peak_memory_remaining_T80":
            fraction_peak_remaining_T80,

        "peak_based_memory_half_life":
            half_life_peak,

        "conditioning_endpoint_residual":
            conditioning_residual
    })


    # ========================================================
    # Save full recovery curve
    # ========================================================

    for (
        t_rec,
        H_t,
        M_t,
        theta_t,
        R_t,
        memory_t
    ) in zip(

        RECOVERY_TIMES,
        H_rec,
        M_rec,
        theta_rec,
        R_rec,
        memory_excess
    ):

        recovery_records.append({

            "sample_id":
                sample_id,

            "recovery_time":
                t_rec,

            "H":
                H_t,

            "M":
                M_t,

            "theta":
                theta_t,

            "R_same_signal":
                R_t,

            "memory_excess":
                memory_t
        })


    completed_history.add(
        sample_id
    )


    if (
        len(
            completed_history
        )
        %
        25
        ==
        0
    ):

        pd.DataFrame(
            history_records
        ).to_csv(
            HISTORY_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


        pd.DataFrame(
            recovery_records
        ).to_csv(
            RECOVERY_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


        print(
            "History checkpoint:",
            len(
                completed_history
            ),
            "/",
            len(
                stable_sets
            )
        )


history_df = pd.DataFrame(
    history_records
)


recovery_df = pd.DataFrame(
    recovery_records
)


history_df.to_csv(
    R2_DIR
    / "R2_stable_monostable_history_results.csv",
    index=False,
    float_format="%.17g"
)


recovery_df.to_csv(
    R2_DIR
    / "R2_all_recovery_curves.csv",
    index=False,
    float_format="%.17g"
)


print(
    "\nHistory simulations complete:",
    len(history_df)
)

In [ ]:
# ============================================================
# PRIMARY R2 ANALYSIS SUBSET
# ============================================================

primary_df = history_df[
    history_df[
        "same_signal_probe_admissible"
    ]
].copy()


primary_df.to_csv(
    R2_DIR
    / "R2_primary_stable_monostable_probe_admissible_subset.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Stable monostable history sets:",
    len(
        history_df
    )
)

print(
    "Probe-admissible primary sets:",
    len(
        primary_df
    )
)

In [ ]:
# ============================================================
# MEMORY CLASSIFICATION
# ============================================================

NUMERICAL_MEMORY_THRESHOLD = 1e-6


primary_df[
    "numerically_nonzero_memory"
] = (

    np.abs(
        primary_df[
            "delta_R_same_signal"
        ]
    )

    >
    NUMERICAL_MEMORY_THRESHOLD
)


primary_df.to_csv(
    R2_DIR
    / "R2_primary_subset_with_memory_flag.csv",
    index=False,
    float_format="%.17g"
)


memory_positive_df = (
    primary_df[
        primary_df[
            "numerically_nonzero_memory"
        ]
    ]
    .copy()
)


print(
    "Numerically nonzero memory:",
    len(
        memory_positive_df
    ),
    "/",
    len(
        primary_df
    )
)

In [ ]:
# ============================================================
# CLASSIFICATION SUMMARY
# ============================================================

n_total = len(
    structural_full_df
)

n_structural_mono = int(
    structural_full_df[
        "structurally_monostable"
    ].sum()
)

n_stable_mono = int(
    structural_full_df[
        "stable_monostable"
    ].sum()
)

n_multistable = int(
    structural_full_df[
        "multistable_detected"
    ].sum()
)

n_probe_admissible = len(
    primary_df
)

n_memory_positive = len(
    memory_positive_df
)


classification_summary_df = pd.DataFrame([

    {
        "category":
            "total_sample",

        "count":
            n_total
    },

    {
        "category":
            "structurally_monostable",

        "count":
            n_structural_mono
    },

    {
        "category":
            "stable_monostable",

        "count":
            n_stable_mono
    },

    {
        "category":
            "multistable_detected",

        "count":
            n_multistable
    },

    {
        "category":
            "stable_monostable_probe_admissible",

        "count":
            n_probe_admissible
    },

    {
        "category":
            "probe_admissible_with_numerically_nonzero_memory",

        "count":
            n_memory_positive
    }
])


classification_summary_df[
    "fraction_of_total"
] = (
    classification_summary_df[
        "count"
    ]
    /
    n_total
)


classification_summary_df.to_csv(
    R2_DIR
    / "R2_classification_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    classification_summary_df
)

In [ ]:
# ============================================================
# PARAMETER-RANGE COVERAGE
# ============================================================

coverage_records = []


declared_ranges = {

    **RANGES_LINEAR,

    "r":
        (0.30, 3.00),

    "epsilon":
        (0.03, 0.30)
}


for parameter in PARAMETER_ORDER:

    declared_min, declared_max = (
        declared_ranges[
            parameter
        ]
    )


    if len(
        memory_positive_df
    ) > 0:

        observed_min = float(
            memory_positive_df[
                parameter
            ].min()
        )

        observed_max = float(
            memory_positive_df[
                parameter
            ].max()
        )


        if parameter in [
            "r",
            "epsilon"
        ]:

            declared_span = (
                np.log10(
                    declared_max
                )
                -
                np.log10(
                    declared_min
                )
            )

            observed_span = (
                np.log10(
                    observed_max
                )
                -
                np.log10(
                    observed_min
                )
            )

        else:

            declared_span = (
                declared_max
                -
                declared_min
            )

            observed_span = (
                observed_max
                -
                observed_min
            )


        coverage_fraction = (
            observed_span
            /
            declared_span
        )

    else:

        observed_min = np.nan
        observed_max = np.nan
        coverage_fraction = np.nan


    coverage_records.append({

        "parameter":
            parameter,

        "declared_min":
            declared_min,

        "declared_max":
            declared_max,

        "memory_positive_observed_min":
            observed_min,

        "memory_positive_observed_max":
            observed_max,

        "range_coverage_fraction":
            coverage_fraction
    })


coverage_df = pd.DataFrame(
    coverage_records
)


coverage_df.to_csv(
    R2_DIR
    / "R2_memory_positive_parameter_range_coverage.csv",
    index=False,
    float_format="%.17g"
)


display(
    coverage_df
)

In [ ]:
# ============================================================
# DESCRIPTIVE PARAMETER-OUTCOME ASSOCIATIONS
# ============================================================

OUTCOMES = [

    "delta_R_same_signal",

    "peak_memory_excess",

    "fraction_peak_memory_remaining_T20",

    "peak_based_memory_half_life"
]


association_records = []


for parameter in PARAMETER_ORDER:

    for outcome in OUTCOMES:

        subset = primary_df[
            [
                parameter,
                outcome
            ]
        ].dropna()


        if len(
            subset
        ) >= 10:

            rho, p_value = spearmanr(

                subset[
                    parameter
                ],

                subset[
                    outcome
                ]
            )

        else:

            rho = np.nan
            p_value = np.nan


        association_records.append({

            "parameter":
                parameter,

            "outcome":
                outcome,

            "n":
                len(
                    subset
                ),

            "spearman_rho":
                rho,

            "p_value":
                p_value
        })


association_df = pd.DataFrame(
    association_records
)


association_df.to_csv(
    R2_DIR
    / "R2_parameter_outcome_spearman.csv",
    index=False,
    float_format="%.17g"
)


display(
    association_df
)

In [ ]:
# ============================================================
# FIGURE R2.1 SOURCE DATA
# ============================================================

fig01_data = classification_summary_df.copy()


fig01_data.to_csv(
    R2_DIR
    / "figR2_01_classification_counts_data.csv",
    index=False,
    float_format="%.17g"
)


# ============================================================
# FIGURE
# ============================================================

plot_data = fig01_data[
    fig01_data[
        "category"
    ].isin([
        "stable_monostable",
        "multistable_detected",
        "stable_monostable_probe_admissible",
        "probe_admissible_with_numerically_nonzero_memory"
    ])
]


fig, ax = plt.subplots(
    figsize=(9.2, 6.5)
)


ax.bar(
    plot_data[
        "category"
    ],
    plot_data[
        "count"
    ]
)


ax.set_ylabel(
    "Number of sampled parameter sets"
)

ax.set_title(
    "Dynamical classification of the prespecified R2 sample"
)

ax.tick_params(
    axis="x",
    rotation=25
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    R2_DIR
    / "figR2_01_classification_counts.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R2.2 SOURCE DATA
# ============================================================

fig02_data = primary_df[[
    "sample_id",
    "delta_R_same_signal"
]].copy()


fig02_data.to_csv(
    R2_DIR
    / "figR2_02_delta_R_distribution_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


ax.hist(
    fig02_data[
        "delta_R_same_signal"
    ],
    bins=40
)


ax.axvline(
    fig02_data[
        "delta_R_same_signal"
    ].median(),
    linestyle="--",
    linewidth=1.2,
    label="Median"
)


ax.set_xlabel(
    "Same-signal response difference, $\\Delta R$"
)

ax.set_ylabel(
    "Number of parameter sets"
)

ax.set_title(
    "Structural robustness of history-dependent response mapping"
)

ax.legend(
    frameon=False
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    R2_DIR
    / "figR2_02_delta_R_distribution.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R2.3 SOURCE DATA
# ============================================================

fig03_data = primary_df[[
    "sample_id",
    "alpha",
    "epsilon",
    "delta_R_same_signal"
]].copy()


fig03_data.to_csv(
    R2_DIR
    / "figR2_03_delta_R_vs_alpha_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


scatter = ax.scatter(

    fig03_data[
        "alpha"
    ],

    fig03_data[
        "delta_R_same_signal"
    ],

    c=np.log10(
        fig03_data[
            "epsilon"
        ]
    ),

    s=18,

    alpha=0.7
)


cbar = fig.colorbar(
    scatter,
    ax=ax
)

cbar.set_label(
    "$\\log_{10}(\\varepsilon)$"
)


ax.set_xlabel(
    "Adaptation strength, $\\alpha$"
)

ax.set_ylabel(
    "Same-signal response difference, $\\Delta R$"
)

ax.set_title(
    "Memory magnitude across structural perturbations"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    R2_DIR
    / "figR2_03_delta_R_vs_alpha.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R2.4 SOURCE DATA
# ============================================================

fig04_data = primary_df[[
    "sample_id",
    "epsilon",
    "fraction_peak_memory_remaining_T20"
]].dropna().copy()


fig04_data.to_csv(
    R2_DIR
    / "figR2_04_persistence_vs_epsilon_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


ax.scatter(

    fig04_data[
        "epsilon"
    ],

    fig04_data[
        "fraction_peak_memory_remaining_T20"
    ],

    s=18,

    alpha=0.7
)


ax.set_xscale(
    "log"
)


ax.set_xlabel(
    "$\\varepsilon$"
)

ax.set_ylabel(
    "Fraction of peak memory remaining at $T=20$"
)

ax.set_title(
    "Persistence across structural parameter perturbations"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    R2_DIR
    / "figR2_04_persistence_vs_epsilon.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R2.5 SOURCE DATA
# ============================================================

fig05_data = association_df[
    association_df[
        "outcome"
    ]
    ==
    "delta_R_same_signal"
].copy()


fig05_data.to_csv(
    R2_DIR
    / "figR2_05_parameter_association_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


ax.bar(
    fig05_data[
        "parameter"
    ],
    fig05_data[
        "spearman_rho"
    ]
)


ax.axhline(
    0.0,
    linewidth=1.0
)


ax.set_xlabel(
    "Parameter"
)

ax.set_ylabel(
    "Spearman $\\rho$ with $\\Delta R$"
)

ax.set_title(
    "Descriptive parameter associations with memory magnitude"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()


fig.savefig(
    R2_DIR
    / "figR2_05_parameter_association.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# R2 SUMMARY
# ============================================================

resolved_half_lives = primary_df[
    "peak_based_memory_half_life"
].dropna()


summary_rows = [

    {
        "metric":
            "number_parameter_sets_total",

        "value":
            n_total
    },

    {
        "metric":
            "number_structurally_monostable",

        "value":
            n_structural_mono
    },

    {
        "metric":
            "fraction_structurally_monostable",

        "value":
            (
                n_structural_mono
                /
                n_total
            )
    },

    {
        "metric":
            "number_stable_monostable",

        "value":
            n_stable_mono
    },

    {
        "metric":
            "fraction_stable_monostable",

        "value":
            (
                n_stable_mono
                /
                n_total
            )
    },

    {
        "metric":
            "number_multistable_detected",

        "value":
            n_multistable
    },

    {
        "metric":
            "fraction_multistable_detected",

        "value":
            (
                n_multistable
                /
                n_total
            )
    },

    {
        "metric":
            "number_primary_probe_admissible",

        "value":
            n_probe_admissible
    },

    {
        "metric":
            "number_primary_with_nonzero_memory",

        "value":
            n_memory_positive
    },

    {
        "metric":
            "fraction_primary_with_nonzero_memory",

        "value":
            (
                n_memory_positive
                /
                n_probe_admissible

                if
                n_probe_admissible
                >
                0

                else
                np.nan
            )
    },

    {
        "metric":
            "median_delta_R_primary",

        "value":
            primary_df[
                "delta_R_same_signal"
            ].median()
    },

    {
        "metric":
            "minimum_delta_R_primary",

        "value":
            primary_df[
                "delta_R_same_signal"
            ].min()
    },

    {
        "metric":
            "maximum_delta_R_primary",

        "value":
            primary_df[
                "delta_R_same_signal"
            ].max()
    },

    {
        "metric":
            "median_fraction_peak_memory_remaining_T20",

        "value":
            primary_df[
                "fraction_peak_memory_remaining_T20"
            ].median()
    },

    {
        "metric":
            "median_resolved_peak_based_half_life",

        "value":
            resolved_half_lives.median()
    },

    {
        "metric":
            "maximum_resolved_peak_based_half_life",

        "value":
            resolved_half_lives.max()
    },

    {
        "metric":
            "number_oscillatory_instability_detected",

        "value":
            int(
                structural_full_df[
                    "oscillatory_instability_detected"
                ]
                .fillna(False)
                .sum()
            )
    }
]


R2_summary_df = pd.DataFrame(
    summary_rows
)


R2_summary_df.to_csv(
    R2_DIR
    / "R2_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    R2_summary_df
)

In [ ]:
# ============================================================
# R2 DECISION FLAGS
# ============================================================

memory_found_in_multiple_structural_sets = bool(
    n_memory_positive
    >
    1
)


memory_majority_of_primary_sample = bool(

    n_probe_admissible
    >
    0

    and

    (
        n_memory_positive
        /
        n_probe_admissible
    )
    >
    0.5
)


all_parameters_broadly_covered = bool(

    np.all(

        coverage_df[
            "range_coverage_fraction"
        ].dropna()
        >
        0.75
    )
)


multistability_also_occurs = bool(
    n_multistable
    >
    0
)


decision_df = pd.DataFrame([{

    "stable_monostable_structural_variants_exist":
        bool(
            n_stable_mono
            >
            0
        ),

    "same_signal_memory_found_in_multiple_structural_variants":
        memory_found_in_multiple_structural_sets,

    "numerical_memory_occurs_in_majority_of_primary_sample":
        memory_majority_of_primary_sample,

    "memory_positive_cases_span_more_than_75_percent_of_each_prespecified_parameter_range":
        all_parameters_broadly_covered,

    "multistable_structural_variants_also_occur_in_same_prespecified_sample":
        multistability_also_occurs
}])


decision_df.to_csv(
    R2_DIR
    / "R2_decision_flags.csv",
    index=False
)


display(
    decision_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
ROBUSTNESS R2
MULTIVARIATE STRUCTURAL ROBUSTNESS
==================================

PURPOSE
-------
Test whether the history-dependent same-signal response
demonstrated in Experiments 4-5 and R1 survives simultaneous
perturbation of the model's structural and timescale
parameters.

SAMPLING DESIGN
---------------
A fixed-seed Latin-hypercube sample of {N_SAMPLES} parameter
sets was generated.

Seed:
    {RNG_SEED}

The sampled fractions are properties of this prespecified
computational sample and must NOT be interpreted as biological
probabilities.

PARAMETER RANGES
----------------
n:
    1.5 to 2.5

q:
    0.75 to 1.50

K_M:
    0.75 to 1.00

theta0:
    0.80 to 1.00

alpha:
    0.50 to 1.50

r:
    0.30 to 3.00
    log-uniform

epsilon:
    0.03 to 0.30
    log-uniform

REFERENCE MODEL
---------------
Experiment 4:

    n       = {EXP4_REFERENCE["n"]}
    q       = {EXP4_REFERENCE["q"]}
    K_M     = {EXP4_REFERENCE["K_M"]}
    theta0  = {EXP4_REFERENCE["theta0"]}
    alpha   = {EXP4_REFERENCE["alpha"]}
    r       = {EXP4_REFERENCE["r"]}
    epsilon = {EXP4_REFERENCE["epsilon"]}

STRUCTURAL CLASSIFICATION
-------------------------
Each sampled parameter set was classified over:

    u = [{U_SCREEN_MIN}, {U_SCREEN_MAX}]

as structurally monostable or multistable using both:

    - equilibrium root counts;
    - fold detection.

Structurally monostable sets were then screened for local
stability using Jacobian eigenvalues.

PRIMARY MEMORY ANALYSIS
-----------------------
Only stable monostable models are used for the primary
"memory without multistability" robustness claim.

History A begins at the exact low-forcing equilibrium.

History B begins from the same equilibrium and receives:

    u = {U_HIGH}
    for {T_CONDITIONING} time units.

Both are evaluated at the same fixed physical signal:

    S* = {S_STAR}

No parameter-specific probe signal or conditioning forcing is
optimized.

PHYSICAL PROBE ADMISSIBILITY
----------------------------
For the primary subset, the external forcing required to
generate S* at each history endpoint must remain nonnegative
and within the prevalidated positive forcing interval.

MEMORY MEASURE
--------------
Primary immediate memory:

    DeltaR =
        F(S*, theta_B)
        -
        F(S*, theta_A)

Recovery is measured after return to:

    u = {U_LOW}

at times:

    {RECOVERY_TIMES.tolist()}

Because R1 revealed a small early post-removal overshoot in
some slow-adaptation cases, the apparent half-life in R2 is
measured relative to the maximum observed post-conditioning
memory value rather than necessarily the value at T=0.

INTERPRETATION
--------------
R2 asks whether adaptive memory is robust to simultaneous
parameter perturbation.

It does NOT estimate the probability that biological
periodontal systems exhibit memory.

MODEL-DEFINITION CAVEAT
-----------------------
The same provisional microbial response function is retained:

    G(H) =
    H^q / (H^q + K_M^q)

Thus R2 establishes parameter robustness within the current
functional architecture.

Functional-form robustness is reserved for R3.

DATA POLICY
-----------
All interpreted numerical outputs are saved as CSV.

Every figure has a companion figure-data CSV.

Checkpoint files allow interrupted Colab runs to resume
without regenerating completed samples.
"""


with open(
    R2_DIR
    / "R2_README.txt",
    "w"
) as f:

    f.write(
        readme
    )


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    R2_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                /
                1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    R2_DIR
    / "R2_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Robustness_R2_multivariate_structural_sample_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(R2_DIR)
)


print(
    "============================================"
)

print(
    "ROBUSTNESS R2 EXPORT COMPLETE"
)

print(
    "============================================"
)

print("\nFolder:")
print(R2_DIR)

print("\nZIP:")
print(zip_path)

print("\nFiles:\n")


for path in sorted(
    R2_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:70s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )

# R3 analysis

In [ ]:
# ============================================================
# ROBUSTNESS R3
# Functional-form robustness
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Biosemiotic_Perio"
)

EXP4_DIR = (
    BASE_DIR
    / "Experiment_04_Same_Signal_Different_History"
)

R2_DIR = (
    BASE_DIR
    / "Robustness_R2_multivariate_structural_sample"
)

R3_DIR = (
    BASE_DIR
    / "Robustness_R3_functional_form"
)

R3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("R3 directory:")
print(R3_DIR)

In [ ]:
# ============================================================
# IMPORTS + PROVENANCE
# ============================================================

import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.stats import spearmanr


# ------------------------------------------------------------
# Experiment 4 provenance
# ------------------------------------------------------------

with open(
    EXP4_DIR
    / "selected_monostable_parameters.json",
    "r"
) as f:

    exp4_params = json.load(f)


with open(
    EXP4_DIR
    / "experiment04_design.json",
    "r"
) as f:

    exp4_design = json.load(f)


probe_df = pd.read_csv(
    EXP4_DIR
    / "probe_definition.csv"
)


P_REF = {

    "n":
        float(exp4_params["n"]),

    "q":
        float(exp4_params["q"]),

    "K_M":
        float(exp4_params["K_M"]),

    "theta0":
        float(exp4_params["theta0"]),

    "alpha":
        float(exp4_params["alpha"]),

    "r":
        float(exp4_params["r"]),

    "epsilon":
        float(exp4_params["epsilon"])
}


U_LOW = float(
    exp4_design[
        "u_low_history"
    ]
)

U_HIGH = float(
    exp4_design[
        "u_high_history"
    ]
)

S_STAR = float(
    probe_df[
        "S_star"
    ].iloc[0]
)

T_CONDITIONING = 200.0


# ------------------------------------------------------------
# Reuse the EXACT R2 parameter sample.
# ------------------------------------------------------------

R2_SAMPLE_FILE = (
    R2_DIR
    / "R2_parameter_sample.csv"
)

if not R2_SAMPLE_FILE.exists():

    raise FileNotFoundError(
        f"Missing R2 sample:\n{R2_SAMPLE_FILE}"
    )


sample_df = pd.read_csv(
    R2_SAMPLE_FILE
)


print("Reference parameters:")
print(P_REF)

print("\nR2 paired sample rows:")
print(len(sample_df))

print("\nProtocol:")
print("U_LOW =", U_LOW)
print("U_HIGH =", U_HIGH)
print("S_STAR =", S_STAR)

In [ ]:
# ============================================================
# PREDECLARED FUNCTIONAL VARIANTS
# ============================================================

VARIANTS = [

    {
        "variant":
            "HillF_HillG",

        "F_form":
            "hill",

        "G_form":
            "hill"
    },

    {
        "variant":
            "HillF_WeibullG",

        "F_form":
            "hill",

        "G_form":
            "weibull"
    },

    {
        "variant":
            "WeibullF_HillG",

        "F_form":
            "weibull",

        "G_form":
            "hill"
    },

    {
        "variant":
            "WeibullF_WeibullG",

        "F_form":
            "weibull",

        "G_form":
            "weibull"
    }
]


variants_df = pd.DataFrame(
    VARIANTS
)

variants_df.to_csv(
    R3_DIR
    / "R3_functional_variants.csv",
    index=False
)


# ------------------------------------------------------------
# Same structural forcing domain used in R2
# ------------------------------------------------------------

U_SCREEN_MIN = 0.02
U_SCREEN_MAX = 1.50

U_SCREEN = np.linspace(
    U_SCREEN_MIN,
    U_SCREEN_MAX,
    41
)

U_STABILITY = np.linspace(
    U_SCREEN_MIN,
    U_SCREEN_MAX,
    21
)


RECOVERY_TIMES = np.array([
    0.0,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0,
    40.0,
    80.0,
    160.0,
    320.0
])


X_MIN = -30.0
X_MAX = 30.0
N_X_SCAN = 8001

STABILITY_TOL = 1e-7

RTOL = 1e-9
ATOL = 1e-11

NUMERICAL_MEMORY_THRESHOLD = 1e-6


CONFIG_R3 = {

    "functional_variants":
        VARIANTS,

    "paired_R2_sample_size":
        len(sample_df),

    "u_screen_min":
        U_SCREEN_MIN,

    "u_screen_max":
        U_SCREEN_MAX,

    "u_low":
        U_LOW,

    "u_high":
        U_HIGH,

    "S_star":
        S_STAR,

    "conditioning_duration":
        T_CONDITIONING,

    "recovery_times":
        RECOVERY_TIMES.tolist(),

    "numerical_memory_threshold":
        NUMERICAL_MEMORY_THRESHOLD,

    "rtol":
        RTOL,

    "atol":
        ATOL
}


with open(
    R3_DIR
    / "R3_parameters.json",
    "w"
) as f:

    json.dump(
        CONFIG_R3,
        f,
        indent=4
    )

In [ ]:
# ============================================================
# RESPONSE FUNCTIONS
# ============================================================

LN2 = np.log(2.0)


def weibull_exponent(
    hill_exponent
):

    # Ensures same midpoint slope as corresponding Hill curve:
    #
    # Hill slope at midpoint = p/(4K)
    #
    # Weibull slope at midpoint =
    # 0.5 * ln(2) * m / K
    #
    # Thus m = p / (2 ln 2)

    return (
        hill_exponent
        /
        (
            2.0
            *
            LN2
        )
    )


def F_response(
    S,
    theta,
    n,
    form
):

    S = np.asarray(
        S
    )

    if form == "hill":

        return (
            np.power(S, n)
            /
            (
                np.power(S, n)
                +
                np.power(theta, n)
            )
        )


    elif form == "weibull":

        m = weibull_exponent(
            n
        )

        z = (
            LN2
            *
            np.power(
                S / theta,
                m
            )
        )

        return (
            1.0
            -
            np.exp(
                -z
            )
        )


    else:

        raise ValueError(
            f"Unknown F form: {form}"
        )


def G_response(
    H,
    K_M,
    q,
    form
):

    H = np.asarray(
        H
    )

    if form == "hill":

        return (
            np.power(H, q)
            /
            (
                np.power(H, q)
                +
                np.power(K_M, q)
            )
        )


    elif form == "weibull":

        m = weibull_exponent(
            q
        )

        z = (
            LN2
            *
            np.power(
                H / K_M,
                m
            )
        )

        return (
            1.0
            -
            np.exp(
                -z
            )
        )


    else:

        raise ValueError(
            f"Unknown G form: {form}"
        )

In [ ]:
# ============================================================
# ANALYTICAL DERIVATIVES
# ============================================================

def dF_dS(
    S,
    theta,
    n,
    form
):

    if form == "hill":

        return (
            n
            *
            np.power(theta, n)
            *
            np.power(S, n - 1.0)
            /
            (
                np.power(S, n)
                +
                np.power(theta, n)
            )**2
        )


    elif form == "weibull":

        m = weibull_exponent(
            n
        )

        ratio = (
            S / theta
        )

        z = (
            LN2
            *
            np.power(
                ratio,
                m
            )
        )

        return (
            np.exp(-z)
            *
            LN2
            *
            m
            *
            np.power(
                ratio,
                m - 1.0
            )
            /
            theta
        )


def dF_dtheta(
    S,
    theta,
    n,
    form
):

    if form == "hill":

        return (
            -n
            *
            np.power(S, n)
            *
            np.power(
                theta,
                n - 1.0
            )
            /
            (
                np.power(S, n)
                +
                np.power(theta, n)
            )**2
        )


    elif form == "weibull":

        m = weibull_exponent(
            n
        )

        ratio = (
            S / theta
        )

        z = (
            LN2
            *
            np.power(
                ratio,
                m
            )
        )

        return (
            -np.exp(-z)
            *
            LN2
            *
            m
            *
            np.power(
                ratio,
                m
            )
            /
            theta
        )


def dG_dH(
    H,
    K_M,
    q,
    form
):

    if form == "hill":

        return (
            q
            *
            np.power(K_M, q)
            *
            np.power(
                H,
                q - 1.0
            )
            /
            (
                np.power(H, q)
                +
                np.power(K_M, q)
            )**2
        )


    elif form == "weibull":

        m = weibull_exponent(
            q
        )

        ratio = (
            H / K_M
        )

        z = (
            LN2
            *
            np.power(
                ratio,
                m
            )
        )

        return (
            np.exp(-z)
            *
            LN2
            *
            m
            *
            np.power(
                ratio,
                m - 1.0
            )
            /
            K_M
        )

In [ ]:
# ============================================================
# EQUILIBRIUM GEOMETRY
# ============================================================

def theta_equilibrium(
    H,
    p
):

    return (
        p["theta0"]
        *
        np.exp(
            -p["alpha"]
            *
            H
        )
    )


def M_equilibrium(
    H,
    p,
    G_form
):

    return G_response(
        H,
        p["K_M"],
        p["q"],
        G_form
    )


def equilibrium_signal_from_H(
    H,
    p,
    F_form
):

    theta = theta_equilibrium(
        H,
        p
    )


    if F_form == "hill":

        return (
            theta
            *
            np.power(
                H
                /
                (1.0 - H),
                1.0
                /
                p["n"]
            )
        )


    elif F_form == "weibull":

        m = weibull_exponent(
            p["n"]
        )

        z = (
            -np.log1p(
                -H
            )
            /
            LN2
        )

        return (
            theta
            *
            np.power(
                z,
                1.0 / m
            )
        )


def u_equilibrium(
    H,
    p,
    F_form,
    G_form
):

    return (
        equilibrium_signal_from_H(
            H,
            p,
            F_form
        )
        -
        M_equilibrium(
            H,
            p,
            G_form
        )
    )

In [ ]:
# ============================================================
# EQUILIBRIUM BRANCH DERIVATIVE
# ============================================================

def dS_equilibrium_dH(
    H,
    p,
    F_form
):

    theta = theta_equilibrium(
        H,
        p
    )


    if F_form == "hill":

        A = np.power(
            H
            /
            (1.0 - H),
            1.0 / p["n"]
        )

        return (
            theta
            *
            A
            *
            (
                1.0
                /
                (
                    p["n"]
                    *
                    H
                    *
                    (1.0 - H)
                )
                -
                p["alpha"]
            )
        )


    elif F_form == "weibull":

        m = weibull_exponent(
            p["n"]
        )

        a = (
            1.0 / m
        )

        z = (
            -np.log1p(
                -H
            )
            /
            LN2
        )

        dz_dH = (
            1.0
            /
            (
                (1.0 - H)
                *
                LN2
            )
        )


        return (
            theta
            *
            (
                -p["alpha"]
                *
                np.power(
                    z,
                    a
                )

                +

                a
                *
                np.power(
                    z,
                    a - 1.0
                )
                *
                dz_dH
            )
        )


def du_dH(
    H,
    p,
    F_form,
    G_form
):

    return (
        dS_equilibrium_dH(
            H,
            p,
            F_form
        )
        -
        dG_dH(
            H,
            p["K_M"],
            p["q"],
            G_form
        )
    )

In [ ]:
# ============================================================
# DYNAMIC MODEL
# ============================================================

def rhs_M1(
    t,
    y,
    u,
    p,
    F_form,
    G_form
):

    H, M, theta = y

    S = (
        M
        +
        u
    )


    dH = (
        -H
        +
        F_response(
            S,
            theta,
            p["n"],
            F_form
        )
    )


    dM = (
        p["r"]
        *
        (
            -M
            +
            G_response(
                H,
                p["K_M"],
                p["q"],
                G_form
            )
        )
    )


    dtheta = (
        p["epsilon"]
        *
        theta
        *
        (
            np.log(
                p["theta0"]
                /
                theta
            )
            -
            p["alpha"]
            *
            H
        )
    )


    return np.array([
        dH,
        dM,
        dtheta
    ])


def jacobian_M1(
    H,
    M,
    theta,
    u,
    p,
    F_form,
    G_form
):

    S = (
        M
        +
        u
    )


    FS = dF_dS(
        S,
        theta,
        p["n"],
        F_form
    )


    Ftheta = dF_dtheta(
        S,
        theta,
        p["n"],
        F_form
    )


    GH = dG_dH(
        H,
        p["K_M"],
        p["q"],
        G_form
    )


    adaptive_bracket = (
        np.log(
            p["theta0"]
            /
            theta
        )
        -
        p["alpha"]
        *
        H
    )


    return np.array([

        [
            -1.0,
            FS,
            Ftheta
        ],

        [
            p["r"] * GH,
            -p["r"],
            0.0
        ],

        [
            -p["epsilon"]
            *
            p["alpha"]
            *
            theta,

            0.0,

            p["epsilon"]
            *
            (
                adaptive_bracket
                -
                1.0
            )
        ]
    ])

In [ ]:
# ============================================================
# ROOT / FOLD HELPERS
# ============================================================

def sigmoid_x(
    x
):

    x = np.clip(
        x,
        -700.0,
        700.0
    )

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


X_SCAN = np.linspace(
    X_MIN,
    X_MAX,
    N_X_SCAN
)

H_SCAN = sigmoid_x(
    X_SCAN
)


def branch_arrays(
    p,
    F_form,
    G_form
):

    u_branch = u_equilibrium(
        H_SCAN,
        p,
        F_form,
        G_form
    )


    derivative_branch = du_dH(
        H_SCAN,
        p,
        F_form,
        G_form
    )


    return (
        u_branch,
        derivative_branch
    )


def approximate_root_count(
    u_target,
    u_branch
):

    values = (
        u_branch
        -
        u_target
    )


    return int(
        np.sum(
            values[:-1]
            *
            values[1:]
            <
            0.0
        )
    )


def exact_equilibrium_roots(
    u_target,
    p,
    F_form,
    G_form,
    u_branch=None
):

    if u_branch is None:

        u_branch = u_equilibrium(
            H_SCAN,
            p,
            F_form,
            G_form
        )


    values = (
        u_branch
        -
        u_target
    )


    indices = np.where(
        values[:-1]
        *
        values[1:]
        <
        0.0
    )[0]


    roots = []


    for i in indices:

        result = root_scalar(

            lambda x:
                (
                    u_equilibrium(
                        sigmoid_x(x),
                        p,
                        F_form,
                        G_form
                    )
                    -
                    u_target
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )


        roots.append(
            sigmoid_x(
                result.root
            )
        )


    return np.asarray(
        roots
    )


def fold_u_values(
    p,
    F_form,
    G_form,
    derivative_branch=None
):

    if derivative_branch is None:

        derivative_branch = du_dH(
            H_SCAN,
            p,
            F_form,
            G_form
        )


    indices = np.where(
        derivative_branch[:-1]
        *
        derivative_branch[1:]
        <
        0.0
    )[0]


    folds = []


    for i in indices:

        result = root_scalar(

            lambda x:
                du_dH(
                    sigmoid_x(x),
                    p,
                    F_form,
                    G_form
                ),

            bracket=(
                X_SCAN[i],
                X_SCAN[i + 1]
            ),

            method="brentq",

            xtol=1e-12
        )


        Hf = sigmoid_x(
            result.root
        )


        folds.append({

            "H_fold":
                Hf,

            "u_fold":
                u_equilibrium(
                    Hf,
                    p,
                    F_form,
                    G_form
                )
        })


    return folds

In [ ]:
# ============================================================
# MATCHING VALIDATION
# ============================================================

matching_records = []


for function_name in [
    "F",
    "G"
]:

    if function_name == "F":

        K = P_REF[
            "theta0"
        ]

        exponent = P_REF[
            "n"
        ]


        hill_value = F_response(
            K,
            K,
            exponent,
            "hill"
        )

        weibull_value = F_response(
            K,
            K,
            exponent,
            "weibull"
        )


        hill_slope = dF_dS(
            K,
            K,
            exponent,
            "hill"
        )

        weibull_slope = dF_dS(
            K,
            K,
            exponent,
            "weibull"
        )


    else:

        K = P_REF[
            "K_M"
        ]

        exponent = P_REF[
            "q"
        ]


        hill_value = G_response(
            K,
            K,
            exponent,
            "hill"
        )

        weibull_value = G_response(
            K,
            K,
            exponent,
            "weibull"
        )


        hill_slope = dG_dH(
            K,
            K,
            exponent,
            "hill"
        )

        weibull_slope = dG_dH(
            K,
            K,
            exponent,
            "weibull"
        )


    matching_records.append({

        "function":
            function_name,

        "midpoint":
            K,

        "Hill_value_at_midpoint":
            hill_value,

        "Weibull_value_at_midpoint":
            weibull_value,

        "absolute_value_difference":
            abs(
                hill_value
                -
                weibull_value
            ),

        "Hill_slope_at_midpoint":
            hill_slope,

        "Weibull_slope_at_midpoint":
            weibull_slope,

        "absolute_slope_difference":
            abs(
                hill_slope
                -
                weibull_slope
            )
    })


matching_df = pd.DataFrame(
    matching_records
)


matching_df.to_csv(
    R3_DIR
    / "R3_function_matching_validation.csv",
    index=False,
    float_format="%.17g"
)


display(
    matching_df
)

In [ ]:
# ============================================================
# FIGURE R3.1 — HOST RESPONSE FUNCTIONS
# ============================================================

S_values = np.linspace(
    0.0,
    1.5,
    1001
)


fig01_data = pd.DataFrame({

    "S":
        S_values,

    "Hill":
        F_response(
            S_values,
            P_REF["theta0"],
            P_REF["n"],
            "hill"
        ),

    "Weibull":
        F_response(
            S_values,
            P_REF["theta0"],
            P_REF["n"],
            "weibull"
        )
})


fig01_data.to_csv(
    R3_DIR
    / "figR3_01_host_response_forms_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.5, 6.2)
)

ax.plot(
    fig01_data["S"],
    fig01_data["Hill"],
    label="Hill"
)

ax.plot(
    fig01_data["S"],
    fig01_data["Weibull"],
    label="Matched Weibull"
)

ax.axvline(
    P_REF["theta0"],
    linestyle="--",
    linewidth=1.0
)

ax.set_xlabel(
    "Physical signal, $S$"
)

ax.set_ylabel(
    "Host response, $F$"
)

ax.set_title(
    "Matched alternative host-response functions"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

fig.savefig(
    R3_DIR
    / "figR3_01_host_response_forms.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE R3.2 — MICROBIAL RESPONSE FUNCTIONS
# ============================================================

H_values = np.linspace(
    0.0,
    1.0,
    1001
)


fig02_data = pd.DataFrame({

    "H":
        H_values,

    "Hill":
        G_response(
            H_values,
            P_REF["K_M"],
            P_REF["q"],
            "hill"
        ),

    "Weibull":
        G_response(
            H_values,
            P_REF["K_M"],
            P_REF["q"],
            "weibull"
        )
})


fig02_data.to_csv(
    R3_DIR
    / "figR3_02_microbial_response_forms_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.5, 6.2)
)

ax.plot(
    fig02_data["H"],
    fig02_data["Hill"],
    label="Hill"
)

ax.plot(
    fig02_data["H"],
    fig02_data["Weibull"],
    label="Matched Weibull"
)

ax.axvline(
    P_REF["K_M"],
    linestyle="--",
    linewidth=1.0
)

ax.set_xlabel(
    "Host state, $H$"
)

ax.set_ylabel(
    "Microbial response, $G$"
)

ax.set_title(
    "Matched alternative microbial-response functions"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

fig.savefig(
    R3_DIR
    / "figR3_02_microbial_response_forms.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SIMULATION HELPERS
# ============================================================

def integrate_constant_u(
    y0,
    u,
    duration,
    p,
    F_form,
    G_form,
    t_eval=None
):

    if t_eval is None:

        t_eval = np.array([
            duration
        ])


    solution = solve_ivp(

        lambda t, y:
            rhs_M1(
                t,
                y,
                u,
                p,
                F_form,
                G_form
            ),

        (
            0.0,
            duration
        ),

        y0,

        t_eval=
            t_eval,

        method=
            "LSODA",

        rtol=
            RTOL,

        atol=
            ATOL
    )


    if not solution.success:

        raise RuntimeError(
            solution.message
        )


    return solution


def peak_based_half_life(
    times,
    values
):

    times = np.asarray(
        times,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )


    if not np.all(
        np.isfinite(values)
    ):

        return np.nan


    peak_index = int(
        np.argmax(values)
    )

    peak_value = float(
        values[
            peak_index
        ]
    )


    if peak_value <= 1e-12:

        return np.nan


    target = (
        0.5
        *
        peak_value
    )


    for i in range(
        peak_index + 1,
        len(times)
    ):

        y1 = values[
            i - 1
        ]

        y2 = values[i]


        if (
            y1 > target
            and
            y2 <= target
        ):

            t1 = times[
                i - 1
            ]

            t2 = times[i]


            if y2 == y1:

                return float(
                    t2
                )


            fraction = (
                target
                -
                y1
            ) / (
                y2
                -
                y1
            )


            return float(
                t1
                +
                fraction
                *
                (
                    t2
                    -
                    t1
                )
            )


    return np.nan

In [ ]:
# ============================================================
# CLASSIFY ONE MODEL
# ============================================================

def classify_model(
    p,
    F_form,
    G_form
):

    u_branch, derivative_branch = (
        branch_arrays(
            p,
            F_form,
            G_form
        )
    )


    root_counts = np.array([

        approximate_root_count(
            u,
            u_branch
        )

        for u in U_SCREEN
    ])


    folds = fold_u_values(
        p,
        F_form,
        G_form,
        derivative_branch
    )


    folds_in_interval = [

        fold

        for fold in folds

        if (
            U_SCREEN_MIN
            <
            fold["u_fold"]
            <
            U_SCREEN_MAX
        )
    ]


    structurally_monostable = bool(

        np.all(
            root_counts == 1
        )

        and

        len(
            folds_in_interval
        )
        ==
        0
    )


    stable_monostable = False

    max_real_eigenvalue = np.nan

    oscillatory_instability = False


    if structurally_monostable:

        local_maxima = []


        for u in U_STABILITY:

            roots = exact_equilibrium_roots(
                u,
                p,
                F_form,
                G_form,
                u_branch
            )


            if len(roots) != 1:

                structurally_monostable = False
                break


            H = float(
                roots[0]
            )

            M = float(
                M_equilibrium(
                    H,
                    p,
                    G_form
                )
            )

            theta = float(
                theta_equilibrium(
                    H,
                    p
                )
            )


            eigvals = np.linalg.eigvals(

                jacobian_M1(
                    H,
                    M,
                    theta,
                    u,
                    p,
                    F_form,
                    G_form
                )
            )


            local_maxima.append(
                np.max(
                    eigvals.real
                )
            )


            if np.any(
                (
                    eigvals.real
                    >
                    STABILITY_TOL
                )
                &
                (
                    np.abs(
                        eigvals.imag
                    )
                    >
                    1e-8
                )
            ):

                oscillatory_instability = True


        if (
            structurally_monostable
            and
            len(local_maxima)
            >
            0
        ):

            max_real_eigenvalue = float(
                np.max(
                    local_maxima
                )
            )


            stable_monostable = bool(
                max_real_eigenvalue
                <
                -STABILITY_TOL
            )


    return {

        "minimum_root_count":
            int(
                np.min(
                    root_counts
                )
            ),

        "maximum_root_count":
            int(
                np.max(
                    root_counts
                )
            ),

        "number_of_folds_in_interval":
            len(
                folds_in_interval
            ),

        "fold_u_values_in_interval":
            ";".join(
                f"{f['u_fold']:.17g}"
                for f in
                folds_in_interval
            ),

        "structurally_monostable":
            structurally_monostable,

        "stable_monostable":
            stable_monostable,

        "maximum_real_eigenvalue":
            max_real_eigenvalue,

        "oscillatory_instability_detected":
            oscillatory_instability
    }

In [ ]:
# ============================================================
# SAME-SIGNAL HISTORY EXPERIMENT
# ============================================================

def run_history_experiment(
    p,
    F_form,
    G_form
):

    u_branch, _ = branch_arrays(
        p,
        F_form,
        G_form
    )


    # --------------------------------------------------------
    # Exact low-forcing equilibrium = History A
    # --------------------------------------------------------

    roots_low = exact_equilibrium_roots(
        U_LOW,
        p,
        F_form,
        G_form,
        u_branch
    )


    if len(roots_low) != 1:

        return None


    H_A = float(
        roots_low[0]
    )

    M_A = float(
        M_equilibrium(
            H_A,
            p,
            G_form
        )
    )

    theta_A = float(
        theta_equilibrium(
            H_A,
            p
        )
    )


    state_A = np.array([
        H_A,
        M_A,
        theta_A
    ])


    R_A = float(
        F_response(
            S_STAR,
            theta_A,
            p["n"],
            F_form
        )
    )


    # --------------------------------------------------------
    # High conditioning
    # --------------------------------------------------------

    solution_B = integrate_constant_u(

        state_A,

        U_HIGH,

        T_CONDITIONING,

        p,

        F_form,

        G_form
    )


    H_B, M_B, theta_B = (
        solution_B.y[
            :,
            -1
        ]
    )


    R_B = float(
        F_response(
            S_STAR,
            theta_B,
            p["n"],
            F_form
        )
    )


    delta_R = (
        R_B
        -
        R_A
    )


    # --------------------------------------------------------
    # Same physical signal feasibility
    # --------------------------------------------------------

    u_required_A = (
        S_STAR
        -
        M_A
    )

    u_required_B = (
        S_STAR
        -
        M_B
    )


    probe_admissible = bool(

        0.0
        <=
        u_required_A
        <=
        U_SCREEN_MAX

        and

        0.0
        <=
        u_required_B
        <=
        U_SCREEN_MAX
    )


    # --------------------------------------------------------
    # Recovery
    # --------------------------------------------------------

    recovery_solution = integrate_constant_u(

        np.array([
            H_B,
            M_B,
            theta_B
        ]),

        U_LOW,

        float(
            RECOVERY_TIMES[-1]
        ),

        p,

        F_form,

        G_form,

        t_eval=
            RECOVERY_TIMES
    )


    H_rec = recovery_solution.y[0]
    M_rec = recovery_solution.y[1]
    theta_rec = recovery_solution.y[2]


    R_rec = F_response(
        S_STAR,
        theta_rec,
        p["n"],
        F_form
    )


    memory_excess = (
        R_rec
        -
        R_A
    )


    peak_index = int(
        np.argmax(
            memory_excess
        )
    )


    peak_memory = float(
        memory_excess[
            peak_index
        ]
    )


    peak_time = float(
        RECOVERY_TIMES[
            peak_index
        ]
    )


    half_life = peak_based_half_life(
        RECOVERY_TIMES,
        memory_excess
    )


    idx20 = int(
        np.where(
            np.isclose(
                RECOVERY_TIMES,
                20.0
            )
        )[0][0]
    )


    memory_T20 = float(
        memory_excess[
            idx20
        ]
    )


    fraction_T20 = (

        memory_T20
        /
        peak_memory

        if
        peak_memory
        >
        1e-12

        else
        np.nan
    )


    residual_B = float(
        np.linalg.norm(
            rhs_M1(
                0.0,
                np.array([
                    H_B,
                    M_B,
                    theta_B
                ]),
                U_HIGH,
                p,
                F_form,
                G_form
            )
        )
    )


    return {

        "H_A":
            H_A,

        "M_A":
            M_A,

        "theta_A":
            theta_A,

        "H_B":
            H_B,

        "M_B":
            M_B,

        "theta_B":
            theta_B,

        "R_A_same_signal":
            R_A,

        "R_B_same_signal":
            R_B,

        "delta_R_same_signal":
            delta_R,

        "u_required_A_for_S_star":
            u_required_A,

        "u_required_B_for_S_star":
            u_required_B,

        "same_signal_probe_admissible":
            probe_admissible,

        "peak_memory_excess":
            peak_memory,

        "time_of_peak_memory":
            peak_time,

        "memory_excess_T20":
            memory_T20,

        "fraction_peak_memory_remaining_T20":
            fraction_T20,

        "peak_based_memory_half_life":
            half_life,

        "conditioning_endpoint_residual":
            residual_B,

        "recovery_H":
            H_rec,

        "recovery_M":
            M_rec,

        "recovery_theta":
            theta_rec,

        "recovery_R":
            R_rec,

        "recovery_memory":
            memory_excess
    }

In [ ]:
# ============================================================
# REFERENCE-POINT FUNCTIONAL-FORM TEST
# ============================================================

reference_records = []
reference_recovery_records = []


for variant in VARIANTS:

    variant_name = (
        variant[
            "variant"
        ]
    )

    F_form = (
        variant[
            "F_form"
        ]
    )

    G_form = (
        variant[
            "G_form"
        ]
    )


    print(
        "Reference test:",
        variant_name
    )


    classification = classify_model(
        P_REF,
        F_form,
        G_form
    )


    history_result = None


    if classification[
        "stable_monostable"
    ]:

        history_result = run_history_experiment(
            P_REF,
            F_form,
            G_form
        )


    record = {

        "variant":
            variant_name,

        "F_form":
            F_form,

        "G_form":
            G_form,

        **classification
    }


    if history_result is not None:

        for key, value in history_result.items():

            if key.startswith(
                "recovery_"
            ):
                continue

            record[key] = value


        for (
            t,
            H,
            M,
            theta,
            R,
            memory
        ) in zip(

            RECOVERY_TIMES,

            history_result[
                "recovery_H"
            ],

            history_result[
                "recovery_M"
            ],

            history_result[
                "recovery_theta"
            ],

            history_result[
                "recovery_R"
            ],

            history_result[
                "recovery_memory"
            ]
        ):

            reference_recovery_records.append({

                "variant":
                    variant_name,

                "recovery_time":
                    t,

                "H":
                    H,

                "M":
                    M,

                "theta":
                    theta,

                "R_same_signal":
                    R,

                "memory_excess":
                    memory
            })


    reference_records.append(
        record
    )


reference_df = pd.DataFrame(
    reference_records
)


reference_recovery_df = pd.DataFrame(
    reference_recovery_records
)


reference_df.to_csv(
    R3_DIR
    / "R3_reference_point_results.csv",
    index=False,
    float_format="%.17g"
)


reference_recovery_df.to_csv(
    R3_DIR
    / "R3_reference_point_recovery_curves.csv",
    index=False,
    float_format="%.17g"
)


display(
    reference_df
)

In [ ]:
# ============================================================
# PAIRED STRUCTURAL CLASSIFICATION
# ============================================================

STRUCTURAL_CHECKPOINT = (
    R3_DIR
    / "R3_structural_checkpoint.csv"
)


if STRUCTURAL_CHECKPOINT.exists():

    existing = pd.read_csv(
        STRUCTURAL_CHECKPOINT
    )

    completed_pairs = set(
        zip(
            existing[
                "variant"
            ].astype(str),

            existing[
                "sample_id"
            ].astype(int)
        )
    )

    structural_records = (
        existing.to_dict(
            "records"
        )
    )

else:

    completed_pairs = set()
    structural_records = []


for variant in VARIANTS:

    variant_name = (
        variant[
            "variant"
        ]
    )

    F_form = (
        variant[
            "F_form"
        ]
    )

    G_form = (
        variant[
            "G_form"
        ]
    )


    print(
        "\n================================="
    )

    print(
        "Variant:",
        variant_name
    )

    print(
        "================================="
    )


    for _, row in sample_df.iterrows():

        sample_id = int(
            row[
                "sample_id"
            ]
        )


        pair_key = (
            variant_name,
            sample_id
        )


        if pair_key in completed_pairs:

            continue


        p = {

            key:
                float(
                    row[key]
                )

            for key in [
                "n",
                "q",
                "K_M",
                "theta0",
                "alpha",
                "r",
                "epsilon"
            ]
        }


        classification = classify_model(
            p,
            F_form,
            G_form
        )


        structural_records.append({

            "variant":
                variant_name,

            "F_form":
                F_form,

            "G_form":
                G_form,

            "sample_id":
                sample_id,

            **p,

            **classification
        })


        completed_pairs.add(
            pair_key
        )


        if (
            len(
                completed_pairs
            )
            %
            100
            ==
            0
        ):

            pd.DataFrame(
                structural_records
            ).to_csv(
                STRUCTURAL_CHECKPOINT,
                index=False,
                float_format="%.17g"
            )


            print(
                "Completed:",
                len(
                    completed_pairs
                ),
                "/",
                len(VARIANTS)
                *
                len(sample_df)
            )


structural_df = (
    pd.DataFrame(
        structural_records
    )
    .drop_duplicates(
        subset=[
            "variant",
            "sample_id"
        ],
        keep="last"
    )
    .sort_values(
        [
            "variant",
            "sample_id"
        ]
    )
)


structural_df.to_csv(
    R3_DIR
    / "R3_paired_structural_classification.csv",
    index=False,
    float_format="%.17g"
)


print(
    structural_df.groupby(
        "variant"
    )[
        "stable_monostable"
    ].sum()
)

In [ ]:
# ============================================================
# PAIRED HISTORY EXPERIMENTS
# ============================================================

HISTORY_CHECKPOINT = (
    R3_DIR
    / "R3_history_checkpoint.csv"
)

RECOVERY_CHECKPOINT = (
    R3_DIR
    / "R3_recovery_checkpoint.csv"
)


if HISTORY_CHECKPOINT.exists():

    existing = pd.read_csv(
        HISTORY_CHECKPOINT
    )

    completed_history = set(
        zip(
            existing[
                "variant"
            ].astype(str),

            existing[
                "sample_id"
            ].astype(int)
        )
    )

    history_records = (
        existing.to_dict(
            "records"
        )
    )

else:

    completed_history = set()
    history_records = []


if RECOVERY_CHECKPOINT.exists():

    existing_recovery = pd.read_csv(
        RECOVERY_CHECKPOINT
    )

    recovery_records = (
        existing_recovery.to_dict(
            "records"
        )
    )

else:

    recovery_records = []


stable_pairs = structural_df[
    structural_df[
        "stable_monostable"
    ]
].copy()


for _, row in stable_pairs.iterrows():

    variant_name = str(
        row[
            "variant"
        ]
    )

    sample_id = int(
        row[
            "sample_id"
        ]
    )


    pair_key = (
        variant_name,
        sample_id
    )


    if pair_key in completed_history:

        continue


    F_form = str(
        row[
            "F_form"
        ]
    )

    G_form = str(
        row[
            "G_form"
        ]
    )


    p = {

        key:
            float(
                row[key]
            )

        for key in [
            "n",
            "q",
            "K_M",
            "theta0",
            "alpha",
            "r",
            "epsilon"
        ]
    }


    result = run_history_experiment(
        p,
        F_form,
        G_form
    )


    if result is None:

        continue


    history_record = {

        "variant":
            variant_name,

        "F_form":
            F_form,

        "G_form":
            G_form,

        "sample_id":
            sample_id,

        **p
    }


    for key, value in result.items():

        if key.startswith(
            "recovery_"
        ):
            continue

        history_record[key] = value


    history_records.append(
        history_record
    )


    for (
        t,
        H,
        M,
        theta,
        R,
        memory
    ) in zip(

        RECOVERY_TIMES,

        result[
            "recovery_H"
        ],

        result[
            "recovery_M"
        ],

        result[
            "recovery_theta"
        ],

        result[
            "recovery_R"
        ],

        result[
            "recovery_memory"
        ]
    ):

        recovery_records.append({

            "variant":
                variant_name,

            "sample_id":
                sample_id,

            "recovery_time":
                t,

            "H":
                H,

            "M":
                M,

            "theta":
                theta,

            "R_same_signal":
                R,

            "memory_excess":
                memory
        })


    completed_history.add(
        pair_key
    )


    if (
        len(
            completed_history
        )
        %
        50
        ==
        0
    ):

        pd.DataFrame(
            history_records
        ).to_csv(
            HISTORY_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


        pd.DataFrame(
            recovery_records
        ).to_csv(
            RECOVERY_CHECKPOINT,
            index=False,
            float_format="%.17g"
        )


        print(
            "History pairs:",
            len(
                completed_history
            ),
            "/",
            len(
                stable_pairs
            )
        )


history_df = pd.DataFrame(
    history_records
)


recovery_df = pd.DataFrame(
    recovery_records
)


history_df[
    "numerically_nonzero_memory"
] = (

    np.abs(
        history_df[
            "delta_R_same_signal"
        ]
    )

    >
    NUMERICAL_MEMORY_THRESHOLD
)


history_df.to_csv(
    R3_DIR
    / "R3_paired_history_results.csv",
    index=False,
    float_format="%.17g"
)


recovery_df.to_csv(
    R3_DIR
    / "R3_all_recovery_curves.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# VARIANT SUMMARY
# ============================================================

summary_records = []


for variant in variants_df[
    "variant"
]:

    structure = structural_df[
        structural_df[
            "variant"
        ]
        ==
        variant
    ]


    histories = history_df[
        history_df[
            "variant"
        ]
        ==
        variant
    ]


    primary = histories[
        histories[
            "same_signal_probe_admissible"
        ]
    ]


    memory_positive = primary[
        primary[
            "numerically_nonzero_memory"
        ]
    ]


    half_lives = primary[
        "peak_based_memory_half_life"
    ].dropna()


    summary_records.append({

        "variant":
            variant,

        "n_total_parameter_vectors":
            len(
                structure
            ),

        "n_stable_monostable":
            int(
                structure[
                    "stable_monostable"
                ].sum()
            ),

        "fraction_stable_monostable":
            structure[
                "stable_monostable"
            ].mean(),

        "n_probe_admissible":
            len(
                primary
            ),

        "n_memory_positive":
            len(
                memory_positive
            ),

        "fraction_memory_positive_among_probe_admissible":
            (
                len(
                    memory_positive
                )
                /
                len(
                    primary
                )

                if
                len(
                    primary
                )
                >
                0

                else
                np.nan
            ),

        "minimum_delta_R":
            primary[
                "delta_R_same_signal"
            ].min(),

        "median_delta_R":
            primary[
                "delta_R_same_signal"
            ].median(),

        "maximum_delta_R":
            primary[
                "delta_R_same_signal"
            ].max(),

        "median_fraction_memory_remaining_T20":
            primary[
                "fraction_peak_memory_remaining_T20"
            ].median(),

        "median_peak_based_half_life":
            half_lives.median(),

        "maximum_peak_based_half_life":
            half_lives.max(),

        "n_oscillatory_instability":
            int(
                structure[
                    "oscillatory_instability_detected"
                ].sum()
            )
    })


variant_summary_df = pd.DataFrame(
    summary_records
)


variant_summary_df.to_csv(
    R3_DIR
    / "R3_variant_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    variant_summary_df
)

In [ ]:
# ============================================================
# COMMON PAIRED SUBSET
# ============================================================

eligible_history_df = history_df[
    history_df[
        "same_signal_probe_admissible"
    ]
].copy()


counts = (
    eligible_history_df
    .groupby(
        "sample_id"
    )[
        "variant"
    ]
    .nunique()
)


common_ids = (
    counts[
        counts
        ==
        len(
            VARIANTS
        )
    ]
    .index
    .to_numpy()
)


common_df = eligible_history_df[
    eligible_history_df[
        "sample_id"
    ].isin(
        common_ids
    )
].copy()


common_df.to_csv(
    R3_DIR
    / "R3_common_stable_monostable_probe_admissible_subset.csv",
    index=False,
    float_format="%.17g"
)


print(
    "Parameter vectors eligible under ALL four forms:",
    len(
        common_ids
    )
)

In [ ]:
# ============================================================
# COMMON-SUBSET SUMMARY
# ============================================================

common_summary_records = []


for variant in variants_df[
    "variant"
]:

    g = common_df[
        common_df[
            "variant"
        ]
        ==
        variant
    ]


    common_summary_records.append({

        "variant":
            variant,

        "n_common_parameter_vectors":
            len(g),

        "fraction_with_nonzero_memory":
            np.mean(
                np.abs(
                    g[
                        "delta_R_same_signal"
                    ]
                )
                >
                NUMERICAL_MEMORY_THRESHOLD
            ),

        "minimum_delta_R":
            g[
                "delta_R_same_signal"
            ].min(),

        "median_delta_R":
            g[
                "delta_R_same_signal"
            ].median(),

        "maximum_delta_R":
            g[
                "delta_R_same_signal"
            ].max(),

        "median_peak_based_half_life":
            g[
                "peak_based_memory_half_life"
            ].median()
    })


common_summary_df = pd.DataFrame(
    common_summary_records
)


common_summary_df.to_csv(
    R3_DIR
    / "R3_common_subset_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    common_summary_df
)

In [ ]:
# ============================================================
# PAIRED CROSS-FORM CORRELATIONS
# ============================================================

wide_delta = (
    common_df
    .pivot(
        index="sample_id",
        columns="variant",
        values="delta_R_same_signal"
    )
)


correlation_records = []


variant_names = list(
    variants_df[
        "variant"
    ]
)


for i in range(
    len(
        variant_names
    )
):

    for j in range(
        i + 1,
        len(
            variant_names
        )
    ):

        A = variant_names[i]
        B = variant_names[j]


        subset = wide_delta[
            [
                A,
                B
            ]
        ].dropna()


        rho, p_value = spearmanr(
            subset[A],
            subset[B]
        )


        correlation_records.append({

            "variant_A":
                A,

            "variant_B":
                B,

            "n":
                len(
                    subset
                ),

            "spearman_rho_delta_R":
                rho,

            "p_value":
                p_value
        })


correlation_df = pd.DataFrame(
    correlation_records
)


correlation_df.to_csv(
    R3_DIR
    / "R3_pairwise_delta_R_correlations.csv",
    index=False,
    float_format="%.17g"
)


display(
    correlation_df
)

In [ ]:
# ============================================================
# PARAMETER-RANGE COVERAGE
# ============================================================

DECLARED_RANGES = {

    "n":
        (1.5, 2.5),

    "q":
        (0.75, 1.5),

    "K_M":
        (0.75, 1.0),

    "theta0":
        (0.8, 1.0),

    "alpha":
        (0.5, 1.5),

    "r":
        (0.3, 3.0),

    "epsilon":
        (0.03, 0.3)
}


coverage_records = []


for variant in variants_df[
    "variant"
]:

    g = history_df[
        (
            history_df[
                "variant"
            ]
            ==
            variant
        )
        &
        (
            history_df[
                "same_signal_probe_admissible"
            ]
        )
        &
        (
            history_df[
                "numerically_nonzero_memory"
            ]
        )
    ]


    for parameter, (
        declared_min,
        declared_max
    ) in DECLARED_RANGES.items():

        if len(g) == 0:

            coverage = np.nan
            observed_min = np.nan
            observed_max = np.nan

        else:

            observed_min = float(
                g[
                    parameter
                ].min()
            )

            observed_max = float(
                g[
                    parameter
                ].max()
            )


            if parameter in [
                "r",
                "epsilon"
            ]:

                coverage = (

                    (
                        np.log10(
                            observed_max
                        )
                        -
                        np.log10(
                            observed_min
                        )
                    )
                    /
                    (
                        np.log10(
                            declared_max
                        )
                        -
                        np.log10(
                            declared_min
                        )
                    )
                )

            else:

                coverage = (

                    (
                        observed_max
                        -
                        observed_min
                    )
                    /
                    (
                        declared_max
                        -
                        declared_min
                    )
                )


        coverage_records.append({

            "variant":
                variant,

            "parameter":
                parameter,

            "declared_min":
                declared_min,

            "declared_max":
                declared_max,

            "observed_min":
                observed_min,

            "observed_max":
                observed_max,

            "coverage_fraction":
                coverage
        })


coverage_df = pd.DataFrame(
    coverage_records
)


coverage_df.to_csv(
    R3_DIR
    / "R3_parameter_range_coverage_by_variant.csv",
    index=False,
    float_format="%.17g"
)

In [ ]:
# ============================================================
# FIGURE R3.3 SOURCE DATA
# ============================================================

fig03_data = reference_df[[
    "variant",
    "stable_monostable",
    "delta_R_same_signal",
    "peak_based_memory_half_life"
]].copy()


fig03_data.to_csv(
    R3_DIR
    / "figR3_03_reference_delta_R_data.csv",
    index=False,
    float_format="%.17g"
)


plot_data = fig03_data[
    fig03_data[
        "stable_monostable"
    ]
]


fig, ax = plt.subplots(
    figsize=(9.0, 6.3)
)


ax.bar(
    plot_data[
        "variant"
    ],
    plot_data[
        "delta_R_same_signal"
    ]
)


ax.set_ylabel(
    "Same-signal response difference, $\\Delta R$"
)

ax.set_title(
    "Functional-form robustness at the Experiment 4 parameter point"
)

ax.tick_params(
    axis="x",
    rotation=25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


fig.savefig(
    R3_DIR
    / "figR3_03_reference_delta_R.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R3.4 SOURCE DATA
# ============================================================

fig04_data = variant_summary_df[[
    "variant",
    "fraction_stable_monostable",
    "fraction_memory_positive_among_probe_admissible"
]].copy()


fig04_data.to_csv(
    R3_DIR
    / "figR3_04_variant_robustness_summary_data.csv",
    index=False,
    float_format="%.17g"
)


x = np.arange(
    len(
        fig04_data
    )
)

width = 0.36


fig, ax = plt.subplots(
    figsize=(9.4, 6.4)
)


ax.bar(
    x - width/2,
    fig04_data[
        "fraction_stable_monostable"
    ],
    width,
    label="Stable monostable"
)


ax.bar(
    x + width/2,
    fig04_data[
        "fraction_memory_positive_among_probe_admissible"
    ],
    width,
    label="Memory positive"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    fig04_data[
        "variant"
    ],
    rotation=25
)

ax.set_ylim(
    0.0,
    1.05
)

ax.set_ylabel(
    "Fraction of prespecified sample"
)

ax.set_title(
    "Functional-form robustness across the paired R2 sample"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


fig.savefig(
    R3_DIR
    / "figR3_04_variant_robustness_summary.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R3.5 SOURCE DATA
# ============================================================

fig05_data = common_df[[
    "sample_id",
    "variant",
    "delta_R_same_signal"
]].copy()


fig05_data.to_csv(
    R3_DIR
    / "figR3_05_common_subset_delta_R_data.csv",
    index=False,
    float_format="%.17g"
)


data_for_boxplot = [

    common_df[
        common_df[
            "variant"
        ]
        ==
        variant
    ][
        "delta_R_same_signal"
    ].values

    for variant in variant_names
]


fig, ax = plt.subplots(
    figsize=(9.5, 6.4)
)


ax.boxplot(
    data_for_boxplot,
    labels=
        variant_names,
    showfliers=False
)


ax.set_ylabel(
    "Same-signal response difference, $\\Delta R$"
)

ax.set_title(
    "Paired functional-form comparison in the common monostable subset"
)

ax.tick_params(
    axis="x",
    rotation=25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


fig.savefig(
    R3_DIR
    / "figR3_05_common_subset_delta_R.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE R3.6 SOURCE DATA
# ============================================================

fig06_data = (
    reference_recovery_df
    .copy()
)


fig06_data.to_csv(
    R3_DIR
    / "figR3_06_reference_memory_decay_data.csv",
    index=False,
    float_format="%.17g"
)


fig, ax = plt.subplots(
    figsize=(8.8, 6.4)
)


for variant, g in (
    fig06_data.groupby(
        "variant"
    )
):

    g = g.sort_values(
        "recovery_time"
    )


    ax.plot(
        g[
            "recovery_time"
        ],
        g[
            "memory_excess"
        ],
        marker="o",
        linewidth=1.8,
        label=variant
    )


ax.set_xlabel(
    "Recovery time"
)

ax.set_ylabel(
    "Same-signal response-memory excess"
)

ax.set_title(
    "Recovery of adaptive memory under alternative response functions"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


fig.savefig(
    R3_DIR
    / "figR3_06_reference_memory_decay.jpg",
    dpi=600,
    format="jpg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# R3 DECISION FLAGS
# ============================================================

reference_eligible = (
    reference_df[
        reference_df[
            "stable_monostable"
        ]
    ]
)


reference_memory_all = bool(

    len(
        reference_eligible
    )
    >
    0

    and

    np.all(
        np.abs(
            reference_eligible[
                "delta_R_same_signal"
            ]
        )
        >
        NUMERICAL_MEMORY_THRESHOLD
    )
)


all_variants_have_primary_models = bool(

    np.all(
        variant_summary_df[
            "n_probe_admissible"
        ]
        >
        0
    )
)


all_variants_majority_memory_positive = bool(

    np.all(
        variant_summary_df[
            "fraction_memory_positive_among_probe_admissible"
        ]
        >
        0.5
    )
)


common_subset_exists = bool(
    len(
        common_ids
    )
    >
    0
)


if common_subset_exists:

    memory_counts_common = (
        common_df
        .assign(
            memory_positive=
                (
                    np.abs(
                        common_df[
                            "delta_R_same_signal"
                        ]
                    )
                    >
                    NUMERICAL_MEMORY_THRESHOLD
                )
        )
        .groupby(
            "sample_id"
        )[
            "memory_positive"
        ]
        .sum()
    )


    all_common_models_memory_positive_all_forms = bool(

        np.all(
            memory_counts_common
            ==
            len(
                VARIANTS
            )
        )
    )

else:

    all_common_models_memory_positive_all_forms = False


minimum_coverage_by_variant = (
    coverage_df
    .groupby(
        "variant"
    )[
        "coverage_fraction"
    ]
    .min()
)


all_variants_broad_parameter_coverage = bool(

    np.all(
        minimum_coverage_by_variant
        >
        0.75
    )
)


decision_df = pd.DataFrame([{

    "reference_point_memory_present_in_all_stable_monostable_functional_variants":
        reference_memory_all,

    "all_functional_variants_have_probe_admissible_stable_monostable_models":
        all_variants_have_primary_models,

    "all_functional_variants_memory_positive_in_majority_of_primary_models":
        all_variants_majority_memory_positive,

    "common_stable_monostable_probe_admissible_subset_exists":
        common_subset_exists,

    "all_common_subset_models_memory_positive_under_all_four_forms":
        all_common_models_memory_positive_all_forms,

    "all_functional_variants_memory_positive_cases_cover_more_than_75_percent_of_each_parameter_range":
        all_variants_broad_parameter_coverage
}])


decision_df.to_csv(
    R3_DIR
    / "R3_decision_flags.csv",
    index=False
)


display(
    decision_df
)

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

summary_rows = []


summary_rows.append({

    "metric":
        "number_functional_variants",

    "value":
        len(
            VARIANTS
        )
})


summary_rows.append({

    "metric":
        "number_paired_parameter_vectors",

    "value":
        len(
            sample_df
        )
})


summary_rows.append({

    "metric":
        "number_common_stable_monostable_probe_admissible_parameter_vectors",

    "value":
        len(
            common_ids
        )
})


for _, row in variant_summary_df.iterrows():

    variant = row[
        "variant"
    ]


    summary_rows.extend([

        {
            "metric":
                f"{variant}_stable_monostable_fraction",

            "value":
                row[
                    "fraction_stable_monostable"
                ]
        },

        {
            "metric":
                f"{variant}_memory_positive_fraction_primary",

            "value":
                row[
                    "fraction_memory_positive_among_probe_admissible"
                ]
        },

        {
            "metric":
                f"{variant}_median_delta_R",

            "value":
                row[
                    "median_delta_R"
                ]
        },

        {
            "metric":
                f"{variant}_median_half_life",

            "value":
                row[
                    "median_peak_based_half_life"
                ]
        }
    ])


R3_summary_df = pd.DataFrame(
    summary_rows
)


R3_summary_df.to_csv(
    R3_DIR
    / "R3_summary.csv",
    index=False,
    float_format="%.17g"
)


display(
    R3_summary_df
)

In [ ]:
# ============================================================
# README
# ============================================================

readme = f"""
ROBUSTNESS R3
FUNCTIONAL-FORM ROBUSTNESS
===========================

PURPOSE
-------
Test whether the principal history-dependent same-signal
response result depends specifically on Hill-function algebra.

FUNCTIONAL FORMS
----------------
Four prespecified host/microbial response combinations were
tested:

1. Hill F / Hill G
2. Hill F / Weibull G
3. Weibull F / Hill G
4. Weibull F / Weibull G

MATCHED WEIBULL FORM
--------------------
The alternative response function is:

    W(x; K, m)
    =
    1 - exp[-ln(2) * (x/K)^m]

with:

    m = p / (2 ln 2)

where p is the corresponding Hill exponent.

This construction gives:

    W(0) = 0
    W(K) = 0.5
    W(infinity) = 1

and matches the Hill function's local midpoint slope:

    W'(K) = p / (4K)

Thus Hill and Weibull curves share zero baseline, midpoint,
midpoint slope, and saturation, while differing in global
functional shape.

ADAPTATION LAW
--------------
The adaptive-threshold law is deliberately unchanged.

Therefore R3 tests RESPONSE-FUNCTION robustness rather than
adaptation-law robustness.

REFERENCE TEST
--------------
All four functional forms are first evaluated using the exact
Experiment 4 numerical parameter values:

{P_REF}

and the original protocol:

    U_LOW = {U_LOW}
    U_HIGH = {U_HIGH}
    conditioning duration = {T_CONDITIONING}
    physical probe S* = {S_STAR}

PAIRED MULTIVARIATE TEST
------------------------
The exact 1,000 parameter vectors generated previously in R2
are reused.

No new parameter sample is generated.

Each parameter vector is therefore evaluated under the same
four alternative functional architectures.

PRIMARY ANALYSIS
----------------
For every functional form:

1. classify equilibrium geometry over
   u = [{U_SCREEN_MIN}, {U_SCREEN_MAX}];

2. retain structurally monostable models;

3. require local equilibrium stability;

4. apply the exact same conditioning protocol;

5. require the same physical probe S* to be achievable using
   nonnegative forcing;

6. calculate same-signal response memory:

       DeltaR
       =
       F(S*, theta_B)
       -
       F(S*, theta_A)

7. quantify recovery and peak-based apparent half-life.

COMMON-SUBSET ANALYSIS
----------------------
A strict paired comparison is also performed using only
parameter vectors that remain:

    stable,
    monostable,
    and probe-admissible

under ALL FOUR functional forms.

This prevents differences in dynamical-regime eligibility from
confounding the cross-form comparison.

INTERPRETATION
--------------
If same-signal adaptive memory persists under the alternative
matched response functions, the result is not specific to Hill
algebra.

The fractions observed in the R2/R3 computational sample are
NOT biological probabilities.

MODEL SCOPE
-----------
R3 changes the static signal-response functions but retains:

    dtheta/dt
    =
    epsilon * theta *
    [ln(theta0/theta) - alpha H]

Thus R3 does not establish robustness to every possible
adaptation law.

DATA POLICY
-----------
All interpreted numerical results are saved as CSV.

Every JPEG has a companion source-data CSV.

All intermediate paired classifications and trajectories are
saved for reproducibility.
"""


with open(
    R3_DIR
    / "R3_README.txt",
    "w"
) as f:

    f.write(
        readme
    )


print(readme)

In [ ]:
# ============================================================
# MANIFEST + ZIP
# ============================================================

inventory = []


for path in sorted(
    R3_DIR.iterdir()
):

    if path.is_file():

        inventory.append({

            "filename":
                path.name,

            "extension":
                path.suffix.lower(),

            "size_bytes":
                path.stat().st_size,

            "size_kB":
                path.stat().st_size
                /
                1024.0,

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()
        })


manifest_df = pd.DataFrame(
    inventory
)


manifest_df.to_csv(
    R3_DIR
    / "R3_manifest.csv",
    index=False,
    float_format="%.17g"
)


zip_base = (
    BASE_DIR
    /
    "Robustness_R3_functional_form_EXPORT"
)


zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(R3_DIR)
)


print(
    "==========================================="
)

print(
    "ROBUSTNESS R3 EXPORT COMPLETE"
)

print(
    "==========================================="
)

print("\nFolder:")
print(R3_DIR)

print("\nZIP:")
print(zip_path)

print("\nFILES:\n")


for path in sorted(
    R3_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:75s}"
            f"{path.stat().st_size / 1024:10.1f} KB"
        )